# NEXUS Alpha v3: Can chart images, SEC filing text, or both predict where a stock goes next?

**A leakage-controlled comparison of 7 computer-vision models and 6 language models (plus baselines) on 50 US stocks in 5 sectors, 2016-10 to 2026-10, at 5, 10, 20 and 40 trading-day horizons.**

Master of Applied Artificial Intelligence, University of Wollongong in Dubai. Final project (Computer Vision + Natural Language Processing).

> Educational research. Not financial advice. A predictive association is not a cause.

**This notebook is the only entry point.** It downloads nothing it cannot verify, trains every model, picks winners on validation data only, freezes those choices, scores the locked test period once, writes a Streamlit dashboard and generates three PDF reports (Computer Vision, NLP, and Multimodal). Every number in the reports is read from files this run saved; no number is typed in by hand.

| Item | Setting |
|---|---|
| Stocks | 50 (10 per sector: Information Technology, Financials, Health Care, Consumer Discretionary, Communication Services) |
| Prices | Verified 1-week, 1-day and 1-hour bars (TriFrame tier-A store, every file SHA-256 checked), plus SPY |
| Text | SEC EDGAR 10-K, 10-Q and 8-K filings of the same 50 companies, downloaded in this notebook with point-in-time acceptance timestamps |
| Decisions | 2016-10-10 to 2026-08 (10 years of decision dates; older bars are used only to draw the first charts) |
| Targets | (a) direction: DOWN / SIDEWAYS / UP, (b) realised volatility, (c) a drawdown of more than 10% — each over 5, 10, 20 and 40 sessions |
| Blocks | train 2016-10 → 2020-12 · inner 2021 (early stopping, calibration) · validation 2022-2023 (all model selection) · **test 2024-01 → 2026-08 (locked, opened once after the freeze)** |
| CV models | Custom CNN (Jiang-Kelly-Xiu style, from scratch), ResNet-50, EfficientNet-B0, ConvNeXt-Tiny, ViT-B/16, Swin-T, YOLOv8 (classification backbone; plus the ready-made YOLOv8 chart-pattern detector as a zero-shot extra) |
| NLP models | TF-IDF + logistic/ridge (baseline), FinBERT, SEC-BERT, RoBERTa-large, Longformer-4096, DeBERTa-v3-large, FinGPT (Llama-2-7B + FinGPT LoRA) |

## 0. Feasibility check of the original plan (read before running)

**Verdict: the project can be done, with six changes.** Each change is forced by data or by how a model works, not by results (no result existed when they were made).

| # | Original plan | What this notebook does | Why |
|---|---|---|---|
| 1 | 50 named stocks | **44 kept, 6 replaced within the same sector**: AVGO→TXN, CRM→ADBE (IT); BKNG→MAR (Consumer Discretionary); CMCSA→META, TMUS→OMC, EA→NYT (Communication Services). GOOGL is represented by **GOOG** (same company, Alphabet class C; prices nearly identical) | The six have no verified 10-year **hourly** bars in our store (Yahoo gives only 730 days of hourly data). Replacing them keeps 10 stocks per sector. 16 same-sector reserves are in the data package; section 3 swaps one in automatically, by a rule fixed in advance, if a stock fails the SEC check |
| 2 | "Candlestick movement" | Interpreted as the **3-class direction** of the price (DOWN / SIDEWAYS / UP) over each horizon, with a volatility-scaled sideways band | A class must be defined before it can be predicted; a fixed band (e.g. ±1%) would make "sideways" mean different things for a utility-like and a volatile stock |
| 3 | YOLOv8 "detects candlestick patterns; ready-made weights" | Two uses: **(a)** YOLOv8s-cls ImageNet backbone fine-tuned on our task exactly like the other six (this is the model in every comparison); **(b)** the public detector `foduucom/stockmarket-pattern-detection-yolov8` (6 chart patterns, published mAP@0.5 ≈ 0.61) used zero-shot as pattern features | A detector outputs boxes, not up/down/volatility. Using its detections as features is the honest way to test "ready-made chart-pattern weights" |
| 4 | "RoBERTa-large (Financial)" | `FacebookAI/roberta-large` (general), fine-tuned on our filings | No established financial RoBERTa-large checkpoint exists; fine-tuning on SEC text is what makes it financial |
| 5 | FinGPT "LoRA fine-tuned" | FinGPT's published LoRA (`FinGPT/fingpt-mt_llama2-7b_lora`) on Llama-2-7B, loaded in 4-bit, used as a **frozen** encoder (embeddings + its native sentiment read-out). Optional QLoRA switch (off) | 7B parameters: a full fine-tune for 12 targets does not fit the budget. Llama-2 is gated (Meta licence accepted on the `HF_TOKEN` account). If loading fails, FinGPT is reported as Not Run and the study has 5 language models |
| 6 | 10 years, 1W/1D/1H charts | Decision dates 2016-10-10 → 2026-08 (labels need 40 sessions after the last decision; data end 2026-10-09). Charts: 52 weekly, 63 daily, 70 hourly candles | Exactly the requested span. The **test period (2024-2026) begins after the pre-training cut-off of every language model used** (Llama-2's data end 2022; DeBERTa-v3 2021; others earlier), so no text model can "remember" test-period news |

**What will probably happen (stated before any result, so the reports cannot be accused of hindsight).** Earlier projects on the same data (TriFrame, NEXUS V1, the SEC filings study) found: volatility is predictable (persistence), drawdown risk is partly predictable, **direction is close to chance**, and chart images or filing text add little once simple numbers are known. This study is designed so that a null result is still a clear, defensible finding.

**Run it in this order.** (1) Upload `nav3_bars_66.zip` to `MyDrive/antigravity_workspace/nexus_alpha_v3/data/`. (2) Colab Secrets: `HF_TOKEN` and `SEC_USER_AGENT` ("Your Name your.email@domain"), *Notebook access* ON. (3) Accept the Llama-2 licence (only needed for FinGPT). (4) Runtime → A100 GPU, High-RAM. (5) Run all with `RUN_MODE = "QUICK_VERIFY"` (about 1-1.5 h, 10 stocks, short training: a pipeline check, stamped "not research results"). (6) Switch to `"FULL_RESEARCH"` and Run all (about 8-10 h the first time; the SEC download is cached). Every stage saves to Drive and resumes after a disconnect.

## 0.1 Where each project question is answered

| Question (from the project brief) | Development answer (validation) | Final answer (locked test) |
|---|---|---|
| Q1. Can CV, NLP, or CV+NLP predict UP/SIDEWAYS/DOWN at 5/10/20/40 days? | §20 | §23.1 |
| Q2. Which CV model is best over the 50 stocks, and why? | §12.1 | §23.2 |
| Q3. Which CV model combination is best (e.g. CNN+ResNet, EfficientNet+ResNet…)? | §12.2 | §23.3 |
| Q4. Which model or combination best predicts (a) direction, (b) volatility, (c) a >10% drawdown? | §12.1-12.2 | §23.3 |
| Q5. Which model works best in which sector; is any network sector-specific? | §12.3 | §23.4 |
| Q6. Which report moves the market more: 10-K, 10-Q or 8-K? | §14 (event study), §18 | §23.5 |
| Q7. In which sectors do filings move prices more? | §14, §18 | §23.5 |
| Q8. What in the reports moves the market? | §15 (words), §18.3 (8-K items, tone, attributions) | §23.5 |
| Q9. Which sectors are sensitive to which report type? | §14 (sector × form) | §23.5 |
| Q10. Does combining the top-3 CV and top-3 NLP models add predictive edge? | §20 | §23.1 |
| Stronger baselines, do CV/NLP add to simple numbers? (post-hoc, after the test was opened) | | §23A |
| Product: Streamlit dashboard (4 stocks × 1W/1D/1H screenshots + filing summary → UP/SIDEWAYS/DOWN + 20-day Monte Carlo) | §21 (simulator check) | §24 |
| Three reports: CV, NLP, Multimodal | | §25 |

## 0.2 Evaluation-checklist mapping (professor's CV and NLP checklists)

| Checklist item | Where |
|---|---|
| Problem statement, real-world relevance, research gap, literature | §1 |
| Dataset selection and analysis (counts, classes, resolution, imbalance, sectors) | §3, §4.2 (labels), §6.3 (images), §13.2 (text) |
| Preprocessing (resizing, normalisation, cleaning, tokenisation, section extraction) | §6, §13 |
| Data augmentation (and why horizontal flips are forbidden for charts) | §10.1 |
| Data splitting and leakage prevention (blocks, purge, locked test, causality tests, pre-training cut-offs) | §4.1, §4.3, §6.2, §22 |
| Baselines (prior, momentum, HAR volatility, trailing-volatility drawdown, TF-IDF, technical LightGBM reference, random-weight CNN, label shuffle) | §8, §15 |
| Model selection, architecture understanding, transfer learning | §1.3, §9, §10, §16-17 |
| Training strategy (loss, optimiser, schedule, batch, epochs, early stopping) | §10.1, §17.1 |
| Hyperparameter tuning (learning-rate probe per model, regularisation grid) | §10.2, §17.2 |
| Over-/under-fitting (training and early-stopping curves) | §10.4, §17.4 |
| Metrics matched to each task (macro-F1, log loss, R², QLIKE, ROC-AUC, PR-AUC, Brier, ECE) | §7 |
| Fair model comparison (same rows, same labels, paired month-block bootstrap, Holm) | §7, §12, §18, §20, §23 |
| Ablations (views, frozen vs fine-tuned, pretrained vs random, augmentation, text length/form) | §9.3, §12.4, §18 |
| Error analysis | §12.5, §18.4, §23.6 |
| Explainable AI (Grad-CAM / token maps, occlusion by time and view, integrated gradients for text) with limits | §12.6, §18.3 |
| Robustness (noise, blur, JPEG, chart style, screenshot resizing, missing view) | §12.7 |
| Computational efficiency (parameters, FLOPs, latency, training minutes) | §12.8, §18.5 |
| Research contribution, limitations, viva preparation | §1.2, §26 |
| Reproducibility (seeds, config hashes, data hashes, package versions, freeze manifest) | §2, §22 |

## 1. Problem, literature, research gap and pre-registered hypotheses

### 1.1 Problem and why it matters
Investors look at charts and read company filings. Two questions follow for AI: can a vision model read a candlestick chart well enough to forecast **where the price goes** (direction), **how much it moves** (volatility) and **whether it falls hard** (a drawdown of more than 10%)? Can a language model do the same from a **10-K, 10-Q or 8-K filing**? And is the combination better than either? Risk teams care most about volatility and drawdown (position sizing, stop-losses); traders care about direction. A clear, leakage-free answer, including a clear "no", is useful because many retail tools claim chart or text "AI signals" without out-of-sample evidence.

### 1.2 What earlier work found, and the gap
* **Chart images.** Jiang, Kelly and Xiu (2023, *J. Finance* 78(6)) trained a CNN on OHLC images of US stocks and reported return predictability beyond momentum, mostly at short horizons and for small firms. Sezer and Ozbayoglu (2019) and Ho and Huang (2021, *Sensors*) used candlestick images with CNNs on few stocks. Modern backbones (ResNet, He et al. 2016; EfficientNet, Tan and Le 2019; ConvNeXt, Liu et al. 2022; ViT, Dosovitskiy et al. 2021; Swin, Liu et al. 2021; YOLO, Redmon et al. 2016 / Ultralytics 2023) were built for photographs; whether ImageNet features transfer to synthetic charts is an open empirical question. Our own TriFrame study found images add ~0 to numeric features for volatility and drawdown on 500 stocks.
* **Filing text.** Loughran and McDonald (2011, 2016) showed finance-specific word lists matter; Tetlock (2007), Cohen, Malloy and Nguyen (2020, "Lazy Prices") and Ke, Kelly and Xiu (2019) found text predicts returns, mainly slowly and for small firms. 8-K items carry news of different importance (Lerman and Livnat 2010). Transformers: FinBERT (Araci 2019; Huang, Wang and Yang 2023), SEC-BERT (Loukas et al. 2022), RoBERTa (Liu et al. 2019), Longformer (Beltagy et al. 2020), DeBERTa-v3 (He et al. 2023), FinGPT (Yang et al. 2023) on Llama-2 (Touvron et al. 2023). **Look-ahead bias** in pre-trained language models is a documented problem (Glasserman and Lin 2023; Sarkar and Vafa 2024): a model pre-trained on text written after the test date can "know" the outcome.
* **Gap.** We found no study that compares, on **identical leakage-controlled rows**, (i) seven vision architectures on **tri-timeframe** (weekly, daily, hourly) candlestick charts with (ii) six language models on **point-in-time** SEC filings, across **four horizons and three targets**, **by sector**, with a test period that starts **after every language model's pre-training cut-off**, and with a deployable dashboard evaluated by the same rules.

**Contribution (what is new and defensible).** A controlled benchmark answering the ten project questions with paired, month-block bootstrap confidence intervals and Holm-corrected tests; an event study of which filing types and 8-K items move prices in which sectors; a reproducible pipeline (hash-verified data, frozen selections, single test opening); and a dashboard whose outputs follow the tested recipe, including a "no call" rule when direction skill is not validated.

### 1.3 Why these models (model selection)
| Model | Inductive bias | Role |
|---|---|---|
| Custom CNN (JKX-style) | small, local filters, trained from scratch | academic baseline: is ImageNet pre-training needed at all? |
| ResNet-50 | residual connections ease deep optimisation | industry-standard pretrained baseline |
| EfficientNet-B0 | compound scaling, depthwise convs: accuracy per FLOP | efficiency end of the spectrum |
| ConvNeXt-Tiny | CNN modernised with transformer design choices | modern CNN |
| ViT-B/16 | global self-attention over 16×16 patches, weak locality prior | tests whether long-range chart structure matters |
| Swin-T | windowed attention, hierarchical | middle ground between CNN locality and ViT globality |
| YOLOv8 | CSP-Darknet backbone built for detection | "pattern detector" family; detector weights used zero-shot |
| TF-IDF + linear | bag of n-grams | the hard-to-beat financial-text baseline |
| FinBERT / SEC-BERT | BERT-base, finance / SEC-domain pre-training | does domain pre-training help? |
| RoBERTa-large / DeBERTa-v3-large | larger general encoders | does scale help? |
| Longformer-4096 | sparse attention, 8× longer input | does reading more of a long filing help? |
| FinGPT (7B) | decoder LLM with finance instruction LoRA | do LLM representations help? |

### 1.4 Pre-registered hypotheses and decision rules (fixed before any data were loaded)
All tests are on the **locked test block (2024-01 → 2026-08)**, after selections are frozen in §22. Confidence intervals: paired bootstrap over calendar-month blocks (B = 1000; Künsch 1989), α = 0.05, Holm correction within each family. "Skill" means: direction macro-F1 above the **shuffled-prediction null** (same predictions, labels permuted within month, which keeps class shares); volatility R² of log realised volatility above 0; drawdown ROC-AUC above 0.5.

| Id | Hypothesis | Supported if |
|---|---|---|
| H1 | The validation-chosen CV model has skill (per target × horizon, 12 tests) | Holm-adjusted CI excludes the null |
| H2 | The validation-chosen NLP model has skill on filing events (12 tests) | same |
| H3 | CV+NLP fusion beats the better single modality (per target × horizon) | paired Δ CI > 0 after Holm |
| H4 | Top-3 CV + top-3 NLP beats top-3 CV alone (Q10) | paired Δ CI > 0 after Holm |
| H5 | Fine-tuning beats frozen features (per CV model, on the composite score) | paired Δ CI > 0 after Holm |
| H6 | Model rankings differ by sector beyond chance (Q5) | stock-level permutation p < 0.05 |
| H7 | Filings move prices more than ordinary days, and the effect differs by form (Q6) | |abnormal move| CI on filing days above non-filing days; Kruskal-Wallis p < 0.05 |
| R1 (reference) | Vision/text models beat the simple baselines (momentum, HAR, trailing volatility) and the technical-indicator LightGBM | reported, not a selection criterion |

**Selection rule (validation only).** Per task (target × horizon) the model with the best validation primary metric wins (macro-F1, R², ROC-AUC). The **overall** CV and NLP rankings use the mean rank over the 12 tasks. Ensembles: equal-weight averages of calibrated outputs (no weights fitted, so no extra over-fitting), members chosen on validation. Fusion weights: grid {0, 0.25, 0.5, 0.75, 1} chosen on validation.

## 2. Install packages, configuration, Drive, secrets and reproducibility tools

Colab ships PyTorch, torchvision, transformers, scikit-learn, pandas, matplotlib and plotly. The cell adds the rest. Versions actually resolved are written to the freeze manifest (§22). GPU: A100 (40 or 80 GB) recommended; L4 works (2-3× slower); without a GPU, deep-model cells are recorded as **Not Run (no GPU)**, never as zeros.

In [ ]:
import subprocess, sys, platform
REQUIREMENTS = [
    "timm>=1.0.15", "transformers>=4.48", "huggingface_hub>=0.26", "accelerate>=1.0", "peft>=0.13", "bitsandbytes>=0.45",
    "sentencepiece>=0.2", "protobuf>=4.25", "ultralytics>=8.3.94,<8.4", "lightgbm>=4.3", "reportlab>=4.2", "streamlit>=1.40",
    "yfinance>=0.2.54", "onnxruntime", "pandas_market_calendars>=4.4", "lxml>=5.0", "pyarrow>=15",
    "scikit-learn>=1.4", "plotly>=5.20",
]
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS], capture_output=True, text=True)
print(_pip.stdout[-1500:] if _pip.returncode == 0 else _pip.stderr[-4000:])
if _pip.returncode != 0:
    raise RuntimeError("Package installation failed (message above). Fix it, then re-run this cell.")
# Screenshot OCR (dashboard only) is optional: try the old package, then its successor; the dashboard falls back to a typed ticker.
OCR_PKG = None
for _pkg in ("rapidocr_onnxruntime", "rapidocr"):
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], capture_output=True, text=True)
    if _r.returncode == 0:
        OCR_PKG = _pkg; break
print("OCR package:", OCR_PKG or "none installed (dashboard will need the ticker typed in)")
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
import os, json, time, math, re, gc, random, hashlib, shutil, warnings, html, io, zipfile, traceback, itertools
import datetime as dt
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import requests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix
import joblib
import lightgbm as lgb
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50)

# ======================================================================== RUN MODE
RUN_MODE = "FULL_RESEARCH"        # "QUICK_VERIFY" first (pipeline check), then "FULL_RESEARCH" (the study). Never mix outputs.
assert RUN_MODE in ("QUICK_VERIFY", "FULL_RESEARCH")
QUICK = RUN_MODE == "QUICK_VERIFY"
FORCE_RERUN = False              # True ignores saved stage manifests and recomputes everything
RUN_FINGPT = True                # FinGPT on Meta's gated Llama-2 (access granted). If it fails it is recorded as Not Run; the rest continues
FINGPT_QLORA = False             # optional QLoRA fine-tune of FinGPT (about +1.5 h on an A100); off by default (frozen use only)

STUDY = {   # sector -> 10 tickers (6 replacements vs the brief: see section 0)
    "Information Technology": ["AAPL", "MSFT", "NVDA", "TXN", "INTC", "CSCO", "ORCL", "QCOM", "AMD", "ADBE"],
    "Financials":             ["JPM", "BAC", "WFC", "C", "GS", "MS", "V", "MA", "BRK.B", "SCHW"],
    "Health Care":            ["UNH", "JNJ", "LLY", "PFE", "MRK", "ABT", "TMO", "AMGN", "GILD", "CVS"],
    "Consumer Discretionary": ["AMZN", "HD", "MCD", "NKE", "SBUX", "F", "LOW", "TJX", "MAR", "ROST"],
    "Communication Services": ["GOOG", "NFLX", "DIS", "T", "VZ", "META", "OMC", "TTWO", "NYT", "LYV"],
}
RESERVES = {"Information Technology": ["ADI", "MU", "IBM", "INTU"], "Financials": ["AXP", "PNC", "USB", "BLK"],
            "Health Care": ["MDT", "BMY", "DHR", "ISRG"], "Consumer Discretionary": ["EBAY", "YUM", "ORLY"],
            "Communication Services": ["IMAX"]}
REPLACED = {"AVGO": "TXN", "CRM": "ADBE", "BKNG": "MAR", "CMCSA": "META", "TMUS": "OMC", "EA": "NYT", "GOOGL": "GOOG"}
# SEC CIKs (first = current filer; later entries = earlier filers of the same company inside the window)
CIKS = {"AAPL": [320193], "MSFT": [789019], "NVDA": [1045810], "TXN": [97476], "INTC": [50863], "CSCO": [858877],
        "ORCL": [1341439], "QCOM": [804328], "AMD": [2488], "ADBE": [796343], "JPM": [19617], "BAC": [70858], "WFC": [72971],
        "C": [831001], "GS": [886982], "MS": [895421], "V": [1403161], "MA": [1141391], "BRK.B": [1067983], "SCHW": [316709],
        "UNH": [731766], "JNJ": [200406], "LLY": [59478], "PFE": [78003], "MRK": [310158], "ABT": [1800], "TMO": [97745],
        "AMGN": [318154], "GILD": [882095], "CVS": [64803], "AMZN": [1018724], "HD": [354950], "MCD": [63908], "NKE": [320187],
        "SBUX": [829224], "F": [37996], "LOW": [60667], "TJX": [109198], "MAR": [1048286], "ROST": [745732],
        "GOOG": [1652044], "NFLX": [1065280], "DIS": [1744489, 1001039], "T": [732717], "VZ": [732712], "META": [1326801],
        "OMC": [29989], "TTWO": [946581], "NYT": [71691], "LYV": [1335258],
        "ADI": [6281], "MU": [723125], "IBM": [51143], "INTU": [896878], "AXP": [4962], "PNC": [713676], "USB": [36104],
        "BLK": [2012383, 1364742], "MDT": [1613103], "BMY": [14272], "DHR": [313616], "ISRG": [1035267], "EBAY": [1065088],
        "YUM": [1041061], "ORLY": [898173], "IMAX": [921582]}
DATA_ZIP_NAME = "nav3_bars_66.zip"
DATA_MANIFEST_SHA256 = "7eb91001d27916661fad13b5d0e7b77f7aa6147504379b4433d6490e15736da5"

CFG = {
    "project": "NEXUS Alpha v3", "run_mode": RUN_MODE,
    "quick_per_sector": 2,
    # ---- targets
    "horizons": [5, 10, 20, 40], "classes": ["DOWN", "SIDEWAYS", "UP"], "k_sideways": 0.5, "sigma_window": 60,
    "dd_threshold": -0.10, "tz": "America/New_York",
    # ---- calendar and blocks (by decision date)
    "warmup_start": "2014-06-01", "decision_start": "2016-10-10", "data_end": "2026-10-09",
    "blocks": {"train": ["2016-10-10", "2020-12-31"], "inner": ["2021-01-01", "2021-12-31"],
               "val": ["2022-01-01", "2023-12-31"], "test": ["2024-01-01", "2026-10-09"]},
    "purge_sessions": 40, "eval_stride": 5, "train_stride": 5 if QUICK else 1,
    # ---- SEC rule for substitutions (section 3) and download
    "sec_min_counts": {"10-K": 8, "10-Q": 20, "8-K": 10},
    "sec": {"forms": ["10-K", "10-Q", "8-K"], "start": "2015-06-01", "public_lag_min": 5, "processing_min": 30,
            "date_only_time_et": "17:30", "max_rps": 8, "max_exhibits_8k": 1, "quick_8k_stride": 3,
            "max_chars": 60000, "lookback_sessions": {"8-K": 20, "periodic": 90}},
    # ---- charts
    "chart": {"size": 224, "bars": {"W": 52, "D": 63, "H": 70}},
    # ---- CV models (timm ids, ImageNet weights; ultralytics YOLOv8s-cls)
    "cv_models": {
        "jkx_cnn":   {"kind": "jkx", "label": "Custom CNN (JKX-style, scratch)", "lr_grid": [3e-4, 1e-3]},
        "resnet50":  {"kind": "timm", "timm": "resnet50.a1_in1k", "label": "ResNet-50", "lr_grid": [1e-4, 3e-4]},
        "effnet_b0": {"kind": "timm", "timm": "efficientnet_b0.ra_in1k", "label": "EfficientNet-B0", "lr_grid": [2e-4, 5e-4]},
        "convnext_t": {"kind": "timm", "timm": "convnext_tiny.in12k_ft_in1k", "label": "ConvNeXt-Tiny", "lr_grid": [5e-5, 1e-4]},
        "vit_b16":   {"kind": "timm", "timm": "vit_base_patch16_224.augreg2_in21k_ft_in1k", "label": "ViT-B/16", "lr_grid": [2e-5, 5e-5]},
        "swin_t":    {"kind": "timm", "timm": "swin_tiny_patch4_window7_224.ms_in1k", "label": "Swin-T", "lr_grid": [3e-5, 1e-4]},
        "yolov8_cls": {"kind": "yolo", "weights": "yolov8s-cls.pt", "label": "YOLOv8s-cls", "lr_grid": [1e-4, 3e-4]},
    },
    "yolo_det_repo": "foduucom/stockmarket-pattern-detection-yolov8",
    "cv_ft": {"epochs": 1 if QUICK else 6, "patience": 2, "batch": 64, "lr_head": 1e-3, "wd": 0.05, "warmup_frac": 0.05,
              "probe_frac": 0.25, "loss_w": {"dir": 1.0, "vol": 0.5, "dd": 0.5}, "aug": True},
    # ---- NLP models (Hugging Face ids)
    "nlp_models": {
        "finbert":   {"hf": "ProsusAI/finbert", "label": "FinBERT", "max_len": 512, "lr_grid": [2e-5, 3e-5], "batch": 16, "accum": 1},
        "secbert":   {"hf": "nlpaueb/sec-bert-base", "label": "SEC-BERT", "max_len": 512, "lr_grid": [2e-5, 3e-5], "batch": 16, "accum": 1},
        "roberta_l": {"hf": "FacebookAI/roberta-large", "label": "RoBERTa-large", "max_len": 512, "lr_grid": [1e-5, 2e-5], "batch": 8, "accum": 2},
        "longformer": {"hf": "allenai/longformer-base-4096", "label": "Longformer-4096", "max_len": 4096, "lr_grid": [2e-5, 3e-5], "batch": 2, "accum": 8},
        "deberta_l": {"hf": "microsoft/deberta-v3-large", "label": "DeBERTa-v3-large", "max_len": 512, "lr_grid": [8e-6, 1.5e-5], "batch": 8, "accum": 2, "amp": False},
        "fingpt":    {"hf": "FinGPT/fingpt-mt_llama2-7b_lora", "base": "meta-llama/Llama-2-7b-hf", "label": "FinGPT (Llama-2-7B LoRA)",
                      "max_len": 1024, "frozen_only": True},
    },
    "nlp_ft": {"epochs": 1 if QUICK else 3, "patience": 1, "lr_head": 1e-3, "wd": 0.01, "warmup_frac": 0.06, "probe_frac": 0.5,
               "frozen_chunks": 4, "chunk_len": 512},
    "tfidf": {"max_features": 50000, "ngram": [1, 2], "min_df": 5, "max_chars": 60000},
    # ---- heads on frozen features
    "probe": {"pca_per_view": 32, "pca_text": 32, "C_grid": [0.003, 0.03, 0.3], "alpha_grid": [1.0, 10.0, 100.0], "max_fit_rows": 15000},
    # ---- inference
    "bootstrap_B": 200 if QUICK else 1000, "alpha": 0.05, "fusion_w_grid": [0.0, 0.25, 0.5, 0.75, 1.0], "top_k": 3,
    "mc": {"nu_grid": [3, 4, 6, 10, 0], "n_paths": 2000, "eval_rows": 600 if QUICK else 2000, "horizon": 20},
    "deploy_max_params_m": 600,
    "seed": 11,
}
HORIZONS = CFG["horizons"]; CLASSES = CFG["classes"]; NY = CFG["tz"]; VIEWS = ["W", "D", "H"]
SECTORS = list(STUDY)
TASKS = [f"{k}_{h}" for k in ("dir", "vol", "dd") for h in HORIZONS]
TASK_LABEL = {"dir": "Direction (3-class)", "vol": "Volatility (log RV)", "dd": "Drawdown > 10%"}
PRIMARY = {"dir": "macro_f1", "vol": "r2", "dd": "auc"}

def set_seed(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_GPU = DEV.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float16
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if HAS_GPU else 0.0
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none"
print(f"RUN_MODE={RUN_MODE} | device={DEV} {GPU_NAME} ({GPU_GB:.0f} GB) | AMP={AMP_DTYPE}")

In [ ]:
# ======================================================================== Drive, folders, secrets
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/antigravity_workspace/nexus_alpha_v3")
    LOCAL_ROOT = Path("/content/nav3_local")
else:
    print("NOTE: not in Colab. The validated path is Colab; local runs are unsupported.")
    DRIVE_ROOT = Path.cwd() / "nav3_drive"; LOCAL_ROOT = Path.cwd() / "nav3_local"
DATA_DIR = DRIVE_ROOT / "data"
ART = DRIVE_ROOT / f"artifacts_{RUN_MODE.lower()}"
SUB = {k: ART / k for k in ["manifests", "preds", "models", "features", "figures", "tables", "results", "reports", "app", "logs"]}
for p_ in [DATA_DIR, LOCAL_ROOT, *SUB.values()]:
    p_.mkdir(parents=True, exist_ok=True)
STAMP = "QUICK_VERIFY - NOT RESEARCH RESULTS" if QUICK else "FULL_RESEARCH"
print("Drive root:", DRIVE_ROOT, "| artifacts:", ART)

def get_secret(name: str):
    """Colab Secrets first, then environment variables. Values are never printed or saved."""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None
HF_TOKEN = get_secret("HF_TOKEN"); SEC_USER_AGENT = get_secret("SEC_USER_AGENT")
print("HF_TOKEN present:", bool(HF_TOKEN), "| SEC_USER_AGENT present:", bool(SEC_USER_AGENT))
if not SEC_USER_AGENT:
    raise RuntimeError("Add the Colab secret SEC_USER_AGENT ('Your Name your.email@domain') and switch Notebook access ON. "
                       "EDGAR refuses anonymous requests.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("WARNING: no HF_TOKEN: public models still download; FinGPT (gated Llama-2) will be Not Run.")

In [ ]:
# ======================================================================== manifests, resume, registry, timers
def cfg_hash(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]

def sha256_file(p) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fingerprint(p) -> str:
    """SHA-256 for files up to 64 MB; larger files: size + first and last MB (fast)."""
    p = Path(p); size = p.stat().st_size; h = hashlib.sha256()
    with open(p, "rb") as f:
        if size <= 64 * 2**20:
            h.update(f.read())
        else:
            h.update(f.read(2**20)); f.seek(size - 2**20); h.update(f.read(2**20)); h.update(str(size).encode())
    return h.hexdigest()[:24]

def now_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

class Stage:
    """A stage is skipped on resume only if its config hash and every output fingerprint still match its manifest."""
    def __init__(self, name: str, config, outputs):
        self.name, self.h = name, cfg_hash(config)
        self.outputs = [Path(o) for o in outputs]
        self.mpath = SUB["manifests"] / f"stage_{name}.json"
        self.t0 = time.time()
    def done(self) -> bool:
        if FORCE_RERUN or not self.mpath.exists():
            return False
        m = json.loads(self.mpath.read_text())
        if m.get("config_hash") != self.h:
            print(f"[{self.name}] configuration changed -> recompute"); return False
        for o in self.outputs:
            if not o.exists() or m.get("outputs", {}).get(str(o)) != fingerprint(o):
                print(f"[{self.name}] output missing or changed ({o.name}) -> recompute"); return False
        print(f"[{self.name}] resumed from saved outputs (config {self.h})"); return True
    def finish(self, **extra):
        m = {"stage": self.name, "config_hash": self.h, "run_mode": RUN_MODE, "finished_utc": now_utc(),
             "seconds": round(time.time() - self.t0, 1), "outputs": {str(o): fingerprint(o) for o in self.outputs}, **extra}
        self.mpath.write_text(json.dumps(m, indent=1, default=str))

REG = {}
REG_PATH = SUB["results"] / "experiments.json"
if REG_PATH.exists() and not FORCE_RERUN:
    REG.update(json.loads(REG_PATH.read_text()))
def register(mid: str, **fields):
    rec = REG.get(mid, {}); rec.update({"model_id": mid, "run_mode": RUN_MODE, "updated_utc": now_utc(), **fields})
    REG[mid] = rec; REG_PATH.write_text(json.dumps(REG, indent=1, default=str)); return rec
def not_run(mid: str, reason: str, **fields):
    """A skipped experiment appears as 'Not Run' in every table, never as a zero."""
    print(f"NOT RUN  {mid}: {reason}")
    return register(mid, status="NOT_RUN", reason=reason, **fields)
def failed(mid: str, err: BaseException, **fields):
    tb = traceback.format_exc()[-3000:]
    print(f"FAILED   {mid}: {type(err).__name__}: {err}\n{tb}")
    (SUB["logs"] / f"failed_{mid}.txt").write_text(tb)
    return register(mid, status="FAILED", reason=f"{type(err).__name__}: {str(err)[:300]}", **fields)

TIMER = {"_t0": time.time()}
def tick(section: str):
    TIMER[section] = round((time.time() - TIMER["_t0"]) / 60, 2); TIMER["_t0"] = time.time()
    (SUB["results"] / "timer.json").write_text(json.dumps({k: v for k, v in TIMER.items() if k != "_t0"}, indent=1))

def savefig(fig, name: str, dpi: int = 130):
    if QUICK:
        fig.text(0.995, 0.005, STAMP, ha="right", va="bottom", fontsize=7, color="#b91c1c")
    fig.savefig(SUB["figures"] / name, dpi=dpi, bbox_inches="tight")

In [ ]:
# ======================================================================== the test vault (procedural seal)
FREEZE_PATH = SUB["results"] / "freeze_manifest.json"
VAULT_LOG = SUB["results"] / "test_openings.json"
class TestVault:
    """Test-block outcomes live here, not in the working tables. Opening needs the freeze manifest written in section 22;
    every opening is appended to test_openings.json (re-openings are disclosed in the reports)."""
    def __init__(self):
        self._store = {}
    def deposit(self, name: str, df: pd.DataFrame):
        """Refuses an all-NaN deposit (that would mean the labels were already hidden, i.e. a cell was re-run out of order)."""
        vals = df.drop(columns=[c for c in ("row", "ev") if c in df.columns])
        if len(df) and not vals.notna().any().any():
            raise RuntimeError(f"Vault deposit '{name}' has no labels: cells were re-run out of order. "
                               "Restart the runtime (Runtime > Restart session) and Run all from the top.")
        self._store[name] = df.copy()
    def has(self, name: str) -> bool:
        return name in self._store
    def open(self, name: str) -> pd.DataFrame:
        if name not in self._store:
            raise RuntimeError(f"The vault has no '{name}' in this session (section 2 was re-run after section 4/14). Restart and Run all.")
        if not FREEZE_PATH.exists():
            raise RuntimeError("The test block is locked: run section 22 (freeze) first.")
        log = json.loads(VAULT_LOG.read_text()) if VAULT_LOG.exists() else []
        log.append({"item": name, "utc": now_utc(), "freeze_sha256": sha256_file(FREEZE_PATH)[:16], "run_mode": RUN_MODE})
        VAULT_LOG.write_text(json.dumps(log, indent=1))
        return self._store[name].copy()
VAULT = TestVault()
print("Registry entries:", len(REG), "| freeze manifest exists:", FREEZE_PATH.exists())
tick("2 setup")

## 3. Universe, data provenance, SEC availability check and price bars

**Price data.** `nav3_bars_66.zip` holds TriFrame's frozen tier-A bars (`final/`, frozen 2026-10-11, data through 2026-10-09) for the 50 study stocks and 16 same-sector reserves, plus SPY daily. Source chain: a Finnhub-derived 1-minute feed cleaned and aggregated to regular-hours hourly bars (09:30-aligned, 7 per session), daily and weekly bars; Yahoo gap fills and the March-October 2026 extension from Yahoo daily and hourly bars. Prices are **split-adjusted, not dividend-adjusted**. TriFrame verified the store against Yahoo (daily close s.d. 0.24 %, hourly 0.04 %) and the NYSE calendar (0 missing sessions). This cell re-checks every file's SHA-256 against the manifest pinned in §2; the notebook never substitutes other price data.

**SEC availability rule (fixed before any download of text).** A study stock is kept if EDGAR lists, between 2016-01-01 and the data end, at least 8 10-K, 20 10-Q and 10 8-K original filings. Otherwise it is replaced by the first reserve of its sector that passes. Every decision is printed and saved.

In [ ]:
BARS_DIR = LOCAL_ROOT / "bars"
def stage_bars_zip():
    zp = DATA_DIR / DATA_ZIP_NAME
    if not zp.exists():                      # 1) look elsewhere in Drive, 2) ask for an upload (Colab)
        hits = []
        for root in (DRIVE_ROOT.parent, DRIVE_ROOT.parent.parent):
            hits = list(root.glob(f"**/{DATA_ZIP_NAME}"))[:1] if root.exists() else []
            if hits:
                break
        if hits:
            print("Found the data package at", hits[0]); shutil.copy(hits[0], zp)
        elif IN_COLAB:
            from google.colab import files as _up
            print(f"{DATA_ZIP_NAME} not found in Drive. Choose it in the dialog below (107 MB, from your Downloads folder):")
            up = _up.upload()
            name = next((k for k in up if k.endswith(".zip")), None)
            if name is None:
                raise RuntimeError("No zip uploaded.")
            Path(name).replace(zp)           # saved into Drive for later runs
        else:
            raise RuntimeError(f"{DATA_ZIP_NAME} not found in {DATA_DIR}.")
    n = 0
    with zipfile.ZipFile(zp) as z:
        for m in z.infolist():
            parts = [x for x in m.filename.replace("\\", "/").split("/") if x]
            if m.is_dir() or not parts:
                continue
            rel = "/".join(parts[1:] if parts[0] == "nav3_bars" else parts)
            dst = BARS_DIR / rel; dst.parent.mkdir(parents=True, exist_ok=True)
            with z.open(m) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
            n += 1
    print(f"Extracted {n} files from {zp.name}")
if not (BARS_DIR / "manifest.json").exists():
    stage_bars_zip()
if sha256_file(BARS_DIR / "manifest.json") != DATA_MANIFEST_SHA256:
    raise RuntimeError("manifest.json does not match the SHA-256 pinned in section 2: the data package was altered or is a different version.")
BARS_MANIFEST = json.loads((BARS_DIR / "manifest.json").read_text())
_bad = [k for k, v in BARS_MANIFEST["files"].items() if not (BARS_DIR / k).exists() or sha256_file(BARS_DIR / k) != v]
if _bad:
    raise RuntimeError(f"{len(_bad)} data files missing or changed, e.g. {_bad[:5]}")
print(f"All {len(BARS_MANIFEST['files'])} data files match the manifest (data end {BARS_MANIFEST['data_end']}).")
PROV = [{"source": "TriFrame tier-A bars + SPY (nav3_bars_66.zip)", "status": "OK",
         "note": f"{len(BARS_MANIFEST['files'])} files SHA-256 verified; root {BARS_MANIFEST['source_root_sha256_prefix']}"}]

In [ ]:
# ---------------------------------------------------------------- EDGAR client (SEC fair-access policy: declared agent, <= 8 req/s)
class SecClient:
    def __init__(self, ua: str, max_rps: float = 8):
        self.s = requests.Session(); self.s.headers.update({"User-Agent": ua, "Accept-Encoding": "gzip, deflate"})
        self.min_dt, self.last, self.n403, self.n_soft_fail = 1.0 / max_rps, 0.0, 0, 0
    def get(self, url: str, as_json: bool = False, tries: int = 5, soft: bool = False):
        tries = 2 if soft else tries
        for k in range(tries):
            wait = self.min_dt - (time.time() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()
            try:
                r = self.s.get(url, timeout=30)
            except requests.RequestException:
                time.sleep(2 ** k); continue
            if r.status_code == 200:
                return r.json() if as_json else r.text
            if r.status_code == 404:
                return None
            if r.status_code == 403:
                self.n403 += 1
                if self.n403 >= 5:
                    raise RuntimeError("EDGAR returned 403 five times: check SEC_USER_AGENT ('Name email').")
            time.sleep(min(60, (2 if soft else 5) * 2 ** k))
        if soft:
            self.n_soft_fail += 1
            return None
        raise RuntimeError(f"EDGAR request failed after {tries} tries: {url}")
SEC = SecClient(SEC_USER_AGENT, CFG["sec"]["max_rps"])
SUBM_DIR = DATA_DIR / "sec" / "submissions"; SUBM_DIR.mkdir(parents=True, exist_ok=True)

def submissions(cik: int, need_before: str) -> pd.DataFrame:
    """All filings of one CIK (recent + older pages as needed), cached on Drive."""
    p = SUBM_DIR / f"CIK{cik:010d}.json"
    if p.exists():
        sub = json.loads(p.read_text())
    else:
        sub = SEC.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json", as_json=True) or {}
        sub["_pages"] = []; p.write_text(json.dumps(sub))
    frames = [pd.DataFrame(sub.get("filings", {}).get("recent", {}))]
    oldest = frames[0]["filingDate"].min() if len(frames[0]) else "9999"
    pages = sub.get("_pages", [])
    if oldest > need_before and not pages and sub.get("filings", {}).get("files"):
        for f in sub["filings"]["files"]:
            pg = SEC.get(f"https://data.sec.gov/submissions/{f['name']}", as_json=True)
            if pg:
                pages.append(pg)
        sub["_pages"] = pages; p.write_text(json.dumps(sub))
    frames += [pd.DataFrame(pg) for pg in pages]
    fl = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(fl):
        fl = fl.drop_duplicates("accessionNumber")
        fl["cik"] = int(cik)
    return fl

def filing_index(t: str) -> pd.DataFrame:
    """10-K / 10-Q / 8-K originals of ticker t from all its CIKs inside the study window."""
    fr = [submissions(c, CFG["sec"]["start"]) for c in CIKS[t]]
    fl = pd.concat([f for f in fr if len(f)], ignore_index=True) if any(len(f) for f in fr) else pd.DataFrame()
    if fl.empty:
        return fl
    fl = fl[fl["form"].isin(CFG["sec"]["forms"]) & (fl["filingDate"] >= CFG["sec"]["start"]) & (fl["filingDate"] <= CFG["data_end"])]
    return fl.drop_duplicates("accessionNumber").sort_values("filingDate").reset_index(drop=True)

# CIK cross-check against SEC's own ticker table (warning only: pinned CIKs include predecessors such as Disney's)
try:
    _ct = SEC.get("https://www.sec.gov/files/company_tickers.json", as_json=True) or {}
    SEC_TICKER_CIK = {v["ticker"].upper().replace("-", "."): int(v["cik_str"]) for v in _ct.values()}
except Exception as e:
    SEC_TICKER_CIK = {}; print("company_tickers.json unavailable:", e)
for t, ciks in CIKS.items():
    if t in SEC_TICKER_CIK and SEC_TICKER_CIK[t] != ciks[0]:
        print(f"WARNING: SEC lists CIK {SEC_TICKER_CIK[t]} for {t}; pinned {ciks}. The SEC value is prepended.")
        CIKS[t] = [SEC_TICKER_CIK[t]] + [c for c in ciks if c != SEC_TICKER_CIK[t]]

In [ ]:
# ---------------------------------------------------------------- the substitution rule
def sec_counts(t: str) -> dict:
    fl = filing_index(t)
    fl = fl[fl["filingDate"] >= "2016-01-01"] if len(fl) else fl
    return {f: int((fl["form"] == f).sum()) if len(fl) else 0 for f in CFG["sec"]["forms"]}
def sec_ok(c: dict) -> bool:
    return all(c[f] >= CFG["sec_min_counts"][f] for f in CFG["sec_min_counts"])

UNIVERSE_LOG, TICKERS, SECTOR_OF = [], [], {}
for sec_name, names in STUDY.items():
    chosen = names[: CFG["quick_per_sector"]] if QUICK else list(names)
    reserves = list(RESERVES[sec_name])
    final = []
    for t in chosen:
        c = sec_counts(t)
        if sec_ok(c):
            final.append(t); UNIVERSE_LOG.append({"sector": sec_name, "ticker": t, "status": "kept", **c})
            continue
        UNIVERSE_LOG.append({"sector": sec_name, "ticker": t, "status": "FAILED SEC rule", **c})
        while reserves:
            r = reserves.pop(0); cr = sec_counts(r)
            if sec_ok(cr):
                final.append(r); UNIVERSE_LOG.append({"sector": sec_name, "ticker": r, "status": f"reserve replacing {t}", **cr}); break
            UNIVERSE_LOG.append({"sector": sec_name, "ticker": r, "status": "reserve failed SEC rule", **cr})
        else:
            raise RuntimeError(f"No reserve left for {sec_name}")
    TICKERS += final
    SECTOR_OF.update({t: sec_name for t in final})
UNIVERSE_DF = pd.DataFrame(UNIVERSE_LOG)
UNIVERSE_DF["replaces_brief_ticker"] = UNIVERSE_DF["ticker"].map({v: k for k, v in REPLACED.items()}).fillna("")
UNIVERSE_DF.to_csv(SUB["tables"] / "universe_sec_check.csv", index=False)
display(UNIVERSE_DF)
print(f"Study universe: {len(TICKERS)} stocks:", {s: [t for t in TICKERS if SECTOR_OF[t] == s] for s in SECTORS})
PROV.append({"source": "SEC EDGAR submissions", "status": "OK", "note": f"{len(UNIVERSE_DF)} tickers checked; rule {CFG['sec_min_counts']}"})

In [ ]:
# ---------------------------------------------------------------- load bars
WARM = pd.Timestamp(CFG["warmup_start"]); DATA_END = pd.Timestamp(CFG["data_end"])
def load_daily(p: Path) -> pd.DataFrame:
    d = pd.read_parquet(p)
    d["date"] = pd.to_datetime(d["date"])
    if d["date"].dt.tz is not None:
        d["date"] = d["date"].dt.tz_localize(None)
    d["date"] = d["date"].dt.normalize().astype("datetime64[ns]")
    d = d[(d["date"] >= WARM) & (d["date"] <= DATA_END)].sort_values("date").reset_index(drop=True)
    return d[["date", "open", "high", "low", "close", "volume"]].astype({c: float for c in ["open", "high", "low", "close", "volume"]})
def load_hourly(p: Path) -> pd.DataFrame:
    h = pd.read_parquet(p)
    h["timestamp"] = pd.to_datetime(h["timestamp"], utc=True).astype("datetime64[ns, UTC]")
    h = h[(h["timestamp"] >= WARM.tz_localize(NY)) & (h["timestamp"] < (DATA_END + pd.Timedelta(days=1)).tz_localize(NY))]
    h = h.sort_values("timestamp").reset_index(drop=True)
    h["session"] = h["timestamp"].dt.tz_convert(NY).dt.tz_localize(None).dt.normalize().astype("datetime64[ns]")
    return h[["timestamp", "session", "open", "high", "low", "close", "volume"]]

BARS_D = {t: load_daily(BARS_DIR / "1day" / f"{t}.parquet") for t in TICKERS}
BARS_H = {t: load_hourly(BARS_DIR / "1hour" / f"{t}.parquet") for t in TICKERS}
# Trading calendar = the study stocks' own sessions (TriFrame verified them against the NYSE calendar: 0 missing, 0 extra).
CAL = pd.DatetimeIndex(sorted(set().union(*[set(d["date"]) for d in BARS_D.values()]))).astype("datetime64[ns]")
# SPY (market reference for abnormal returns and 3 technical features). The bundled file ends 2022-12-30 (NEXUS V1 sealed
# design), so later sessions are added from Yahoo, scaled to the bundled close on the last common day; if Yahoo fails,
# an equal-weight index of the study stocks is used after 2022. The source of every SPY row is recorded.
SPY_D = load_daily(BARS_DIR / "etf" / "SPY_1day.parquet"); SPY_D["source"] = "bundled"
SPY_CACHE = DATA_DIR / "spy_extension.parquet"
if SPY_D["date"].max() < DATA_END - pd.Timedelta(days=7):
    last = SPY_D["date"].max(); ext = None
    try:
        if SPY_CACHE.exists():
            ext = pd.read_parquet(SPY_CACHE)
        else:
            import yfinance as yf
            y_ = yf.Ticker("SPY").history(start=str((last - pd.Timedelta(days=10)).date()), end=str((DATA_END + pd.Timedelta(days=2)).date()),
                                          interval="1d", auto_adjust=False, actions=False)
            yi = pd.DatetimeIndex(y_.index); yi = yi.tz_convert(NY).tz_localize(None) if yi.tz is not None else yi
            ext = pd.DataFrame({"date": yi.normalize().astype("datetime64[ns]"), "open": y_["Open"].values, "high": y_["High"].values,
                                "low": y_["Low"].values, "close": y_["Close"].values, "volume": y_["Volume"].values.astype(float)}).dropna()
            ext.to_parquet(SPY_CACHE, index=False)
        k = float(SPY_D.set_index("date").loc[last, "close"]) / float(ext.set_index("date").loc[last, "close"])
        ext = ext[(ext["date"] > last) & (ext["date"] <= DATA_END)].copy()
        ext[["open", "high", "low", "close"]] *= k; ext["source"] = "yahoo (scaled)"
    except Exception as e:
        print(f"Yahoo SPY unavailable ({type(e).__name__}: {e}); using an equal-weight index of the study stocks after {last.date()}.")
        cl = pd.DataFrame({t: BARS_D[t].set_index("date")["close"] for t in TICKERS}).sort_index()
        ew = (cl.pct_change().mean(1).fillna(0) + 1).cumprod(); ew = ew / ew.loc[last] * float(SPY_D.set_index("date").loc[last, "close"])
        ew = ew[(ew.index > last) & (ew.index <= DATA_END)]
        ext = pd.DataFrame({"date": ew.index, "open": ew.values, "high": ew.values, "low": ew.values, "close": ew.values, "volume": 0.0,
                            "source": "equal-weight study stocks"})
    SPY_D = pd.concat([SPY_D, ext[SPY_D.columns]], ignore_index=True).sort_values("date").reset_index(drop=True)
_spy_missing = len(CAL[CAL >= pd.Timestamp(CFG["decision_start"]) - pd.Timedelta(days=400)].difference(pd.DatetimeIndex(SPY_D["date"])))
print("SPY rows by source:", SPY_D["source"].value_counts().to_dict(), "| SPY last date:", SPY_D["date"].max().date(),
      "| study sessions without SPY:", _spy_missing)
assert SPY_D["date"].max() >= DATA_END - pd.Timedelta(days=7) and _spy_missing <= 5, "SPY does not cover the study window"
PROV.append({"source": "SPY", "status": "OK", "note": str(SPY_D["source"].value_counts().to_dict())})
print(f"{len(TICKERS)} stocks loaded; daily rows {sum(len(d) for d in BARS_D.values()):,}; hourly rows {sum(len(h) for h in BARS_H.values()):,}; "
      f"calendar {CAL[0].date()} .. {CAL[-1].date()} ({len(CAL)} sessions)")

### 3.1 Bar validation tests (stop on failure)
Per stock: unique sorted dates; OHLC consistency and positive prices; hourly bars start at :30 New York time with at most 7 per session; every hourly session is a daily session; daily close equals the last hourly close (≥ 95 % of sessions within 0.1 %); daily sessions equal the SPY calendar (no missing session); bars reach the data end.

In [ ]:
BAR_CHECKS = []
for t in TICKERS:
    d, h = BARS_D[t], BARS_H[t]
    c = {"ticker": t, "sector": SECTOR_OF[t], "first": str(d.date.min().date()), "last": str(d.date.max().date()),
         "daily_rows": len(d), "hourly_rows": len(h)}
    c["daily_sorted_unique"] = bool(d["date"].is_unique and d["date"].is_monotonic_increasing)
    c["ohlc_ok"] = bool(((d.high >= d[["open", "close"]].max(axis=1) - 1e-9) & (d.low <= d[["open", "close"]].min(axis=1) + 1e-9) & (d.low > 0)).all())
    hny = h["timestamp"].dt.tz_convert(NY)
    c["hourly_minute_30"] = bool((hny.dt.minute == 30).all())
    per = h.groupby("session").size()
    c["hourly_max_per_session"] = int(per.max())
    c["hourly_subset_of_daily"] = bool(set(per.index) <= set(d["date"]))
    lh = h.groupby("session")["close"].last(); dc = d.set_index("date")["close"].reindex(lh.index)
    c["close_match_share"] = float((np.abs(dc - lh) / dc < 1e-3).mean())
    win = d["date"] >= pd.Timestamp(CFG["decision_start"]) - pd.Timedelta(days=120)
    c["sessions_without_hourly_in_window"] = int((~d.loc[win, "date"].isin(per.index)).sum())
    cal_w = CAL[(CAL >= d.date.min()) & (CAL <= d.date.max())]
    c["missing_vs_calendar"] = int(len(cal_w.difference(pd.DatetimeIndex(d.date))))
    BAR_CHECKS.append(c)
BAR_CHECKS_DF = pd.DataFrame(BAR_CHECKS); BAR_CHECKS_DF.to_csv(SUB["tables"] / "bar_validation.csv", index=False)
display(BAR_CHECKS_DF)
for c in BAR_CHECKS:
    assert c["daily_sorted_unique"] and c["ohlc_ok"], f"bar integrity failed: {c['ticker']}"
    assert c["hourly_minute_30"] and c["hourly_max_per_session"] <= 7 and c["hourly_subset_of_daily"], f"hourly layout failed: {c['ticker']}"
    assert c["close_match_share"] > 0.95, f"daily/hourly disagreement: {c['ticker']}"
    assert c["missing_vs_calendar"] == 0, f"missing sessions: {c['ticker']}"
    assert pd.Timestamp(c["last"]) >= DATA_END - pd.Timedelta(days=7), f"bars end early: {c['ticker']}"
    assert pd.Timestamp(c["first"]) <= pd.Timestamp(CFG["decision_start"]) - pd.Timedelta(days=380), f"too little warm-up history: {c['ticker']}"
print("Bar validation passed for all stocks. Sessions without hourly bars inside the window:",
      int(BAR_CHECKS_DF["sessions_without_hourly_in_window"].sum()))

### 3.2 Dataset analysis (stocks, sectors, market path)

In [ ]:
close = pd.DataFrame({t: BARS_D[t].set_index("date")["close"] for t in TICKERS}).sort_index()
rets = close.pct_change()
fig, ax = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [3, 1.2]})
BLOCK_COLORS = {"train": "#dbeafe", "inner": "#fef3c7", "val": "#dcfce7", "test": "#fee2e2"}
for s in SECTORS:
    cols = [t for t in TICKERS if SECTOR_OF[t] == s]
    idx = (rets[cols].loc[CFG["decision_start"]:].mean(1).fillna(0) + 1).cumprod()
    ax[0].plot(idx.index, idx.values, lw=1, label=s)
for b, (a, z) in CFG["blocks"].items():
    ax[0].axvspan(pd.Timestamp(a), pd.Timestamp(z), color=BLOCK_COLORS[b], alpha=0.6, lw=0)
ax[0].set_yscale("log"); ax[0].legend(fontsize=7, ncol=3); ax[0].set_title("Equal-weight price index by sector (blocks shaded: train / inner / val / test)")
ax[0].grid(alpha=0.3)
vol = rets.loc[CFG["decision_start"]:].std() * np.sqrt(252)
pd.DataFrame({"vol": vol, "sector": pd.Series(SECTOR_OF)}).boxplot(column="vol", by="sector", ax=ax[1], rot=60, fontsize=7)
ax[1].set_title("Annualised daily volatility per stock"); fig.suptitle("")
plt.tight_layout(); savefig(fig, "eda_universe.png"); plt.show()
DATASET_TABLE = pd.DataFrame([{"ticker": t, "sector": SECTOR_OF[t], "first_bar": BARS_D[t].date.min().date(),
                               "sessions_in_window": int((BARS_D[t].date >= pd.Timestamp(CFG["decision_start"])).sum()),
                               "hourly_bars": len(BARS_H[t]), "ann_vol": round(float(vol[t]), 3),
                               "median_dollar_volume_m": round(float((BARS_D[t].close * BARS_D[t].volume).median() / 1e6), 1)}
                              for t in TICKERS])
DATASET_TABLE.to_csv(SUB["tables"] / "dataset_stocks.csv", index=False); display(DATASET_TABLE.head(12))
tick("3 data")

## 4. Decisions, targets, blocks and leakage control

**Decision time.** The close of session *t* (after the last hourly bar). Every input (chart, numbers, filings) uses only information available at that moment.

**Targets for each horizon h ∈ {5, 10, 20, 40} sessions** (defined before any data were seen):
* **Direction (3 classes).** r = log(C<sub>t+h</sub> / C<sub>t</sub>). σ<sub>t</sub> = standard deviation of the last 60 daily log returns (known at *t*). **UP** if r > 0.5 σ<sub>t</sub> √h, **DOWN** if r < −0.5 σ<sub>t</sub> √h, else **SIDEWAYS**. Scaling by σ<sub>t</sub> makes "sideways" mean the same thing for a calm and a volatile stock; k = 0.5 gives roughly balanced classes for normal returns (38 % sideways).
* **Volatility.** Log of annualised realised volatility over sessions t+1 … t+h, measured from **hourly** bars (overnight return² + hourly return²): much less noisy than h daily returns.
* **Drawdown > 10 %.** 1 if the lowest low in sessions t+1 … t+h is more than 10 % below C<sub>t</sub>.

**Rows.** Evaluation rows use every 5th session of a common calendar (so 5-session labels do not overlap within a stock). Training rows use every session in FULL mode (more chart variety; overlapping labels are fine for fitting, never for evaluation).

**Blocks and purge.** A row belongs to the block of its decision date. A row is dropped if its 40-session label window reaches into the next block (purge = longest horizon), so no training label overlaps validation or test outcomes.

**Locked test.** After the labels are built, every test-block label is moved into `VAULT` and replaced by NaN in the working table. Nothing before §23 can read them; the vault opens only after the freeze manifest (§22) exists, and each opening is logged.

In [ ]:
K_SIDE, SIGW, DDT = CFG["k_sideways"], CFG["sigma_window"], CFG["dd_threshold"]
BLK = {b: (pd.Timestamp(a), pd.Timestamp(z)) for b, (a, z) in CFG["blocks"].items()}
BLOCK_ORDER = ["train", "inner", "val", "test"]
CAL_POS = {d: i for i, d in enumerate(CAL)}
G0 = int(np.searchsorted(CAL.values, np.datetime64(pd.Timestamp(CFG["decision_start"]))))
HMAX = max(HORIZONS)

def session_rv(d: pd.DataFrame, hb: pd.DataFrame) -> np.ndarray:
    """Realised variance of each daily session: overnight log return^2 + sum of hourly log returns^2
    (first hourly bar measured from the session open). Sessions without hourly bars use the daily close-to-close return^2."""
    c = d["close"].values; o = d["open"].values
    prev_c = np.r_[np.nan, c[:-1]]
    on = np.log(o / prev_c) ** 2
    hb = hb[["session", "open", "close"]].copy()
    hb["prev"] = hb.groupby("session")["close"].shift(1)
    hb["prev"] = hb["prev"].fillna(hb["open"])
    hb["r2"] = np.log(hb["close"] / hb["prev"]) ** 2
    intra = hb.groupby("session")["r2"].sum()
    intra_v = intra.reindex(pd.DatetimeIndex(d["date"])).values
    cc = np.log(c / prev_c) ** 2
    return np.where(np.isfinite(intra_v), np.nan_to_num(on) + intra_v, cc)

def build_stock_rows(t: str) -> pd.DataFrame:
    d, hb = BARS_D[t], BARS_H[t]
    pos = np.array([CAL_POS.get(x, -1) for x in d["date"]])
    c, lo = d["close"].values, d["low"].values
    lr = np.r_[np.nan, np.diff(np.log(c))]
    sig = pd.Series(lr).rolling(SIGW, min_periods=SIGW).std().values
    rvs = session_rv(d, hb); crv = np.r_[0.0, np.nancumsum(rvs)]
    n = len(d); idx = np.arange(n)
    keep = (pos >= G0) & (idx + HMAX < n) & np.isfinite(sig)
    out = {"ticker": t, "sector": SECTOR_OF[t], "date": d["date"].values, "t_idx": idx, "g": pos, "sigma": sig, "close": c}
    for h in HORIZONS:
        j = np.minimum(idx + h, n - 1)
        r = np.log(c[j] / c)
        thr = K_SIDE * sig * np.sqrt(h)
        out[f"ret_{h}"] = r
        out[f"dir_{h}"] = np.where(r > thr, 2, np.where(r < -thr, 0, 1)).astype(float)
        rv = (crv[np.minimum(idx + h + 1, n)] - crv[np.minimum(idx + 1, n)])
        out[f"vol_{h}"] = 0.5 * np.log(np.maximum(rv, 1e-12) * 252.0 / h)
        win = np.lib.stride_tricks.sliding_window_view(np.r_[lo, np.full(HMAX + 1, np.nan)], h)[1: n + 1]
        out[f"dd_{h}"] = (np.nanmin(win, axis=1) / c - 1 <= DDT).astype(float)
        for k in ("ret", "dir", "vol", "dd"):
            out[f"{k}_{h}"] = np.where(idx + h < n, out[f"{k}_{h}"], np.nan)
    df = pd.DataFrame(out)
    return df[keep]

P_ALL = pd.concat([build_stock_rows(t) for t in TICKERS], ignore_index=True)
P_ALL["date"] = pd.to_datetime(P_ALL["date"])
P_ALL["block"] = None
for b in BLOCK_ORDER:
    P_ALL.loc[(P_ALL.date >= BLK[b][0]) & (P_ALL.date <= BLK[b][1]), "block"] = b
P_ALL = P_ALL.dropna(subset=["block"])
# purge: the 40-session label window must end before the next block starts
end_g = P_ALL["g"].values + HMAX
for b, nxt in zip(BLOCK_ORDER[:-1], BLOCK_ORDER[1:]):
    g_next = int(np.searchsorted(CAL.values, np.datetime64(BLK[nxt][0])))
    P_ALL = P_ALL[~((P_ALL["block"] == b) & (P_ALL["g"] + HMAX >= g_next))]
rel = P_ALL["g"].values - G0
eval_row = rel % CFG["eval_stride"] == 0
train_row = rel % CFG["train_stride"] == 0
P = P_ALL[np.where(P_ALL["block"].values == "train", train_row, eval_row)].reset_index(drop=True)
P["eval_stride_row"] = ((P["g"] - G0) % CFG["eval_stride"] == 0)
P["month"] = P["date"].dt.to_period("M").astype(str)
P["row"] = np.arange(len(P))
N = len(P)
TR = (P.block == "train").values; IN = (P.block == "inner").values; VA = (P.block == "val").values; TE = (P.block == "test").values
TR_EVAL = TR & P["eval_stride_row"].values          # non-overlapping training rows (for diagnostics only)
LABEL_COLS = [f"{k}_{h}" for k in ("ret", "dir", "vol", "dd") for h in HORIZONS]
print(f"Panel rows: {N:,} (train {TR.sum():,}, inner {IN.sum():,}, val {VA.sum():,}, test {TE.sum():,}); stocks {P.ticker.nunique()}; "
      f"last decision {P.date.max().date()}")
assert TR.sum() and IN.sum() and VA.sum() and TE.sum(), "a block has no rows: check the calendar / data end printed in section 3"
assert P.loc[TE, "date"].max() >= pd.Timestamp("2026-06-01"), "the test block stops early: the data do not reach 2026"

# ---------------------------------------------------------------- lock the test labels (same cell as the build: a re-run always starts from fresh labels)
VAULT.deposit("panel_labels", P.loc[TE, ["row", *LABEL_COLS]])
P.loc[TE, LABEL_COLS] = np.nan
Y = {k: P[k].values.copy() for k in LABEL_COLS}       # working labels (test = NaN)

### 4.1 Leakage tests (must pass)

In [ ]:
LEAK = {}
for b, nxt in zip(BLOCK_ORDER[:-1], BLOCK_ORDER[1:]):
    last_end = CAL[int(P.loc[P.block == b, "g"].max()) + HMAX]
    LEAK[f"purge_{b}_before_{nxt}"] = bool(last_end < BLK[nxt][0])
LEAK["test_labels_hidden"] = bool(np.isnan(P.loc[TE, LABEL_COLS].values).all())
LEAK["unique_stock_dates"] = bool(not P.duplicated(["ticker", "date"]).any())
LEAK["eval_rows_non_overlapping_5"] = bool(P.loc[~TR].groupby("ticker")["g"].diff().dropna().ge(CFG["eval_stride"]).all())
# labels depend only on bars after t: changing bars up to t (except the anchor close) must not change the volatility/drawdown windows
_t = TICKERS[0]; _d = BARS_D[_t].copy(); _keep = BARS_D[_t]
_i = int(P.loc[P.ticker == _t, "t_idx"].iloc[len(P.loc[P.ticker == _t]) // 2])
_d.loc[: _i - 1, ["open", "high", "low"]] *= 1.5
BARS_D[_t] = _d; _rows = build_stock_rows(_t); BARS_D[_t] = _keep
_r0 = build_stock_rows(_t)
LEAK["labels_ignore_past_bars"] = bool(np.allclose(_rows.loc[_rows.t_idx == _i, [f"dd_{h}" for h in HORIZONS]].values,
                                                   _r0.loc[_r0.t_idx == _i, [f"dd_{h}" for h in HORIZONS]].values, equal_nan=True))
print(LEAK)
assert all(LEAK.values()), f"leakage test failed: {LEAK}"
pd.Series(LEAK).rename_axis("check").rename("passed").to_csv(SUB["tables"] / "leakage_tests.csv")

### 4.2 Label analysis (class balance and base rates, development blocks only)

In [ ]:
dev = ~TE
rows = []
for h in HORIZONS:
    for b in ["train", "inner", "val"]:
        m = (P.block == b).values & P["eval_stride_row"].values
        y = Y[f"dir_{h}"][m]
        rows.append({"horizon": h, "block": b, "n": int(m.sum()), **{f"share_{c}": float(np.mean(y == k)) for k, c in enumerate(CLASSES)},
                     "dd_rate": float(np.nanmean(Y[f"dd_{h}"][m])), "mean_vol": float(np.exp(np.nanmean(Y[f"vol_{h}"][m])))})
LABEL_BAL = pd.DataFrame(rows); LABEL_BAL.to_csv(SUB["tables"] / "label_balance.csv", index=False); display(LABEL_BAL.round(3))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
tr_lb = LABEL_BAL[LABEL_BAL.block == "train"].set_index("horizon")
tr_lb[[f"share_{c}" for c in CLASSES]].plot.bar(stacked=True, ax=ax[0], color=["#dc2626", "#9ca3af", "#16a34a"]); ax[0].set_title("Direction classes (train)")
LABEL_BAL.pivot(index="horizon", columns="block", values="dd_rate").plot.bar(ax=ax[1]); ax[1].set_title("Drawdown > 10 % base rate")
for h in HORIZONS:
    ax[2].hist(Y[f"vol_{h}"][TR & np.isfinite(Y[f"vol_{h}"])], bins=60, histtype="step", label=f"{h}d", density=True)
ax[2].legend(); ax[2].set_title("log annualised realised volatility (train)")
plt.tight_layout(); savefig(fig, "eda_labels.png"); plt.show()
sec_bal = pd.DataFrame([{"sector": s, "horizon": h, "dd_rate": float(np.nanmean(Y[f"dd_{h}"][dev & (P.sector == s).values])),
                         "share_UP": float(np.nanmean(Y[f"dir_{h}"][dev & (P.sector == s).values] == 2))}
                        for s in SECTORS for h in HORIZONS])
sec_bal.to_csv(SUB["tables"] / "label_balance_by_sector.csv", index=False)
display(sec_bal.pivot(index="sector", columns="horizon", values="dd_rate").round(3))
tick("4 labels")

## 5. Technical-indicator features (reference model only)

The CV models see **only images** and the NLP models **only text**. A gradient-boosting model on 20 classic indicators is reported as a **reference** ("what simple numbers already achieve"), because earlier work on this data found numbers hard to beat. It is never used to select CV or NLP models. All features use bars up to the decision close; the test below recomputes them on data **truncated at t** and requires identical values.

In [ ]:
def wilder(x: np.ndarray, n: int) -> np.ndarray:
    out = np.full(len(x), np.nan); s = np.nan
    for i, v in enumerate(x):
        if not np.isfinite(v):
            continue
        s = v if not np.isfinite(s) else s + (v - s) / n
        out[i] = s
    return out

def tech_frame(d: pd.DataFrame, hb: pd.DataFrame, spy: pd.DataFrame) -> pd.DataFrame:
    c, o, hi, lo, v = (d[k].values.astype(float) for k in ("close", "open", "high", "low", "volume"))
    lc = np.log(c); lr = np.r_[np.nan, np.diff(lc)]
    s = pd.Series
    f = {}
    for k in (1, 5, 20, 60, 120):
        f[f"ret_{k}d"] = lc - np.r_[np.full(k, np.nan), lc[:-k]]
    for k in (5, 20, 60):
        f[f"lrv_{k}d"] = np.log(s(lr).rolling(k, min_periods=k).std().values * np.sqrt(252) + 1e-6)
    rvs = session_rv(d, hb)
    f["lrv_hourly_5d"] = 0.5 * np.log(s(rvs).rolling(5, min_periods=5).sum().values * 252 / 5 + 1e-12)
    prev_c = np.r_[np.nan, c[:-1]]
    tr = np.nanmax(np.vstack([hi - lo, np.abs(hi - prev_c), np.abs(lo - prev_c)]), axis=0)
    f["atr14_pct"] = wilder(tr, 14) / c
    up, dn = np.clip(np.r_[np.nan, np.diff(c)], 0, None), np.clip(-np.r_[np.nan, np.diff(c)], 0, None)
    au, ad = wilder(up, 14), wilder(dn, 14)
    f["rsi14"] = 100 - 100 / (1 + au / np.maximum(ad, 1e-12))
    for k in (20, 50, 200):
        f[f"dist_sma{k}"] = c / s(c).rolling(k, min_periods=k).mean().values - 1
    sd20 = s(c).rolling(20, min_periods=20).std(ddof=0).values
    f["bb_pos20"] = (c - s(c).rolling(20, min_periods=20).mean().values) / np.maximum(2 * sd20, 1e-12)
    f["log_vol_ratio20"] = np.log((v + 1) / (s(v).rolling(20, min_periods=20).mean().values + 1))
    f["dd_252"] = c / s(hi).rolling(252, min_periods=120).max().values - 1
    mn, mx = s(lo).rolling(20, min_periods=20).min().values, s(hi).rolling(20, min_periods=20).max().values
    f["range_pos20"] = (c - mn) / np.maximum(mx - mn, 1e-12)
    f["gap_1d"] = np.log(o / prev_c)
    sp = spy.set_index("date")["close"].reindex(pd.DatetimeIndex(d["date"])).ffill().values
    lsp = np.log(sp); spr = np.r_[np.nan, np.diff(lsp)]
    f["spy_ret_5d"] = lsp - np.r_[np.full(5, np.nan), lsp[:-5]]
    f["spy_ret_20d"] = lsp - np.r_[np.full(20, np.nan), lsp[:-20]]
    f["spy_lrv_20d"] = np.log(s(spr).rolling(20, min_periods=20).std().values * np.sqrt(252) + 1e-6)
    return pd.DataFrame(f)

TECH_COLS = list(tech_frame(BARS_D[TICKERS[0]].iloc[:300], BARS_H[TICKERS[0]], SPY_D).columns)
TECH_FULL = {t: tech_frame(BARS_D[t], BARS_H[t], SPY_D) for t in TICKERS}
X_TECH = np.vstack([TECH_FULL[t].values[P.loc[P.ticker == t, "t_idx"].values] for t in TICKERS])
_order = np.concatenate([np.flatnonzero((P.ticker == t).values) for t in TICKERS])
X_TECH = X_TECH[np.argsort(_order)].astype(np.float32)
print("TECH features:", X_TECH.shape, "| missing share:", float(np.isnan(X_TECH).mean().round(4)))

# causality: recompute on data truncated at the decision session
for r in np.random.default_rng(1).choice(np.flatnonzero(VA | TE), 6, replace=False):
    t, i = P.at[r, "ticker"], int(P.at[r, "t_idx"])
    d_cut = BARS_D[t].iloc[: i + 1]
    h_cut = BARS_H[t][BARS_H[t]["session"] <= d_cut["date"].iloc[-1]]
    v_cut = tech_frame(d_cut, h_cut, SPY_D[SPY_D["date"] <= d_cut["date"].iloc[-1]]).values[-1]
    assert np.allclose(v_cut, X_TECH[r], equal_nan=True, rtol=1e-5, atol=1e-6), f"TECH feature uses future data ({t}, row {r})"
print("TECH causality test passed (truncated recomputation equals stored features).")
tick("5 tech")

## 6. Shared core (renderer, model architectures, calibration, Monte Carlo) and chart images

The code in `CORE_SRC` is executed by this notebook **and** written unchanged to the dashboard as `nav3_core.py`, so the dashboard draws charts and computes predictions with exactly the code that was trained and tested (§24 checks this numerically).

**Rendering contract (train = validation = test = dashboard).** 224 × 224 px, dark theme, teal up-candles, red down-candles, volume panel at the bottom; **no text, axes, dates or tickers** (nothing in the image can encode a date or the answer). 52 weekly candles (the last one is the week in progress, built from days ≤ t), 63 daily, 70 hourly (10 sessions). Vertical scale = the window's lowest low to highest high (price level is invisible, as in Jiang, Kelly and Xiu 2023). Images are stored as palette indices (1 byte per pixel); the colour palette is applied on the GPU, which makes the "chart style" robustness test (§12.7) a palette swap.

In [ ]:
CORE_SRC = r'''
"""NEXUS Alpha v3 shared core: chart rendering, model architectures, calibration and Monte Carlo.
Executed by the notebook and imported by the Streamlit dashboard (identical code in both)."""
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

HORIZONS = [5, 10, 20, 40]
CLASSES = ["DOWN", "SIDEWAYS", "UP"]
VIEWS = ["W", "D", "H"]
CHART_BARS = {"W": 52, "D": 63, "H": 70}
# palette index: 0 background, 1 up candle, 2 down candle, 3 up volume, 4 down volume
PALETTE_DARK = np.array([[19, 23, 34], [38, 166, 154], [239, 83, 80], [26, 110, 103], [160, 56, 54]], np.uint8)
PALETTE_LIGHT = np.array([[255, 255, 255], [8, 153, 129], [242, 54, 69], [150, 210, 200], [245, 170, 170]], np.uint8)


# ------------------------------------------------------------------ bars -> arrays
def prepare_arrays(d, hb):
    """d: daily bars (date, open, high, low, close, volume), oldest first; hb: hourly bars with a 'session' column
    (New York date of the bar). Returns plain arrays used by views_at (fast, no pandas in the loop)."""
    dates = pd.DatetimeIndex(d["date"]).values.astype("datetime64[D]")
    days = dates.astype(np.int64)
    wkey = days - (days + 3) % 7                      # Monday of each week (1970-01-01 was a Thursday)
    A = {k: d[k].values.astype(float) for k in ("open", "high", "low", "close", "volume")}
    ukeys, wstart = np.unique(wkey, return_index=True)
    wend = np.r_[wstart[1:], len(days)]
    W = {"open": A["open"][wstart], "close": A["close"][wend - 1],
         "high": np.maximum.reduceat(A["high"], wstart), "low": np.minimum.reduceat(A["low"], wstart),
         "volume": np.add.reduceat(A["volume"], wstart)}
    hs = pd.DatetimeIndex(hb["session"]).values.astype("datetime64[D]")
    sidx = np.searchsorted(dates, hs)
    ok = (sidx < len(dates)) & (dates[np.minimum(sidx, len(dates) - 1)] == hs)
    Hh = {k: hb[k].values.astype(float)[ok] for k in ("open", "high", "low", "close", "volume")}
    return {"dates": dates, "D": A, "wkey": wkey, "ukeys": ukeys, "wstart": wstart, "W": W, "H": Hh, "hsidx": sidx[ok]}


def views_at(S, i, bars=None):
    """(o, h, l, c, v) arrays for the weekly, daily and hourly chart of as-of daily row i. Uses nothing after row i."""
    bars = bars or CHART_BARS
    A, W = S["D"], S["W"]
    sl = slice(max(0, i - bars["D"] + 1), i + 1)
    out = {"D": tuple(A[k][sl] for k in ("open", "high", "low", "close", "volume"))}
    wp = int(np.searchsorted(S["ukeys"], S["wkey"][i]))
    a = max(0, wp - bars["W"] + 1)
    s0 = int(S["wstart"][wp])
    part = (A["open"][s0], A["high"][s0: i + 1].max(), A["low"][s0: i + 1].min(), A["close"][i], A["volume"][s0: i + 1].sum())
    out["W"] = tuple(np.r_[W[k][a:wp], part[j]] for j, k in enumerate(("open", "high", "low", "close", "volume")))
    e = int(np.searchsorted(S["hsidx"], i, side="right"))
    hs = slice(max(0, e - bars["H"]), e)
    out["H"] = tuple(S["H"][k][hs] for k in ("open", "high", "low", "close", "volume"))
    return out


def render_candles(o, h, l, c, v, size=224):
    """Palette-index candlestick chart (uint8, size x size) from bar arrays in time order (oldest first)."""
    o, h, l, c, v = (np.asarray(x, dtype=float) for x in (o, h, l, c, v))
    n = len(c)
    img = np.zeros((size, size), np.uint8)
    if n == 0:
        return img
    top, bot = 4, int(round(size * 0.79))
    vtop, vbot = int(round(size * 0.82)), size - 4
    lo, hi = float(np.nanmin(l)), float(np.nanmax(h))
    span = hi - lo if hi > lo else max(abs(hi), 1.0) * 1e-6
    def ypix(p):
        return top + (bot - 1 - top) * (hi - np.asarray(p, dtype=float)) / span
    pxpb = size / n
    xs = ((np.arange(n) + 0.5) * pxpb).astype(int)
    half = max(1, int(pxpb * 0.35))
    yh, yl, yo, yc = (np.clip(np.round(ypix(a)), top, bot - 1).astype(int) for a in (h, l, o, c))
    up = c >= o
    for k in range(n):
        col = 1 if up[k] else 2
        img[yh[k]: yl[k] + 1, xs[k]] = col
        b0, b1 = min(yo[k], yc[k]), max(yo[k], yc[k])
        img[b0: b1 + 1, max(0, xs[k] - half): min(size, xs[k] + half + 1)] = col
    vv = np.nan_to_num(v)
    vmax = float(vv.max()) if len(vv) else 0.0
    if vmax > 0:
        hgt = np.round(vv / vmax * (vbot - vtop)).astype(int)
        for k in range(n):
            if hgt[k] > 0:
                img[vbot - hgt[k]: vbot, max(0, xs[k] - half): min(size, xs[k] + half + 1)] = 3 if up[k] else 4
    return img


def render_views(S, i, size=224):
    """(3, size, size) palette-index stack in the order W, D, H."""
    vw = views_at(S, i)
    return np.stack([render_candles(*vw[k], size=size) for k in VIEWS])


def trailing_sigma(close, window=60):
    lr = np.diff(np.log(np.asarray(close, dtype=float)))
    return float(np.std(lr[-window:], ddof=1)) if len(lr) >= window else float("nan")


def to_float_images(idx, palette):
    """idx: uint8 tensor (B, V, H, W) of palette indices; palette: float tensor (P, 3). Returns (B, V, 3, H, W) in [0, 1]."""
    x = palette[idx.long()]
    return x.permute(0, 1, 4, 2, 3).contiguous() / 255.0


# ------------------------------------------------------------------ vision models
class JKXNet(nn.Module):
    """CNN in the style of Jiang, Kelly and Xiu (2023): (5x3) convolutions, BatchNorm, LeakyReLU and max-pooling,
    the first layer with vertical stride 3 and dilation 2; widened to four blocks for 224-px charts; trained from scratch."""
    def __init__(self, chans=(64, 128, 256, 512)):
        super().__init__()
        layers, cin = [], 3
        for j, co in enumerate(chans):
            if j == 0:
                conv = nn.Conv2d(cin, co, (5, 3), stride=(3, 1), dilation=(2, 1), padding=(12, 1))
            else:
                conv = nn.Conv2d(cin, co, (5, 3), padding=(2, 1))
            layers += [conv, nn.BatchNorm2d(co), nn.LeakyReLU(0.01), nn.MaxPool2d((2, 2))]
            cin = co
        self.body = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feat_dim = chans[-1]
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.pool(self.body(x)).flatten(1)


class YoloFeat(nn.Module):
    """Backbone of an Ultralytics YOLOv8 classification model; returns the pooled features that feed its linear layer."""
    def __init__(self, weights="yolov8s-cls.pt"):
        super().__init__()
        import os
        os.environ.setdefault("YOLO_VERBOSE", "False")
        from ultralytics import YOLO
        seq = YOLO(weights).model.model
        head = seq[-1]
        self.layers = nn.ModuleList(list(seq)[:-1])
        self.conv, self.pool = head.conv, head.pool
        self.feat_dim = int(head.linear.in_features)
        for p in self.parameters():
            p.requires_grad_(True)
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.pool(self.conv(x)).flatten(1)


def build_backbone(spec, pretrained=True):
    """(module, feature_dim, mean, std) for a CV model spec from the configuration."""
    if spec["kind"] == "timm":
        import timm
        from timm.data import resolve_data_config
        m = timm.create_model(spec["timm"], pretrained=pretrained, num_classes=0)
        dc = resolve_data_config({}, model=m)
        return m, int(m.num_features), tuple(dc["mean"]), tuple(dc["std"])
    if spec["kind"] == "jkx":
        m = JKXNet()
        return m, m.feat_dim, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    if spec["kind"] == "yolo":
        m = YoloFeat(spec["weights"])
        return m, m.feat_dim, (0.0, 0.0, 0.0), (1.0, 1.0, 1.0)
    raise ValueError(spec["kind"])


class MultiTaskHead(nn.Module):
    """Shared MLP, then direction logits (n_h x 3), standardised log-volatility (n_h) and drawdown logits (n_h)."""
    def __init__(self, in_dim, n_h=4, hidden=512, p=0.2):
        super().__init__()
        self.n_h = n_h
        self.body = nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(p), nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(p))
        self.dir = nn.Linear(hidden, n_h * 3)
        self.vol = nn.Linear(hidden, n_h)
        self.dd = nn.Linear(hidden, n_h)
    def forward(self, z):
        b = self.body(z)
        return {"dir": self.dir(b).view(-1, self.n_h, 3), "vol": self.vol(b), "dd": self.dd(b)}


class ChartNet(nn.Module):
    """One backbone shared by the weekly, daily and hourly charts; the three feature vectors are concatenated."""
    def __init__(self, spec, pretrained=True, n_views=3, n_h=4):
        super().__init__()
        self.backbone, self.feat_dim, mean, std = build_backbone(spec, pretrained)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))
        self.n_views = n_views
        self.head = MultiTaskHead(self.feat_dim * n_views, n_h)
    def embed(self, x):
        b, v = x.shape[:2]
        z = self.backbone((x.flatten(0, 1) - self.mean) / self.std)
        return z.view(b, v, -1)
    def forward(self, x):
        return self.head(self.embed(x).flatten(1))


# ------------------------------------------------------------------ text models
class TextNet(nn.Module):
    """Hugging Face encoder, attention-masked mean pooling, multi-task head."""
    def __init__(self, hf_id, n_h=4, from_config=False, grad_ckpt=False):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        if from_config:
            self.enc = AutoModel.from_config(AutoConfig.from_pretrained(hf_id))
        else:
            self.enc = AutoModel.from_pretrained(hf_id, torch_dtype=torch.float32)   # some checkpoints are stored in fp16
        self.enc = self.enc.float()
        if grad_ckpt and hasattr(self.enc, "gradient_checkpointing_enable"):
            self.enc.gradient_checkpointing_enable()
        self.is_longformer = getattr(self.enc.config, "model_type", "") == "longformer"
        self.head = MultiTaskHead(int(self.enc.config.hidden_size), n_h)
    def pooled(self, input_ids=None, attention_mask=None, inputs_embeds=None):
        kw = {}
        if self.is_longformer:
            g = torch.zeros_like(attention_mask); g[:, 0] = 1
            kw["global_attention_mask"] = g
        if inputs_embeds is not None:
            out = self.enc(inputs_embeds=inputs_embeds, attention_mask=attention_mask, **kw).last_hidden_state
        else:
            out = self.enc(input_ids=input_ids, attention_mask=attention_mask, **kw).last_hidden_state
        out = out.float()
        m = attention_mask.unsqueeze(-1).float()
        return (out * m).sum(1) / m.sum(1).clamp(min=1.0)
    def forward(self, input_ids, attention_mask):
        return self.head(self.pooled(input_ids, attention_mask))


def text_input(form, section, body, max_chars=60000):
    """The exact string a text model reads: form and section tags, then the text."""
    return f"[{form}] [{section}] " + " ".join(str(body or "").split())[:max_chars]


# ------------------------------------------------------------------ raw outputs -> calibrated predictions
def softmax_np(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def apply_calibration(raw, cal):
    """raw: {'dir_h': (n,3) logits, 'vol_h': (n,) log-vol, 'dd_h': (n,) logit}; cal from calibrate_fit (notebook).
    Returns {'dir_h': (n,3) probabilities, 'vol_h': (n,), 'dd_h': (n,) probabilities}."""
    out = {}
    for h in HORIZONS:
        T = cal.get(f"T_{h}", 1.0)
        out[f"dir_{h}"] = softmax_np(np.asarray(raw[f"dir_{h}"], dtype=float) / T)
        a, b = cal.get(f"vol_ab_{h}", (0.0, 1.0))
        out[f"vol_{h}"] = a + b * np.asarray(raw[f"vol_{h}"], dtype=float)
        a, b = cal.get(f"dd_ab_{h}", (0.0, 1.0))
        out[f"dd_{h}"] = 1.0 / (1.0 + np.exp(-(a + b * np.asarray(raw[f"dd_{h}"], dtype=float))))
    return out


def _default_amp():
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16


@torch.no_grad()
def chart_raw_outputs(model, idx_u8, palette_t, vol_mean, vol_std, device, batch=128, amp_dtype=None):
    """Raw outputs of a ChartNet for palette-index images (n, 3, H, W) as numpy uint8."""
    model.eval()
    res = {"dir": [], "vol": [], "dd": []}
    for s in range(0, len(idx_u8), batch):
        x = to_float_images(torch.from_numpy(np.ascontiguousarray(idx_u8[s: s + batch])).to(device), palette_t)
        with torch.autocast(device_type=device.type, dtype=amp_dtype or _default_amp(), enabled=device.type == "cuda"):
            o = model(x)
        for k in res:
            res[k].append(o[k].float().cpu().numpy())
    d, v, q = (np.concatenate(res[k]) for k in ("dir", "vol", "dd"))
    out = {}
    for j, h in enumerate(HORIZONS):
        out[f"dir_{h}"] = d[:, j]
        out[f"vol_{h}"] = v[:, j] * vol_std[j] + vol_mean[j]
        out[f"dd_{h}"] = q[:, j]
    return out


@torch.no_grad()
def text_raw_outputs(model, tok, texts, max_len, vol_mean, vol_std, device, batch=8, amp_dtype=None, use_amp=True):
    model.eval()
    res = {"dir": [], "vol": [], "dd": []}
    for s in range(0, len(texts), batch):
        enc = tok(list(texts[s: s + batch]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype or _default_amp(), enabled=use_amp and device.type == "cuda"):
            o = model(enc["input_ids"], enc["attention_mask"])
        for k in res:
            res[k].append(o[k].float().cpu().numpy())
    d, v, q = (np.concatenate(res[k]) for k in ("dir", "vol", "dd"))
    out = {}
    for j, h in enumerate(HORIZONS):
        out[f"dir_{h}"] = d[:, j]
        out[f"vol_{h}"] = v[:, j] * vol_std[j] + vol_mean[j]
        out[f"dd_{h}"] = q[:, j]
    return out


def raw_from_probs(pred):
    """Calibrated predictions back to 'raw' form (log-probabilities, logits, log-volatility) so an ensemble or blend
    can be re-calibrated like any single model."""
    raw = {}
    for t, v in pred.items():
        v = np.asarray(v, dtype=float)
        if t.startswith("dir"):
            raw[t] = np.log(np.clip(v, 1e-9, 1))
        elif t.startswith("dd"):
            q = np.clip(v, 1e-6, 1 - 1e-6)
            raw[t] = np.log(q / (1 - q))
        else:
            raw[t] = v
    return raw


def average_predictions(preds):
    """Equal-weight ensemble of calibrated predictions (probabilities and log-volatility are averaged)."""
    keys = preds[0].keys()
    return {k: np.mean([np.asarray(p[k], dtype=float) for p in preds], axis=0) for k in keys}


def blend(p_a, p_b, w):
    """w * p_a + (1 - w) * p_b for every task (w can differ by task: dict task -> weight)."""
    return {k: (w[k] if isinstance(w, dict) else w) * np.asarray(p_a[k]) + (1 - (w[k] if isinstance(w, dict) else w)) * np.asarray(p_b[k])
            for k in p_a}


# ------------------------------------------------------------------ Monte Carlo price paths
def mc_simulate(p0, sigma_t, vol_ann, p_dir, nu=6, n_days=20, n_paths=4000, k=0.5, dd=-0.10, seed=0):
    """Daily log returns = mu + s * e, e ~ standardised Student-t(nu) (nu = 0 -> normal), s = vol_ann / sqrt(252).
    mu is chosen on a grid so that the simulated DOWN / SIDEWAYS / UP shares at n_days (same thresholds as the labels:
    +-k * sigma_t * sqrt(n_days)) are closest (KL divergence) to the model's calibrated probabilities p_dir.
    Intraday lows between consecutive closes come from the exact Brownian-bridge minimum, giving P(drawdown > 10 %)."""
    rng = np.random.default_rng(seed)
    s = float(vol_ann) / math.sqrt(252.0)
    if nu and nu > 2:
        e = rng.standard_t(nu, size=(n_paths, n_days)) * math.sqrt((nu - 2) / nu)
    else:
        e = rng.standard_normal((n_paths, n_days))
    thr = k * float(sigma_t) * math.sqrt(n_days)
    S = e.sum(1)
    grid = np.linspace(-0.5, 0.5, 201) * s
    R = grid[:, None] * n_days + s * S[None, :]
    pu, pdn = (R > thr).mean(1), (R < -thr).mean(1)
    q = np.clip(np.vstack([pdn, 1 - pu - pdn, pu]).T, 1e-4, 1.0)
    p = np.clip(np.asarray(p_dir, dtype=float), 1e-4, 1.0)
    mu = float(grid[int(np.argmin((p * np.log(p / q)).sum(1)))])
    lr = mu + s * e
    logp = math.log(p0) + np.cumsum(lr, axis=1)
    a = np.concatenate([np.full((n_paths, 1), math.log(p0)), logp[:, :-1]], axis=1)
    u = rng.random((n_paths, n_days))
    lows = (a + logp - np.sqrt((logp - a) ** 2 - 2 * s ** 2 * np.log(u))) / 2
    p_dd = float((lows.min(1) <= math.log(p0) + math.log(1 + dd)).mean())
    R_end = logp[:, -1] - math.log(p0)
    sim_dir = [float((R_end < -thr).mean()), float(((R_end >= -thr) & (R_end <= thr)).mean()), float((R_end > thr).mean())]
    qs = np.percentile(np.exp(logp), [5, 25, 50, 75, 95], axis=0)
    return {"paths": np.exp(logp), "mu_daily": mu, "sd_daily": s, "p_dd": p_dd, "sim_dir": sim_dir, "quantiles": qs,
            "thr": thr, "ret_q": np.percentile(R_end, [5, 10, 25, 50, 75, 90, 95])}
'''
exec(compile(CORE_SRC, "nav3_core.py", "exec"), globals())
print("Core loaded:", ", ".join(n for n in ["prepare_arrays", "views_at", "render_candles", "ChartNet", "TextNet", "mc_simulate"]))

In [ ]:
# ---------------------------------------------------------------- render every panel row (local Colab disk; memmap)
S_ARR = {t: prepare_arrays(BARS_D[t], BARS_H[t]) for t in TICKERS}
IMG_PATH = LOCAL_ROOT / f"images_{cfg_hash([CFG['chart'], N, P[['ticker', 't_idx']].values.tolist()])}.npy"
IMG_MARK = SUB["manifests"] / "images_rendered.json"
def _render_ticker(t):
    rows = np.flatnonzero((P.ticker == t).values)
    mm = np.load(IMG_PATH, mmap_mode="r+")
    S_t = S_ARR[t]
    for r, i in zip(rows, P.loc[rows, "t_idx"].values):
        mm[r] = render_views(S_t, int(i))
    mm.flush(); del mm
    return len(rows)
if not IMG_PATH.exists():
    t0 = time.time()
    _mm = np.lib.format.open_memmap(IMG_PATH, mode="w+", dtype=np.uint8, shape=(N, 3, 224, 224)); del _mm
    import multiprocessing as mp
    with mp.get_context("fork").Pool(min(12, os.cpu_count() or 2)) as pool:
        done = pool.map(_render_ticker, TICKERS)
    print(f"Rendered {sum(done):,} rows x 3 charts in {time.time() - t0:.0f}s -> {IMG_PATH} ({N * 3 * 224 * 224 / 1e9:.1f} GB)")
    IMG_MARK.write_text(json.dumps({"rows": N, "path": str(IMG_PATH), "utc": now_utc()}))
_avail = __import__("psutil").virtual_memory().available
IMG = np.load(IMG_PATH, mmap_mode=None if N * 3 * 224 * 224 < 0.35 * _avail else "r")
print("Images:", IMG.shape, "| in RAM" if isinstance(IMG, np.ndarray) and not isinstance(IMG, np.memmap) else "| memory-mapped")
PAL_DARK_T = torch.tensor(PALETTE_DARK, dtype=torch.float32, device=DEV)
PAL_LIGHT_T = torch.tensor(PALETTE_LIGHT, dtype=torch.float32, device=DEV)

### 6.2 Image tests (determinism, causality, stored = fresh render, no blank charts)

In [ ]:
r0 = int(np.flatnonzero(VA)[len(np.flatnonzero(VA)) // 2])
t0_, i0 = P.at[r0, "ticker"], int(P.at[r0, "t_idx"])
a1 = render_views(S_ARR[t0_], i0)
assert np.array_equal(a1, render_views(S_ARR[t0_], i0)), "renderer is not deterministic"
_d = BARS_D[t0_].copy(); _d.loc[_d.index > i0, ["open", "high", "low", "close"]] *= 3.0
_h = BARS_H[t0_].copy(); _h.loc[_h["session"] > P.at[r0, "date"], ["open", "high", "low", "close"]] *= 3.0
assert np.array_equal(a1, render_views(prepare_arrays(_d, _h), i0)), "future bars changed the image (leakage)"
assert np.array_equal(np.asarray(IMG[r0]), a1), "stored image differs from a fresh render"
_samp = np.linspace(0, N - 1, 300).astype(int)
nonbg = np.array([(np.asarray(IMG[r]) > 0).mean() for r in _samp])
assert nonbg.min() > 0.01, "a blank chart was rendered"
_vw = views_at(S_ARR[t0_], i0)
print(f"Image tests passed. Bars per view at a validation row: { {k: len(v[3]) for k, v in _vw.items()} }; "
      f"non-background share median {np.median(nonbg):.3f}")

### 6.3 Image dataset analysis

In [ ]:
rng = np.random.default_rng(3)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i_, k in enumerate(range(3)):
    cand = np.flatnonzero(TR & (Y["dir_20"] == k))
    r = int(rng.choice(cand))
    for j, v in enumerate(VIEWS):
        ax = axes[i_, j]; ax.imshow(PALETTE_DARK[np.asarray(IMG[r, j])]); ax.set_xticks([]); ax.set_yticks([])
        if i_ == 0:
            ax.set_title({"W": "weekly (52)", "D": "daily (63)", "H": "hourly (70)"}[v])
    axes[i_, 0].set_ylabel(f"{P.at[r, 'ticker']} {P.at[r, 'date'].date()}\n20-day label: {CLASSES[k]}")
plt.suptitle("Model inputs: one training example per 20-day class (no text, axes or dates in any image)")
plt.tight_layout(); savefig(fig, "chart_gallery.png", dpi=110); plt.show()
def up_share(a):
    a = np.asarray(a); u, d_ = int((a == 1).sum()), int((a == 2).sum()); return u / max(1, u + d_)
_tr = np.flatnonzero(TR)[:: max(1, TR.sum() // 800)]
IMG_STATS = pd.DataFrame({"label_20": [CLASSES[int(Y["dir_20"][r])] for r in _tr], **{f"up_pixel_share_{v}": [up_share(IMG[r, j]) for r in _tr] for j, v in enumerate(VIEWS)}})
display(IMG_STATS.groupby("label_20").mean().round(3))
IMG_STATS.groupby("label_20").mean().to_csv(SUB["tables"] / "image_stats_by_label.csv")
pd.DataFrame([{"rows": N, "views": 3, "height": 224, "width": 224, "colours": len(PALETTE_DARK), "bytes_gb": round(N * 3 * 224 * 224 / 1e9, 2),
               "train_rows": int(TR.sum()), "inner_rows": int(IN.sum()), "val_rows": int(VA.sum()), "test_rows": int(TE.sum())}]).to_csv(
    SUB["tables"] / "image_dataset.csv", index=False)
tick("6 render")

## 7. Evaluation toolkit (metrics, uncertainty, calibration, prediction store)

**Metrics matched to each task.**
* **Direction (3 classes):** macro-F1 (primary: every class counts equally, so predicting only "UP" in a rising market scores badly), accuracy, balanced accuracy, log loss and multi-class Brier score (probability quality), ROC-AUC of UP-vs-rest and DOWN-vs-rest, confusion matrix. Skill = macro-F1 minus the **within-month shuffled-prediction null** (same predictions, labels permuted inside each month; keeps the class mix, removes any link to the future).
* **Volatility (log annualised realised volatility):** R² (primary), Spearman correlation, QLIKE (Patton 2011: the robust loss for volatility forecasts), MAE.
* **Drawdown > 10 %:** ROC-AUC (primary: ranking quality, insensitive to the rare base rate), PR-AUC (average precision: what matters when positives are rare), Brier skill score against the training base rate, expected calibration error (ECE, 10 bins).

**Uncertainty.** Rows are resampled in **calendar-month blocks** (all stocks of a month together), because returns of different stocks in the same month are correlated; B = 1000 (200 in QUICK). Model-vs-model comparisons are **paired** (same resampled rows for both). p-values: share of bootstrap differences on the wrong side of zero, doubled. Families of tests use **Holm**.

**Calibration.** Every model's raw outputs are calibrated on the **inner block (2021)** only: temperature scaling for direction (Guo et al. 2017), Platt scaling for drawdown, a linear map for volatility. Validation and test never touch a calibrator's fit.

In [ ]:
from scipy.stats import rankdata, spearmanr
from scipy.optimize import minimize_scalar

def auc_np(y, s):
    y = np.asarray(y).astype(bool); n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def ap_np(y, s):
    y = np.asarray(y).astype(bool)
    if y.sum() == 0:
        return np.nan
    o = np.argsort(-np.asarray(s), kind="stable"); yy = y[o]
    tp = np.cumsum(yy); prec = tp / np.arange(1, len(yy) + 1)
    return float(prec[yy].sum() / yy.sum())

def ece_np(y, p, bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    e = np.linspace(0, 1, bins + 1); k = np.clip(np.digitize(p, e[1:-1]), 0, bins - 1); out = 0.0
    for b in range(bins):
        m = k == b
        if m.any():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def dir_metrics(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, float); n = len(y)
    if n == 0:
        return {}
    yh = p.argmax(1)
    cm = np.zeros((3, 3)); np.add.at(cm, (y, yh), 1)
    tp = np.diag(cm); prec = np.divide(tp, cm.sum(0), out=np.zeros(3), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(3), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    oh = np.eye(3)[y]
    return {"macro_f1": float(f1.mean()), "accuracy": float(tp.sum() / n), "balanced_acc": float(rec.mean()),
            "log_loss": float(-np.mean(np.log(np.clip(p[np.arange(n), y], 1e-9, 1)))),
            "brier": float(np.mean(((p - oh) ** 2).sum(1))), "auc_up": auc_np(y == 2, p[:, 2]), "auc_down": auc_np(y == 0, p[:, 0]),
            "f1_down": float(f1[0]), "f1_side": float(f1[1]), "f1_up": float(f1[2]), "n": n}

def vol_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if len(y) < 3:
        return {}
    e = y - p
    return {"r2": float(1 - (e ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-12)), "spearman": float(spearmanr(y, p)[0]),
            "qlike": float(np.mean(np.exp(2 * e) - 2 * e - 1)), "mae": float(np.abs(e).mean()), "n": len(y)}

def dd_metrics(y, p, prior):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if len(y) == 0:
        return {}
    b = float(np.mean((p - y) ** 2)); b0 = float(np.mean((prior - y) ** 2))
    return {"auc": auc_np(y, p), "pr_auc": ap_np(y, p), "brier": b, "bss": float(1 - b / b0) if b0 > 0 else np.nan,
            "ece": ece_np(y, p), "base_rate": float(y.mean()), "n": len(y)}

def primary_value(task, y, p):
    kind = task.split("_")[0]
    y = np.asarray(y)
    if len(y) == 0:
        return np.nan
    if kind == "dir":                   # fast path (called thousands of times by the bootstrap): macro-F1 from a 3x3 confusion count
        yi = y.astype(int); yh = np.asarray(p, float).argmax(1)
        cm = np.bincount(yi * 3 + yh, minlength=9).reshape(3, 3).astype(float)
        tp = np.diag(cm); cs, rs = cm.sum(0), cm.sum(1)
        prec = np.divide(tp, cs, out=np.zeros(3), where=cs > 0); rec = np.divide(tp, rs, out=np.zeros(3), where=rs > 0)
        return float(np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0).mean())
    if kind == "vol":
        e = np.asarray(y, float) - np.asarray(p, float); yy = np.asarray(y, float)
        return float(1 - (e ** 2).sum() / max(((yy - yy.mean()) ** 2).sum(), 1e-12))
    return auc_np(y, p)

In [ ]:
class EvalCtx:
    """Labels, months, blocks and sectors of one evaluation table (the panel, or the filing events).
    Test labels are NaN until the vault is opened in section 23."""
    def __init__(self, name, labels, months, blocks, sectors, tickers):
        self.name, self.Y = name, labels
        self.months, self.blocks = np.asarray(months), np.asarray(blocks)
        self.sectors, self.tickers = np.asarray(sectors), np.asarray(tickers)
        self.n = len(self.months)
        self.mask = {b: self.blocks == b for b in BLOCK_ORDER}
        self.prior = {}
    def fit_priors(self, fit_mask):
        for h in HORIZONS:
            y = self.Y[f"dir_{h}"][fit_mask]; y = y[np.isfinite(y)].astype(int)
            self.prior[f"dir_{h}"] = np.bincount(y, minlength=3) / max(1, len(y))
            self.prior[f"dd_{h}"] = float(np.nanmean(self.Y[f"dd_{h}"][fit_mask]))
            self.prior[f"vol_{h}"] = float(np.nanmean(self.Y[f"vol_{h}"][fit_mask]))
    def rows(self, task, mask, pred=None):
        """Row indices inside mask with a finite label (and a finite prediction, if given)."""
        y = self.Y[task]; m = mask & np.isfinite(y)
        if pred is not None:
            pr = np.asarray(pred)
            m &= np.isfinite(pr).all(1) if pr.ndim == 2 else np.isfinite(pr)
        return np.flatnonzero(m)

def metrics_for(ctx, task, pred, mask):
    r = ctx.rows(task, mask, pred)
    y, p = ctx.Y[task][r], np.asarray(pred)[r]
    kind = task.split("_")[0]
    if kind == "dir":
        return dir_metrics(y, p)
    if kind == "vol":
        return vol_metrics(y, p)
    return dd_metrics(y, p, ctx.prior.get(task, float(np.nanmean(y)) if len(y) else 0.0))

def month_groups(months_sub):
    u, inv = np.unique(months_sub, return_inverse=True)
    return [np.flatnonzero(inv == k) for k in range(len(u))]

BOOT = {"B": None}       # Phase A (section 23A) raises the number of bootstrap draws for the many-test ablation families, then resets this to None

def bootstrap(fn, groups, B=None, seed=0):
    """Month-block bootstrap of fn(row_positions); returns an array of B statistics."""
    B = B or BOOT["B"] or CFG["bootstrap_B"]; rng = np.random.default_rng(seed); out = np.empty(B)
    G = len(groups)
    for b in range(B):
        pick = rng.integers(0, G, G)
        out[b] = fn(np.concatenate([groups[g] for g in pick]))
    return out

def dir_null_f1(y, p, months, n_perm=20, seed=0):
    """Mean macro-F1 when labels are shuffled within months (keeps class shares, breaks the link to the future)."""
    rng = np.random.default_rng(seed); groups = month_groups(months); vals = []
    for _ in range(n_perm):
        ys = y.copy()
        for g in groups:
            ys[g] = ys[g][rng.permutation(len(g))]
        vals.append(primary_value("dir_0", ys, p))
    return float(np.mean(vals))

def skill_test(ctx, task, pred, mask, seed=0):
    """Point estimate, 95 % CI and one-sided p of skill: macro-F1 - shuffled null / R^2 - 0 / AUC - 0.5."""
    r = ctx.rows(task, mask, pred)
    if len(r) < 30:
        return {"n": len(r), "skill": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan}
    y, p, mo = ctx.Y[task][r], np.asarray(pred)[r], ctx.months[r]
    kind = task.split("_")[0]
    null = dir_null_f1(y.astype(int), p, mo, seed=seed) if kind == "dir" else (0.0 if kind == "vol" else 0.5)
    fn = lambda ix: primary_value(task, y[ix], p[ix]) - null
    bs = bootstrap(fn, month_groups(mo), seed=seed)
    est = fn(np.arange(len(r)))
    return {"n": len(r), "null": null, "skill": float(est), "lo": float(np.nanpercentile(bs, 2.5)), "hi": float(np.nanpercentile(bs, 97.5)),
            "p": float((1 + np.sum(bs <= 0)) / (1 + np.isfinite(bs).sum()))}

def paired_delta(ctx, task, pa, pb, mask, seed=0):
    """Delta of the primary metric (A - B) on rows where both predict; 95 % CI and two-sided p (paired month bootstrap)."""
    pa, pb = np.asarray(pa), np.asarray(pb)
    r = np.intersect1d(ctx.rows(task, mask, pa), ctx.rows(task, mask, pb))
    if len(r) < 30:
        return {"n": len(r), "delta": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan}
    y = ctx.Y[task][r]; A, Bp = pa[r], pb[r]
    fn = lambda ix: primary_value(task, y[ix], A[ix]) - primary_value(task, y[ix], Bp[ix])
    bs = bootstrap(fn, month_groups(ctx.months[r]), seed=seed)
    est = fn(np.arange(len(r)))
    p2 = 2 * min((1 + np.sum(bs <= 0)) / (1 + len(bs)), (1 + np.sum(bs >= 0)) / (1 + len(bs)))
    return {"n": len(r), "delta": float(est), "lo": float(np.nanpercentile(bs, 2.5)), "hi": float(np.nanpercentile(bs, 97.5)), "p": float(min(1.0, p2))}

def best_rows(df, keys, col="primary"):
    """Row with the highest `col` per group of `keys`; groups with no valid value are skipped (small QUICK-mode groups)."""
    d = df.dropna(subset=[col])
    return d.loc[d.groupby(keys)[col].idxmax()] if len(d) else d

def holm(pvals):
    p = np.asarray(pvals, float); out = np.full(len(p), np.nan); ok = np.isfinite(p)
    idx = np.flatnonzero(ok); o = idx[np.argsort(p[idx])]; m = len(o); run = 0.0
    for k, i in enumerate(o):
        run = max(run, min(1.0, (m - k) * p[i])); out[i] = run
    return out

In [ ]:
# ---------------------------------------------------------------- calibration (fit on the inner block only)
def calibrate_fit(ctx, raw, fit_mask):
    cal = {}
    for h in HORIZONS:
        t = f"dir_{h}"; r = ctx.rows(t, fit_mask, raw[t])
        if len(r) >= 50:
            z = np.asarray(raw[t], float)[r]; y = ctx.Y[t][r].astype(int)
            nll = lambda T: -np.mean(np.log(np.clip(softmax_np(z / T)[np.arange(len(y)), y], 1e-9, 1)))
            cal[f"T_{h}"] = float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)
        t = f"vol_{h}"; r = ctx.rows(t, fit_mask, raw[t])
        if len(r) >= 50:
            x = np.asarray(raw[t], float)[r]; y = ctx.Y[t][r]
            b = float(np.cov(x, y)[0, 1] / max(np.var(x, ddof=1), 1e-12)); a = float(y.mean() - b * x.mean())
            cal[f"vol_ab_{h}"] = (a, b)
        t = f"dd_{h}"; r = ctx.rows(t, fit_mask, raw[t])
        if len(r) >= 50 and 0 < ctx.Y[t][r].sum() < len(r):
            x = np.asarray(raw[t], float)[r].reshape(-1, 1)
            lr_ = LogisticRegression(C=100.0).fit(x, ctx.Y[t][r].astype(int))
            cal[f"dd_ab_{h}"] = (float(lr_.intercept_[0]), float(lr_.coef_[0, 0]))
    return cal

# ---------------------------------------------------------------- prediction store
PRED = {"panel": {}, "event": {}}
RAW = {"panel": {}, "event": {}}
CALS = {"panel": {}, "event": {}}
def _pred_path(ctxname, mid):
    (SUB["preds"] / ctxname).mkdir(exist_ok=True); return SUB["preds"] / ctxname / f"{mid}.npz"
def save_pred(ctxname, mid, pred, raw, cal):
    np.savez_compressed(_pred_path(ctxname, mid), **{f"p__{k}": v for k, v in pred.items()}, **{f"r__{k}": v for k, v in raw.items()})
    (SUB["preds"] / ctxname / f"{mid}.cal.json").write_text(json.dumps(cal))
    PRED[ctxname][mid], RAW[ctxname][mid], CALS[ctxname][mid] = pred, raw, cal
def load_pred(ctxname, mid):
    if mid in PRED[ctxname]:
        return PRED[ctxname][mid]
    p_ = _pred_path(ctxname, mid)
    if not p_.exists():
        return None
    z = np.load(p_)
    expected = N if ctxname == "panel" else (len(EV) if "EV" in globals() else None)
    sizes = {len(z[k]) for k in z.files}
    if expected is not None and sizes != {expected}:
        print(f"[{ctxname}/{mid}] saved predictions have {sorted(sizes)} rows, this run has {expected}: stale, recomputing")
        return None
    PRED[ctxname][mid] = {k[3:]: z[k] for k in z.files if k.startswith("p__")}
    RAW[ctxname][mid] = {k[3:]: z[k] for k in z.files if k.startswith("r__")}
    cp = SUB["preds"] / ctxname / f"{mid}.cal.json"
    CALS[ctxname][mid] = json.loads(cp.read_text()) if cp.exists() else {}
    return PRED[ctxname][mid]
def have_pred(ctxname, mid):
    return load_pred(ctxname, mid) is not None and REG.get(mid, {}).get("status") == "RUN"

def summarize(ctx, pred, mask):
    return {t: metrics_for(ctx, t, pred[t], mask) for t in TASKS if t in pred}

def finalize(ctx, mid, raw, family, **meta):
    """Calibrate on inner, store, and register inner/validation metrics (test is scored only in section 23)."""
    cal = calibrate_fit(ctx, raw, ctx.mask["inner"])
    pred = apply_calibration(raw, cal)
    save_pred(ctx.name, mid, pred, raw, cal)
    mv = summarize(ctx, pred, ctx.mask["val"])
    register(mid, status="RUN", family=family, ctx=ctx.name,
             val_primary={t: (mv[t].get(PRIMARY[t.split('_')[0]]) if mv.get(t) else None) for t in TASKS}, **meta)
    return pred

def leaderboard(ctx, mids, mask, primary_only=False):
    rows = []
    for mid in mids:
        pr = load_pred(ctx.name, mid)
        if pr is None or REG.get(mid, {}).get("status") != "RUN":
            rows.append({"model": mid, "status": REG.get(mid, {}).get("status", "NOT_RUN")}); continue
        for t in TASKS:
            m = metrics_for(ctx, t, pr[t], mask)
            if not m:
                continue
            row = {"model": mid, "status": "RUN", "task": t, "target": t.split("_")[0], "horizon": int(t.split("_")[1]),
                   "primary": m.get(PRIMARY[t.split("_")[0]])}
            if not primary_only:
                row.update(m)
            rows.append(row)
    return pd.DataFrame(rows, columns=None) if rows else pd.DataFrame(columns=["model", "status", "task", "target", "horizon", "primary"])

def mean_rank(lb):
    """Mean rank over the 12 tasks (1 = best) from a leaderboard with columns model, task, primary."""
    ok = lb[lb.status == "RUN"].dropna(subset=["primary"])
    if ok.empty:
        return pd.Series(dtype=float)
    r = ok.pivot_table(index="model", columns="task", values="primary").rank(ascending=False)
    return r.mean(1).sort_values()

In [ ]:
# ---------------------------------------------------------------- generic heads on features (linear or LightGBM)
GBM = dict(learning_rate=0.03, n_estimators=2000, num_leaves=15, min_child_samples=200, feature_fraction=0.8,
           bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, n_jobs=8)
def fit_heads(ctx, X, fit_mask, sel_mask, kind="linear", seed=0, tasks=None):
    """One model per task on fit_mask rows; the regularisation is chosen on sel_mask (inner) by loss.
    Returns raw outputs for all rows: direction log-probabilities, volatility predictions, drawdown logits."""
    rng = np.random.default_rng(seed); raw = {}
    X = np.asarray(X, dtype=np.float32)
    okX = np.isfinite(X).all(1) if kind == "linear" else np.ones(len(X), bool)
    tasks = tasks or TASKS
    for t in tasks:
        kd = t.split("_")[0]
        fr = np.intersect1d(ctx.rows(t, fit_mask), np.flatnonzero(okX))
        if len(fr) > CFG["probe"]["max_fit_rows"]:
            fr = np.sort(rng.choice(fr, CFG["probe"]["max_fit_rows"], replace=False))
        sr = np.intersect1d(ctx.rows(t, sel_mask), np.flatnonzero(okX))
        y_f, y_s = ctx.Y[t][fr], ctx.Y[t][sr]
        out = np.full((len(X), 3) if kd == "dir" else len(X), np.nan)
        if len(fr) < 100 or len(sr) < 30 or (kd != "vol" and len(np.unique(y_f)) < (3 if kd == "dir" else 2)):
            raw[t] = out; continue            # too few rows or a missing class: the task is left empty (shows as '-' / Not Run)
        if kind == "linear":
            sc = StandardScaler().fit(X[fr]); Xf, Xs, Xa = sc.transform(X[fr]), sc.transform(X[sr]), sc.transform(np.nan_to_num(X))
            best, best_loss = None, np.inf
            for c in (CFG["probe"]["alpha_grid"] if kd == "vol" else CFG["probe"]["C_grid"]):
                if kd == "vol":
                    m = Ridge(alpha=c).fit(Xf, y_f); loss = float(np.mean((m.predict(Xs) - y_s) ** 2))
                else:
                    m = LogisticRegression(C=c, max_iter=2000).fit(Xf, y_f.astype(int))
                    ps = np.clip(m.predict_proba(Xs), 1e-9, 1)
                    yi = y_s.astype(int) if kd == "dir" else y_s.astype(int)
                    loss = float(-np.mean(np.log(ps[np.arange(len(yi)), yi])))
                if loss < best_loss:
                    best, best_loss = m, loss
            if kd == "vol":
                out = best.predict(Xa)
            elif kd == "dir":
                out = np.log(np.clip(best.predict_proba(Xa), 1e-9, 1))
            else:
                pp = np.clip(best.predict_proba(Xa)[:, 1], 1e-6, 1 - 1e-6); out = np.log(pp / (1 - pp))
            out = np.where(okX[:, None], out, np.nan) if out.ndim == 2 else np.where(okX, out, np.nan)
        else:
            cb = [lgb.early_stopping(100, verbose=False)]
            if kd == "vol":
                m = lgb.LGBMRegressor(**GBM, random_state=seed).fit(X[fr], y_f, eval_set=[(X[sr], y_s)], callbacks=cb)
                out = m.predict(X)
            elif kd == "dir":
                m = lgb.LGBMClassifier(**GBM, objective="multiclass", random_state=seed).fit(X[fr], y_f.astype(int), eval_set=[(X[sr], y_s.astype(int))], callbacks=cb)
                out = np.log(np.clip(m.predict_proba(X), 1e-9, 1))
            else:
                m = lgb.LGBMClassifier(**GBM, random_state=seed).fit(X[fr], y_f.astype(int), eval_set=[(X[sr], y_s.astype(int))], callbacks=cb)
                pp = np.clip(m.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6); out = np.log(pp / (1 - pp))
        raw[t] = out
    return raw

PANEL = EvalCtx("panel", Y, P["month"].values, P["block"].values, P["sector"].values, P["ticker"].values)
PANEL.fit_priors(TR)
print("Panel priors (train):", {k: (np.round(v, 3).tolist() if isinstance(v, np.ndarray) else round(v, 3)) for k, v in PANEL.prior.items() if k.startswith(("dir_20", "dd_20"))})
tick("7 eval")

## 8. Baselines (every deep model must beat these to claim anything)

| Id | What it is | Why |
|---|---|---|
| `prior` | training class shares / mean volatility / drawdown base rate, then calibrated | the "know nothing" floor |
| `baseline_simple` | direction: logistic on 20/60/120-day momentum; volatility: **HAR** regression on 5/20/60-day and hourly realised volatility (Corsi 2009); drawdown: logistic on trailing volatility, distance from 52-week high and 20-day return | the classic finance baselines for each target |
| `tech_gbm` | LightGBM on all 20 technical indicators (§5), early-stopped on the inner block | reference: what simple numbers already achieve |

CV-specific controls (random-weight backbone, label shuffle) are in §9; the NLP baseline (TF-IDF) is in §15.

In [ ]:
def prior_raw(ctx):
    raw = {}
    for h in HORIZONS:
        raw[f"dir_{h}"] = np.tile(np.log(np.clip(ctx.prior[f"dir_{h}"], 1e-6, 1)), (ctx.n, 1))
        raw[f"vol_{h}"] = np.full(ctx.n, ctx.prior[f"vol_{h}"])
        q = min(max(ctx.prior[f"dd_{h}"], 1e-4), 1 - 1e-4)
        raw[f"dd_{h}"] = np.full(ctx.n, np.log(q / (1 - q)))
    return raw
finalize(PANEL, "prior", prior_raw(PANEL), "baseline", label="Prior (base rates)")

_fi = {c: i for i, c in enumerate(TECH_COLS)}
SIMPLE_FEATS = {"dir": ["ret_20d", "ret_60d", "ret_120d"], "vol": ["lrv_5d", "lrv_20d", "lrv_60d", "lrv_hourly_5d"],
                "dd": ["lrv_20d", "lrv_hourly_5d", "dd_252", "ret_20d"]}
raw_simple = {}
for kd, cols in SIMPLE_FEATS.items():
    Xs = X_TECH[:, [_fi[c] for c in cols]]
    raw_simple.update(fit_heads(PANEL, Xs, TR, IN, "linear", tasks=[f"{kd}_{h}" for h in HORIZONS]))
finalize(PANEL, "baseline_simple", raw_simple, "baseline", label="Momentum / HAR / trailing-vol baselines", features=SIMPLE_FEATS)

st_tech = Stage("tech_gbm", [TECH_COLS, GBM, N, CFG["blocks"]], [_pred_path("panel", "tech_gbm")])
if not (st_tech.done() and have_pred("panel", "tech_gbm")):
    t0 = time.time()
    finalize(PANEL, "tech_gbm", fit_heads(PANEL, X_TECH, TR, IN, "gbm"), "reference", label="Technical-indicator LightGBM",
             train_minutes=round((time.time() - t0) / 60, 2))
    st_tech.finish()
BASELINES = ["prior", "baseline_simple", "tech_gbm"]
LB_BASE = leaderboard(PANEL, BASELINES, PANEL.mask["val"])
display(LB_BASE.pivot_table(index="model", columns="task", values="primary").round(3))
LB_BASE.to_csv(SUB["tables"] / "baselines_val.csv", index=False)
tick("8 baselines")

## 9. Computer-vision models, part 1: architectures and frozen pretrained features

### 9.1 How the models process a chart
* **Convolutional networks** (Custom CNN, ResNet-50, EfficientNet-B0, ConvNeXt-Tiny, YOLOv8 backbone) slide small learned filters over the image; early layers respond to edges (candle bodies, wicks), deeper layers to arrangements of candles (runs, gaps, volatility clusters). Pooling and strides shrink the map so deeper filters "see" more bars. ResNet's skip connections (y = x + F(x)) let gradients pass through 50 layers; EfficientNet uses depthwise-separable convolutions scaled jointly in depth, width and resolution; ConvNeXt uses 7×7 depthwise convolutions, LayerNorm and an inverted bottleneck borrowed from transformers.
* **Vision Transformers** cut the 224 × 224 chart into 196 patches of 16 × 16 px (ViT) — about 4-5 candles each — embed each patch as a token, and let every patch attend to every other: a token for the last week can directly compare itself with one from a year ago. Swin restricts attention to shifted 7 × 7 windows and merges patches stage by stage (CNN-like hierarchy at lower cost).
* **Our multi-view network.** One backbone (shared weights) encodes the weekly, daily and hourly chart separately; the three feature vectors are concatenated and a small multi-task head outputs, for each of the 4 horizons, 3 direction logits, one volatility value and one drawdown logit (12 tasks from one model).

### 9.2 Transfer learning, step 1: frozen features
Each ImageNet-pretrained backbone is used **unchanged** as a feature extractor; per view the features are reduced to 32 principal components (fitted on training rows) and a regularised linear model per task is trained on top (regularisation chosen on the inner block). This tells us how much a photo-trained network "already sees" in a chart. Controls: the same ResNet-50 with **random weights** (if it matches the pretrained one, pre-training is not what helps), and a probe trained on **shuffled labels** (must score at chance; checks the pipeline for leakage).

In [ ]:
def extract_embeddings(name, spec, pretrained=True, seed=0, batch=128):
    set_seed(seed)
    net = ChartNet(spec, pretrained=pretrained).to(DEV).eval()
    path = LOCAL_ROOT / f"emb_{name}{'' if pretrained else '_random'}_{N}.npy"
    if path.exists():
        return path
    mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16, shape=(N, 3, net.feat_dim))
    with torch.no_grad():
        for s in range(0, N, batch):
            x = to_float_images(torch.from_numpy(np.ascontiguousarray(IMG[s: s + batch])).to(DEV), PAL_DARK_T)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                z = net.embed(x)
            mm[s: s + batch] = z.float().cpu().numpy().astype(np.float16)
    mm.flush(); del mm, net; torch.cuda.empty_cache()
    return path

def pca_per_view(E, fit_rows, k):
    rng = np.random.default_rng(0)
    fr = np.sort(rng.choice(fit_rows, min(len(fit_rows), 20000), replace=False))
    out = np.zeros((E.shape[0], E.shape[1] * k), np.float32); pcas = []
    for v in range(E.shape[1]):
        pca = PCA(k, random_state=0).fit(np.asarray(E[fr, v], dtype=np.float32))
        for s in range(0, E.shape[0], 20000):
            out[s: s + 20000, v * k: (v + 1) * k] = pca.transform(np.asarray(E[s: s + 20000, v], dtype=np.float32))
        pcas.append(pca)
    return out, pcas

FROZEN_CV = [n for n, s in CFG["cv_models"].items() if s["kind"] != "jkx"]      # the scratch CNN has no pretrained weights
CV_PCA = {}
k_pca = CFG["probe"]["pca_per_view"]
for name in FROZEN_CV + ["resnet50_random"]:
    base = "resnet50" if name == "resnet50_random" else name
    spec = CFG["cv_models"][base]; pretrained = name != "resnet50_random"
    fpath = SUB["features"] / f"cvpca_{name}_{N}.npy"
    st = Stage(f"cv_frozen_{name}", [spec, pretrained, k_pca, N, CFG["probe"]], [fpath])
    try:
        if not (st.done() and fpath.exists()):
            if not HAS_GPU:
                not_run(f"{name}_frozen", "no GPU"); continue
            t0 = time.time()
            E = np.load(extract_embeddings(base, spec, pretrained), mmap_mode="r")
            Z, _ = pca_per_view(E, np.flatnonzero(TR), k_pca)
            np.save(fpath, Z); st.finish(extract_minutes=round((time.time() - t0) / 60, 2)); del E
        CV_PCA[name] = np.load(fpath)
        mid = f"{name}_frozen" if pretrained else "resnet50_random_frozen"
        if not have_pred("panel", mid) or FORCE_RERUN:
            finalize(PANEL, mid, fit_heads(PANEL, CV_PCA[name], TR, IN, "linear"), "cv_frozen", backbone=base, pretrained=pretrained)
        print(f"{mid}: val primary", {t: round(v, 3) for t, v in REG[mid]["val_primary"].items() if v is not None and t.endswith("_20")})
    except Exception as e:
        failed(f"{name}_frozen", e)
    gc.collect(); torch.cuda.empty_cache() if HAS_GPU else None

### 9.3 Controls and the view ablation (which timeframe carries the signal?)

In [ ]:
if "resnet50" in CV_PCA:
    # label-shuffle control: labels permuted within month on the TRAINING rows only; evaluated against the true labels
    rng = np.random.default_rng(5); Ysh = {k: v.copy() for k, v in Y.items()}
    for g in month_groups(P["month"].values[TR]):
        gi = np.flatnonzero(TR)[g]
        perm = gi[rng.permutation(len(gi))]
        for k in Ysh:
            Ysh[k][gi] = Y[k][perm]
    SH = EvalCtx("panel", Ysh, P["month"].values, P["block"].values, P["sector"].values, P["ticker"].values)
    raw_sh = fit_heads(SH, CV_PCA["resnet50"], TR, IN, "linear")
    finalize(PANEL, "resnet50_frozen_shuffled", raw_sh, "control", note="probe fitted on within-month shuffled training labels")
VIEW_ABL = []
for name in FROZEN_CV:
    if name not in CV_PCA:
        continue
    for j, v in enumerate(VIEWS):
        mid = f"{name}_frozen_{v}"
        if not have_pred("panel", mid) or FORCE_RERUN:
            finalize(PANEL, mid, fit_heads(PANEL, CV_PCA[name][:, j * k_pca: (j + 1) * k_pca], TR, IN, "linear"), "cv_view_ablation",
                     backbone=name, view=v)
        for t in TASKS:
            VIEW_ABL.append({"backbone": name, "view": v, "task": t, "val_primary": REG.get(mid, {}).get("val_primary", {}).get(t)})
    for t in TASKS:
        VIEW_ABL.append({"backbone": name, "view": "W+D+H", "task": t, "val_primary": REG.get(f"{name}_frozen", {}).get("val_primary", {}).get(t)})
VIEW_ABL_DF = pd.DataFrame(VIEW_ABL)
if len(VIEW_ABL_DF):
    VIEW_ABL_DF.to_csv(SUB["tables"] / "cv_view_ablation_val.csv", index=False)
    piv = VIEW_ABL_DF.assign(target=VIEW_ABL_DF.task.str.split("_").str[0]).groupby(["view", "target"])["val_primary"].mean().unstack()
    display(piv.round(4))
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
    for a_, kd in zip(ax, ["dir", "vol", "dd"]):
        sub = VIEW_ABL_DF[VIEW_ABL_DF.task.str.startswith(kd)]
        sub.groupby(["view", "backbone"])["val_primary"].mean().unstack().plot.bar(ax=a_, legend=(kd == "dd"))
        a_.set_title(f"{TASK_LABEL[kd]}: frozen probe, mean over horizons"); a_.set_ylabel(PRIMARY[kd])
    if ax[2].get_legend():
        ax[2].legend(fontsize=6)
    plt.tight_layout(); savefig(fig, "cv_view_ablation.png"); plt.show()
tick("9 cv frozen")

## 10. Computer-vision models, part 2: fine-tuning (transfer learning, step 2)

### 10.1 Training strategy (identical for all seven models)
* **Input:** the three charts of a row (W, D, H), palette → RGB, normalised with each backbone's own pretraining mean/std.
* **Loss:** multi-task sum over the 4 horizons: cross-entropy (direction) + 0.5 × mean squared error on standardised log-volatility + 0.5 × binary cross-entropy (drawdown). No class weights (they would distort calibrated probabilities; macro-F1 is handled by calibration and evaluation).
* **Optimiser:** AdamW, weight decay 0.05 (not on norms and biases), backbone learning rate from the probe below, head learning rate 1e-3, 5 % linear warm-up then cosine decay, gradient clipping at 1.0, bfloat16 mixed precision, batch 64 rows (192 images).
* **Epochs and early stopping:** up to 6 epochs; after each epoch the loss on the **inner block (2021)** is computed and the best epoch's weights are kept (patience 2). Validation (2022-23) is never used for stopping, so it stays a fair selection set.
* **Augmentation (training only), chosen to respect what a chart means:** small vertical rescale/shift (±5 % / ±4 %: price scale is arbitrary), dropping up to 8 % of the oldest bars on the left (the newest bars, which carry the decision, are never cut), brightness ±10 %, light pixel noise. **Forbidden:** horizontal flips (they reverse time), vertical flips (they turn rallies into crashes), random crops of the right edge, rotations.
* **Hyperparameter tuning (10.2):** for each model, two backbone learning rates (from the usual range for that architecture) are each trained for one epoch on a random 25 % of the training rows; the one with the lower inner-block loss is used for the full run. Probe results are saved.
* **Ablations:** ResNet-50 without augmentation; ResNet-50 from random initialisation (pretrained vs scratch).

In [ ]:
FT = CFG["cv_ft"]
MICRO = FT["batch"] if GPU_GB >= 30 else FT["batch"] // 2
VOL_MU = np.array([np.nanmean(Y[f"vol_{h}"][TR]) for h in HORIZONS]); VOL_SD = np.array([np.nanstd(Y[f"vol_{h}"][TR]) for h in HORIZONS])
Y_DIR = np.stack([Y[f"dir_{h}"] for h in HORIZONS], 1).astype(np.float32)
Y_VOLZ = ((np.stack([Y[f"vol_{h}"] for h in HORIZONS], 1) - VOL_MU) / VOL_SD).astype(np.float32)
Y_DD = np.stack([Y[f"dd_{h}"] for h in HORIZONS], 1).astype(np.float32)

def mt_loss(out, yd, yv, yq, w=FT["loss_w"]):
    md, mv, mq = torch.isfinite(yd), torch.isfinite(yv), torch.isfinite(yq)
    l = out["dir"].new_zeros(())
    if md.any():
        l = l + w["dir"] * F.cross_entropy(out["dir"][md].float(), yd[md].long())
    if mv.any():
        l = l + w["vol"] * F.mse_loss(out["vol"][mv].float(), yv[mv])
    if mq.any():
        l = l + w["dd"] * F.binary_cross_entropy_with_logits(out["dd"][mq].float(), yq[mq])
    return l

def label_tensors(rb):
    return tuple(torch.from_numpy(a[rb]).to(DEV) for a in (Y_DIR, Y_VOLZ, Y_DD))

def augment(x):
    """x: (B, V, 3, H, W) in [0, 1]. Vertical rescale/shift, left-side (oldest-bar) crop, brightness and noise."""
    b, v = x.shape[:2]; z = x.flatten(0, 1); n = z.shape[0]; d = z.device
    sy = 1 + (torch.rand(n, device=d) * 0.10 - 0.05); ty = torch.rand(n, device=d) * 0.08 - 0.04
    crop = torch.rand(n, device=d) * 0.08
    theta = torch.zeros(n, 2, 3, device=d)
    theta[:, 0, 0] = 1 - crop; theta[:, 0, 2] = crop; theta[:, 1, 1] = sy; theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(z.shape), align_corners=False)
    z = F.grid_sample(z, grid, mode="bilinear", padding_mode="border", align_corners=False)
    z = z * (0.9 + 0.2 * torch.rand(n, 1, 1, 1, device=d)) + 0.01 * torch.randn_like(z)
    return z.clamp(0, 1).view(b, v, *z.shape[1:])

def batch_images(rb, palette=None):
    return to_float_images(torch.from_numpy(np.ascontiguousarray(IMG[rb])).to(DEV, non_blocking=True), PAL_DARK_T if palette is None else palette)

@torch.no_grad()
def eval_loss(net, rows, batch=256):
    net.eval(); tot, n = 0.0, 0
    for s in range(0, len(rows), batch):
        rb = rows[s: s + batch]
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = net(batch_images(rb))
        tot += float(mt_loss(out, *label_tensors(rb))) * len(rb); n += len(rb)
    return tot / max(1, n)

def param_groups(net, lr_backbone, lr_head, wd):
    g = {"bd": [], "bn": [], "hd": [], "hn": []}
    for n_, p in net.named_parameters():
        if not p.requires_grad:
            continue
        key = ("h" if n_.startswith("head.") else "b") + ("n" if p.ndim <= 1 else "d")
        g[key].append(p)
    return [{"params": g["bd"], "lr": lr_backbone, "weight_decay": wd}, {"params": g["bn"], "lr": lr_backbone, "weight_decay": 0.0},
            {"params": g["hd"], "lr": lr_head, "weight_decay": wd}, {"params": g["hn"], "lr": lr_head, "weight_decay": 0.0}]

def train_chartnet(spec, lr, rows_tr, rows_es, epochs, pretrained=True, aug=True, seed=0, tag=""):
    set_seed(seed); rng = np.random.default_rng(seed)
    net = ChartNet(spec, pretrained=pretrained).to(DEV)
    opt = torch.optim.AdamW(param_groups(net, lr, FT["lr_head"], FT["wd"]))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)     # needed only on fp16 GPUs (T4)
    B = FT["batch"]; spe = math.ceil(len(rows_tr) / B); total = max(1, epochs * spe); warm = max(1, int(FT["warmup_frac"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    hist, best, best_state, bad, t0 = [], np.inf, None, 0, time.time()
    for ep in range(epochs):
        net.train(); perm = rng.permutation(rows_tr); run, k = 0.0, 0
        for s in range(0, len(perm), B):
            rb_all = np.sort(perm[s: s + B]); opt.zero_grad(set_to_none=True); tot = 0.0
            for c0 in range(0, len(rb_all), MICRO):            # micro-batches: same effective batch on smaller GPUs
                rb = rb_all[c0: c0 + MICRO]
                x = batch_images(rb)
                if aug:
                    x = augment(x)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    loss = mt_loss(net(x), *label_tensors(rb)) * (len(rb) / len(rb_all))
                scaler.scale(loss).backward(); tot += float(loss)
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            run += tot; k += 1
        es = eval_loss(net, rows_es)
        hist.append({"epoch": ep + 1, "train_loss": run / max(1, k), "inner_loss": es, "minutes": round((time.time() - t0) / 60, 2)})
        print(f"  {tag} epoch {ep + 1}/{epochs}: train {run / max(1, k):.4f} | inner {es:.4f} | {hist[-1]['minutes']} min")
        if es < best - 1e-5:
            best, bad = es, 0
            best_state = {k_: v.detach().to("cpu", copy=True) for k_, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= FT["patience"]:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    return net, hist, best, round((time.time() - t0) / 60, 2)

ROWS_TR = np.flatnonzero(TR & np.isfinite(Y_DIR).all(1))
ROWS_IN = np.flatnonzero(IN)
ROWS_PRED = np.flatnonzero(IN | VA | TE)          # rows that receive predictions (test outputs stay unscored until section 23)
CV_FT_IDS = {name: f"{name}_ft" for name in CFG["cv_models"]}
LR_PROBE, CURVES = [], {}

def run_cv_finetune(name, spec, mid, pretrained=True, aug=True, lr_fixed=None, seed=0):
    mpath = SUB["models"] / f"cv_{mid}.pt"
    st = Stage(f"cv_{mid}", [spec, FT, pretrained, aug, lr_fixed, N, CFG["blocks"], int(TR.sum())], [mpath, _pred_path("panel", mid)])
    if st.done() and have_pred("panel", mid):
        CURVES[mid] = REG[mid].get("history", []); return
    if not HAS_GPU:
        not_run(mid, "no GPU"); return
    try:
        if lr_fixed is not None:
            lr = lr_fixed
        elif QUICK or len(spec["lr_grid"]) == 1:
            lr = spec["lr_grid"][0]
        else:
            rng = np.random.default_rng(1)
            sub_rows = np.sort(rng.choice(ROWS_TR, int(FT["probe_frac"] * len(ROWS_TR)), replace=False))
            losses = {}
            for lr_ in spec["lr_grid"]:
                net, _, l_, mins = train_chartnet(spec, lr_, sub_rows, ROWS_IN, 1, pretrained, aug, seed, tag=f"{mid} probe lr={lr_:g}")
                losses[lr_] = l_; LR_PROBE.append({"model": mid, "lr": lr_, "inner_loss": l_, "minutes": mins})
                del net; torch.cuda.empty_cache()
            lr = min(losses, key=lambda z: losses[z])
        torch.cuda.reset_peak_memory_stats()
        net, hist, best, mins = train_chartnet(spec, lr, ROWS_TR, ROWS_IN, FT["epochs"], pretrained, aug, seed, tag=mid)
        peak = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        torch.save({"state_dict": net.state_dict(), "spec": spec, "vol_mu": VOL_MU, "vol_sd": VOL_SD, "lr": lr, "pretrained": pretrained,
                    "aug": aug, "history": hist}, mpath)
        out = chart_raw_outputs(net, IMG[ROWS_PRED], PAL_DARK_T, VOL_MU, VOL_SD, DEV, amp_dtype=AMP_DTYPE)
        raw = {}
        for t in TASKS:
            a = np.full((N, 3) if t.startswith("dir") else N, np.nan); a[ROWS_PRED] = out[t]; raw[t] = a
        n_par = sum(p.numel() for p in net.parameters())
        finalize(PANEL, mid, raw, "cv_ft", backbone=name, lr=lr, epochs_run=len(hist), best_inner_loss=best, history=hist,
                 train_minutes=mins, peak_gpu_gb=peak, params_m=round(n_par / 1e6, 2), pretrained=pretrained, aug=aug,
                 label=spec["label"] + ("" if pretrained else " (random init)") + ("" if aug else " (no augmentation)"))
        CURVES[mid] = hist; st.finish()
        print(f"{mid}: val primary (20d)", {t: round(v, 3) for t, v in REG[mid]["val_primary"].items() if v is not None and t.endswith("_20")})
        del net
    except Exception as e:
        failed(mid, e)
    gc.collect(); torch.cuda.empty_cache()

for name, spec in CFG["cv_models"].items():
    run_cv_finetune(name, spec, CV_FT_IDS[name])
if not QUICK:
    run_cv_finetune("resnet50", CFG["cv_models"]["resnet50"], "resnet50_ft_noaug", aug=False, lr_fixed=REG.get("resnet50_ft", {}).get("lr"))
    run_cv_finetune("resnet50", CFG["cv_models"]["resnet50"], "resnet50_scratch", pretrained=False, lr_fixed=1e-3)
if LR_PROBE:
    pd.DataFrame(LR_PROBE).to_csv(SUB["tables"] / "cv_lr_probe.csv", index=False); display(pd.DataFrame(LR_PROBE))

### 10.4 Training curves (over-/under-fitting check)
A falling training loss with a rising inner loss means over-fitting; early stopping keeps the epoch with the lowest inner loss. Flat curves at the prior's loss mean the network found nothing to learn.

In [ ]:
_ft = [m for m in list(CV_FT_IDS.values()) + ["resnet50_ft_noaug", "resnet50_scratch"] if REG.get(m, {}).get("history")]
if _ft:
    fig, axes = plt.subplots(1, len(_ft), figsize=(3.0 * len(_ft), 2.8), squeeze=False)
    for a_, m in zip(axes[0], _ft):
        h_ = pd.DataFrame(REG[m]["history"])
        a_.plot(h_.epoch, h_.train_loss, "o-", label="train"); a_.plot(h_.epoch, h_.inner_loss, "s-", label="inner (early stop)")
        a_.set_title(m, fontsize=8); a_.set_xlabel("epoch"); a_.grid(alpha=0.3)
    axes[0][0].legend(fontsize=7); plt.tight_layout(); savefig(fig, "cv_training_curves.png"); plt.show()
    pd.concat([pd.DataFrame(REG[m]["history"]).assign(model=m) for m in _ft]).to_csv(SUB["tables"] / "cv_training_curves.csv", index=False)
tick("10 cv finetune")

## 11. The ready-made YOLOv8 chart-pattern detector (zero-shot)

`foduucom/stockmarket-pattern-detection-yolov8` is a YOLOv8s **object detector** trained by its authors on annotated chart screenshots (6 classes such as head-and-shoulders top/bottom, M-head, W-bottom, triangle; published mAP@0.5 ≈ 0.61; licence AGPL-3.0 through Ultralytics). It outputs boxes, not forecasts. We run it **unchanged** on our charts (light theme, closest to its training screenshots), turn its detections into features (for each pattern and view: highest confidence and number of boxes) and fit LightGBM heads on them. This answers "do ready-made chart-pattern weights carry predictive information?" without any fine-tuning on our labels. Its features are computed on non-overlapping rows (every 5th session) to save time.

In [ ]:
def run_yolo_detector():
    mid = "yolov8_det_patterns"
    fpath = SUB["features"] / f"yolo_det_{N}.npy"
    st = Stage("yolo_det", [CFG["yolo_det_repo"], N, CFG["eval_stride"]], [fpath])
    if not HAS_GPU:
        not_run(mid, "no GPU"); return None
    from huggingface_hub import list_repo_files, hf_hub_download
    from ultralytics import YOLO
    files = list_repo_files(CFG["yolo_det_repo"])
    pt = sorted(f for f in files if f.endswith(".pt"))
    if not pt:
        not_run(mid, f"no .pt weights in {CFG['yolo_det_repo']}"); return None
    det = YOLO(hf_hub_download(CFG["yolo_det_repo"], pt[0]))
    names = det.names if isinstance(det.names, dict) else dict(enumerate(det.names))
    rows = np.flatnonzero(P["eval_stride_row"].values)
    if st.done() and fpath.exists():
        Xd = np.load(fpath)
    else:
        Xd = np.full((N, 3 * len(names) * 2), np.nan, np.float32)
        for s in range(0, len(rows), 64):
            rb = rows[s: s + 64]
            imgs = [PALETTE_LIGHT[np.asarray(IMG[r, j])][:, :, ::-1].copy() for r in rb for j in range(3)]   # BGR for Ultralytics
            res = det.predict(imgs, imgsz=640, conf=0.15, verbose=False, half=True)
            for q, r in enumerate(rb):
                feats = []
                for j in range(3):
                    bx = res[q * 3 + j].boxes
                    cls = bx.cls.cpu().numpy().astype(int) if bx is not None and len(bx) else np.array([], int)
                    cf = bx.conf.cpu().numpy() if bx is not None and len(bx) else np.array([])
                    for c in sorted(names):
                        sel = cls == c
                        feats += [float(cf[sel].max()) if sel.any() else 0.0, float(sel.sum())]
                Xd[r] = feats
        np.save(fpath, Xd); st.finish(n_rows=len(rows))
    cols = [f"{v}_{names[c]}_{s_}" for v in VIEWS for c in sorted(names) for s_ in ("maxconf", "count")]
    fit_rows = TR & P["eval_stride_row"].values
    finalize(PANEL, mid, fit_heads(PANEL, Xd, fit_rows, IN, "gbm"), "cv_zero_shot", label="YOLOv8 pattern detector (zero-shot features)",
             patterns=list(names.values()))
    # descriptive: how often each pattern fires (daily view) and the mean 20-day log return after it (training rows)
    dcol = [i for i, c in enumerate(cols) if c.startswith("D_") and c.endswith("maxconf")]
    tab = []
    for i in dcol:
        hit = fit_rows & (Xd[:, i] > 0)
        tab.append({"pattern": cols[i][2:-8], "share_of_rows": float(hit.sum() / max(1, fit_rows.sum())),
                    "mean_ret20_when_detected": float(np.nanmean(Y["ret_20"][hit])) if hit.any() else np.nan,
                    "mean_ret20_otherwise": float(np.nanmean(Y["ret_20"][fit_rows & ~(Xd[:, i] > 0)]))})
    YOLO_PAT = pd.DataFrame(tab); YOLO_PAT.to_csv(SUB["tables"] / "yolo_patterns_train.csv", index=False); display(YOLO_PAT.round(4))
    ex = [int(r) for r in rows[np.isin(rows, np.flatnonzero(VA))][:200] if np.nansum(Xd[r, : len(names) * 2: 2]) > 0][:4]
    if ex:
        fig, axes = plt.subplots(1, len(ex), figsize=(4 * len(ex), 4), squeeze=False)
        for a_, r in zip(axes[0], ex):
            im = PALETTE_LIGHT[np.asarray(IMG[r, 0])][:, :, ::-1].copy()
            a_.imshow(det.predict([im], imgsz=640, conf=0.15, verbose=False)[0].plot()[:, :, ::-1]); a_.axis("off")
            a_.set_title(f"{P.at[r, 'ticker']} {P.at[r, 'date'].date()} (weekly)", fontsize=8)
        plt.suptitle("YOLOv8 pattern detector on our weekly charts (validation examples)"); plt.tight_layout(); savefig(fig, "yolo_detections.png"); plt.show()
    return Xd

try:
    X_YOLO_DET = run_yolo_detector()
except Exception as e:
    X_YOLO_DET = None; failed("yolov8_det_patterns", e)
gc.collect(); torch.cuda.empty_cache() if HAS_GPU else None
tick("11 yolo det")

## 12. Computer-vision analysis on the validation block (2022-2023)

Everything in this section uses **validation** rows only; it is where models are compared and chosen. The same tables are recomputed once on the locked test block in §23.

### 12.1 Leaderboard and the best CV model (Q2, Q4)

In [ ]:
CV_MAIN = [m for m in CV_FT_IDS.values() if REG.get(m, {}).get("status") == "RUN"]
CV_OTHER = [m for m in [f"{n}_frozen" for n in FROZEN_CV] + ["resnet50_random_frozen", "resnet50_frozen_shuffled", "yolov8_det_patterns",
                         "resnet50_ft_noaug", "resnet50_scratch"] if REG.get(m, {}).get("status") == "RUN"]
LABEL_OF = {m: REG.get(m, {}).get("label", m) for m in REG}
LB_CV_VAL = leaderboard(PANEL, CV_MAIN + CV_OTHER + BASELINES, PANEL.mask["val"])
LB_CV_VAL.to_csv(SUB["tables"] / "cv_leaderboard_val.csv", index=False)
PIV_CV = LB_CV_VAL[LB_CV_VAL.status == "RUN"].pivot_table(index="model", columns="task", values="primary").reindex(columns=TASKS)
CV_MAIN = [m for m in CV_MAIN if m in PIV_CV.index]
display(PIV_CV.round(3))
CV_RANK = mean_rank(LB_CV_VAL[LB_CV_VAL.model.isin(CV_MAIN)])
CV_BEST = CV_RANK.index[0] if len(CV_RANK) else None
BEST_CV_TASK = {t: PIV_CV.loc[CV_MAIN, t].idxmax() for t in TASKS if PIV_CV.loc[CV_MAIN, t].notna().any()} if CV_MAIN else {}
print("Mean rank over the 12 tasks (validation, 1 = best):"); display(CV_RANK.round(2).to_frame("mean_rank"))
CV_RANK.to_frame("mean_rank").to_csv(SUB["tables"] / "cv_rank_val.csv")
if CV_MAIN:
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(PIV_CV) + 1.5))
    Z = PIV_CV.copy()
    for kd in ("dir", "vol", "dd"):        # colour by rank within each task column, annotate raw values
        cols = [t for t in TASKS if t.startswith(kd)]
        Z[cols] = PIV_CV[cols].rank(pct=True)
    ax.imshow(Z.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    for i in range(PIV_CV.shape[0]):
        for j in range(PIV_CV.shape[1]):
            v = PIV_CV.values[i, j]
            ax.text(j, i, "" if not np.isfinite(v) else f"{v:.3f}", ha="center", va="center", fontsize=6.5)
    ax.set_xticks(range(len(TASKS))); ax.set_xticklabels(TASKS, rotation=45, fontsize=7)
    ax.set_yticks(range(len(PIV_CV))); ax.set_yticklabels(PIV_CV.index, fontsize=7)
    ax.set_title("Validation primary metric (dir: macro-F1, vol: R², dd: ROC-AUC); colour = rank within column")
    plt.tight_layout(); savefig(fig, "cv_leaderboard_val.png"); plt.show()
SKILL_CV_VAL = pd.DataFrame([{"task": t, "model": m, **skill_test(PANEL, t, load_pred("panel", m)[t], PANEL.mask["val"])}
                             for t, m in BEST_CV_TASK.items()])
if len(SKILL_CV_VAL):
    SKILL_CV_VAL.to_csv(SUB["tables"] / "cv_skill_val.csv", index=False); display(SKILL_CV_VAL.round(4))

def why_best(best, rank, piv, main):
    """Evidence-based explanation of why the best model ranks first (generated from the tables, not typed in)."""
    if best is None:
        return "No CV model ran."
    lines = [f"**{LABEL_OF.get(best, best)}** has the best mean rank ({rank[best]:.2f}) over the 12 validation tasks."]
    for kd in ("dir", "vol", "dd"):
        cols = [t for t in TASKS if t.startswith(kd) and t in piv.columns]
        if not cols:
            continue
        gap = (piv.loc[best, cols] - piv.loc[[m for m in main if m != best], cols].max()).mean()
        lines.append(f"{TASK_LABEL[kd]}: mean {PRIMARY[kd]} {piv.loc[best, cols].mean():.3f} "
                     f"({'ahead of' if gap > 0 else 'behind'} the next-best model by {abs(gap):.3f} on average).")
    fz = best.replace("_ft", "_frozen")
    if fz != best and fz in piv.index:
        lines.append(f"Fine-tuning changed its mean primary metric by {float((piv.loc[best] - piv.loc[fz]).mean()):+.3f} versus its frozen features, "
                     "so most of its edge " + ("comes from adapting to charts." if float((piv.loc[best] - piv.loc[fz]).mean()) > 0.005 else "was already in the pre-trained features or is small."))
    tm = REG.get(best, {}).get("train_minutes")
    lines.append(f"Size: {REG.get(best, {}).get('params_m', '?')} M parameters" + (f", {tm} training minutes." if tm is not None else "."))
    lines.append("Caution: validation was also used to choose it, so its validation lead is optimistic; §23 gives the unbiased test result.")
    return " ".join(lines)
WHY_CV = why_best(CV_BEST, CV_RANK, PIV_CV, CV_MAIN)
display(Markdown(WHY_CV)); (SUB["results"] / "why_cv_best_val.md").write_text(WHY_CV)

### 12.2 Model combinations (Q3) and the best model or ensemble per target (Q4)
Every pair (21) and triple (35) of the seven fine-tuned CV models is averaged with **equal weights** (calibrated probabilities; log-volatility); nothing is fitted, so a combination can only win by genuinely complementary errors. The top-3 ensemble per task uses the three best single models on validation.

In [ ]:
COMBO_ROWS = []
if len(CV_MAIN) >= 2:
    for k in (2, 3):
        for combo in itertools.combinations(CV_MAIN, k):
            avg = average_predictions([load_pred("panel", m) for m in combo])
            for t in TASKS:
                r = PANEL.rows(t, PANEL.mask["val"], avg[t])
                COMBO_ROWS.append({"combo": " + ".join(c.replace("_ft", "") for c in combo), "size": k, "task": t,
                                   "primary": primary_value(t, PANEL.Y[t][r], avg[t][r])})
    for m in CV_MAIN:
        for t in TASKS:
            COMBO_ROWS.append({"combo": m.replace("_ft", ""), "size": 1, "task": t, "primary": PIV_CV.loc[m, t]})
COMBO_VAL = pd.DataFrame(COMBO_ROWS)
if len(COMBO_VAL):
    COMBO_VAL.to_csv(SUB["tables"] / "cv_combos_val.csv", index=False)
    cpiv = COMBO_VAL.pivot_table(index="combo", columns="task", values="primary")
    combo_rank = cpiv.rank(ascending=False).mean(1).sort_values()
    BEST_COMBO = combo_rank.index[0]
    print("Best 10 single models and combinations by mean rank (validation):"); display(combo_rank.head(10).round(2).to_frame("mean_rank"))
    for named in ["jkx_cnn + resnet50", "effnet_b0 + resnet50", "resnet50 + effnet_b0"]:
        if named in cpiv.index:
            print(f"  {named}: rank {int((combo_rank.index == named).argmax()) + 1} of {len(combo_rank)}")
    best_per_task = best_rows(COMBO_VAL, ["task"]).set_index("task")[["combo", "size", "primary"]]
    display(best_per_task)
    best_per_task.to_csv(SUB["tables"] / "cv_best_per_task_val.csv")
    combo_rank.to_frame("mean_rank").to_csv(SUB["tables"] / "cv_combo_rank_val.csv")
TOP3_CV = {t: list(PIV_CV.loc[CV_MAIN, t].dropna().sort_values(ascending=False).index[: CFG["top_k"]]) for t in TASKS} if CV_MAIN else {}
BEST_COMBO = combo_rank.index[0] if len(COMBO_VAL) else None

In [ ]:
def ensemble_model(ctx, mid, members_by_task, family, **meta):
    """Equal-weight average of the members' calibrated predictions per task, then re-calibrated on the inner block."""
    avg = {}
    for t in TASKS:
        mem = members_by_task.get(t) or []
        avg[t] = (np.mean([np.asarray(load_pred(ctx.name, m)[t], dtype=float) for m in mem], axis=0) if mem
                  else np.full((ctx.n, 3) if t.startswith("dir") else ctx.n, np.nan))
    return finalize(ctx, mid, raw_from_probs(avg), family, members=members_by_task, **meta)
if len(CV_MAIN) >= 3:
    ensemble_model(PANEL, "cv_top3", TOP3_CV, "cv_ensemble", label="Top-3 CV ensemble (per task, chosen on validation)")
    ensemble_model(PANEL, "cv_all7", {t: CV_MAIN for t in TASKS}, "cv_ensemble", label="All fine-tuned CV models (equal weights)")
    display(leaderboard(PANEL, ["cv_top3", "cv_all7", CV_BEST], PANEL.mask["val"]).pivot_table(index="model", columns="task", values="primary").round(3))

### 12.3 Sectors (Q5): which model works best where, and is any network sector-specific?
**Interaction test (H6).** Statistic: for each sector, subtract the sector's mean metric over models (removes "some sectors are easier"); then sum over models the variance of these centred values across sectors. A large value means rankings change by sector. Null: the 50 stocks are reassigned to sectors at random (10 per sector), keeping each stock's rows together; 200 permutations.

In [ ]:
def sector_table(ctx, models, mask):
    rows = []
    for s in SECTORS:
        ms = mask & (ctx.sectors == s)
        for m in models:
            pr = load_pred(ctx.name, m)
            for t in TASKS:
                r = ctx.rows(t, ms, pr[t])
                rows.append({"sector": s, "model": m, "task": t, "target": t.split("_")[0], "n": len(r),
                             "primary": primary_value(t, ctx.Y[t][r], pr[t][r]) if len(r) >= 20 else np.nan})
    return pd.DataFrame(rows)

def sector_interaction_test(ctx, models, mask, kind, n_perm=None, seed=0):
    n_perm = n_perm or (50 if QUICK else 200)
    tasks = [t for t in TASKS if t.startswith(kind)]
    preds = {m: load_pred(ctx.name, m) for m in models}
    stocks = np.array(sorted(set(ctx.tickers[mask])))
    true_sec = {t: SECTOR_OF[t] for t in stocks}
    def stat(sec_map):
        sec_arr = np.array([sec_map.get(t, "") for t in ctx.tickers])
        M = np.full((len(SECTORS), len(models)), np.nan)
        for i, s in enumerate(SECTORS):
            ms = mask & (sec_arr == s)
            for j, m in enumerate(models):
                vals = []
                for t in tasks:
                    r = ctx.rows(t, ms, preds[m][t])
                    if len(r) >= 20:
                        vals.append(primary_value(t, ctx.Y[t][r], preds[m][t][r]))
                M[i, j] = np.mean(vals) if vals else np.nan
        C = M - np.nanmean(M, axis=1, keepdims=True)
        return float(np.nansum(np.nanvar(C, axis=0))), M
    obs, M = stat(true_sec)
    rng = np.random.default_rng(seed); null = []
    labels = np.array([true_sec[t] for t in stocks])
    for _ in range(n_perm):
        null.append(stat(dict(zip(stocks, rng.permutation(labels))))[0])
    return {"target": kind, "statistic": obs, "p": float((1 + np.sum(np.array(null) >= obs)) / (1 + n_perm)), "n_perm": n_perm}, M

if CV_MAIN:
    SEC_CV_VAL = sector_table(PANEL, CV_MAIN, PANEL.mask["val"]); SEC_CV_VAL.to_csv(SUB["tables"] / "cv_sector_val.csv", index=False)
    sec_mean = SEC_CV_VAL.groupby(["sector", "target", "model"])["primary"].mean().reset_index()
    BEST_BY_SECTOR_VAL = best_rows(sec_mean, ["sector", "target"]).pivot(index="sector", columns="target", values="model")
    print("Best fine-tuned CV model per sector and target (validation; mean primary over horizons):"); display(BEST_BY_SECTOR_VAL)
    BEST_BY_SECTOR_VAL.to_csv(SUB["tables"] / "cv_best_by_sector_val.csv")
    H6_VAL = []
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
    for a_, kd in zip(axes, ["dir", "vol", "dd"]):
        res, M = sector_interaction_test(PANEL, CV_MAIN, PANEL.mask["val"], kd)
        H6_VAL.append(res)
        a_.imshow(M - np.nanmean(M, 1, keepdims=True), cmap="coolwarm", aspect="auto")
        a_.set_xticks(range(len(CV_MAIN))); a_.set_xticklabels([m.replace("_ft", "") for m in CV_MAIN], rotation=60, fontsize=7)
        a_.set_yticks(range(len(SECTORS))); a_.set_yticklabels(SECTORS, fontsize=7)
        a_.set_title(f"{TASK_LABEL[kd]}: centred {PRIMARY[kd]} (perm. p = {res['p']:.3f})", fontsize=8)
    plt.tight_layout(); savefig(fig, "cv_sector_interaction_val.png"); plt.show()
    pd.DataFrame(H6_VAL).to_csv(SUB["tables"] / "cv_sector_interaction_val.csv", index=False); display(pd.DataFrame(H6_VAL))

### 12.4 Ablations: fine-tuned vs frozen, pretrained vs scratch vs random, augmentation, label shuffle
Paired month-block bootstrap on validation; Δ = first minus second, on each target's primary metric averaged over the four horizons.

In [ ]:
def paired_delta_family(ctx, kind, pa, pb, mask, seed=0):
    tasks = [t for t in TASKS if t.startswith(kind)]
    r = ctx.rows(tasks[0], mask, pa[tasks[0]])
    for t in tasks:
        r = np.intersect1d(r, np.intersect1d(ctx.rows(t, mask, pa[t]), ctx.rows(t, mask, pb[t])))
    if len(r) < 30:
        return {"n": len(r), "delta": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan}
    def fn(ix):
        rr = r[ix]
        return float(np.mean([primary_value(t, ctx.Y[t][rr], pa[t][rr]) - primary_value(t, ctx.Y[t][rr], pb[t][rr]) for t in tasks]))
    bs = bootstrap(fn, month_groups(ctx.months[r]), seed=seed); est = fn(np.arange(len(r)))
    p2 = 2 * min((1 + np.sum(bs <= 0)) / (1 + len(bs)), (1 + np.sum(bs >= 0)) / (1 + len(bs)))
    return {"n": len(r), "delta": est, "lo": float(np.nanpercentile(bs, 2.5)), "hi": float(np.nanpercentile(bs, 97.5)), "p": float(min(1, p2))}

ABL_PAIRS = [(f"{n}_ft", f"{n}_frozen", "fine-tuned vs frozen") for n in FROZEN_CV] + [
    ("resnet50_ft", "resnet50_scratch", "ImageNet init vs random init (both fine-tuned)"),
    ("resnet50_ft", "resnet50_ft_noaug", "with vs without augmentation"),
    ("resnet50_frozen", "resnet50_random_frozen", "pretrained vs random frozen features"),
    ("resnet50_frozen_shuffled", "prior", "label-shuffle control vs prior (should be ~0 or negative)"),
    ("tech_gbm", CV_BEST or "prior", "technical LightGBM vs best CV model (reference)")]
ABL = []
for a, b, what in ABL_PAIRS:
    if REG.get(a, {}).get("status") != "RUN" or REG.get(b, {}).get("status") != "RUN":
        continue
    pa, pb = load_pred("panel", a), load_pred("panel", b)
    for kd in ("dir", "vol", "dd"):
        ABL.append({"comparison": what, "A": a, "B": b, "target": kd, **paired_delta_family(PANEL, kd, pa, pb, PANEL.mask["val"])})
ABL_VAL = pd.DataFrame(ABL)
if len(ABL_VAL):
    ABL_VAL["p_holm"] = holm(ABL_VAL["p"].values); ABL_VAL.to_csv(SUB["tables"] / "cv_ablations_val.csv", index=False)
    display(ABL_VAL.round(4))

### 12.5 Error analysis (best CV model, validation)

In [ ]:
def load_chartnet(mid):
    ck = torch.load(SUB["models"] / f"cv_{mid}.pt", map_location="cpu", weights_only=False)
    net = ChartNet(ck["spec"], pretrained=False); net.load_state_dict(ck["state_dict"])
    return net.to(DEV).eval(), ck

if CV_BEST:
    pb_ = load_pred("panel", CV_BEST); vm = PANEL.mask["val"]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    for a_, h in zip(axes, HORIZONS):
        r = PANEL.rows(f"dir_{h}", vm, pb_[f"dir_{h}"])
        cm = confusion_matrix(PANEL.Y[f"dir_{h}"][r].astype(int), pb_[f"dir_{h}"][r].argmax(1), labels=[0, 1, 2], normalize="true")
        a_.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                a_.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8)
        a_.set_xticks(range(3)); a_.set_xticklabels(CLASSES, fontsize=7); a_.set_yticks(range(3)); a_.set_yticklabels(CLASSES, fontsize=7)
        a_.set_title(f"{h}-day direction (rows = truth)", fontsize=8)
    plt.suptitle(f"Confusion matrices, {CV_BEST}, validation"); plt.tight_layout(); savefig(fig, "cv_confusion_val.png"); plt.show()
    r = PANEL.rows("dir_20", vm, pb_["dir_20"])
    E = pd.DataFrame({"correct": pb_["dir_20"][r].argmax(1) == PANEL.Y["dir_20"][r], "sector": P.sector.values[r],
                      "sigma_decile": pd.qcut(P.sigma.values[r], 10, labels=False, duplicates="drop"),
                      "market_20d": np.where(X_TECH[r, _fi["spy_ret_20d"]] > 0, "SPY up", "SPY down"),
                      "quarter": P.date.dt.to_period("Q").astype(str).values[r],
                      "true": [CLASSES[int(c)] for c in PANEL.Y["dir_20"][r]]})
    ERR_TABLES = {k: E.groupby(k)["correct"].agg(["mean", "size"]).rename(columns={"mean": "accuracy"}) for k in ["sector", "sigma_decile", "market_20d", "quarter", "true"]}
    for k, v in ERR_TABLES.items():
        v.to_csv(SUB["tables"] / f"cv_errors_by_{k}_val.csv")
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for a_, k in zip(axes, ["sigma_decile", "sector", "quarter", "true"]):
        ERR_TABLES[k]["accuracy"].plot.bar(ax=a_, color="#2563eb"); a_.axhline(E["correct"].mean(), color="k", ls="--", lw=0.8)
        a_.set_title(f"20-day direction accuracy by {k}", fontsize=8); a_.tick_params(labelsize=6)
    plt.tight_layout(); savefig(fig, "cv_errors_val.png"); plt.show()
    rd = PANEL.rows("dd_20", vm, pb_["dd_20"]); pos = rd[PANEL.Y["dd_20"][rd] == 1]
    thr_ = np.quantile(pb_["dd_20"][rd], 0.8) if len(rd) else 1.0
    DD_FN = pd.DataFrame({"sector": P.sector.values[pos], "flagged_top20pct": pb_["dd_20"][pos] >= thr_}).groupby("sector")["flagged_top20pct"].mean()
    print("Share of real >10 % drawdowns (20 d) that were in the model's top-20 % risk bucket, by sector:"); display(DD_FN.round(3))
    DD_FN.to_csv(SUB["tables"] / "cv_dd_recall_by_sector_val.csv")
    wrong = r[(pb_["dir_20"][r].argmax(1) != PANEL.Y["dir_20"][r])]
    conf = pb_["dir_20"][wrong].max(1); top = wrong[np.argsort(-conf)[:4]]
    if len(top):
        fig, axes = plt.subplots(len(top), 3, figsize=(9, 3 * len(top)), squeeze=False)
        for i, rr in enumerate(top):
            for j in range(3):
                axes[i][j].imshow(PALETTE_DARK[np.asarray(IMG[rr, j])]); axes[i][j].axis("off")
            axes[i][0].set_title(f"{P.at[rr, 'ticker']} {P.at[rr, 'date'].date()}: predicted {CLASSES[int(pb_['dir_20'][rr].argmax())]} "
                                 f"({pb_['dir_20'][rr].max():.2f}), truth {CLASSES[int(PANEL.Y['dir_20'][rr])]}", fontsize=8, loc="left")
        plt.tight_layout(); savefig(fig, "cv_confident_errors_val.png", dpi=100); plt.show()

### 12.6 Explainability: Grad-CAM and occlusion by view and by time
**Grad-CAM** (Selvaraju et al. 2017) weights the last convolutional (or last transformer-block) feature map by the gradient of one output (here the 20-day UP logit and the 20-day volatility output) and shows where that output is most sensitive. **Limits:** a heat map shows sensitivity, not reasoning; it is coarse (7 × 7 or 14 × 14 cells over 52-70 candles); and on a near-chance target (direction) it highlights what the network reacts to, not a reliable pattern. The **occlusion** table is more quantitative: it blanks a whole view, its newest quarter of bars or its oldest quarter, and measures how much predictions move.

In [ ]:
def target_layer(net):
    bb = net.backbone; kind = net.__class__
    for attr in ("layer4", "stages", "conv_head", "blocks", "layers", "body"):
        if hasattr(bb, attr):
            m = getattr(bb, attr)
            if attr in ("stages", "blocks", "layers") and isinstance(m, (nn.Sequential, nn.ModuleList)):
                return m[-1]
            if attr == "body":
                return m[-2]
            return m
    return None

def grad_cam(net, x, which="dir", h_index=2):
    """x: (1, 3, 3, H, W) float. Returns (3, 224, 224) CAMs (one per view) for the chosen output."""
    layer = target_layer(net); store = {}
    def fwd(_, __, out):
        out = out[0] if isinstance(out, tuple) else out
        out.retain_grad(); store["a"] = out
    hd = layer.register_forward_hook(fwd)
    net.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=False):
        o = net(x)
    y = o["dir"][0, h_index, 2] if which == "dir" else o["vol"][0, h_index]
    y.backward(); hd.remove()
    a, g = store["a"].float(), store["a"].grad.float()
    if a.dim() == 3:                                     # transformer tokens (B, N, C): drop prefix tokens, reshape to a grid
        n = a.shape[1]; s = int(math.isqrt(n)); a, g = a[:, n - s * s:], g[:, n - s * s:]
        a = a.transpose(1, 2).reshape(a.shape[0], -1, s, s); g = g.transpose(1, 2).reshape(g.shape[0], -1, s, s)
    elif a.dim() == 4 and a.shape[-1] > a.shape[1] and a.shape[1] == a.shape[2]:   # NHWC (Swin)
        a, g = a.permute(0, 3, 1, 2), g.permute(0, 3, 1, 2)
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[:, 0]
    cam = cam / cam.flatten(1).max(1)[0].clamp(min=1e-9)[:, None, None]
    return cam.detach().cpu().numpy()

_cnns = [m for m in CV_RANK.index if m.replace("_ft", "") in ("jkx_cnn", "resnet50", "effnet_b0", "convnext_t", "yolov8_cls")]
_trf = [m for m in CV_RANK.index if m.replace("_ft", "") in ("vit_b16", "swin_t")]
XAI_MODELS = [m for m in (_cnns[:1] + _trf[:1]) if m]
if XAI_MODELS and HAS_GPU:
    ex_rows = [int(x) for x in np.random.default_rng(7).choice(np.flatnonzero(VA), 3, replace=False)]
    fig, axes = plt.subplots(len(XAI_MODELS) * len(ex_rows), 6, figsize=(16, 2.7 * len(XAI_MODELS) * len(ex_rows)), squeeze=False)
    k = 0
    for m in XAI_MODELS:
        try:
            net, _ = load_chartnet(m)
            for rr in ex_rows:
                x = batch_images(np.array([rr])).requires_grad_(False)
                cd = grad_cam(net, x, "dir", HORIZONS.index(20)); cv_ = grad_cam(net, x, "vol", HORIZONS.index(20))
                for j in range(3):
                    rgb = PALETTE_DARK[np.asarray(IMG[rr, j])]
                    axes[k][2 * j].imshow(rgb); axes[k][2 * j].imshow(cd[j], cmap="jet", alpha=0.45); axes[k][2 * j].axis("off")
                    axes[k][2 * j + 1].imshow(rgb); axes[k][2 * j + 1].imshow(cv_[j], cmap="jet", alpha=0.45); axes[k][2 * j + 1].axis("off")
                    axes[k][2 * j].set_title(f"{VIEWS[j]}: UP-20d", fontsize=7); axes[k][2 * j + 1].set_title(f"{VIEWS[j]}: vol-20d", fontsize=7)
                axes[k][0].text(-10, 112, f"{m}\n{P.at[rr, 'ticker']} {P.at[rr, 'date'].date()}", fontsize=7, ha="right", va="center")
                k += 1
            del net; torch.cuda.empty_cache()
        except Exception as e:
            print(f"Grad-CAM skipped for {m}: {type(e).__name__}: {e}")
    plt.suptitle("Grad-CAM (red = most sensitive region); sensitivity, not proof of reasoning"); plt.tight_layout()
    savefig(fig, "cv_gradcam.png", dpi=95); plt.show()

@torch.no_grad()
def raw_from_float(net, xf, batch=96):
    res = {"dir": [], "vol": [], "dd": []}
    for s in range(0, len(xf), batch):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            o = net(xf[s: s + batch])
        for kk in res:
            res[kk].append(o[kk].float().cpu().numpy())
    d, v, q = (np.concatenate(res[kk]) for kk in ("dir", "vol", "dd"))
    return {**{f"dir_{h}": d[:, j] for j, h in enumerate(HORIZONS)}, **{f"vol_{h}": v[:, j] * VOL_SD[j] + VOL_MU[j] for j, h in enumerate(HORIZONS)},
            **{f"dd_{h}": q[:, j] for j, h in enumerate(HORIZONS)}}

def occlude(xf, view, part):
    x = xf.clone(); bg = PAL_DARK_T[0] / 255.0
    cols = {"all": slice(0, 224), "newest25": slice(168, 224), "oldest25": slice(0, 56)}[part]
    x[:, view, :, :, cols] = bg.view(1, 3, 1, 1)
    return x

OCC = []
if CV_MAIN and HAS_GPU:
    occ_rows = np.sort(np.random.default_rng(8).choice(np.flatnonzero(VA), min(400, int(VA.sum())), replace=False))
    for m in list(CV_RANK.index[:3]):
        try:
            net, _ = load_chartnet(m); cal = CALS["panel"].get(m) or json.loads((SUB["preds"] / "panel" / f"{m}.cal.json").read_text())
            xf = batch_images(occ_rows)
            base = apply_calibration(raw_from_float(net, xf), cal)
            for j, v in enumerate(VIEWS):
                for part in ("all", "newest25", "oldest25"):
                    pr = apply_calibration(raw_from_float(net, occlude(xf, j, part)), cal)
                    OCC.append({"model": m, "view": v, "blanked": part,
                                "mean_abs_change_P_up20": float(np.abs(pr["dir_20"][:, 2] - base["dir_20"][:, 2]).mean()),
                                "mean_abs_change_logvol20": float(np.abs(pr["vol_20"] - base["vol_20"]).mean()),
                                "mean_abs_change_P_dd20": float(np.abs(pr["dd_20"] - base["dd_20"]).mean())})
            del net, xf; torch.cuda.empty_cache()
        except Exception as e:
            print(f"Occlusion skipped for {m}: {type(e).__name__}: {e}")
OCC_DF = pd.DataFrame(OCC)
if len(OCC_DF):
    OCC_DF.to_csv(SUB["tables"] / "cv_occlusion_val.csv", index=False)
    display(OCC_DF.pivot_table(index=["view", "blanked"], columns="model", values="mean_abs_change_logvol20").round(4))

### 12.7 Robustness (corruptions a real screenshot pipeline could introduce)
Top-3 CV models on 1,000 validation rows: Gaussian pixel noise, blur, JPEG compression (quality 30), a **light chart theme** (style shift never seen in training), screenshot-like down-and-up resizing, and a missing hourly chart.

In [ ]:
def jpeg(xf, q=30):
    from PIL import Image as PILImage
    out = xf.clone(); arr = (xf.clamp(0, 1) * 255).byte().cpu().numpy()
    for i in range(arr.shape[0]):
        for v in range(arr.shape[1]):
            buf = io.BytesIO(); PILImage.fromarray(arr[i, v].transpose(1, 2, 0)).save(buf, format="JPEG", quality=q)
            out[i, v] = torch.from_numpy(np.asarray(PILImage.open(io.BytesIO(buf.getvalue()))).transpose(2, 0, 1).copy()).float().to(xf.device) / 255
    return out
def blur(xf, k=5, s=1.0):
    ax_ = torch.arange(k, device=xf.device) - k // 2; g = torch.exp(-(ax_ ** 2) / (2 * s * s)); g = g / g.sum()
    w = (g[:, None] * g[None, :]).expand(3, 1, k, k)
    z = xf.flatten(0, 1); z = F.conv2d(F.pad(z, (k // 2,) * 4, mode="replicate"), w, groups=3)
    return z.view_as(xf)
def resize_screenshot(xf):
    z = xf.flatten(0, 1); z = F.interpolate(F.interpolate(z, scale_factor=0.5, mode="bilinear", align_corners=False), size=(224, 224), mode="bilinear", align_corners=False)
    return z.view_as(xf)
CORRUPT = {"clean": lambda x, rows: x, "gaussian_noise_0.05": lambda x, rows: (x + 0.05 * torch.randn_like(x)).clamp(0, 1),
           "blur_5px": lambda x, rows: blur(x), "jpeg_q30": lambda x, rows: jpeg(x), "light_theme": lambda x, rows: batch_images(rows, PAL_LIGHT_T),
           "screenshot_resize_0.5x": lambda x, rows: resize_screenshot(x), "hourly_missing": lambda x, rows: occlude(x, 2, "all")}
ROB = []
if CV_MAIN and HAS_GPU:
    rob_rows = np.sort(np.random.default_rng(9).choice(np.flatnonzero(VA), min(1000, int(VA.sum())), replace=False))
    for m in list(CV_RANK.index[:3]):
        try:
            net, _ = load_chartnet(m); cal = CALS["panel"].get(m) or {}
            for cname, fn in CORRUPT.items():
                preds = []
                for s in range(0, len(rob_rows), 200):
                    rb = rob_rows[s: s + 200]; preds.append(apply_calibration(raw_from_float(net, fn(batch_images(rb), rb)), cal))
                pr = {t: np.concatenate([p_[t] for p_ in preds]) for t in preds[0]}
                row = {"model": m, "corruption": cname}
                for t in ("dir_20", "vol_20", "dd_20"):
                    y = PANEL.Y[t][rob_rows]; ok = np.isfinite(y)
                    row[f"{t}_{PRIMARY[t.split('_')[0]]}"] = primary_value(t, y[ok], pr[t][ok])
                ROB.append(row)
            del net; torch.cuda.empty_cache()
        except Exception as e:
            print(f"Robustness skipped for {m}: {type(e).__name__}: {e}")
ROB_DF = pd.DataFrame(ROB)
if len(ROB_DF):
    for c in [c for c in ROB_DF.columns if c not in ("model", "corruption")]:
        ROB_DF[c + "_delta"] = ROB_DF[c] - ROB_DF.groupby("model")[c].transform(lambda s: s[ROB_DF.loc[s.index, "corruption"] == "clean"].iloc[0])
    ROB_DF.to_csv(SUB["tables"] / "cv_robustness_val.csv", index=False); display(ROB_DF.round(4))

### 12.8 Computational efficiency

In [ ]:
EFF = []
if HAS_GPU:
    from torch.utils.flop_counter import FlopCounterMode
    for m in CV_MAIN:
        try:
            net, ck = load_chartnet(m)
            x1 = batch_images(np.array([int(np.flatnonzero(VA)[0])]))
            with FlopCounterMode(display=False) as fc:
                with torch.no_grad():
                    net(x1)
            flops = fc.get_total_flops()
            with torch.no_grad():
                for _ in range(3):
                    net(x1)
                torch.cuda.synchronize(); ts = []
                for _ in range(20):
                    t0 = time.perf_counter(); net(x1); torch.cuda.synchronize(); ts.append(time.perf_counter() - t0)
                x64 = batch_images(np.flatnonzero(VA)[:64]); torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    net(x64)
                torch.cuda.synchronize(); thr = 64 / (time.perf_counter() - t0)
            EFF.append({"model": m, "params_m": round(sum(p.numel() for p in net.parameters()) / 1e6, 2), "gflops_per_row_3_views": round(flops / 1e9, 2),
                        "latency_ms_batch1_fp32": round(1000 * float(np.median(ts)), 1), "rows_per_s_batch64_bf16": round(thr, 1),
                        "train_minutes": REG[m].get("train_minutes"), "checkpoint_mb": round((SUB["models"] / f"cv_{m}.pt").stat().st_size / 1e6, 1),
                        "val_mean_rank": round(float(CV_RANK.get(m, np.nan)), 2)})
            del net; torch.cuda.empty_cache()
        except Exception as e:
            print(f"Efficiency skipped for {m}: {type(e).__name__}: {e}")
EFF_CV = pd.DataFrame(EFF)
if len(EFF_CV):
    EFF_CV.to_csv(SUB["tables"] / "cv_efficiency.csv", index=False); display(EFF_CV)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(EFF_CV.gflops_per_row_3_views, EFF_CV.val_mean_rank, s=EFF_CV.params_m * 3 + 10, alpha=0.6)
    for _, r_ in EFF_CV.iterrows():
        ax.annotate(r_.model.replace("_ft", ""), (r_.gflops_per_row_3_views, r_.val_mean_rank), fontsize=7)
    ax.invert_yaxis(); ax.set_xlabel("GFLOPs per row (3 charts)"); ax.set_ylabel("validation mean rank (top = better)")
    ax.set_title("Accuracy vs compute (marker size = parameters)"); ax.grid(alpha=0.3)
    plt.tight_layout(); savefig(fig, "cv_efficiency.png"); plt.show()
tick("12 cv analysis")

## 13. SEC filings: download, point-in-time timestamps, sections and text analysis

**Which filings.** Original 10-K, 10-Q and 8-K filings (amendments excluded) of the study companies, 2015-06 to the data end (filings before the first decision are needed only as "latest filing" context). QUICK mode keeps every 10-K and 10-Q and every 3rd 8-K.

**When a filing becomes usable (point-in-time rule from the SEC study).** EDGAR's JSON `acceptanceDateTime` is unreliable (about 70 % are New York wall time mislabelled as UTC), so the acceptance time is read from each filing's SGML header (quality A); otherwise the later of the two JSON readings is used (quality C). Availability = acceptance + 5 min (public dissemination) + 30 min (processing). A periodic report is never treated as available at its period end.

**Text and sections.** HTML → text (scripts, inline-XBRL headers and mostly-numeric tables removed). 10-K: Item 7, *Management's Discussion and Analysis* (MD&A); 10-Q: Part I Item 2 (MD&A); banks often put the MD&A in Exhibit 13 (the annual report), which is fetched too. 8-K: the item text plus the first EX-99 exhibit (usually the press release); the 8-K item codes (e.g. 2.02 results, 5.02 officer change) come from EDGAR's index. If an MD&A heading is not found (or is shorter than 2,000 characters), the whole document is used minus its first 3 % (cover page and table of contents). Each model reads `[form] [section] text`.

In [ ]:
import lxml.html
_WS = re.compile(r"\s+"); _DIGIT_RE = re.compile(r"\d")
_BLOCK_TAGS = {"p", "div", "br", "tr", "li", "h1", "h2", "h3", "h4", "h5", "h6", "table"}
def html_to_text(raw: str, is_html: bool = True) -> str:
    if not raw:
        return ""
    if not is_html:
        return _WS.sub(" ", raw).strip()
    try:
        doc = lxml.html.fromstring(raw.encode("utf-8", "ignore"))
    except Exception:
        return _WS.sub(" ", re.sub(r"<[^>]+>", " ", raw)).strip()
    for bad in doc.xpath("//script|//style|//head|//*[name()='ix:header']"):
        bad.drop_tree()
    for tb in doc.xpath("//table"):
        s = tb.text_content()
        if s and len(_DIGIT_RE.findall(s)) / max(1, len(s)) > 0.25:
            tb.drop_tree()
    for el in doc.iter():
        if isinstance(el.tag, str) and el.tag.lower() in _BLOCK_TAGS:
            el.tail = "\n" + (el.tail or "")
    txt = re.sub(r"[ \t\xa0]+", " ", doc.text_content())
    return re.sub(r"\n\s*\n+", "\n\n", txt).strip()

_ACC_RE = re.compile(r"<ACCEPTANCE-DATETIME>\s*(\d{14})|ACCEPTANCE-DATETIME:\s*(\d{14})")
def parse_sgml_header(txt: str):
    t = html.unescape(txt or ""); m = _ACC_RE.search(t)
    acc = (m.group(1) or m.group(2)) if m else None
    docs = []
    for block in re.split(r"<DOCUMENT>", t)[1:]:
        ty, fn = re.search(r"<TYPE>\s*([^\s<]+)", block), re.search(r"<FILENAME>\s*([^\s<]+)", block)
        if ty and fn:
            docs.append((ty.group(1).strip().upper(), fn.group(1).strip()))
    return acc, docs

def acceptance_time(header14, json_raw, filing_date):
    if header14:
        t = pd.Timestamp(f"{header14[:4]}-{header14[4:6]}-{header14[6:8]} {header14[8:10]}:{header14[10:12]}:{header14[12:14]}")
        return t.tz_localize(NY, ambiguous=True, nonexistent="shift_forward"), "A"
    if json_raw is not None and str(json_raw).strip() not in ("", "nan", "None"):
        t = pd.Timestamp(str(json_raw)[:19])
        later = max(t.tz_localize("UTC").tz_convert(NY), t.tz_localize(NY, ambiguous=True, nonexistent="shift_forward"))
        if not (later.hour == 0 and later.minute == 0):
            return later, "C"
    return pd.Timestamp(filing_date).tz_localize(NY) + pd.Timedelta(CFG["sec"]["date_only_time_et"] + ":00"), "B"

_SEC_PAT = {
    "10-K": (r"item\s*7\s*[\.\:\-–—]?\s*management[’'`]?s?\s+discussion",
             r"item\s*7a\s*[\.\:\-–—]?\s*quantitative|item\s*8\s*[\.\:\-–—]?\s*financial\s+statements"),
    "10-Q": (r"item\s*2\s*[\.\:\-–—]?\s*management[’'`]?s?\s+discussion",
             r"item\s*3\s*[\.\:\-–—]?\s*quantitative|item\s*4\s*[\.\:\-–—]?\s*controls"),
}
def extract_section(form: str, text: str):
    """(section name, section text). MD&A for 10-K / 10-Q (longest heading-to-next-item span), else a fallback."""
    if form == "8-K":
        return "8-K body + EX-99", text
    st_p, en_p = _SEC_PAT[form]
    best = ""
    for m in re.finditer(st_p, text, flags=re.I):
        e = re.search(en_p, text[m.end(): m.end() + 600000], flags=re.I)
        span = text[m.start(): m.end() + (e.start() if e else 300000)]
        if len(span) > len(best):
            best = span
    if len(best) >= 2000:
        return "MD&A", best
    return "full text (no MD&A heading)", text[int(0.03 * len(text)):]

In [ ]:
SEC_DIR = DATA_DIR / "sec"; TEXT_DIR = SEC_DIR / "texts"; TEXT_DIR.mkdir(parents=True, exist_ok=True)
SEC_KEY = cfg_hash([CFG["sec"], QUICK, CFG["data_end"], "v3"])
FIL_COLS = ["ticker", "cik", "accession", "form", "filing_date", "acceptance_et", "available_utc", "ts_quality", "items", "n_chars",
            "section", "section_chars", "docs"]

def download_ticker(t: str) -> pd.DataFrame:
    meta_p, text_p = TEXT_DIR / f"{t}_{SEC_KEY}_meta.parquet", TEXT_DIR / f"{t}_{SEC_KEY}_text.parquet"
    if meta_p.exists() and text_p.exists() and not FORCE_RERUN:
        return pd.read_parquet(meta_p)
    fl = filing_index(t)
    if QUICK and len(fl):
        k8 = fl[fl.form == "8-K"].iloc[:: CFG["sec"]["quick_8k_stride"]]
        fl = pd.concat([fl[fl.form != "8-K"], k8]).sort_values("filingDate")
    rows, texts, t0 = [], [], time.time()
    for _, f in fl.iterrows():
        acc = f["accessionNumber"]; nod = acc.replace("-", ""); cik = int(f["cik"])
        base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{nod}/"
        hdr14, docs = parse_sgml_header(SEC.get(base + f"{acc}-index-headers.html", soft=True) or "")
        acc_et, q = acceptance_time(hdr14, f.get("acceptanceDateTime"), f["filingDate"])
        avail = acc_et + pd.Timedelta(minutes=CFG["sec"]["public_lag_min"] + CFG["sec"]["processing_min"])
        want = [f["primaryDocument"]] if str(f.get("primaryDocument") or "").strip() else []
        if f["form"] == "8-K":
            want += [fn for ty, fn in docs if ty.startswith("EX-99")][: CFG["sec"]["max_exhibits_8k"]]
        else:
            want += [fn for ty, fn in docs if ty.startswith("EX-13")][:1]
        parts = [html_to_text(SEC.get(base + fn, soft=True) or "", is_html=not fn.lower().endswith(".txt")) for fn in want]
        body = "\n\n".join(x for x in parts if x)[:400000]
        sec_name, sec_txt = extract_section(f["form"], body)
        rows.append({"ticker": t, "cik": cik, "accession": acc, "form": f["form"], "filing_date": str(f["filingDate"]),
                     "acceptance_et": str(acc_et), "available_utc": str(avail.tz_convert("UTC")), "ts_quality": q,
                     "items": str(f.get("items") or ""), "n_chars": len(body), "section": sec_name, "section_chars": len(sec_txt),
                     "docs": "|".join(want)})
        texts.append({"accession": acc, "section_text": sec_txt[: CFG["sec"]["max_chars"] * 2]})
    meta = pd.DataFrame(rows, columns=FIL_COLS)
    pd.DataFrame(texts, columns=["accession", "section_text"]).to_parquet(text_p, index=False, compression="zstd")
    meta.to_parquet(meta_p, index=False)                     # written last: marks the ticker complete
    print(f"  {t}: {meta.form.value_counts().to_dict()} in {time.time() - t0:.0f}s")
    return meta

t0 = time.time()
FIL = pd.concat([download_ticker(t) for t in TICKERS], ignore_index=True)
FIL["available_utc"] = pd.to_datetime(FIL["available_utc"], utc=True)
FIL = FIL.sort_values(["ticker", "available_utc"]).reset_index(drop=True)
print(f"SEC filings ready in {time.time() - t0:.0f}s: {FIL.form.value_counts().to_dict()}; optional requests that fell back: {SEC.n_soft_fail}")
_TXT = {}
for t in TICKERS:
    tp = pd.read_parquet(TEXT_DIR / f"{t}_{SEC_KEY}_text.parquet"); _TXT.update(dict(zip(tp["accession"], tp["section_text"])))
FIL["section_text"] = FIL["accession"].map(_TXT).fillna(""); del _TXT
_ITEM_RE = re.compile(r"\bItem\s+[1-9]\.\d{2}\b", re.I)
def strip_8k_cover(t):
    """Start an 8-K at its first item heading (e.g. 'Item 2.02 Results of Operations'): the cover page is boilerplate."""
    m = _ITEM_RE.search(t)
    return t[m.start():] if m and m.start() < 0.5 * len(t) else t
_is8 = FIL["form"] == "8-K"
FIL.loc[_is8, "section_text"] = FIL.loc[_is8, "section_text"].map(strip_8k_cover)
TEXT_KEY = cfg_hash([SEC_KEY, "8k_cover_strip_v1"])          # changes whenever the text a model reads changes
acc_ny = pd.to_datetime(FIL["acceptance_et"].astype(str), utc=True).dt.tz_convert(NY)
mins = acc_ny.dt.hour * 60 + acc_ny.dt.minute
FIL["session_bucket"] = np.where(mins < 570, "pre-market", np.where(mins < 960, "intraday", "after close"))
FIL["sector"] = FIL["ticker"].map(SECTOR_OF)
PROV.append({"source": "SEC EDGAR filings", "status": "OK", "note": f"{len(FIL)} filings; timestamp quality {FIL.ts_quality.value_counts().to_dict()}"})

### 13.2 Text dataset analysis

In [ ]:
FIL["year"] = pd.to_datetime(FIL["filing_date"]).dt.year
cnt = FIL.pivot_table(index="year", columns="form", values="accession", aggfunc="count").fillna(0).astype(int)
cnt.to_csv(SUB["tables"] / "sec_per_year.csv"); display(cnt.T)
sec_cnt = FIL.pivot_table(index="sector", columns="form", values="accession", aggfunc="count").fillna(0).astype(int)
sec_cnt.to_csv(SUB["tables"] / "sec_per_sector.csv"); display(sec_cnt)
TEXT_STATS = FIL.groupby("form").agg(filings=("accession", "size"), median_doc_chars=("n_chars", "median"),
                                     median_section_chars=("section_chars", "median"),
                                     mdna_found_share=("section", lambda s: float((s == "MD&A").mean())),
                                     empty_share=("section_chars", lambda s: float((s < 500).mean())))
TEXT_STATS.to_csv(SUB["tables"] / "sec_text_stats.csv"); display(TEXT_STATS)
items8 = FIL.loc[FIL.form == "8-K", "items"].str.split(",").explode().str.strip()
ITEM_COUNTS = items8[items8 != ""].value_counts(); ITEM_COUNTS.to_csv(SUB["tables"] / "sec_8k_item_counts.csv")
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
for f_ in CFG["sec"]["forms"]:
    np.log10(FIL.loc[FIL.form == f_, "section_chars"].clip(lower=10)).hist(bins=40, ax=ax[0], alpha=0.5, label=f_)
ax[0].legend(); ax[0].set_title("log10(characters in the section a model reads)")
FIL.groupby(["form", "session_bucket"]).size().unstack().plot.bar(ax=ax[1]); ax[1].set_title("Acceptance time vs trading session")
ITEM_COUNTS.head(15).plot.barh(ax=ax[2]); ax[2].invert_yaxis(); ax[2].set_title("Most frequent 8-K items")
plt.tight_layout(); savefig(fig, "eda_sec.png"); plt.show()
print("Filings with < 500 characters of usable text:", int((FIL.section_chars < 500).sum()))
tick("13 sec text")

## 14. Filing events: tradeable labels and the event study (Q6, Q7, Q9)

**Entry rule (no look-ahead).** A filing becomes usable at its availability time. Entry is the **open of the first hourly bar that starts after it** (bars start 09:30, 10:30 … 15:30 New York); a filing available after 15:30 or outside market hours enters at the next session's 09:30 open. Event targets for horizon h use the same definitions as the panel, measured from the entry price: direction over sessions *s … s+h−1* (close of the last one), realised volatility of the h sessions after the entry session, and a >10 % drawdown from the entry price (lows of the remaining bars of the entry session and of the next h−1 sessions). The sideways band uses σ known before the filing.

**Market reaction (event study, descriptive).** "Pre" = the last close before availability. Abnormal return AR₂ = log return of the stock minus SPY from pre to the close of the session after the entry session (2 sessions); scaled by the stock's pre-event daily abnormal volatility, |z| = |AR₂| / (σ_AR √2) measures surprise in "normal-day units". Also: abnormal volume (log of entry-session volume over the 20-session median before) and the post-event volatility ratio (log of mean session RV over the 5 sessions from entry vs the 20 before). **Control:** the same measures on random **non-filing** sessions of the same stocks and years (no filing within ±3 sessions).

The study below uses development blocks only (train + inner + validation); the test block is replicated in §23.5.

In [ ]:
CLOSE_T = pd.Timedelta(hours=16)
EVR = []
for t in TICKERS:
    d, hb = BARS_D[t], BARS_H[t]
    dates = pd.DatetimeIndex(d["date"]); c, lo, vol_ = d["close"].values, d["low"].values, d["volume"].values
    close_utc = (dates + CLOSE_T).tz_localize(NY).tz_convert("UTC")
    hstart = hb["timestamp"].values.astype("datetime64[ns]"); hs_sess = pd.DatetimeIndex(hb["session"])
    h_didx = np.searchsorted(dates.values, hs_sess.values)
    lr = np.r_[np.nan, np.diff(np.log(c))]
    spy = SPY_D.set_index("date")["close"].reindex(dates).ffill().values
    ar = lr - np.r_[np.nan, np.diff(np.log(spy))]
    sig = pd.Series(lr).rolling(SIGW, min_periods=SIGW).std().values
    sig_ar = pd.Series(ar).rolling(SIGW, min_periods=SIGW).std().values
    rvs = session_rv(d, hb)
    for fid, f in FIL[FIL.ticker == t].iterrows():
        av = f["available_utc"].tz_convert("UTC").tz_localize(None).to_datetime64()
        pre = int(np.searchsorted(close_utc.tz_localize(None).values, av, side="right")) - 1
        k = int(np.searchsorted(hstart, av, side="left"))
        if pre < SIGW or k >= len(hstart):
            continue
        s = int(h_didx[k]); entry = float(hb["open"].values[k])
        row = {"fil_id": int(fid), "ticker": t, "form": f["form"], "sector": SECTOR_OF[t], "items": f["items"], "section": f["section"],
               "entry_date": dates[s], "entry_hour": pd.Timestamp(hstart[k]).tz_localize("UTC").tz_convert(NY).hour, "pre_idx": pre,
               "s_idx": s, "g": CAL_POS.get(dates[s], -1), "sigma": sig[pre], "session_bucket": f["session_bucket"]}
        for h in HORIZONS:
            e = s + h - 1
            if e + 1 >= len(c):
                row.update({f"ret_{h}": np.nan, f"dir_{h}": np.nan, f"vol_{h}": np.nan, f"dd_{h}": np.nan}); continue
            r = math.log(c[e] / entry); thr = K_SIDE * sig[pre] * math.sqrt(h)
            row[f"ret_{h}"] = r; row[f"dir_{h}"] = 2.0 if r > thr else (0.0 if r < -thr else 1.0)
            row[f"vol_{h}"] = 0.5 * math.log(max(float(np.nansum(rvs[s + 1: s + h + 1])), 1e-12) * 252.0 / h)
            low_same = hb["low"].values[k: np.searchsorted(h_didx, s, side="right")]
            mn = min(float(low_same.min()) if len(low_same) else entry, float(lo[s + 1: e + 1].min()) if e >= s + 1 else entry)
            row[f"dd_{h}"] = float(mn / entry - 1 <= DDT)
        if s + 1 < len(c) and np.isfinite(sig_ar[pre]):
            ar2 = float(np.nansum(ar[pre + 1: s + 2]))
            row.update({"ar2": ar2, "abs_z": abs(ar2) / (sig_ar[pre] * math.sqrt(max(1, s + 1 - pre))),
                        "abn_volume": math.log((vol_[s] + 1) / (np.median(vol_[max(0, pre - 19): pre + 1]) + 1)),
                        "post_vol_ratio": math.log((np.nanmean(rvs[s: s + 5]) + 1e-12) / (np.nanmean(rvs[max(0, pre - 19): pre + 1]) + 1e-12))})
        EVR.append(row)
EV = pd.DataFrame(EVR)
EV = EV[EV["g"] >= 0].copy()
EV["block"] = None
for b in BLOCK_ORDER:
    EV.loc[(EV.entry_date >= BLK[b][0]) & (EV.entry_date <= BLK[b][1]), "block"] = b
EV["block"] = EV["block"].fillna("pre_study")          # filings before 2016-10-10: context for the panel only, no labels used
for b, nxt in zip(BLOCK_ORDER[:-1], BLOCK_ORDER[1:]):
    g_next = int(np.searchsorted(CAL.values, np.datetime64(BLK[nxt][0])))
    EV.loc[(EV.block == b) & (EV.g + HMAX >= g_next), "block"] = f"{b}_purged"
EV = EV.reset_index(drop=True); EV["ev"] = np.arange(len(EV))
EV["month"] = EV["entry_date"].dt.to_period("M").astype(str)
EV_LABELS = [f"{k}_{h}" for k in ("ret", "dir", "vol", "dd") for h in HORIZONS]
REACT_COLS = ["ar2", "abs_z", "abn_volume", "post_vol_ratio"]
for c_ in REACT_COLS:
    if c_ not in EV:
        EV[c_] = np.nan
TE_EV = (EV.block == "test").values
VAULT.deposit("event_labels", EV.loc[TE_EV, ["ev", *EV_LABELS, *REACT_COLS]])
EV.loc[TE_EV, EV_LABELS + REACT_COLS] = np.nan
YE = {k: EV[k].values.astype(float).copy() for k in EV_LABELS}
EVENT = EvalCtx("event", YE, EV["month"].values, EV["block"].values, EV["sector"].values, EV["ticker"].values)
EVENT.fit_priors(EVENT.mask["train"])
EV_TEXT = [text_input(f, s_, FIL.at[i, "section_text"], CFG["sec"]["max_chars"]) for f, s_, i in zip(EV.form, EV.section, EV.fil_id)]
print("Events by block:", EV.block.value_counts().to_dict())
display(EV.groupby(["block", "form"]).size().unstack().fillna(0).astype(int))
assert np.isnan(EV.loc[TE_EV, EV_LABELS].values).all(), "test event labels are not hidden"
assert (EV.entry_date.values >= np.array([FIL.at[i, "available_utc"].tz_convert(NY).tz_localize(None).normalize() for i in EV.fil_id], dtype="datetime64[ns]")).all(), "entry before availability"

In [ ]:
# ---------------------------------------------------------------- matched non-filing sessions (control for H7)
rng = np.random.default_rng(11); CTRL = []
for t in TICKERS:
    d = BARS_D[t]; c = d["close"].values; dates = pd.DatetimeIndex(d["date"])
    lr = np.r_[np.nan, np.diff(np.log(c))]
    spy = SPY_D.set_index("date")["close"].reindex(dates).ffill().values
    ar = lr - np.r_[np.nan, np.diff(np.log(spy))]
    sig_ar = pd.Series(ar).rolling(SIGW, min_periods=SIGW).std().values
    busy = np.zeros(len(d), bool)
    for s in EV.loc[EV.ticker == t, "s_idx"]:
        busy[max(0, s - 3): s + 4] = True
    ok = np.flatnonzero(~busy & (dates >= BLK["train"][0]) & (dates <= BLK["val"][1]) & np.isfinite(sig_ar))
    ok = ok[ok + 2 < len(d)]
    for s in rng.choice(ok, min(len(ok), 4 * int((EV.ticker == t).sum())), replace=False):
        ar2 = float(np.nansum(ar[s: s + 2]))
        CTRL.append({"ticker": t, "sector": SECTOR_OF[t], "form": "no filing", "month": str(dates[s].to_period("M")),
                     "abs_z": abs(ar2) / (sig_ar[s - 1] * math.sqrt(2)), "abn_volume": math.log((d["volume"].values[s] + 1) / (np.median(d["volume"].values[max(0, s - 20): s]) + 1))})
CTRL = pd.DataFrame(CTRL)

### 14.1 Event study on the development blocks

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
def mean_ci(df, col, B=None, seed=0):
    x = df[col].values.astype(float); ok = np.isfinite(x)
    if ok.sum() < 10:
        return np.nan, np.nan, np.nan
    groups = month_groups(df["month"].values[ok]); xs = x[ok]
    bs = bootstrap(lambda ix: float(np.mean(xs[ix])), groups, B=B or min(CFG["bootstrap_B"], 500), seed=seed)
    return float(xs.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

def event_study(ev, ctrl):
    rows = []
    for name, g in list(ev.groupby("form")) + [("no filing (control)", ctrl)]:
        m, lo_, hi_ = mean_ci(g, "abs_z")
        rows.append({"group": name, "n": int(np.isfinite(g["abs_z"]).sum()), "mean_abs_z": m, "ci_lo": lo_, "ci_hi": hi_,
                     "share_abs_z_gt_2": float((g["abs_z"] > 2).mean()), "mean_abn_volume": float(g["abn_volume"].mean()),
                     "mean_post_vol_ratio": float(g["post_vol_ratio"].mean()) if "post_vol_ratio" in g else np.nan})
    tab = pd.DataFrame(rows)
    fx = [g["abs_z"].dropna().values for _, g in ev.groupby("form")]
    kw = kruskal(*fx) if len(fx) >= 2 and all(len(x) > 5 for x in fx) else None
    mw = mannwhitneyu(ev["abs_z"].dropna(), ctrl["abs_z"].dropna(), alternative="greater") if len(ctrl) else None
    return tab, (float(kw.pvalue) if kw else np.nan), (float(mw.pvalue) if mw else np.nan)

DEV_EV = EV[EV.block.isin(["train", "inner", "val"])]
ES_FORM, KW_P_DEV, MW_P_DEV = event_study(DEV_EV, CTRL)
ES_FORM.to_csv(SUB["tables"] / "event_study_by_form_dev.csv", index=False); display(ES_FORM.round(3))
print(f"H7 (development): filing days vs non-filing days, one-sided Mann-Whitney p = {MW_P_DEV:.2g}; forms differ (Kruskal-Wallis) p = {KW_P_DEV:.2g}")
ES_SECTOR = DEV_EV.groupby(["sector", "form"])["abs_z"].agg(["mean", "size"]).reset_index()
ctrl_sec = CTRL.groupby("sector")["abs_z"].mean()
ES_SECTOR["excess_over_non_filing_days"] = ES_SECTOR["mean"] - ES_SECTOR["sector"].map(ctrl_sec)
ES_SECTOR.to_csv(SUB["tables"] / "event_study_sector_form_dev.csv", index=False)
items_ev = DEV_EV[DEV_EV.form == "8-K"].assign(item=DEV_EV["items"].str.split(",")).explode("item")
items_ev["item"] = items_ev["item"].str.strip()
items_ev = items_ev[items_ev["item"].isin(ITEM_COUNTS.index[:14]) & (items_ev["item"] != "9.01")]
ES_ITEMS = pd.DataFrame([{"item": it, "n": len(g), **dict(zip(["mean_abs_z", "ci_lo", "ci_hi"], mean_ci(g, "abs_z"))),
                          "mean_abn_volume": float(g["abn_volume"].mean())} for it, g in items_ev.groupby("item") if len(g) >= 15],
                        columns=["item", "n", "mean_abs_z", "ci_lo", "ci_hi", "mean_abn_volume"]).sort_values("mean_abs_z", ascending=False)
ES_ITEMS.to_csv(SUB["tables"] / "event_study_8k_items_dev.csv", index=False); display(ES_ITEMS.round(3))
ES_BUCKET = DEV_EV.groupby(["form", "session_bucket"])["abs_z"].agg(["mean", "size"]).round(3)
ES_BUCKET.to_csv(SUB["tables"] / "event_study_timing_dev.csv")
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].errorbar(ES_FORM.group, ES_FORM.mean_abs_z, yerr=[ES_FORM.mean_abs_z - ES_FORM.ci_lo, ES_FORM.ci_hi - ES_FORM.mean_abs_z], fmt="o", capsize=4)
ax[0].set_title("Mean |abnormal 2-day move| in normal-day units (95 % CI)"); ax[0].grid(alpha=0.3)
hm = ES_SECTOR.pivot(index="sector", columns="form", values="excess_over_non_filing_days")
ax[1].imshow(hm.values, cmap="Reds", aspect="auto")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax[1].text(j, i, f"{hm.values[i, j]:+.2f}", ha="center", va="center", fontsize=8)
ax[1].set_xticks(range(hm.shape[1])); ax[1].set_xticklabels(hm.columns); ax[1].set_yticks(range(hm.shape[0])); ax[1].set_yticklabels(hm.index, fontsize=7)
ax[1].set_title("Sector × form: mean |z| minus non-filing days")
if len(ES_ITEMS):
    ax[2].barh(ES_ITEMS["item"].astype(str), ES_ITEMS["mean_abs_z"]); ax[2].invert_yaxis(); ax[2].set_title("8-K items: mean |z|")
plt.tight_layout(); savefig(fig, "event_study_dev.png"); plt.show()
tick("14 events")

## 15. NLP baseline: TF-IDF + logistic / ridge regression (and which words go with big moves, Q8)

Word and two-word n-grams (sub-linear TF, document frequency ≥ 5, 50,000 features) fitted on training filings; logistic regression for direction and drawdown, ridge regression for volatility; regularisation chosen on the inner block. In finance this simple model is notoriously hard to beat (Loughran and McDonald 2016): if a transformer cannot beat it, the problem is the signal or the pipeline, not the model size.

**Q8, words.** A separate ridge model on the development filings predicts the **size** of the market reaction (|z|) and its **sign** (AR₂) from the same n-grams; its largest coefficients show which words go with big or positive/negative reactions. These are associations inside the sample, not causes; boilerplate legal phrases appear when they mark a filing type.

In [ ]:
import scipy.sparse as sp
def fit_heads_sparse(ctx, Xs, fit_mask, sel_mask, tasks=None, keep=None):
    """As fit_heads (linear), for sparse TF-IDF matrices. keep: optional dict that receives the fitted model per task (for deployment)."""
    raw = {}
    for t in (tasks or TASKS):
        kd = t.split("_")[0]; fr, sr = ctx.rows(t, fit_mask), ctx.rows(t, sel_mask)
        out = np.full((Xs.shape[0], 3) if kd == "dir" else Xs.shape[0], np.nan)
        if len(fr) < 80 or len(sr) < 20:
            raw[t] = out; continue
        y_f, y_s = ctx.Y[t][fr], ctx.Y[t][sr]; best, bl = None, np.inf
        for c in (CFG["probe"]["alpha_grid"] if kd == "vol" else [0.1, 1.0, 10.0]):
            if kd == "vol":
                m = Ridge(alpha=c).fit(Xs[fr], y_f); loss = float(np.mean((m.predict(Xs[sr]) - y_s) ** 2))
            else:
                if len(np.unique(y_f)) < 2:
                    continue
                m = LogisticRegression(C=c, max_iter=3000).fit(Xs[fr], y_f.astype(int))
                ps = np.clip(m.predict_proba(Xs[sr]), 1e-9, 1); yi = y_s.astype(int)
                if ps.shape[1] != (3 if kd == "dir" else 2):
                    continue
                loss = float(-np.mean(np.log(ps[np.arange(len(yi)), yi])))
            if loss < bl:
                best, bl = m, loss
        if best is None:
            raw[t] = out; continue
        if keep is not None:
            keep[t] = best
        if kd == "vol":
            out = best.predict(Xs)
        elif kd == "dir":
            out = np.log(np.clip(best.predict_proba(Xs), 1e-9, 1))
        else:
            q = np.clip(best.predict_proba(Xs)[:, 1], 1e-6, 1 - 1e-6); out = np.log(q / (1 - q))
        raw[t] = out
    return raw

TF = CFG["tfidf"]
TFIDF = TfidfVectorizer(ngram_range=tuple(TF["ngram"]), min_df=TF["min_df"], max_features=TF["max_features"], sublinear_tf=True,
                        lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{2,}\b", dtype=np.float32)
_tr_ev = EVENT.mask["train"]
TFIDF.fit([EV_TEXT[i] for i in np.flatnonzero(_tr_ev)])
X_TFIDF = TFIDF.transform(EV_TEXT).tocsr()
print("TF-IDF matrix:", X_TFIDF.shape)
TFIDF_MODELS = {}
finalize(EVENT, "tfidf_lr", fit_heads_sparse(EVENT, X_TFIDF, EVENT.mask["train"], EVENT.mask["inner"], keep=TFIDF_MODELS), "nlp_baseline",
         label="TF-IDF + logistic / ridge", params_m=round(X_TFIDF.shape[1] * 15 / 1e6, 3))
finalize(EVENT, "prior_event", prior_raw(EVENT), "baseline", label="Prior (event base rates)")
joblib.dump({"vectorizer": TFIDF, "models": TFIDF_MODELS}, SUB["models"] / "tfidf_lr.joblib")

# Q8: which n-grams go with large and with positive / negative reactions (development filings)
vocab = np.array(TFIDF.get_feature_names_out())
dev_ev = np.flatnonzero(EV.block.isin(["train", "inner", "val"]).values & np.isfinite(EV["abs_z"].values))
WORDS = {}
for col, lab in (("abs_z", "size of reaction |z|"), ("ar2", "signed abnormal return")):
    y = EV[col].values[dev_ev]; y = np.clip(y, *np.nanpercentile(y, [1, 99]))
    rg = Ridge(alpha=10.0).fit(X_TFIDF[dev_ev], (y - y.mean()) / (y.std() + 1e-12))
    o = np.argsort(rg.coef_)
    WORDS[col] = pd.DataFrame({"top_positive": vocab[o[::-1][:25]], "coef_pos": rg.coef_[o[::-1][:25]].round(3),
                               "top_negative": vocab[o[:25]], "coef_neg": rg.coef_[o[:25]].round(3)})
    WORDS[col].to_csv(SUB["tables"] / f"nlp_words_{col}_dev.csv", index=False)
    print(f"Words most associated with {lab} (development filings, ridge on TF-IDF):"); display(WORDS[col].head(15))
tick("15 tfidf")

## 16. Language models, part 1: architectures and frozen embeddings

### 16.1 How the models read a filing
* **BERT-family encoders** (FinBERT, SEC-BERT, RoBERTa-large, DeBERTa-v3-large) split text into sub-word tokens, add position information and pass them through 12 (base) or 24 (large) self-attention layers; each token's vector is updated from every other token in a **512-token** window. FinBERT continued BERT pre-training on financial news and was tuned for sentiment; SEC-BERT was pre-trained from scratch on 260,773 10-K filings with its own vocabulary; RoBERTa drops next-sentence prediction and trains longer on more data; DeBERTa-v3 separates content and position in attention ("disentangled attention") and is pre-trained as a discriminator (replaced-token detection).
* **Longformer** replaces full attention by a sliding window plus a few global tokens, so cost grows linearly with length and it reads **4,096 tokens** (about 8 pages) of the MD&A instead of 512.
* **FinGPT** is a decoder-only LLM (Llama-2-7B) with a LoRA adapter trained by its authors on financial instruction tasks (sentiment, headline classification, NER). We load it in 4-bit NF4 (QLoRA-style), read 1,024 tokens, and use (a) the mean of its last hidden layer as an embedding and (b) its own answer probabilities for "negative / neutral / positive" to a sentiment prompt.

### 16.2 Frozen use
Documents longer than one window are split into up to 4 chunks of 512 tokens; each chunk is encoded and mean-pooled (attention-masked), and the document vector is the mean of its chunks. 32 principal components (fitted on training filings) feed regularised linear models per task, exactly as for the CV probes. **Length ablation:** the same probe on the first chunk only.

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
NLPC = CFG["nlp_models"]; NCH, CL = CFG["nlp_ft"]["frozen_chunks"], CFG["nlp_ft"]["chunk_len"]
N_EV = len(EV)
ENCODERS = [m for m, s in NLPC.items() if m != "fingpt"]
TEXT_EMB = {}

@torch.no_grad()
def encode_frozen(spec):
    tok = AutoTokenizer.from_pretrained(spec["hf"])
    enc = AutoModel.from_pretrained(spec["hf"], torch_dtype=torch.bfloat16).to(DEV).eval()
    long_ = spec["max_len"] > 512
    clen = spec["max_len"] - 2
    ids_all = tok([t_[: 10 * NCH * clen] for t_ in EV_TEXT], add_special_tokens=False, truncation=True, max_length=(clen if long_ else NCH * clen))["input_ids"]
    cls_id = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id      # [CLS] / <s>
    sep_id = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id      # [SEP] / </s>
    chunks, owner, first = [], [], []
    for i, ids in enumerate(ids_all):
        ids = ids or [tok.unk_token_id]
        for k, c0 in enumerate(range(0, len(ids), clen)):
            chunks.append([cls_id] + list(ids[c0: c0 + clen]) + [sep_id]); owner.append(i); first.append(k == 0)
    owner, first = np.array(owner), np.array(first)
    order = np.argsort([len(c) for c in chunks])
    d = int(enc.config.hidden_size); Z = np.zeros((len(chunks), d), np.float32)
    bs = 4 if long_ else (64 if d <= 768 else 32)
    for s in range(0, len(order), bs):
        ix = order[s: s + bs]; L = max(len(chunks[j]) for j in ix)
        inp = torch.full((len(ix), L), tok.pad_token_id, dtype=torch.long); att = torch.zeros((len(ix), L), dtype=torch.long)
        for r, j in enumerate(ix):
            inp[r, : len(chunks[j])] = torch.tensor(chunks[j]); att[r, : len(chunks[j])] = 1
        inp, att = inp.to(DEV), att.to(DEV)
        kw = {}
        if getattr(enc.config, "model_type", "") == "longformer":
            g = torch.zeros_like(att); g[:, 0] = 1; kw["global_attention_mask"] = g
        h = enc(input_ids=inp, attention_mask=att, **kw).last_hidden_state.float()
        m = att.unsqueeze(-1).float(); Z[ix] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    mean_emb = np.zeros((N_EV, d), np.float32); cnt = np.bincount(owner, minlength=N_EV).astype(np.float32)
    np.add.at(mean_emb, owner, Z); mean_emb /= np.maximum(cnt, 1)[:, None]
    first_emb = np.zeros((N_EV, d), np.float32); first_emb[owner[first]] = Z[first]
    n_par = sum(p.numel() for p in enc.parameters())
    del enc; torch.cuda.empty_cache()
    return mean_emb, first_emb, n_par, float(np.mean(cnt))

def text_pca(E, fit_mask, k):
    fr = np.flatnonzero(fit_mask); pca = PCA(min(k, len(fr) - 1, E.shape[1]), random_state=0).fit(E[fr])
    return pca.transform(E).astype(np.float32), pca

TEXT_PCA, TEXT_PCAOBJ = {}, {}
for m in ENCODERS:
    spec = NLPC[m]; fpath = SUB["features"] / f"textemb_{m}_{N_EV}_{TEXT_KEY}.npz"
    st = Stage(f"text_emb_{m}", [spec, NCH, CL, N_EV, TEXT_KEY], [fpath])
    try:
        if not (st.done() and fpath.exists()):
            if not HAS_GPU:
                not_run(f"{m}_frozen", "no GPU"); continue
            t0 = time.time(); E, E1, npar, nch = encode_frozen(spec)
            np.savez_compressed(fpath, mean=E.astype(np.float16), first=E1.astype(np.float16))
            st.finish(minutes=round((time.time() - t0) / 60, 2), params_m=round(npar / 1e6, 1), mean_chunks=nch)
        z = np.load(fpath); TEXT_EMB[m] = z["mean"].astype(np.float32)
        TEXT_PCA[m], TEXT_PCAOBJ[m] = text_pca(TEXT_EMB[m], EVENT.mask["train"], CFG["probe"]["pca_text"])
        if not have_pred("event", f"{m}_frozen") or FORCE_RERUN:
            finalize(EVENT, f"{m}_frozen", fit_heads(EVENT, TEXT_PCA[m], EVENT.mask["train"], EVENT.mask["inner"], "linear"), "nlp_frozen",
                     encoder=m, label=spec["label"] + " (frozen)")
        if spec["max_len"] <= 512 and (not have_pred("event", f"{m}_frozen_1chunk") or FORCE_RERUN):
            Z1, _ = text_pca(z["first"].astype(np.float32), EVENT.mask["train"], CFG["probe"]["pca_text"])
            finalize(EVENT, f"{m}_frozen_1chunk", fit_heads(EVENT, Z1, EVENT.mask["train"], EVENT.mask["inner"], "linear"),
                     "nlp_length_ablation", encoder=m, note="first 512 tokens only")
        print(f"{m}_frozen: val primary (20d)", {t: round(v, 3) for t, v in REG[f"{m}_frozen"]["val_primary"].items() if v is not None and t.endswith("_20")})
    except Exception as e:
        failed(f"{m}_frozen", e)
    gc.collect(); torch.cuda.empty_cache() if HAS_GPU else None

In [ ]:
# ---------------------------------------------------------------- FinGPT (Llama-2-7B + FinGPT LoRA), 4-bit, frozen
FINGPT_PROMPT = "Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.\nInput: {text}\nAnswer:"
def load_fingpt(trainable=False):
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    from huggingface_hub import hf_hub_download
    spec = NLPC["fingpt"]
    ac = json.loads(Path(hf_hub_download(spec["hf"], "adapter_config.json", token=HF_TOKEN)).read_text())
    base = str(ac.get("base_model_name_or_path") or "")
    if not base.startswith("meta-llama/"):          # the adapter records the authors' local path (e.g. base_models/Llama-2-7b-hf)
        base = spec["base"]
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(base, token=HF_TOKEN); tok.pad_token = tok.pad_token or tok.eos_token; tok.padding_side = "right"
    lm = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map={"": 0}, token=HF_TOKEN, torch_dtype=torch.bfloat16)
    if trainable:
        from peft import prepare_model_for_kbit_training
        lm = prepare_model_for_kbit_training(lm, use_gradient_checkpointing=True)
    lm = PeftModel.from_pretrained(lm, spec["hf"], token=HF_TOKEN, is_trainable=trainable)
    return tok, lm, base

def fingpt_inputs(tok, texts, max_len):
    budget = max_len - 48
    cut = [tok.decode(tok(t_[: 8 * budget], add_special_tokens=False, truncation=True, max_length=budget)["input_ids"]) for t_ in texts]
    return [FINGPT_PROMPT.replace("{text}", c) for c in cut]

@torch.no_grad()
def fingpt_embed(tok, lm, max_len=1024, bs=4):
    prompts = fingpt_inputs(tok, EV_TEXT, max_len)
    ans = [tok(w, add_special_tokens=False)["input_ids"][0] for w in ("negative", "neutral", "positive")]   # first sub-word ("▁neg...")
    E = np.zeros((N_EV, lm.config.hidden_size), np.float32); S = np.zeros((N_EV, 3), np.float32)
    order = np.argsort([len(p_) for p_ in prompts])
    for s in range(0, N_EV, bs):
        ix = order[s: s + bs]
        enc = tok([prompts[i] for i in ix], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(DEV)
        out = lm(**enc, output_hidden_states=True)
        h = out.hidden_states[-1].float(); m = enc["attention_mask"].unsqueeze(-1).float()
        E[ix] = ((h * m).sum(1) / m.sum(1)).cpu().numpy()
        last = enc["attention_mask"].sum(1) - 1
        lg = out.logits[torch.arange(len(ix), device=DEV), last][:, ans].float()
        S[ix] = torch.softmax(lg, -1).cpu().numpy()
    return E, S

def run_fingpt():
    mid = "fingpt_frozen"
    if not RUN_FINGPT:
        not_run(mid, "RUN_FINGPT=False"); return
    if not HAS_GPU or not HF_TOKEN:
        not_run(mid, "needs a GPU and HF_TOKEN with the Llama-2 licence accepted"); return
    fpath = SUB["features"] / f"textemb_fingpt_{N_EV}_{TEXT_KEY}.npz"
    st = Stage("text_emb_fingpt", [NLPC["fingpt"], N_EV, TEXT_KEY, FINGPT_PROMPT], [fpath])
    if not (st.done() and fpath.exists()):
        try:
            tok, lm, base = load_fingpt()
        except Exception as e:
            not_run(mid, f"could not load FinGPT/Llama-2 ({type(e).__name__}: {str(e)[:160]}). Accept the licence at "
                         "https://huggingface.co/meta-llama/Llama-2-7b-hf with the HF_TOKEN account"); return
        t0 = time.time(); E, S = fingpt_embed(tok, lm, NLPC["fingpt"]["max_len"])
        np.savez_compressed(fpath, mean=E.astype(np.float16), sentiment=S)
        st.finish(minutes=round((time.time() - t0) / 60, 2), base=base); del lm; torch.cuda.empty_cache()
    z = np.load(fpath); TEXT_EMB["fingpt"] = z["mean"].astype(np.float32)
    Zp, TEXT_PCAOBJ["fingpt"] = text_pca(TEXT_EMB["fingpt"], EVENT.mask["train"], CFG["probe"]["pca_text"])
    TEXT_PCA["fingpt"] = np.hstack([Zp, z["sentiment"].astype(np.float32)])
    finalize(EVENT, mid, fit_heads(EVENT, TEXT_PCA["fingpt"], EVENT.mask["train"], EVENT.mask["inner"], "linear"), "nlp_frozen",
             encoder="fingpt", label=NLPC["fingpt"]["label"] + " (frozen, 4-bit)", params_m=6738)
    FINGPT_SENT = pd.DataFrame(z["sentiment"], columns=["p_negative", "p_neutral", "p_positive"]).assign(form=EV.form.values, block=EV.block.values)
    FINGPT_SENT.groupby(["form"])[["p_negative", "p_neutral", "p_positive"]].mean().to_csv(SUB["tables"] / "fingpt_sentiment_by_form.csv")
try:
    run_fingpt()
except Exception as e:
    failed("fingpt_frozen", e)
gc.collect(); torch.cuda.empty_cache() if HAS_GPU else None

In [ ]:
# ---------------------------------------------------------------- FinBERT sentence tone (for Q8): mean P(positive) - P(negative) of the first 40 sentences
TONE_PATH = SUB["features"] / f"finbert_tone_{N_EV}_{TEXT_KEY}.npy"
if TONE_PATH.exists():
    TONE = np.load(TONE_PATH)
elif HAS_GPU:
    tk = AutoTokenizer.from_pretrained(NLPC["finbert"]["hf"])
    clf = AutoModelForSequenceClassification.from_pretrained(NLPC["finbert"]["hf"]).to(DEV).eval()
    lab = {v.lower(): int(k) for k, v in clf.config.id2label.items()}
    sents, own = [], []
    for i, tx in enumerate(EV_TEXT):
        ss = [s_ for s_ in re.split(r"(?<=[\.\!\?])\s+", tx[:20000]) if 30 <= len(s_) <= 600][:40]
        sents += ss; own += [i] * len(ss)
    own = np.array(own); pr = np.zeros((len(sents), 3), np.float32)
    with torch.no_grad():
        for s in range(0, len(sents), 256):
            enc = tk(sents[s: s + 256], truncation=True, max_length=96, padding=True, return_tensors="pt").to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                pr[s: s + 256] = torch.softmax(clf(**enc).logits.float(), -1).cpu().numpy()
    net_ = pr[:, lab.get("positive", 0)] - pr[:, lab.get("negative", 1)]
    TONE = np.full(N_EV, np.nan, np.float32)
    if len(own):
        sums = np.bincount(own, weights=net_, minlength=N_EV); cnts = np.bincount(own, minlength=N_EV)
        TONE = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan).astype(np.float32)
    np.save(TONE_PATH, TONE); del clf; torch.cuda.empty_cache()
else:
    TONE = np.full(N_EV, np.nan, np.float32)
EV["finbert_tone"] = TONE
tick("16 nlp frozen")

## 17. Language models, part 2: fine-tuning on filing events

### 17.1 Training strategy (same for all five encoders)
Input `[form] [section] text`, truncated to the model's window (512 tokens; Longformer 4,096). Attention-masked mean pooling, the same multi-task head and loss as the CV models (direction CE + 0.5 × volatility MSE + 0.5 × drawdown BCE, 4 horizons). AdamW (weight decay 0.01), encoder learning rate from the probe, head 1e-3, 6 % warm-up then linear-cosine decay, bfloat16, gradient clipping 1.0, gradient checkpointing for the large and long models, effective batch 16 (micro-batch × accumulation). Up to 3 epochs with early stopping on the **inner-block filings** (patience 1). Fine-tuning a 355-435 M-parameter model on a few thousand noisy labels can over-fit fast: the probe, early stopping and the frozen comparison in §18 are the guard.

### 17.2 Hyperparameter tuning
Two encoder learning rates per model (typical ranges: base 2-3e-5, large 0.8-2e-5), one epoch each on half of the training filings, chosen by inner loss.

In [ ]:
NFT = CFG["nlp_ft"]
VOLE_MU = np.array([np.nanmean(YE[f"vol_{h}"][EVENT.mask["train"]]) for h in HORIZONS])
VOLE_SD = np.array([np.nanstd(YE[f"vol_{h}"][EVENT.mask["train"]]) for h in HORIZONS])
YE_DIR = np.stack([YE[f"dir_{h}"] for h in HORIZONS], 1).astype(np.float32)
YE_VOLZ = ((np.stack([YE[f"vol_{h}"] for h in HORIZONS], 1) - VOLE_MU) / VOLE_SD).astype(np.float32)
YE_DD = np.stack([YE[f"dd_{h}"] for h in HORIZONS], 1).astype(np.float32)
ROWS_TR_EV = np.flatnonzero(EVENT.mask["train"] & np.isfinite(YE_DIR).all(1))
ROWS_IN_EV = np.flatnonzero(EVENT.mask["inner"] & np.isfinite(YE_DIR).all(1))
NLP_FT_IDS = {m: f"{m}_ft" for m in ENCODERS}
NLP_LR_PROBE = []

def ev_labels(rb):
    return tuple(torch.from_numpy(a[rb]).to(DEV) for a in (YE_DIR, YE_VOLZ, YE_DD))

def text_batches(tok, texts, rows, max_len, bs):
    for s in range(0, len(rows), bs):
        rb = rows[s: s + bs]
        enc = tok([texts[i] for i in rb], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        yield rb, enc["input_ids"].to(DEV), enc["attention_mask"].to(DEV)

@torch.no_grad()
def text_eval_loss(net, tok, texts, rows, max_len, bs, amp=True):
    net.eval(); tot, n = 0.0, 0
    for rb, ii, am in text_batches(tok, texts, rows, max_len, bs * 2):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=amp):
            o = net(ii, am)
        tot += float(mt_loss(o, *ev_labels(rb))) * len(rb); n += len(rb)
    return tot / max(1, n)

def train_textnet(spec, lr, rows_tr, rows_es, epochs, seed=0, tag=""):
    set_seed(seed); rng = np.random.default_rng(seed)
    tok = AutoTokenizer.from_pretrained(spec["hf"])
    big = spec["max_len"] > 512 or "large" in spec["hf"]
    net = TextNet(spec["hf"], grad_ckpt=big).to(DEV)
    texts = [t_[: spec["max_len"] * 8] for t_ in EV_TEXT]
    groups = [{"params": [p for n_, p in net.named_parameters() if not n_.startswith("head.")], "lr": lr},
              {"params": net.head.parameters(), "lr": NFT["lr_head"]}]
    opt = torch.optim.AdamW(groups, weight_decay=NFT["wd"])
    amp = spec.get("amp", True)                               # DeBERTa-v3 trains in full precision (unstable in 16-bit)
    scaler = torch.amp.GradScaler("cuda", enabled=amp and AMP_DTYPE == torch.float16)
    bs, acc = spec["batch"], spec["accum"]; n_bad = 0
    total = max(1, epochs * math.ceil(len(rows_tr) / (bs * acc))); warm = max(1, int(NFT["warmup_frac"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    hist, best, best_state, bad, t0 = [], np.inf, None, 0, time.time()
    for ep in range(epochs):
        net.train(); run, k, j = 0.0, 0, 0
        opt.zero_grad(set_to_none=True)
        for rb, ii, am in text_batches(tok, texts, rng.permutation(rows_tr), spec["max_len"], bs):
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=amp):
                loss = mt_loss(net(ii, am), *ev_labels(rb)) / acc
            if not torch.isfinite(loss):                       # numerical blow-up: skip the batch, count it, stop if persistent
                n_bad += 1; opt.zero_grad(set_to_none=True)
                if n_bad > 20:
                    raise RuntimeError(f"{tag}: non-finite loss in more than 20 batches (training diverged)")
                continue
            scaler.scale(loss).backward(); run += float(loss) * acc; k += 1; j += 1
            if j % acc == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
        if j % acc:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
        es = text_eval_loss(net, tok, texts, rows_es, spec["max_len"], bs, amp)
        if not np.isfinite(es):
            raise RuntimeError(f"{tag}: inner-block loss is not finite (training diverged)")
        hist.append({"epoch": ep + 1, "train_loss": run / max(1, k), "inner_loss": es, "minutes": round((time.time() - t0) / 60, 2)})
        print(f"  {tag} epoch {ep + 1}/{epochs}: train {run / max(1, k):.4f} | inner {es:.4f} | {hist[-1]['minutes']} min")
        if es < best - 1e-5:
            best, bad = es, 0; best_state = {k_: v.detach().to("cpu", copy=True) for k_, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= NFT["patience"]:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    return net, tok, texts, hist, best, round((time.time() - t0) / 60, 2)

def run_nlp_finetune(m):
    spec, mid = NLPC[m], NLP_FT_IDS[m]; mpath = SUB["models"] / f"nlp_{mid}.pt"
    st = Stage(f"nlp_{mid}", [spec, NFT, N_EV, TEXT_KEY, len(ROWS_TR_EV)], [mpath, _pred_path("event", mid)])
    if st.done() and have_pred("event", mid):
        return
    if not HAS_GPU:
        not_run(mid, "no GPU"); return
    try:
        if QUICK or len(spec["lr_grid"]) == 1:
            lr = spec["lr_grid"][0]
        else:
            sub = np.sort(np.random.default_rng(2).choice(ROWS_TR_EV, int(NFT["probe_frac"] * len(ROWS_TR_EV)), replace=False))
            losses = {}
            for lr_ in spec["lr_grid"]:
                net, *_, l_, mins = train_textnet(spec, lr_, sub, ROWS_IN_EV, 1, tag=f"{mid} probe lr={lr_:g}")
                losses[lr_] = l_; NLP_LR_PROBE.append({"model": mid, "lr": lr_, "inner_loss": l_, "minutes": mins}); del net; torch.cuda.empty_cache()
            lr = min(losses, key=lambda z: losses[z])
        torch.cuda.reset_peak_memory_stats()
        net, tok, texts, hist, best, mins = train_textnet(spec, lr, ROWS_TR_EV, ROWS_IN_EV, NFT["epochs"], tag=mid)
        torch.save({"state_dict": {k: v.half() if v.is_floating_point() else v for k, v in net.state_dict().items()}, "hf": spec["hf"],
                    "max_len": spec["max_len"], "vol_mu": VOLE_MU, "vol_sd": VOLE_SD, "lr": lr, "history": hist, "amp": spec.get("amp", True)}, mpath)
        t1 = time.time()
        raw = text_raw_outputs(net, tok, texts, spec["max_len"], VOLE_MU, VOLE_SD, DEV, batch=spec["batch"] * 2, amp_dtype=AMP_DTYPE, use_amp=spec.get("amp", True))
        docs_per_s = N_EV / max(1e-6, time.time() - t1)
        finalize(EVENT, mid, raw, "nlp_ft", encoder=m, lr=lr, history=hist, best_inner_loss=best, train_minutes=mins,
                 peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2), params_m=round(sum(p.numel() for p in net.parameters()) / 1e6, 1),
                 docs_per_s=round(docs_per_s, 1), max_len=spec["max_len"], label=spec["label"] + " (fine-tuned)")
        st.finish()
        print(f"{mid}: val primary (20d)", {t: round(v, 3) for t, v in REG[mid]["val_primary"].items() if v is not None and t.endswith("_20")})
        del net
    except Exception as e:
        failed(mid, e)
    gc.collect(); torch.cuda.empty_cache()

for m in ENCODERS:
    run_nlp_finetune(m)
if NLP_LR_PROBE:
    pd.DataFrame(NLP_LR_PROBE).to_csv(SUB["tables"] / "nlp_lr_probe.csv", index=False); display(pd.DataFrame(NLP_LR_PROBE))

In [ ]:
# ---------------------------------------------------------------- optional: continue training FinGPT's LoRA (QLoRA) with the multi-task head
class FinGPTMT(nn.Module):
    def __init__(self, lm):
        super().__init__(); self.lm = lm; self.head = MultiTaskHead(int(lm.config.hidden_size)).to(DEV)
    def forward(self, input_ids, attention_mask):
        h = self.lm(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True).hidden_states[-1].float()
        m = attention_mask.unsqueeze(-1).float(); return self.head((h * m).sum(1) / m.sum(1).clamp(min=1))
if FINGPT_QLORA and RUN_FINGPT and HAS_GPU and REG.get("fingpt_frozen", {}).get("status") == "RUN" and not have_pred("event", "fingpt_qlora"):
    try:
        tok_g, lm_g, _ = load_fingpt(trainable=True); net_g = FinGPTMT(lm_g)
        prom = fingpt_inputs(tok_g, EV_TEXT, NLPC["fingpt"]["max_len"])
        opt = torch.optim.AdamW([{"params": [p for p in lm_g.parameters() if p.requires_grad], "lr": 1e-4},
                                 {"params": net_g.head.parameters(), "lr": 1e-3}], weight_decay=0.0)
        rng = np.random.default_rng(0); t0 = time.time(); net_g.train(); j = 0
        for rb, ii, am in text_batches(tok_g, prom, rng.permutation(ROWS_TR_EV), NLPC["fingpt"]["max_len"], 2):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = mt_loss(net_g(ii, am), *ev_labels(rb)) / 8
            loss.backward(); j += 1
            if j % 8 == 0:
                torch.nn.utils.clip_grad_norm_([p for p in net_g.parameters() if p.requires_grad], 1.0); opt.step(); opt.zero_grad(set_to_none=True)
        net_g.eval(); res = {"dir": [], "vol": [], "dd": []}
        with torch.no_grad():
            for rb, ii, am in text_batches(tok_g, prom, np.arange(N_EV), NLPC["fingpt"]["max_len"], 4):
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    o = net_g(ii, am)
                for k_ in res:
                    res[k_].append(o[k_].float().cpu().numpy())
        d_, v_, q_ = (np.concatenate(res[k_]) for k_ in ("dir", "vol", "dd"))
        raw = {**{f"dir_{h}": d_[:, i] for i, h in enumerate(HORIZONS)}, **{f"vol_{h}": v_[:, i] * VOLE_SD[i] + VOLE_MU[i] for i, h in enumerate(HORIZONS)},
               **{f"dd_{h}": q_[:, i] for i, h in enumerate(HORIZONS)}}
        finalize(EVENT, "fingpt_qlora", raw, "nlp_ft", encoder="fingpt", train_minutes=round((time.time() - t0) / 60, 1),
                 label="FinGPT (QLoRA, 1 epoch)", params_m=6738)
        lm_g.save_pretrained(str(SUB["models"] / "fingpt_qlora_adapter")); del net_g, lm_g
    except Exception as e:
        failed("fingpt_qlora", e)
    gc.collect(); torch.cuda.empty_cache()

### 17.4 Training curves

In [ ]:
_nft = [m for m in NLP_FT_IDS.values() if REG.get(m, {}).get("history")]
if _nft:
    fig, axes = plt.subplots(1, len(_nft), figsize=(3.2 * len(_nft), 2.8), squeeze=False)
    for a_, m in zip(axes[0], _nft):
        h_ = pd.DataFrame(REG[m]["history"])
        a_.plot(h_.epoch, h_.train_loss, "o-", label="train"); a_.plot(h_.epoch, h_.inner_loss, "s-", label="inner")
        a_.set_title(m, fontsize=8); a_.grid(alpha=0.3)
    axes[0][0].legend(fontsize=7); plt.tight_layout(); savefig(fig, "nlp_training_curves.png"); plt.show()
    pd.concat([pd.DataFrame(REG[m]["history"]).assign(model=m) for m in _nft]).to_csv(SUB["tables"] / "nlp_training_curves.csv", index=False)
tick("17 nlp finetune")

## 18. NLP analysis on validation filings (2022-2023)

### 18.1 Leaderboard, best model, and skill by filing type (Q6) and sector (Q7, Q9)

In [ ]:
NLP_MAIN = [m for m in list(NLP_FT_IDS.values()) + ["fingpt_frozen", "fingpt_qlora", "tfidf_lr"] if REG.get(m, {}).get("status") == "RUN"]
NLP_OTHER = [m for m in [f"{e}_frozen" for e in ENCODERS] + [f"{e}_frozen_1chunk" for e in ENCODERS] + ["prior_event"]
             if REG.get(m, {}).get("status") == "RUN"]
LABEL_OF.update({m: REG.get(m, {}).get("label", m) for m in REG})
LB_NLP_VAL = leaderboard(EVENT, NLP_MAIN + NLP_OTHER, EVENT.mask["val"]); LB_NLP_VAL.to_csv(SUB["tables"] / "nlp_leaderboard_val.csv", index=False)
PIV_NLP = LB_NLP_VAL[LB_NLP_VAL.status == "RUN"].pivot_table(index="model", columns="task", values="primary")
PIV_NLP = PIV_NLP[[t for t in TASKS if t in PIV_NLP.columns]]
NLP_MAIN = [m for m in NLP_MAIN if m in PIV_NLP.index]
display(PIV_NLP.round(3))
NLP_RANK = mean_rank(LB_NLP_VAL[LB_NLP_VAL.model.isin(NLP_MAIN)])
NLP_BEST = NLP_RANK.index[0] if len(NLP_RANK) else None
BEST_NLP_TASK = {t: PIV_NLP.loc[NLP_MAIN, t].idxmax() for t in TASKS if t in PIV_NLP and PIV_NLP.loc[NLP_MAIN, t].notna().any()} if NLP_MAIN else {}
display(NLP_RANK.round(2).to_frame("mean_rank")); NLP_RANK.to_frame("mean_rank").to_csv(SUB["tables"] / "nlp_rank_val.csv")
SKILL_NLP_VAL = pd.DataFrame([{"task": t, "model": m, **skill_test(EVENT, t, load_pred("event", m)[t], EVENT.mask["val"])} for t, m in BEST_NLP_TASK.items()])
if len(SKILL_NLP_VAL):
    SKILL_NLP_VAL.to_csv(SUB["tables"] / "nlp_skill_val.csv", index=False); display(SKILL_NLP_VAL.round(4))
WHY_NLP = why_best(NLP_BEST, NLP_RANK, PIV_NLP, NLP_MAIN) if NLP_BEST else "No NLP model ran."
display(Markdown(WHY_NLP)); (SUB["results"] / "why_nlp_best_val.md").write_text(WHY_NLP)
if NLP_MAIN:
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(PIV_NLP) + 1.5))
    Zr = PIV_NLP.rank(pct=True)
    ax.imshow(Zr.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    for i in range(PIV_NLP.shape[0]):
        for j in range(PIV_NLP.shape[1]):
            v = PIV_NLP.values[i, j]; ax.text(j, i, "" if not np.isfinite(v) else f"{v:.3f}", ha="center", va="center", fontsize=6.5)
    ax.set_xticks(range(PIV_NLP.shape[1])); ax.set_xticklabels(PIV_NLP.columns, rotation=45, fontsize=7)
    ax.set_yticks(range(len(PIV_NLP))); ax.set_yticklabels(PIV_NLP.index, fontsize=7)
    ax.set_title("NLP validation primary metric on filing events"); plt.tight_layout(); savefig(fig, "nlp_leaderboard_val.png"); plt.show()

def by_group(ctx, models, mask, col_values, group_name):
    rows = []
    for gval in sorted(set(col_values[mask])):
        mg = mask & (col_values == gval)
        for m in models:
            pr = load_pred(ctx.name, m)
            for t in TASKS:
                r = ctx.rows(t, mg, pr[t])
                rows.append({group_name: gval, "model": m, "task": t, "target": t.split("_")[0], "n": len(r),
                             "primary": primary_value(t, ctx.Y[t][r], pr[t][r]) if len(r) >= 25 else np.nan})
    return pd.DataFrame(rows)
if NLP_MAIN:
    NLP_FORM_VAL = by_group(EVENT, NLP_MAIN, EVENT.mask["val"], EV["form"].values, "form"); NLP_FORM_VAL.to_csv(SUB["tables"] / "nlp_by_form_val.csv", index=False)
    display(NLP_FORM_VAL.groupby(["form", "target"])["primary"].max().unstack().round(3))
    NLP_SECTOR_VAL = by_group(EVENT, NLP_MAIN, EVENT.mask["val"], EV["sector"].values, "sector"); NLP_SECTOR_VAL.to_csv(SUB["tables"] / "nlp_by_sector_val.csv", index=False)
    _sm = NLP_SECTOR_VAL.groupby(["sector", "target", "model"])["primary"].mean().reset_index()
    BEST_NLP_BY_SECTOR_VAL = best_rows(_sm, ["sector", "target"]).pivot(index="sector", columns="target", values="model")
    display(BEST_NLP_BY_SECTOR_VAL); BEST_NLP_BY_SECTOR_VAL.to_csv(SUB["tables"] / "nlp_best_by_sector_val.csv")

### 18.2 Ablations: fine-tuned vs frozen, domain pre-training, length

In [ ]:
NABL = []
pairs = [(f"{e}_ft", f"{e}_frozen", "fine-tuned vs frozen") for e in ENCODERS] + [(f"{e}_frozen", f"{e}_frozen_1chunk", "4 chunks vs first 512 tokens") for e in ENCODERS] + [
    ("finbert_ft", "roberta_l_ft", "FinBERT (finance, base) vs RoBERTa-large (general, large)"), ("secbert_ft", "finbert_ft", "SEC-BERT vs FinBERT (domain)"),
    ("longformer_ft", "finbert_ft", "Longformer 4096 vs FinBERT 512 (length)"), ((NLP_BEST or "tfidf_lr"), "tfidf_lr", "best NLP vs TF-IDF baseline")]
for a, b, what in pairs:
    if a == b or REG.get(a, {}).get("status") != "RUN" or REG.get(b, {}).get("status") != "RUN":
        continue
    pa, pb = load_pred("event", a), load_pred("event", b)
    for kd in ("dir", "vol", "dd"):
        NABL.append({"comparison": what, "A": a, "B": b, "target": kd, **paired_delta_family(EVENT, kd, pa, pb, EVENT.mask["val"])})
NLP_ABL_VAL = pd.DataFrame(NABL)
if len(NLP_ABL_VAL):
    NLP_ABL_VAL["p_holm"] = holm(NLP_ABL_VAL["p"].values); NLP_ABL_VAL.to_csv(SUB["tables"] / "nlp_ablations_val.csv", index=False); display(NLP_ABL_VAL.round(4))

### 18.3 What in the filings goes with market moves (Q8)
Three complementary views (development filings): **(a)** 8-K item types and their mean |z| (§14), **(b)** n-grams associated with large or signed reactions (§15), **(c)** FinBERT sentence tone vs the reaction, and **(d)** integrated-gradient token attributions of the best fine-tuned encoder for a few large-reaction validation filings. All are associations, not causes.

In [ ]:
TONE_ROWS = []
dev_m = EV.block.isin(["train", "inner", "val"]).values
for f_ in CFG["sec"]["forms"] + ["all"]:
    m = dev_m & np.isfinite(EV["finbert_tone"].values) & np.isfinite(EV["ar2"].values) & ((EV.form == f_).values if f_ != "all" else True)
    if m.sum() >= 30:
        TONE_ROWS.append({"form": f_, "n": int(m.sum()), "spearman_tone_vs_signed_AR": float(spearmanr(EV.finbert_tone[m], EV.ar2[m])[0]),
                          "spearman_tone_vs_abs_z": float(spearmanr(EV.finbert_tone[m], EV.abs_z[m])[0]),
                          "mean_tone": float(EV.finbert_tone[m].mean())})
TONE_TAB = pd.DataFrame(TONE_ROWS); TONE_TAB.to_csv(SUB["tables"] / "nlp_tone_vs_reaction_dev.csv", index=False); display(TONE_TAB.round(3))

def integrated_gradients(net, tok, text, max_len, task="vol", h_index=2, steps=24):
    enc = tok([text], truncation=True, max_length=min(max_len, 512), return_tensors="pt").to(DEV)
    emb = net.enc.get_input_embeddings()(enc["input_ids"]).detach().float()
    tot = torch.zeros_like(emb)
    for a in torch.linspace(1.0 / steps, 1.0, steps):
        x = (a * emb).requires_grad_(True)
        o = net.head(net.pooled(attention_mask=enc["attention_mask"], inputs_embeds=x.to(next(net.parameters()).dtype)))
        y = o["vol"][0, h_index] if task == "vol" else o["dir"][0, h_index, 2] - o["dir"][0, h_index, 0]
        g, = torch.autograd.grad(y, x); tot += g
    attr = (emb * tot / steps).sum(-1)[0].detach().cpu().numpy()
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
    return toks, attr
IG_ROWS = []
ig_model = next((m for m in NLP_RANK.index if m.endswith("_ft") and m.replace("_ft", "") in ENCODERS), None) if len(NLP_RANK) else None
if ig_model and HAS_GPU:
    try:
        ck = torch.load(SUB["models"] / f"nlp_{ig_model}.pt", map_location="cpu", weights_only=False)
        net = TextNet(ck["hf"], from_config=True); net.load_state_dict({k: v.float() for k, v in ck["state_dict"].items()}); net = net.to(DEV).eval()
        tok = AutoTokenizer.from_pretrained(ck["hf"])
        big = EV[EVENT.mask["val"]].sort_values("abs_z", ascending=False).head(4)
        for i, r_ in big.iterrows():
            for task in ("vol", "dir"):
                toks, attr = integrated_gradients(net, tok, EV_TEXT[i], ck["max_len"], task)
                o = np.argsort(-attr)[:12]
                IG_ROWS.append({"event": f"{r_.ticker} {r_.form} {r_.entry_date.date()}", "abs_z": round(float(r_.abs_z), 2), "output": task,
                                "top_tokens_raising_output": " ".join(t_.replace("\u2581", "").replace("\u0120", "") for t_ in np.array(toks)[o] if t_ not in ("[CLS]", "[SEP]", "<s>", "</s>"))})
        del net; torch.cuda.empty_cache()
    except Exception as e:
        print("Integrated gradients skipped:", type(e).__name__, e)
IG_TAB = pd.DataFrame(IG_ROWS)
if len(IG_TAB):
    IG_TAB.to_csv(SUB["tables"] / "nlp_integrated_gradients_val.csv", index=False); display(IG_TAB)

### 18.4 Error analysis (best NLP model, validation)

In [ ]:
if NLP_BEST:
    pr = load_pred("event", NLP_BEST); vm = EVENT.mask["val"]
    r = EVENT.rows("dir_20", vm, pr["dir_20"])
    E2 = pd.DataFrame({"correct": pr["dir_20"][r].argmax(1) == YE["dir_20"][r], "form": EV.form.values[r], "sector": EV.sector.values[r],
                       "timing": EV.session_bucket.values[r], "length_decile": pd.qcut(np.log1p(FIL.section_chars.values[EV.fil_id.values[r]]), 5, labels=False, duplicates="drop")})
    NLP_ERR = {k: E2.groupby(k)["correct"].agg(["mean", "size"]).rename(columns={"mean": "accuracy"}) for k in ["form", "sector", "timing", "length_decile"]}
    for k, v in NLP_ERR.items():
        v.to_csv(SUB["tables"] / f"nlp_errors_by_{k}_val.csv")
    fig, axes = plt.subplots(1, 5, figsize=(19, 3.2))
    for a_, k in zip(axes[:4], NLP_ERR):
        NLP_ERR[k]["accuracy"].plot.bar(ax=a_, color="#7c3aed"); a_.axhline(E2.correct.mean(), color="k", ls="--", lw=0.8); a_.set_title(f"20-day accuracy by {k}", fontsize=8)
    cm = confusion_matrix(YE["dir_20"][r].astype(int), pr["dir_20"][r].argmax(1), labels=[0, 1, 2], normalize="true")
    axes[4].imshow(cm, cmap="Purples", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            axes[4].text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8)
    axes[4].set_xticks(range(3)); axes[4].set_xticklabels(CLASSES, fontsize=7); axes[4].set_yticks(range(3)); axes[4].set_yticklabels(CLASSES, fontsize=7)
    axes[4].set_title(f"{NLP_BEST}: 20-day confusion", fontsize=8)
    plt.tight_layout(); savefig(fig, "nlp_errors_val.png"); plt.show()

### 18.5 Efficiency

In [ ]:
EFF_NLP = pd.DataFrame([{"model": m, "params_m": REG[m].get("params_m"), "max_tokens": REG[m].get("max_len", NLPC.get(REG[m].get("encoder", ""), {}).get("max_len")),
                         "train_minutes": REG[m].get("train_minutes"), "docs_per_s_inference": REG[m].get("docs_per_s"),
                         "val_mean_rank": round(float(NLP_RANK.get(m, np.nan)), 2)} for m in NLP_MAIN])
EFF_NLP.to_csv(SUB["tables"] / "nlp_efficiency.csv", index=False); display(EFF_NLP)
tick("18 nlp analysis")

## 19. Text on the decision panel (so CV, NLP and fusion are compared on identical rows)

For each panel row (stock, decision close *t*), the **latest periodic report** (10-K or 10-Q) and the **latest 8-K** whose availability is before 16:00 New York on *t*; older than 90 / 20 sessions counts as "no filing". Age = sessions since the filing became tradeable.

Two kinds of panel NLP model:
* **Frozen panel probes** `<model>_panel_frozen`: linear models on the panel training rows using 32 principal components of the latest periodic filing's embedding, 32 of the latest 8-K's (zero if none), flags and log-ages. Embeddings carry no labels, so training rows are clean.
* **Fine-tuned transfers** `<model>_panel_ft` (also TF-IDF): the event model's prediction for the most recent filing, used while the filing's own horizon is still open (age ≤ h), otherwise the training base rate. Only filings that are themselves outside the training block are used (their predictions are out-of-sample). Then calibrated on the inner block like every model.

**Fresh-text rows** (a filing in the last 5 sessions) mimic the dashboard, where the user pastes a recent filing summary.

In [ ]:
dec_utc = (pd.DatetimeIndex(P["date"]) + CLOSE_T).tz_localize(NY).tz_convert("UTC").tz_localize(None).values
EV_AVAIL = np.array([FIL.at[i, "available_utc"].tz_convert("UTC").tz_localize(None).to_datetime64() for i in EV.fil_id], dtype="datetime64[ns]")
IS_PER = EV.form.isin(["10-K", "10-Q"]).values
LATEST = {"per": np.full(N, -1), "8k": np.full(N, -1)}; AGE = {"per": np.full(N, np.nan), "8k": np.full(N, np.nan)}
for t in TICKERS:
    pr_rows = np.flatnonzero((P.ticker == t).values)
    for key, msk, lim in (("per", IS_PER, CFG["sec"]["lookback_sessions"]["periodic"]), ("8k", ~IS_PER, CFG["sec"]["lookback_sessions"]["8-K"])):
        ev_i = np.flatnonzero((EV.ticker == t).values & msk)
        if not len(ev_i):
            continue
        o = ev_i[np.argsort(EV_AVAIL[ev_i])]; av = EV_AVAIL[o]
        k = np.searchsorted(av, dec_utc[pr_rows], side="right") - 1
        ok = k >= 0
        cand = np.where(ok, o[np.clip(k, 0, None)], -1)
        age = np.where(ok, P["g"].values[pr_rows] - EV["g"].values[np.clip(cand, 0, None)], np.nan)
        ok &= (age >= 0) & (age <= lim)
        LATEST[key][pr_rows] = np.where(ok, cand, -1); AGE[key][pr_rows] = np.where(ok, age, np.nan)
HAS_PER, HAS_8K = LATEST["per"] >= 0, LATEST["8k"] >= 0
age_any = np.fmin(np.where(HAS_PER, AGE["per"], np.inf), np.where(HAS_8K, AGE["8k"], np.inf))
FRESH = np.isfinite(age_any) & (age_any <= 5)
COVER = pd.DataFrame([{"block": b, "rows": int(PANEL.mask[b].sum()), "has_periodic": float(HAS_PER[PANEL.mask[b]].mean()),
                       "has_8k_20s": float(HAS_8K[PANEL.mask[b]].mean()), "fresh_text_5s": float(FRESH[PANEL.mask[b]].mean())} for b in BLOCK_ORDER])
COVER.to_csv(SUB["tables"] / "panel_text_coverage.csv", index=False); display(COVER.round(3))
# leakage check: every attached filing was available before the decision close
for key in ("per", "8k"):
    rr = np.flatnonzero(LATEST[key] >= 0)
    assert (EV_AVAIL[LATEST[key][rr]] <= dec_utc[rr]).all(), f"a {key} filing is attached before it was available"
print("Panel text attachment passed the availability check.")

def panel_text_features(m):
    Z = TEXT_PCA[m]; k = Z.shape[1]
    Xp = np.where(HAS_PER[:, None], Z[np.clip(LATEST["per"], 0, None)], 0.0)
    X8 = np.where(HAS_8K[:, None], Z[np.clip(LATEST["8k"], 0, None)], 0.0)
    return np.hstack([Xp, X8, HAS_PER[:, None], HAS_8K[:, None], np.log1p(np.nan_to_num(AGE["per"], nan=99))[:, None],
                      np.log1p(np.nan_to_num(AGE["8k"], nan=99))[:, None]]).astype(np.float32)
NLP_PANEL_FROZEN = []
for m in list(TEXT_PCA):
    mid = f"{m}_panel_frozen"
    try:
        if not have_pred("panel", mid) or FORCE_RERUN:
            finalize(PANEL, mid, fit_heads(PANEL, panel_text_features(m), TR, IN, "linear"), "nlp_panel_frozen", encoder=m,
                     label=f"{LABEL_OF.get(m + '_frozen', m)} on the panel (frozen)")
        NLP_PANEL_FROZEN.append(mid)
    except Exception as e:
        failed(mid, e)

EV_BLOCK = EV["block"].values
def panel_transfer(event_mid):
    pe = load_pred("event", event_mid); raw = {}
    usable = ~np.isin(EV_BLOCK, ["train", "train_purged", "pre_study"])
    cand = np.where(HAS_8K & (~HAS_PER | (np.nan_to_num(AGE["8k"], nan=1e9) <= np.nan_to_num(AGE["per"], nan=1e9))), LATEST["8k"], LATEST["per"])
    age = np.where(cand == LATEST["8k"], AGE["8k"], AGE["per"])
    ok0 = (cand >= 0) & usable[np.clip(cand, 0, None)]
    for t in TASKS:
        h = int(t.split("_")[1]); ok = ok0 & (np.nan_to_num(age, nan=1e9) <= h)
        pri = PANEL.prior[t]
        if t.startswith("dir"):
            a = np.tile(np.asarray(pri, float), (N, 1)); a[ok] = pe[t][cand[ok]]
        else:
            a = np.full(N, float(pri)); a[ok] = pe[t][cand[ok]]
        raw[t] = a
    return raw_from_probs(raw)
NLP_PANEL_FT = []
for em in [m for m in NLP_MAIN if REG.get(m, {}).get("status") == "RUN"]:
    mid = em.replace("_ft", "").replace("_frozen", "").replace("_lr", "") + ("_panel_ft" if em.endswith("_ft") or em == "fingpt_qlora" else "_panel_transfer")
    try:
        finalize(PANEL, mid, panel_transfer(em), "nlp_panel_transfer", event_model=em, label=f"{LABEL_OF.get(em, em)} carried to the panel")
        NLP_PANEL_FT.append(mid)
    except Exception as e:
        failed(mid, e)
NLP_PANEL_ALL = NLP_PANEL_FROZEN + NLP_PANEL_FT
LB_NLPP_VAL = leaderboard(PANEL, NLP_PANEL_ALL, PANEL.mask["val"]); LB_NLPP_VAL.to_csv(SUB["tables"] / "nlp_panel_leaderboard_val.csv", index=False)
PIV_NLPP = LB_NLPP_VAL[LB_NLPP_VAL.status == "RUN"].pivot_table(index="model", columns="task", values="primary")
display(PIV_NLPP.round(3))
NLPP_RANK = mean_rank(LB_NLPP_VAL)
tick("19 panel text")

## 20. Fusion: does combining CV and NLP help? (Q1, Q10) — validation

| Id | Fusion | What is fitted |
|---|---|---|
| `fusion_late_top3` | weight × (top-3 CV ensemble) + (1 − weight) × (top-3 NLP panel ensemble) | the weight per task, from {0, .25, .5, .75, 1}, on validation |
| `fusion_late_best` | the same with the single best CV and best NLP model per task | the weight per task |
| `fusion_early` | one linear model per task on [best frozen CV features + best frozen NLP panel features] | linear model on training rows |
| `fusion_tech` (reference) | LightGBM on [technical indicators + the same CV and NLP features] | trees on training rows |
| `deploy_recipe` | what the dashboard runs: top-3 CV ensemble; on rows with a filing in the last 5 sessions, blended with the top-3 *deployable* NLP models (fine-tuned encoders ≤ 600 M parameters, or TF-IDF) | the fresh-text weight per task |

A weight of 1 means "CV only", 0 means "NLP only": the chosen weights are themselves an answer to Q1.

In [ ]:
def ens_panel(mid, members_by_task, family, **meta):
    return ensemble_model(PANEL, mid, members_by_task, family, **meta)
def choose_weights(pa, pb, mask, grid=None):
    grid = grid or CFG["fusion_w_grid"]; W = {}
    for t in TASKS:
        sc = []
        for w in grid:
            p_ = w * np.asarray(pa[t]) + (1 - w) * np.asarray(pb[t]); r = PANEL.rows(t, mask, p_)
            sc.append(primary_value(t, PANEL.Y[t][r], p_[r]) if len(r) >= 30 else -np.inf)
        W[t] = float(grid[int(np.argmax(sc))]) if np.isfinite(sc).any() else 1.0      # nothing scorable -> CV only
    return W
VM = PANEL.mask["val"]
BEST_NLPP_TASK = {t: PIV_NLPP[t].idxmax() for t in TASKS if t in PIV_NLPP and PIV_NLPP[t].notna().any()}
TOP3_NLPP = {t: (list(PIV_NLPP[t].sort_values(ascending=False).dropna().index[: CFG["top_k"]]) if t in PIV_NLPP else []) for t in TASKS}
FUSION_W = {}
if any(TOP3_NLPP.values()):
    ens_panel("nlp_panel_top3", TOP3_NLPP, "nlp_panel_ensemble", label="Top-3 NLP panel ensemble")
HAVE_CV3 = REG.get("cv_top3", {}).get("status") == "RUN"
if HAVE_CV3 and have_pred("panel", "nlp_panel_top3"):
    pa, pb = load_pred("panel", "cv_top3"), load_pred("panel", "nlp_panel_top3")
    FUSION_W["late_top3"] = choose_weights(pa, pb, VM)
    finalize(PANEL, "fusion_late_top3", raw_from_probs(blend(pa, pb, FUSION_W["late_top3"])), "fusion", weights=FUSION_W["late_top3"],
             label="Late fusion: top-3 CV + top-3 NLP")
if len(BEST_CV_TASK) == len(TASKS) and len(BEST_NLPP_TASK) == len(TASKS):
    pa = {t: load_pred("panel", BEST_CV_TASK[t])[t] for t in TASKS}; pb = {t: load_pred("panel", BEST_NLPP_TASK[t])[t] for t in TASKS}
    FUSION_W["late_best"] = choose_weights(pa, pb, VM)
    finalize(PANEL, "fusion_late_best", raw_from_probs(blend(pa, pb, FUSION_W["late_best"])), "fusion", weights=FUSION_W["late_best"],
             label="Late fusion: best CV + best NLP")
_cv_fz = [m for m in CV_OTHER if m.endswith("_frozen") and not m.startswith("resnet50_random")]
_r_cv = mean_rank(leaderboard(PANEL, _cv_fz, VM)) if _cv_fz else pd.Series(dtype=float)
_r_nl = mean_rank(leaderboard(PANEL, NLP_PANEL_FROZEN, VM)) if NLP_PANEL_FROZEN else pd.Series(dtype=float)
BEST_CV_FROZEN = _r_cv.index[0].replace("_frozen", "") if len(_r_cv) else None
BEST_NLP_FROZEN = _r_nl.index[0].replace("_panel_frozen", "") if len(_r_nl) else None
if BEST_CV_FROZEN in CV_PCA and BEST_NLP_FROZEN in TEXT_PCA:
    X_EARLY = np.hstack([CV_PCA[BEST_CV_FROZEN], panel_text_features(BEST_NLP_FROZEN)])
    finalize(PANEL, "fusion_early", fit_heads(PANEL, X_EARLY, TR, IN, "linear"), "fusion", cv=BEST_CV_FROZEN, nlp=BEST_NLP_FROZEN,
             label="Early fusion: frozen CV + frozen NLP features")
    st_ft = Stage("fusion_tech", [BEST_CV_FROZEN, BEST_NLP_FROZEN, N, GBM], [_pred_path("panel", "fusion_tech")])
    if not (st_ft.done() and have_pred("panel", "fusion_tech")):
        finalize(PANEL, "fusion_tech", fit_heads(PANEL, np.hstack([X_TECH, X_EARLY]), TR, IN, "gbm"), "reference",
                 label="Reference: technical + CV + NLP features (LightGBM)"); st_ft.finish()
# ---- the dashboard recipe
DEPLOYABLE_NLP = [m for m in NLP_MAIN if (m.endswith("_ft") and (REG[m].get("params_m") or 1e9) <= CFG["deploy_max_params_m"]) or m == "tfidf_lr"]
def _transfer_id(em):
    return em.replace("_ft", "").replace("_lr", "") + ("_panel_ft" if em.endswith("_ft") else "_panel_transfer")
NLP_DEPLOY_TOP3 = {}
if DEPLOYABLE_NLP:
    ev_piv = PIV_NLP.loc[[m for m in DEPLOYABLE_NLP if m in PIV_NLP.index]]
    NLP_DEPLOY_TOP3 = {t: (list(ev_piv[t].sort_values(ascending=False).dropna().index[: CFG["top_k"]]) if t in ev_piv.columns else []) for t in TASKS}
    ens_panel("nlp_deploy_top3", {t: [_transfer_id(m) for m in NLP_DEPLOY_TOP3[t]] for t in TASKS}, "nlp_panel_ensemble",
              label="Top-3 deployable NLP models (panel transfer)")
if HAVE_CV3:
    pcv = load_pred("panel", "cv_top3")
    if have_pred("panel", "nlp_deploy_top3"):
        pnl = load_pred("panel", "nlp_deploy_top3")
        FUSION_W["deploy_fresh"] = choose_weights(pcv, pnl, VM & FRESH)
        mixed = blend(pcv, pnl, FUSION_W["deploy_fresh"])
        dep = {t: np.where((FRESH[:, None] if np.asarray(pcv[t]).ndim == 2 else FRESH), mixed[t], pcv[t]) for t in TASKS}
    else:
        FUSION_W["deploy_fresh"] = {t: 1.0 for t in TASKS}; dep = pcv
    finalize(PANEL, "deploy_recipe", raw_from_probs(dep), "deploy", weights_fresh=FUSION_W["deploy_fresh"], cv_members=TOP3_CV,
             nlp_members=NLP_DEPLOY_TOP3, label="Dashboard recipe")
(SUB["results"] / "fusion_weights.json").write_text(json.dumps(FUSION_W, indent=1))

In [ ]:
# ---------------------------------------------------------------- Q1 and Q10 on validation
FUSION_IDS = [m for m in ["fusion_late_top3", "fusion_late_best", "fusion_early"] if REG.get(m, {}).get("status") == "RUN"]
def val_primary(m, t):
    pr = load_pred("panel", m)[t]; r = PANEL.rows(t, VM, pr)
    return primary_value(t, PANEL.Y[t][r], pr[r]) if len(r) >= 30 else -np.inf
def q1_table(mask, tag):
    """Q1 roles per task; 'best fusion' is always chosen on validation, whatever block is being scored."""
    rows = []
    for t in TASKS:
        cands = {"prior": "prior", "technical (reference)": "tech_gbm", "best CV": BEST_CV_TASK.get(t), "best NLP (panel)": BEST_NLPP_TASK.get(t),
                 "top-3 CV": "cv_top3", "top-3 NLP": "nlp_panel_top3", "best fusion": None, "dashboard recipe": "deploy_recipe"}
        fz = {m: val_primary(m, t) for m in FUSION_IDS}
        cands["best fusion"] = max(fz, key=lambda k: fz[k]) if fz else None
        for role, m in cands.items():
            if not m or REG.get(m, {}).get("status") != "RUN":
                continue
            pr = load_pred("panel", m)[t]; r = PANEL.rows(t, mask, pr)
            rows.append({"task": t, "role": role, "model": m, "n": len(r), "primary": primary_value(t, PANEL.Y[t][r], pr[r]) if len(r) >= 30 else np.nan})
    df = pd.DataFrame(rows); df.to_csv(SUB["tables"] / f"q1_{tag}.csv", index=False)
    return df
Q1_VAL = q1_table(VM, "val"); Q1_VAL_FRESH = q1_table(VM & FRESH, "val_fresh")
display(Q1_VAL.pivot_table(index="role", columns="task", values="primary").round(3))
print("Fresh-text rows only:"); display(Q1_VAL_FRESH.pivot_table(index="role", columns="task", values="primary").round(3))
Q10_VAL = pd.DataFrame([{"task": t, **paired_delta(PANEL, t, load_pred("panel", "fusion_late_top3")[t], load_pred("panel", "cv_top3")[t], VM)}
                        for t in TASKS]) if "fusion_late_top3" in FUSION_IDS and HAVE_CV3 else pd.DataFrame()
if len(Q10_VAL):
    Q10_VAL["p_holm"] = holm(Q10_VAL.p.values); Q10_VAL.to_csv(SUB["tables"] / "q10_val.csv", index=False)
    print("Q10 (validation): top-3 CV + top-3 NLP minus top-3 CV"); display(Q10_VAL.round(4))
if FUSION_W:
    display(pd.DataFrame(FUSION_W).rename_axis("task").round(2))
tick("20 fusion")

## 21. Monte Carlo price paths (the dashboard's 20-day simulation), checked on validation
For one stock at one date, 20 daily log returns are simulated as drift + scale × standardised Student-t noise. The scale comes from the recipe's 20-day volatility forecast; the drift is solved so that the simulated DOWN / SIDEWAYS / UP shares at day 20 match the recipe's calibrated probabilities; intraday lows come from the exact Brownian-bridge minimum. The tail thickness ν is chosen on validation from {3, 4, 6, 10, normal} so that the 80 % and 90 % ranges of the 20-day return contain the real outcome about 80 % and 90 % of the time. Test coverage is reported in §23.

In [ ]:
def mc_eval(mid, mask, nu, n_rows, seed=0, scale=1.0):
    """scale multiplies the forecast volatility before simulating (1.0 = as forecast; section 23A may calibrate it on validation)."""
    pr = load_pred("panel", mid); rows = PANEL.rows("ret_20", mask, pr["vol_20"])
    rows = np.sort(np.random.default_rng(seed).choice(rows, min(n_rows, len(rows)), replace=False))
    out = []
    for r in rows:
        sim = mc_simulate(1.0, P.sigma.values[r], math.exp(pr["vol_20"][r]) * scale, pr["dir_20"][r], nu=nu, n_days=20, n_paths=CFG["mc"]["n_paths"], seed=int(r))
        q = sim["ret_q"]; y = PANEL.Y["ret_20"][r]
        out.append({"row": r, "in80": q[1] <= y <= q[5], "in90": q[0] <= y <= q[6], "p_dd_mc": sim["p_dd"], "dd": PANEL.Y["dd_20"][r], "p_dd_head": pr["dd_20"][r]})
    return pd.DataFrame(out)
MC_CFG = {"nu": 6, "recipe": "deploy_recipe"}
if REG.get("deploy_recipe", {}).get("status") == "RUN":
    MC_VAL = []
    for nu in CFG["mc"]["nu_grid"]:
        d_ = mc_eval("deploy_recipe", VM, nu, CFG["mc"]["eval_rows"])
        MC_VAL.append({"nu": nu if nu else "normal", "coverage80": float(d_.in80.mean()), "coverage90": float(d_.in90.mean()),
                       "brier_dd_mc": float(np.mean((d_.p_dd_mc - d_.dd) ** 2)), "brier_dd_head": float(np.mean((d_.p_dd_head - d_.dd) ** 2)), "n": len(d_)})
    MC_VAL = pd.DataFrame(MC_VAL); MC_VAL["score"] = (MC_VAL.coverage80 - 0.8).abs() + (MC_VAL.coverage90 - 0.9).abs()
    MC_CFG["nu"] = int(CFG["mc"]["nu_grid"][int(MC_VAL.score.values.argmin())])
    MC_VAL.to_csv(SUB["tables"] / "mc_calibration_val.csv", index=False); display(MC_VAL.round(3))
    r_ex = int(np.flatnonzero(VM)[len(np.flatnonzero(VM)) // 2]); pr_ = load_pred("panel", "deploy_recipe")
    sim = mc_simulate(float(P.close.values[r_ex]), P.sigma.values[r_ex], math.exp(pr_["vol_20"][r_ex]), pr_["dir_20"][r_ex], nu=MC_CFG["nu"], n_paths=4000)
    fig, ax = plt.subplots(figsize=(8, 3.6)); x_ = np.arange(1, 21)
    ax.fill_between(x_, sim["quantiles"][0], sim["quantiles"][4], alpha=0.2, label="5-95 %"); ax.fill_between(x_, sim["quantiles"][1], sim["quantiles"][3], alpha=0.35, label="25-75 %")
    ax.plot(x_, sim["quantiles"][2], label="median"); ax.plot(x_, sim["paths"][:25].T, lw=0.4, color="grey", alpha=0.5)
    ax.set_title(f"Monte Carlo example: {P.ticker.values[r_ex]} {P.date.values[r_ex].astype('datetime64[D]')} (nu = {MC_CFG['nu'] or 'normal'}, P(DD>10%) = {sim['p_dd']:.2f})")
    ax.legend(fontsize=7); ax.grid(alpha=0.3); plt.tight_layout(); savefig(fig, "mc_example_val.png"); plt.show()
(SUB["results"] / "mc_config.json").write_text(json.dumps(MC_CFG))
tick("21 monte carlo")

## 22. Freeze: every choice and every prediction is fixed before the test block opens

The freeze manifest records: all selections (best models, top-3 members, combinations, fusion weights, Monte Carlo ν), the SHA-256 of every prediction file (test predictions already exist; their labels do not), the configuration hash, the data-package hash, the shared-core code hash and the package versions. §23 refuses to run without it. Re-freezing after a change is allowed but recorded in `freeze_history` and disclosed in the reports.

In [ ]:
import importlib.metadata as ilm
def _ver(p):
    try:
        return ilm.version(p)
    except Exception:
        return None
SELECTIONS = {
    "universe": TICKERS, "sector_of": SECTOR_OF, "replaced": REPLACED,
    "cv": {"rank_val": CV_RANK.round(4).to_dict(), "best": CV_BEST, "best_per_task": BEST_CV_TASK, "top3": TOP3_CV, "best_combo": BEST_COMBO},
    "nlp": {"rank_val": NLP_RANK.round(4).to_dict(), "best": NLP_BEST, "best_per_task": BEST_NLP_TASK, "panel_best_per_task": BEST_NLPP_TASK,
            "panel_top3": TOP3_NLPP, "deploy_top3": NLP_DEPLOY_TOP3},
    "fusion": {"ids": FUSION_IDS, "weights": FUSION_W}, "mc": MC_CFG,
}
POST_FREEZE_IDS = {"company_event", "simple_event", "tech_event", "fusion_tech_cv", "fusion_tech_nlp", "fusion_tech_cal", "fusion_tech_cal_co"}      # created in section 23A, after the test block was opened: never part of the freeze
pred_hashes = {f"{p_.parent.name}/{p_.name}": sha256_file(p_) for p_ in sorted(SUB["preds"].glob("*/*.npz")) if p_.stem not in POST_FREEZE_IDS}
FREEZE = {"project": CFG["project"], "run_mode": RUN_MODE, "frozen_utc": now_utc(), "config_hash": cfg_hash(CFG),
          "data_manifest_sha256": DATA_MANIFEST_SHA256, "core_sha256": hashlib.sha256(CORE_SRC.encode()).hexdigest(),
          "selections": SELECTIONS, "selections_sha256": cfg_hash(SELECTIONS), "prediction_files": pred_hashes,
          "registry_models": {k: v.get("status") for k, v in REG.items()},
          "packages": {p_: _ver(p_) for p_ in ["torch", "timm", "transformers", "peft", "bitsandbytes", "ultralytics", "lightgbm", "scikit-learn",
                                                "numpy", "pandas", "streamlit", "yfinance", "reportlab"]},
          "gpu": GPU_NAME, "python": platform.python_version()}
_hist = []
if FREEZE_PATH.exists():
    old = json.loads(FREEZE_PATH.read_text())
    if old.get("selections_sha256") == FREEZE["selections_sha256"] and old.get("config_hash") == FREEZE["config_hash"]:
        # a resumed run with identical selections and configuration is not a re-freeze: keep the original freeze (time, history, file hashes)
        FREEZE = old; FREEZE["resumed_utc"] = old.get("resumed_utc", []) + [now_utc()]; _hist = old.get("freeze_history", [])
        print(f"Same selections and configuration as the freeze of {old.get('frozen_utc')}: original freeze kept (resumed {len(FREEZE['resumed_utc'])} time(s)).")
    else:
        _hist = old.get("freeze_history", []) + [{k: old.get(k) for k in ("frozen_utc", "selections_sha256", "config_hash")}]
        print("NOTE: selections or configuration differ from the previous freeze; this re-freeze is recorded and disclosed in the reports.")
FREEZE["freeze_history"] = _hist
FREEZE_PATH.write_text(json.dumps(FREEZE, indent=1, default=str))
print(f"Freeze manifest written: {len(pred_hashes)} prediction files, selections {FREEZE['selections_sha256']}, previous freezes {len(_hist)}")
tick("22 freeze")

## 23. The locked test block (2024-01 → 2026-08): opened once, everything answered

In [ ]:
_lab = VAULT.open("panel_labels")
for c_ in LABEL_COLS:
    Y[c_][_lab["row"].values] = _lab[c_].values
_le = VAULT.open("event_labels")
for c_ in EV_LABELS:
    YE[c_][_le["ev"].values] = _le[c_].values
for c_ in REACT_COLS:
    EV.loc[_le["ev"].values, c_] = _le[c_].values
TEM, TEE = PANEL.mask["test"], EVENT.mask["test"]
_diag = {"test_rows": int(TEM.sum()), "vault_rows": len(_lab), "finite_dir20": int(np.isfinite(PANEL.Y["dir_20"][TEM]).sum()),
         "finite_vol20": int(np.isfinite(PANEL.Y["vol_20"][TEM]).sum()), "test_events": int(TEE.sum()),
         "last_decision_date": str(P.loc[TEM, "date"].max().date()) if TEM.any() else None,
         "pred_finite_prior": int(np.isfinite(load_pred("panel", "prior")["vol_20"][TEM]).sum())}
print("Test diagnostics:", _diag)
assert _diag["finite_dir20"] > 0, f"No test labels after opening the vault: {_diag}. Send this line to Claude Code."
print(f"Test opened: {int(np.isfinite(Y['dir_20'][TEM]).sum()):,} panel rows, {int(np.isfinite(YE['dir_20'][TEE]).sum()):,} filing events. "
      f"Openings logged: {len(json.loads(VAULT_LOG.read_text()))}")

def skill_rows(ctx, model_by_task, mask, family):
    rows = [{"family": family, "task": t, "model": m, **skill_test(ctx, t, load_pred(ctx.name, m)[t], mask)} for t, m in model_by_task.items()
            if m and REG.get(m, {}).get("status") == "RUN"]
    df = pd.DataFrame(rows)
    if len(df):
        df["p_holm"] = holm(df["p"].values); df["supported"] = (df["p_holm"] < CFG["alpha"]) & (df["lo"] > 0)
    return df
def delta_rows(ctx, pairs_by_task, mask, family):
    rows = [{"family": family, "task": t, "A": a, "B": b, **paired_delta(ctx, t, load_pred(ctx.name, a)[t], load_pred(ctx.name, b)[t], mask)}
            for t, (a, b) in pairs_by_task.items() if REG.get(a, {}).get("status") == "RUN" and REG.get(b, {}).get("status") == "RUN"]
    df = pd.DataFrame(rows)
    if len(df):
        df["p_holm"] = holm(df["p"].values); df["supported"] = (df["p_holm"] < CFG["alpha"]) & (df["delta"] > 0)
    return df

### 23.1 Q1 and Q10: CV vs NLP vs CV+NLP at 5, 10, 20, 40 days (H1-H4)

In [ ]:
Q1_TEST = q1_table(TEM, "test"); Q1_TEST_FRESH = q1_table(TEM & FRESH, "test_fresh")
display(Q1_TEST.pivot_table(index="role", columns="task", values="primary").round(3))
H1 = skill_rows(PANEL, BEST_CV_TASK, TEM, "H1 CV skill")
H2 = skill_rows(EVENT, BEST_NLP_TASK, TEE, "H2 NLP skill (filing events)")
H2p = skill_rows(PANEL, BEST_NLPP_TASK, TEM, "H2b NLP skill (panel)")
_bf = {t: Q1_TEST[(Q1_TEST.task == t) & (Q1_TEST.role == "best fusion")]["model"].iloc[0] for t in TASKS
       if ((Q1_TEST.task == t) & (Q1_TEST.role == "best fusion")).any()}
_better = {t: (BEST_CV_TASK[t] if val_primary(BEST_CV_TASK[t], t) >= val_primary(BEST_NLPP_TASK.get(t, BEST_CV_TASK[t]), t) else BEST_NLPP_TASK[t])
           for t in TASKS if t in BEST_CV_TASK}
H3 = delta_rows(PANEL, {t: (_bf[t], _better[t]) for t in TASKS if t in _bf and t in _better}, TEM, "H3 fusion vs better single modality")
H4 = delta_rows(PANEL, {t: ("fusion_late_top3", "cv_top3") for t in TASKS}, TEM, "H4 top-3 CV + top-3 NLP vs top-3 CV (Q10)")
SKILL_DEPLOY = skill_rows(PANEL, {t: "deploy_recipe" for t in TASKS}, TEM, "dashboard recipe skill")
for nm, df in [("H1", H1), ("H2", H2), ("H2b", H2p), ("H3", H3), ("H4", H4), ("deploy", SKILL_DEPLOY)]:
    if len(df):
        df.to_csv(SUB["tables"] / f"test_{nm}.csv", index=False); print(nm); display(df.round(4))

### 23.2 - 23.3 Q2, Q3, Q4 on test: CV leaderboard, combinations and the best per target

In [ ]:
LB_CV_TEST = leaderboard(PANEL, CV_MAIN + CV_OTHER + BASELINES + ["cv_top3", "cv_all7"], TEM); LB_CV_TEST.to_csv(SUB["tables"] / "cv_leaderboard_test.csv", index=False)
PIV_CV_TEST = LB_CV_TEST[LB_CV_TEST.status == "RUN"].pivot_table(index="model", columns="task", values="primary")
CV_RANK_TEST = mean_rank(LB_CV_TEST[LB_CV_TEST.model.isin(CV_MAIN)]); CV_RANK_TEST.to_frame("mean_rank_test").to_csv(SUB["tables"] / "cv_rank_test.csv")
display(pd.concat([CV_RANK.rename("val_rank"), CV_RANK_TEST.rename("test_rank")], axis=1).round(2))
COMBO_TEST = []
if len(CV_MAIN) >= 2:
    for k in (1, 2, 3):
        for combo in itertools.combinations(CV_MAIN, k):
            avg = average_predictions([load_pred("panel", m) for m in combo])
            for t in TASKS:
                r = PANEL.rows(t, TEM, avg[t])
                COMBO_TEST.append({"combo": " + ".join(c.replace("_ft", "") for c in combo), "size": k, "task": t, "primary": primary_value(t, PANEL.Y[t][r], avg[t][r])})
COMBO_TEST = pd.DataFrame(COMBO_TEST)
if len(COMBO_TEST):
    COMBO_TEST.to_csv(SUB["tables"] / "cv_combos_test.csv", index=False)
    cr_t = COMBO_TEST.pivot_table(index="combo", columns="task", values="primary").rank(ascending=False).mean(1).sort_values()
    cr_t.to_frame("mean_rank_test").to_csv(SUB["tables"] / "cv_combo_rank_test.csv")
    print("Validation-chosen best combination:", BEST_COMBO, "| its test rank:", int((cr_t.index == BEST_COMBO).argmax()) + 1 if BEST_COMBO in cr_t.index else "n/a",
          "of", len(cr_t)); display(cr_t.head(10).round(2))
BEST_PER_TARGET = []
for kd in ("dir", "vol", "dd"):
    for t in [x for x in TASKS if x.startswith(kd)]:
        cands = {m: PIV_CV_TEST.loc[m, t] for m in CV_MAIN + ["cv_top3", "cv_all7"] if m in PIV_CV_TEST.index}
        BEST_PER_TARGET.append({"target": kd, "task": t, "val_choice": BEST_CV_TASK.get(t), "val_choice_test_score": PIV_CV_TEST.loc[BEST_CV_TASK[t], t] if BEST_CV_TASK.get(t) in PIV_CV_TEST.index else np.nan,
                                "top3_test_score": cands.get("cv_top3"), "test_best_ex_post": max(cands, key=lambda z: cands[z]) if cands else None})
BEST_PER_TARGET = pd.DataFrame(BEST_PER_TARGET); BEST_PER_TARGET.to_csv(SUB["tables"] / "cv_best_per_target_test.csv", index=False); display(BEST_PER_TARGET.round(3))
H5 = []
for n in FROZEN_CV:
    a, b = f"{n}_ft", f"{n}_frozen"
    if REG.get(a, {}).get("status") == "RUN" and REG.get(b, {}).get("status") == "RUN":
        for kd in ("dir", "vol", "dd"):
            H5.append({"family": "H5 fine-tuned vs frozen", "model": n, "target": kd, **paired_delta_family(PANEL, kd, load_pred("panel", a), load_pred("panel", b), TEM)})
H5 = pd.DataFrame(H5)
if len(H5):
    H5["p_holm"] = holm(H5.p.values); H5["supported"] = (H5.p_holm < CFG["alpha"]) & (H5.delta > 0); H5.to_csv(SUB["tables"] / "test_H5.csv", index=False); display(H5.round(4))

### 23.4 Q5 on test: sectors (H6)

In [ ]:
if CV_MAIN:
    SEC_CV_TEST = sector_table(PANEL, CV_MAIN, TEM); SEC_CV_TEST.to_csv(SUB["tables"] / "cv_sector_test.csv", index=False)
    _sm = SEC_CV_TEST.groupby(["sector", "target", "model"])["primary"].mean().reset_index()
    BEST_BY_SECTOR_TEST = best_rows(_sm, ["sector", "target"]).pivot(index="sector", columns="target", values="model")
    BEST_BY_SECTOR_TEST.to_csv(SUB["tables"] / "cv_best_by_sector_test.csv")
    agree = (BEST_BY_SECTOR_TEST == BEST_BY_SECTOR_VAL.reindex_like(BEST_BY_SECTOR_TEST)).values.mean()
    print(f"Best CV model per sector and target on test (agreement with the validation choice: {agree:.0%}):"); display(BEST_BY_SECTOR_TEST)
    H6 = pd.DataFrame([sector_interaction_test(PANEL, CV_MAIN, TEM, kd)[0] for kd in ("dir", "vol", "dd")]).assign(family="H6 sector x model interaction")
    H6["supported"] = H6.p < CFG["alpha"]; H6.to_csv(SUB["tables"] / "test_H6.csv", index=False); display(H6)
    NLP_SECTOR_TEST = by_group(EVENT, NLP_MAIN, TEE, EV["sector"].values, "sector") if NLP_MAIN else pd.DataFrame()
    if len(NLP_SECTOR_TEST):
        NLP_SECTOR_TEST.to_csv(SUB["tables"] / "nlp_by_sector_test.csv", index=False)
else:
    H6 = pd.DataFrame()

### 23.5 Q6-Q9 on test: which reports move prices, where, and what in them (H7, replication)

In [ ]:
rng = np.random.default_rng(12); CTRL_T = []
for t in TICKERS:
    d = BARS_D[t]; c = d["close"].values; dates = pd.DatetimeIndex(d["date"])
    lr = np.r_[np.nan, np.diff(np.log(c))]; spy = SPY_D.set_index("date")["close"].reindex(dates).ffill().values
    ar = lr - np.r_[np.nan, np.diff(np.log(spy))]; sig_ar = pd.Series(ar).rolling(SIGW, min_periods=SIGW).std().values
    busy = np.zeros(len(d), bool)
    for s in EV.loc[EV.ticker == t, "s_idx"]:
        busy[max(0, s - 3): s + 4] = True
    ok = np.flatnonzero(~busy & (dates >= BLK["test"][0]) & np.isfinite(sig_ar)); ok = ok[ok + 2 < len(d)]
    for s in rng.choice(ok, min(len(ok), 4 * int(((EV.ticker == t) & TE_EV).sum())), replace=False):
        ar2 = float(np.nansum(ar[s: s + 2]))
        CTRL_T.append({"ticker": t, "sector": SECTOR_OF[t], "form": "no filing", "month": str(dates[s].to_period("M")),
                       "abs_z": abs(ar2) / (sig_ar[s - 1] * math.sqrt(2)), "abn_volume": math.log((d["volume"].values[s] + 1) / (np.median(d["volume"].values[max(0, s - 20): s]) + 1))})
CTRL_T = pd.DataFrame(CTRL_T)
TEST_EV = EV[TE_EV]
ES_FORM_TEST, KW_P_TEST, MW_P_TEST = event_study(TEST_EV, CTRL_T); ES_FORM_TEST.to_csv(SUB["tables"] / "event_study_by_form_test.csv", index=False)
display(ES_FORM_TEST.round(3))
H7 = pd.DataFrame([{"family": "H7 event study", "test": "filing days move more than non-filing days (Mann-Whitney, one-sided)", "p": MW_P_TEST, "p_dev": MW_P_DEV},
                   {"family": "H7 event study", "test": "reaction size differs by form (Kruskal-Wallis)", "p": KW_P_TEST, "p_dev": KW_P_DEV}])
H7["supported"] = H7.p < CFG["alpha"]; H7.to_csv(SUB["tables"] / "test_H7.csv", index=False); display(H7)
ES_SECTOR_TEST = TEST_EV.groupby(["sector", "form"])["abs_z"].agg(["mean", "size"]).reset_index()
ES_SECTOR_TEST["excess_over_non_filing_days"] = ES_SECTOR_TEST["mean"] - ES_SECTOR_TEST["sector"].map(CTRL_T.groupby("sector")["abs_z"].mean())
ES_SECTOR_TEST.to_csv(SUB["tables"] / "event_study_sector_form_test.csv", index=False)
_it = TEST_EV[TEST_EV.form == "8-K"].assign(item=TEST_EV["items"].str.split(",")).explode("item"); _it["item"] = _it["item"].str.strip()
_it = _it[_it.item.isin(ES_ITEMS["item"])] if len(ES_ITEMS) else _it.iloc[0:0]
ES_ITEMS_TEST = _it.groupby("item")["abs_z"].agg(["mean", "size"]).sort_values("mean", ascending=False)
ES_ITEMS_TEST.to_csv(SUB["tables"] / "event_study_8k_items_test.csv")
LB_NLP_TEST = leaderboard(EVENT, NLP_MAIN + NLP_OTHER, TEE); LB_NLP_TEST.to_csv(SUB["tables"] / "nlp_leaderboard_test.csv", index=False)
NLP_RANK_TEST = mean_rank(LB_NLP_TEST[LB_NLP_TEST.model.isin(NLP_MAIN)]); NLP_RANK_TEST.to_frame("mean_rank_test").to_csv(SUB["tables"] / "nlp_rank_test.csv")
NLP_FORM_TEST = by_group(EVENT, NLP_MAIN, TEE, EV["form"].values, "form") if NLP_MAIN else pd.DataFrame()
if len(NLP_FORM_TEST):
    NLP_FORM_TEST.to_csv(SUB["tables"] / "nlp_by_form_test.csv", index=False); display(NLP_FORM_TEST.groupby(["form", "target"])["primary"].max().unstack().round(3))
LB_NLPP_TEST = leaderboard(PANEL, NLP_PANEL_ALL + ["nlp_panel_top3", "nlp_deploy_top3"], TEM); LB_NLPP_TEST.to_csv(SUB["tables"] / "nlp_panel_leaderboard_test.csv", index=False)
LB_FUSION_TEST = leaderboard(PANEL, FUSION_IDS + ["fusion_tech", "deploy_recipe", "cv_top3", "nlp_panel_top3", "tech_gbm", "prior"], TEM)
LB_FUSION_TEST.to_csv(SUB["tables"] / "fusion_leaderboard_test.csv", index=False)

### 23.6 Monte Carlo coverage and recipe errors on test

In [ ]:
MC_TEST = pd.DataFrame()
if REG.get("deploy_recipe", {}).get("status") == "RUN":
    d_ = mc_eval("deploy_recipe", TEM, MC_CFG["nu"], CFG["mc"]["eval_rows"], seed=1)
    MC_TEST = pd.DataFrame([{"nu": MC_CFG["nu"] or "normal", "coverage80": float(d_.in80.mean()), "coverage90": float(d_.in90.mean()),
                             "brier_dd_mc": float(np.mean((d_.p_dd_mc - d_.dd) ** 2)), "brier_dd_head": float(np.mean((d_.p_dd_head - d_.dd) ** 2)), "n": len(d_)}])
    MC_TEST.to_csv(SUB["tables"] / "mc_calibration_test.csv", index=False); display(MC_TEST.round(3))
    pr = load_pred("panel", "deploy_recipe")
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    for a_, h in zip(axes, HORIZONS):
        r = PANEL.rows(f"dir_{h}", TEM, pr[f"dir_{h}"])
        cm = confusion_matrix(PANEL.Y[f"dir_{h}"][r].astype(int), pr[f"dir_{h}"][r].argmax(1), labels=[0, 1, 2], normalize="true")
        a_.imshow(cm, cmap="Greens", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                a_.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8)
        a_.set_xticks(range(3)); a_.set_xticklabels(CLASSES, fontsize=7); a_.set_yticks(range(3)); a_.set_yticklabels(CLASSES, fontsize=7)
        a_.set_title(f"Dashboard recipe, {h}-day, test", fontsize=8)
    plt.tight_layout(); savefig(fig, "deploy_confusion_test.png"); plt.show()

### 23.7 Verdicts: hypotheses and the ten project questions

In [ ]:
def fam_summary(df, name):
    if df is None or not len(df):
        return {"family": name, "tests": 0, "supported": 0, "verdict": "Not Run"}
    k = int(df["supported"].sum())
    return {"family": name, "tests": len(df), "supported": k, "verdict": "supported" if k == len(df) else ("partly supported" if k else "not supported")}
HYP = pd.DataFrame([fam_summary(H1, "H1 CV has skill"), fam_summary(H2, "H2 NLP has skill (events)"), fam_summary(H2p, "H2b NLP has skill (panel)"),
                    fam_summary(H3, "H3 fusion beats the better single modality"), fam_summary(H4, "H4 top-3 + top-3 beats top-3 CV"),
                    fam_summary(H5, "H5 fine-tuning beats frozen"), fam_summary(H6, "H6 sector-specific rankings"), fam_summary(H7, "H7 filings move prices")])
HYP.to_csv(SUB["tables"] / "hypotheses_test.csv", index=False); display(HYP)

def _skill_line(df, t):
    if df is None or not len(df) or t not in set(df.task):
        return "Not Run"
    r = df.set_index("task").loc[t]
    return f"{r['model']}: skill {r['skill']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]{' *' if r['supported'] else ''}"
QANS = []
for h in HORIZONS:
    for kd in ("dir", "vol", "dd"):
        t = f"{kd}_{h}"
        QANS.append({"question": "Q1", "task": t, "CV": _skill_line(H1, t), "NLP (filings)": _skill_line(H2, t), "NLP (panel)": _skill_line(H2p, t),
                     "fusion vs best single": (lambda d: "Not Run" if d is None or not len(d) or t not in set(d.task) else
                                               f"{d.set_index('task').loc[t, 'delta']:+.4f} [{d.set_index('task').loc[t, 'lo']:+.4f}, {d.set_index('task').loc[t, 'hi']:+.4f}]")(H3)})
Q1_ANS = pd.DataFrame(QANS); Q1_ANS.to_csv(SUB["tables"] / "q1_answers_test.csv", index=False)
best_test_cv = CV_RANK_TEST.index[0] if len(CV_RANK_TEST) else None
ANSWERS = {
    "Q1": "See q1_answers_test.csv: '*' marks Holm-significant skill on the locked test block. " +
          f"Direction skill supported for {int(H1[H1.task.str.startswith('dir')]['supported'].sum()) if len(H1) else 0}/4 horizons (CV), "
          f"{int(H2p[H2p.task.str.startswith('dir')]['supported'].sum()) if len(H2p) else 0}/4 (NLP panel); "
          f"fusion beat the better single modality in {int(H3['supported'].sum()) if len(H3) else 0}/{len(H3)} tasks.",
    "Q2": f"Validation-chosen best CV model: {CV_BEST} (validation mean rank {CV_RANK.get(CV_BEST, float('nan')):.2f}); on test it ranks "
          f"{int((CV_RANK_TEST.index == CV_BEST).argmax()) + 1 if CV_BEST in CV_RANK_TEST.index else 'n/a'} of {len(CV_RANK_TEST)}; best on test ex post: {best_test_cv}. " + WHY_CV,
    "Q3": f"Best combination on validation: {BEST_COMBO}; see cv_combos_test.csv for its test standing and all 63 combinations.",
    "Q4": "Best per target and horizon (validation choice and its test score): cv_best_per_target_test.csv.",
    "Q5": "Best model per sector: cv_best_by_sector_test.csv; interaction test (H6) p-values: " + (", ".join(f"{r.target} {r.p:.3f}" for r in H6.itertuples()) if len(H6) else "Not Run"),
    "Q6": "Reaction size by form (test): " + (", ".join(f"{r.group} |z| {r.mean_abs_z:.2f}" for r in ES_FORM_TEST.itertuples()) if len(ES_FORM_TEST) else "Not Run"),
    "Q7": "Sector x form excess reaction over non-filing days: event_study_sector_form_test.csv (development: event_study_sector_form_dev.csv).",
    "Q8": "8-K items ranked by reaction (event_study_8k_items_*.csv), n-grams (nlp_words_*.csv), FinBERT tone (nlp_tone_vs_reaction_dev.csv), attributions (nlp_integrated_gradients_val.csv).",
    "Q9": "Same tables as Q7, read row by row: a sector is 'sensitive' to a form when its excess |z| CI is above zero in both development and test.",
    "Q10": (f"Top-3 CV + top-3 NLP minus top-3 CV: supported in {int(H4['supported'].sum())}/{len(H4)} tasks after Holm." if len(H4) else "Not Run"),
}
NO_CALL = {f"dir_{h}": not bool(SKILL_DEPLOY.set_index("task").loc[f"dir_{h}", "supported"]) if len(SKILL_DEPLOY) and f"dir_{h}" in set(SKILL_DEPLOY.task) else True
           for h in HORIZONS}
FINAL = {"stamp": STAMP, "answers": ANSWERS, "hypotheses": HYP.to_dict("records"), "no_call_direction": NO_CALL,
         "deploy_skill": SKILL_DEPLOY.to_dict("records") if len(SKILL_DEPLOY) else [], "mc_test": MC_TEST.to_dict("records"),
         "test_openings": len(json.loads(VAULT_LOG.read_text())), "freeze_sha": FREEZE["selections_sha256"], "freeze_history": len(FREEZE["freeze_history"])}
(SUB["results"] / "final_results.json").write_text(json.dumps(FINAL, indent=1, default=str))
for k, v in ANSWERS.items():
    display(Markdown(f"**{k}.** {v}"))
tick("23 test")

## 23A. Addendum after the test: stronger baselines, incremental value and high-resolution tests (post-hoc)

**Status.** Sections 4-23 are the confirmatory analysis: every selection was frozen in §22 and the test block was opened after that. This section was written **afterwards**, in response to a methodological review, and is labelled *post-hoc* in every table and report. It changes no earlier number.

| Step | Question | What is computed | Data that decide the choice |
|---|---|---|---|
| A1 | Does the best CV / NLP / fusion model beat the strongest **numbers-only** baseline, not just the prior? | paired month-block bootstrap, Holm-corrected, same rows | baseline chosen per task on validation |
| A3 | Is the filing-text signal only a company-identity signal? | three no-text baselines on the filing events: per-company base rates, momentum/HAR/trailing-volatility + form, LightGBM on 20 indicators + form | trained on the training block, tuned on 2021 |
| H8 | Do charts and filing text add information to the technical-indicator model? | LightGBM on technical features alone (frozen in §22) vs the same learner with CV features, NLP features, or both | the first model was frozen before the test; the three extended models use the same learner and settings |
| A4 | Can the many-test ablation families reach significance? | the same paired tests with 5,000 bootstrap draws instead of 1,000 (with 1,000 draws the smallest Holm-corrected p was about 2m/1001, i.e. about 0.07 for 36-44 tests) | none: only the precision of the p-values changes |
| B8 | Are the Monte Carlo ranges too narrow? | one volatility scale factor, chosen on validation, applied to the dashboard simulation | validation coverage |

**Rules kept.** New models use the training block for fitting and the inner block for tuning; no test label influences any choice; baselines are selected on validation only; the original tables are not overwritten; the reproducibility check at the end compares this execution with the archived first run.

In [ ]:
POST_HOC_TABLES = []
def _save_post_hoc(df, name, **kw):
    df.to_csv(SUB["tables"] / name, **kw); POST_HOC_TABLES.append(name)

def company_prior_raw(ctx, shrink=25.0):
    """Per-company training-block base rates (class shares, mean log-volatility, drawdown rate), shrunk towards the overall prior."""
    raw = {t: (np.zeros((ctx.n, 3)) if t.startswith("dir") else np.zeros(ctx.n)) for t in TASKS}
    for t in TASKS:
        kd = t.split("_")[0]; y = ctx.Y[t]; ok_tr = ctx.mask["train"] & np.isfinite(y); g = ctx.prior[t]
        for tk in TICKERS:
            ii = ctx.tickers == tk; ytr = y[ok_tr & ii]
            if kd == "dir":
                cnt = np.bincount(ytr.astype(int), minlength=3).astype(float)
                p_ = (cnt + shrink * np.asarray(g, dtype=float)) / (cnt.sum() + shrink)
                raw[t][ii] = np.log(np.clip(p_, 1e-6, 1.0))
            elif kd == "vol":
                raw[t][ii] = (float(ytr.sum()) + shrink * float(g)) / (len(ytr) + shrink)
            else:
                q_ = float(np.clip((float(ytr.sum()) + shrink * float(g)) / (len(ytr) + shrink), 1e-4, 1 - 1e-4))
                raw[t][ii] = math.log(q_ / (1 - q_))
    return raw

### 23A.1 No-text baselines on the filing events and the extended technical models (trained on training + inner blocks only)

In [ ]:
try:
    FORM_OH = np.stack([(EV["form"].values == f_) for f_ in CFG["sec"]["forms"]], 1).astype(np.float32)
    EV_TECH = np.full((len(EV), len(TECH_COLS)), np.nan, dtype=np.float32)
    for t_ in TICKERS:
        ii_ = np.flatnonzero((EV["ticker"] == t_).values)
        if len(ii_):
            EV_TECH[ii_] = TECH_FULL[t_].values[EV["pre_idx"].values[ii_].astype(int)]        # indicators at the last close before the filing was available
    fi_ = {c: i for i, c in enumerate(TECH_COLS)}
    raw_se = {}
    for kd_, cols_ in SIMPLE_FEATS.items():
        Xs_ = np.hstack([EV_TECH[:, [fi_[c] for c in cols_]], FORM_OH])
        raw_se.update(fit_heads(EVENT, Xs_, EVENT.mask["train"], EVENT.mask["inner"], "linear", tasks=[f"{kd_}_{h}" for h in HORIZONS]))
    finalize(EVENT, "simple_event", raw_se, "baseline_event", label="Numbers-only baseline on filing events (momentum / HAR / trailing volatility + form)")
    finalize(EVENT, "company_event", company_prior_raw(EVENT), "baseline_event", label="Per-company base rates (training block, shrunk)")
    st_te = Stage("phase_a_tech_event", [TECH_COLS, GBM, len(EV), CFG["blocks"]], [_pred_path("event", "tech_event")])
    if not (st_te.done() and have_pred("event", "tech_event")):
        finalize(EVENT, "tech_event", fit_heads(EVENT, np.hstack([EV_TECH, FORM_OH]), EVENT.mask["train"], EVENT.mask["inner"], "gbm"), "baseline_event",
                 label="Technical LightGBM + form on filing events")
        st_te.finish()
    print("Event baselines ready:", [m for m in ("prior_event", "company_event", "simple_event", "tech_event") if REG.get(m, {}).get("status") == "RUN"])
except Exception as e_:
    failed("phase_a_event_baselines", e_)

try:
    if BEST_CV_FROZEN in CV_PCA and BEST_NLP_FROZEN in TEXT_PCA:
        X_CV_ONLY = CV_PCA[BEST_CV_FROZEN]; X_NLP_ONLY = panel_text_features(BEST_NLP_FROZEN)
        st_pa = Stage("phase_a_fusion_parts", [BEST_CV_FROZEN, BEST_NLP_FROZEN, N, GBM], [_pred_path("panel", "fusion_tech_cv"), _pred_path("panel", "fusion_tech_nlp")])
        if not (st_pa.done() and have_pred("panel", "fusion_tech_cv") and have_pred("panel", "fusion_tech_nlp")):
            finalize(PANEL, "fusion_tech_cv", fit_heads(PANEL, np.hstack([X_TECH, X_CV_ONLY]), TR, IN, "gbm"), "reference", label="Reference: technical + CV features (LightGBM)")
            finalize(PANEL, "fusion_tech_nlp", fit_heads(PANEL, np.hstack([X_TECH, X_NLP_ONLY]), TR, IN, "gbm"), "reference", label="Reference: technical + NLP features (LightGBM)")
            st_pa.finish()
        print("Extended technical models ready:", [m for m in ("fusion_tech_cv", "fusion_tech_nlp", "fusion_tech") if REG.get(m, {}).get("status") == "RUN"])
    else:
        print("Extended technical models skipped: no frozen CV / NLP features available.")
except Exception as e_:
    failed("phase_a_fusion_parts", e_)

### 23A.2 Locked-test comparisons against the strongest numbers-only baselines (A1, A3) and the incremental-value tests (H8)

In [ ]:
H1C = H2C_EV = H2C_PANEL = H3C = H8 = H8B = H8C = None
try:
    def _vp(ctx, m, t):
        pr = load_pred(ctx.name, m)[t]; r = ctx.rows(t, ctx.mask["val"], pr)
        return primary_value(t, ctx.Y[t][r], pr[r]) if len(r) >= 30 else -np.inf
    def _verdict(df):
        if df is None or not len(df):
            return df
        df = df.copy()
        better = (df["p_holm"] < CFG["alpha"]) & (df["delta"] > 0); worse = (df["p_holm"] < CFG["alpha"]) & (df["delta"] < 0)
        df["verdict"] = np.where(better, "better", np.where(worse, "worse", "no clear difference"))
        return df
    NUM_BASES = [m for m in ("baseline_simple", "tech_gbm") if REG.get(m, {}).get("status") == "RUN"]
    EV_BASES = [m for m in ("prior_event", "company_event", "simple_event", "tech_event") if REG.get(m, {}).get("status") == "RUN"]
    BEST_NUM_BASE = {t: max(NUM_BASES, key=lambda m: _vp(PANEL, m, t)) for t in TASKS}          # chosen on validation only
    BEST_EV_BASE = {t: max(EV_BASES, key=lambda m: _vp(EVENT, m, t)) for t in TASKS}
    BASE_CHOICE = pd.DataFrame({"task": TASKS, "panel_baseline": [BEST_NUM_BASE[t] for t in TASKS], "event_baseline": [BEST_EV_BASE[t] for t in TASKS]})
    _save_post_hoc(BASE_CHOICE, "phase_a_baseline_choice.csv", index=False)
    H1C = _verdict(delta_rows(PANEL, {t: (BEST_CV_TASK[t], BEST_NUM_BASE[t]) for t in TASKS if t in BEST_CV_TASK}, TEM, "H1c best CV vs strongest numbers-only baseline"))
    H2C_PANEL = _verdict(delta_rows(PANEL, {t: (BEST_NLPP_TASK[t], BEST_NUM_BASE[t]) for t in TASKS if t in BEST_NLPP_TASK}, TEM, "H2c best NLP (panel) vs strongest numbers-only baseline"))
    H2C_EV = _verdict(delta_rows(EVENT, {t: (BEST_NLP_TASK[t], BEST_EV_BASE[t]) for t in TASKS if t in BEST_NLP_TASK}, TEE, "H2c best NLP (filing events) vs strongest no-text baseline"))
    H3C = _verdict(delta_rows(PANEL, {t: (_bf[t], BEST_NUM_BASE[t]) for t in TASKS if t in _bf}, TEM, "H3c best fusion vs strongest numbers-only baseline"))
    H8 = _verdict(delta_rows(PANEL, {t: ("fusion_tech", "tech_gbm") for t in TASKS}, TEM, "H8 technical + CV + NLP vs technical alone"))
    H8B = _verdict(delta_rows(PANEL, {t: ("fusion_tech_cv", "tech_gbm") for t in TASKS}, TEM, "H8b technical + CV vs technical alone"))
    H8C = _verdict(delta_rows(PANEL, {t: ("fusion_tech_nlp", "tech_gbm") for t in TASKS}, TEM, "H8c technical + NLP vs technical alone"))
    for nm_, df_ in [("H1c", H1C), ("H2c_events", H2C_EV), ("H2c_panel", H2C_PANEL), ("H3c", H3C), ("H8", H8), ("H8b", H8B), ("H8c", H8C)]:
        if df_ is not None and len(df_):
            _save_post_hoc(df_, f"test_{nm_}.csv", index=False); print(nm_); display(df_[["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"]].round(4))
    def _fam_counts(df, name):
        if df is None or not len(df):
            return {"family": name, "tests": 0, "better": 0, "no clear difference": 0, "worse": 0}
        vc = df["verdict"].value_counts()
        return {"family": name, "tests": len(df), "better": int(vc.get("better", 0)), "no clear difference": int(vc.get("no clear difference", 0)), "worse": int(vc.get("worse", 0))}
    HYP_A = pd.DataFrame([_fam_counts(H1C, "H1c best CV vs numbers-only baseline"), _fam_counts(H2C_PANEL, "H2c best NLP (panel) vs numbers-only baseline"),
                          _fam_counts(H2C_EV, "H2c best NLP (events) vs no-text baseline"), _fam_counts(H3C, "H3c best fusion vs numbers-only baseline"),
                          _fam_counts(H8, "H8 technical + CV + NLP vs technical"), _fam_counts(H8B, "H8b technical + CV vs technical"),
                          _fam_counts(H8C, "H8c technical + NLP vs technical")])
    _save_post_hoc(HYP_A, "hypotheses_phase_a.csv", index=False); display(HYP_A)
    _pan = [m for m in ["prior", "baseline_simple", "tech_gbm", "fusion_tech_cv", "fusion_tech_nlp", "fusion_tech", "cv_top3", "nlp_panel_top3", "fusion_late_top3", "deploy_recipe"]
            if REG.get(m, {}).get("status") == "RUN"]
    _evt = [m for m in ["prior_event", "company_event", "simple_event", "tech_event"] + list(NLP_MAIN) if REG.get(m, {}).get("status") == "RUN"]
    PIV_A_PANEL = leaderboard(PANEL, _pan, TEM, primary_only=True).pivot_table(index="model", columns="task", values="primary").reindex(columns=TASKS)
    PIV_A_EVENT = leaderboard(EVENT, _evt, TEE, primary_only=True).pivot_table(index="model", columns="task", values="primary").reindex(columns=TASKS)
    _save_post_hoc(PIV_A_PANEL.round(4), "phase_a_panel_test.csv"); _save_post_hoc(PIV_A_EVENT.round(4), "phase_a_event_test.csv")
    print("Test primary metrics, panel rows:"); display(PIV_A_PANEL.round(3)); print("Test primary metrics, filing events:"); display(PIV_A_EVENT.round(3))
except Exception as e_:
    failed("phase_a_comparisons", e_)

### 23A.2b Is the text gain only the filing calendar or the company? And a fair direction comparison (Phase A2, post-hoc)

**Calendar control.** The panel text features contain, besides the text embeddings, four non-text columns: whether a periodic report / an 8-K exists and how many sessions ago it was filed. Volatility rises around earnings filings, so those columns alone could explain H8c. Two more LightGBM models with the same settings: technical + calendar columns (`fusion_tech_cal`), and technical + calendar + company one-hot (`fusion_tech_cal_co`). **H8d** = calendar vs technical alone; **H8e** = technical + text (with calendar) vs technical + calendar, i.e. the content of the text; **H8f** = the same against technical + calendar + company, i.e. text content beyond the calendar and the company's identity.

**Fair direction comparison.** Macro-F1 punishes a model that predicts mostly one class, which the numbers-only baselines do. **H1d / H2d** compare *skills*: each model's macro-F1 minus its own within-month shuffled null (the H1 definition), so a model is not rewarded for spreading its predictions over the three classes.

In [ ]:
H8D = H8E = H8F = H1D = H2D = None
def _verdict2(df):
    if df is None or not len(df):
        return df
    df = df.copy()
    better = (df["p_holm"] < CFG["alpha"]) & (df["delta"] > 0); worse = (df["p_holm"] < CFG["alpha"]) & (df["delta"] < 0)
    df["verdict"] = np.where(better, "better", np.where(worse, "worse", "no clear difference"))
    return df
try:
    X_CAL = np.column_stack([HAS_PER, HAS_8K, np.log1p(np.nan_to_num(AGE["per"], nan=99)), np.log1p(np.nan_to_num(AGE["8k"], nan=99))]).astype(np.float32)
    _tk = P["ticker"].astype(str).to_numpy(dtype=object)                      # plain NumPy array (pandas 3 returns an Arrow array from .values, which has no 2-D indexing)
    X_CO = np.stack([(_tk == t_) for t_ in TICKERS], axis=1).astype(np.float32)
    st_cal = Stage("phase_a2_calendar", [N, GBM, TICKERS, "cal_v1"], [_pred_path("panel", "fusion_tech_cal"), _pred_path("panel", "fusion_tech_cal_co")])
    if not (st_cal.done() and have_pred("panel", "fusion_tech_cal") and have_pred("panel", "fusion_tech_cal_co")):
        finalize(PANEL, "fusion_tech_cal", fit_heads(PANEL, np.hstack([X_TECH, X_CAL]), TR, IN, "gbm"), "reference", label="Reference: technical + filing-calendar columns (LightGBM)")
        finalize(PANEL, "fusion_tech_cal_co", fit_heads(PANEL, np.hstack([X_TECH, X_CAL, X_CO]), TR, IN, "gbm"), "reference",
                 label="Reference: technical + filing calendar + company one-hot (LightGBM)")
        st_cal.finish()
    H8D = _verdict2(delta_rows(PANEL, {t: ("fusion_tech_cal", "tech_gbm") for t in TASKS}, TEM, "H8d technical + filing calendar vs technical alone"))
    H8E = _verdict2(delta_rows(PANEL, {t: ("fusion_tech_nlp", "fusion_tech_cal") for t in TASKS}, TEM, "H8e technical + text vs technical + filing calendar (text content)"))
    H8F = _verdict2(delta_rows(PANEL, {t: ("fusion_tech_nlp", "fusion_tech_cal_co") for t in TASKS}, TEM, "H8f technical + text vs technical + calendar + company (text beyond identity)"))
except Exception as e_:
    failed("phase_a2_calendar", e_)

def paired_skill_delta(ctx, task, pa, pb, mask, seed=0):
    """Direction only: (macro-F1 of A minus A's own within-month shuffled null) minus the same for B; paired month-block bootstrap."""
    pa, pb = np.asarray(pa), np.asarray(pb)
    r = np.intersect1d(ctx.rows(task, mask, pa), ctx.rows(task, mask, pb))
    if len(r) < 30:
        return {"n": len(r), "skill_A": np.nan, "skill_B": np.nan, "delta": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan}
    y = ctx.Y[task][r].astype(int); A, Bp = pa[r], pb[r]; mo = ctx.months[r]
    nA, nB = dir_null_f1(y, A, mo, seed=seed), dir_null_f1(y, Bp, mo, seed=seed)
    fn = lambda ix: (primary_value(task, y[ix], A[ix]) - nA) - (primary_value(task, y[ix], Bp[ix]) - nB)
    bs = bootstrap(fn, month_groups(mo), seed=seed); est = fn(np.arange(len(r)))
    p2 = 2 * min((1 + np.sum(bs <= 0)) / (1 + len(bs)), (1 + np.sum(bs >= 0)) / (1 + len(bs)))
    return {"n": len(r), "skill_A": float(primary_value(task, y, A) - nA), "skill_B": float(primary_value(task, y, Bp) - nB), "delta": float(est),
            "lo": float(np.nanpercentile(bs, 2.5)), "hi": float(np.nanpercentile(bs, 97.5)), "p": float(min(1.0, p2))}
def skill_delta_rows(ctx, pairs_by_task, mask, family):
    rows = [{"family": family, "task": t, "A": a, "B": b, **paired_skill_delta(ctx, t, load_pred(ctx.name, a)[t], load_pred(ctx.name, b)[t], mask)}
            for t, (a, b) in pairs_by_task.items() if REG.get(a, {}).get("status") == "RUN" and REG.get(b, {}).get("status") == "RUN"]
    df = pd.DataFrame(rows)
    if len(df):
        df["p_holm"] = holm(df["p"].values)
    return _verdict2(df)
try:
    _dir = [f"dir_{h}" for h in HORIZONS]
    H1D = skill_delta_rows(PANEL, {t: (BEST_CV_TASK[t], BEST_NUM_BASE[t]) for t in _dir if t in BEST_CV_TASK}, TEM, "H1d direction skill: best CV vs strongest numbers-only baseline")
    H2D = skill_delta_rows(EVENT, {t: (BEST_NLP_TASK[t], BEST_EV_BASE[t]) for t in _dir if t in BEST_NLP_TASK}, TEE, "H2d direction skill: best NLP vs strongest no-text baseline (events)")
except Exception as e_:
    failed("phase_a2_direction_skill", e_)
try:
    for nm_, df_ in [("H8d", H8D), ("H8e", H8E), ("H8f", H8F), ("H1d", H1D), ("H2d_events", H2D)]:
        if df_ is not None and len(df_):
            _save_post_hoc(df_, f"test_{nm_}.csv", index=False); print(nm_); display(df_.drop(columns=["family"]).round(4))
    _cnt = lambda df, name: ({"family": name, "tests": 0, "better": 0, "no clear difference": 0, "worse": 0} if df is None or not len(df) else
                             {"family": name, "tests": len(df), **{k: int((df["verdict"] == k).sum()) for k in ("better", "no clear difference", "worse")}})
    _new = pd.DataFrame([_cnt(H1D, "H1d direction skill: best CV vs numbers-only baseline (own shuffled nulls)"),
                         _cnt(H2D, "H2d direction skill: best NLP vs no-text baseline (events, own shuffled nulls)"),
                         _cnt(H8D, "H8d technical + filing calendar vs technical"), _cnt(H8E, "H8e technical + text vs technical + calendar"),
                         _cnt(H8F, "H8f technical + text vs technical + calendar + company")])
    _old = globals().get("HYP_A")
    HYP_A = pd.concat([_old, _new], ignore_index=True) if _old is not None else _new
    _save_post_hoc(HYP_A, "hypotheses_phase_a.csv", index=False); display(HYP_A)
except Exception as e_:
    failed("phase_a2_save", e_)

### 23A.3 High-resolution versions of the many-test families (5,000 bootstrap draws)

In [ ]:
CV_ABL_HI = NLP_ABL_HI = H5_HI = None
BOOT["B"] = 5000 if not QUICK else CFG["bootstrap_B"]
try:
    t0_ = time.time(); rows_ = []
    for a_, b_, what_ in ABL_PAIRS:
        if REG.get(a_, {}).get("status") != "RUN" or REG.get(b_, {}).get("status") != "RUN":
            continue
        pa_, pb_ = load_pred("panel", a_), load_pred("panel", b_)
        for kd_ in ("dir", "vol", "dd"):
            rows_.append({"comparison": what_, "A": a_, "B": b_, "target": kd_, **paired_delta_family(PANEL, kd_, pa_, pb_, PANEL.mask["val"])})
    CV_ABL_HI = pd.DataFrame(rows_)
    if len(CV_ABL_HI):
        CV_ABL_HI["p_holm"] = holm(CV_ABL_HI["p"].values); CV_ABL_HI["B"] = BOOT["B"]; _save_post_hoc(CV_ABL_HI, "cv_ablations_val_hiB.csv", index=False)
    print(f"CV ablations ({len(CV_ABL_HI)} tests, B={BOOT['B']}): {(time.time() - t0_) / 60:.1f} min")
except Exception as e_:
    failed("phase_a_cv_ablations", e_)
try:
    t0_ = time.time(); rows_ = []
    pairs_ = ([(f"{e}_ft", f"{e}_frozen", "fine-tuned vs frozen") for e in ENCODERS] + [(f"{e}_frozen", f"{e}_frozen_1chunk", "4 chunks vs first 512 tokens") for e in ENCODERS] +
              [("finbert_ft", "roberta_l_ft", "FinBERT (finance, base) vs RoBERTa-large (general, large)"), ("secbert_ft", "finbert_ft", "SEC-BERT vs FinBERT (domain)"),
               ("longformer_ft", "finbert_ft", "Longformer 4096 vs FinBERT 512 (length)"), ((NLP_BEST or "tfidf_lr"), "tfidf_lr", "best NLP vs TF-IDF baseline")])
    for a_, b_, what_ in pairs_:
        if a_ == b_ or REG.get(a_, {}).get("status") != "RUN" or REG.get(b_, {}).get("status") != "RUN":
            continue
        pa_, pb_ = load_pred("event", a_), load_pred("event", b_)
        for kd_ in ("dir", "vol", "dd"):
            rows_.append({"comparison": what_, "A": a_, "B": b_, "target": kd_, **paired_delta_family(EVENT, kd_, pa_, pb_, EVENT.mask["val"])})
    NLP_ABL_HI = pd.DataFrame(rows_)
    if len(NLP_ABL_HI):
        NLP_ABL_HI["p_holm"] = holm(NLP_ABL_HI["p"].values); NLP_ABL_HI["B"] = BOOT["B"]; _save_post_hoc(NLP_ABL_HI, "nlp_ablations_val_hiB.csv", index=False)
    print(f"NLP ablations ({len(NLP_ABL_HI)} tests, B={BOOT['B']}): {(time.time() - t0_) / 60:.1f} min")
except Exception as e_:
    failed("phase_a_nlp_ablations", e_)
try:
    t0_ = time.time(); rows_ = []
    for n_ in FROZEN_CV:
        a_, b_ = f"{n_}_ft", f"{n_}_frozen"
        if REG.get(a_, {}).get("status") == "RUN" and REG.get(b_, {}).get("status") == "RUN":
            for kd_ in ("dir", "vol", "dd"):
                rows_.append({"family": "H5 fine-tuned vs frozen", "model": n_, "target": kd_, **paired_delta_family(PANEL, kd_, load_pred("panel", a_), load_pred("panel", b_), TEM)})
    H5_HI = pd.DataFrame(rows_)
    if len(H5_HI):
        H5_HI["p_holm"] = holm(H5_HI["p"].values); H5_HI["supported"] = (H5_HI["p_holm"] < CFG["alpha"]) & (H5_HI["delta"] > 0); H5_HI["B"] = BOOT["B"]
        _save_post_hoc(H5_HI, "test_H5_hiB.csv", index=False)
    print(f"H5 on the test block ({len(H5_HI)} tests, B={BOOT['B']}): {(time.time() - t0_) / 60:.1f} min")
except Exception as e_:
    failed("phase_a_h5", e_)
finally:
    BOOT["B"] = None

### 23A.4 Monte Carlo ranges: one volatility scale factor, chosen on validation

In [ ]:
MC_SCALE_VAL = MC_TEST_SCALED = None
try:
    if REG.get("deploy_recipe", {}).get("status") == "RUN":
        grid_ = [1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.4]; rows_ = []
        for k_ in grid_:
            d_ = mc_eval("deploy_recipe", VM, MC_CFG["nu"], CFG["mc"]["eval_rows"], seed=0, scale=k_)
            rows_.append({"scale": k_, "coverage80": float(d_.in80.mean()), "coverage90": float(d_.in90.mean()),
                          "brier_dd_mc": float(np.mean((d_.p_dd_mc - d_.dd) ** 2)), "n": len(d_)})
        MC_SCALE_VAL = pd.DataFrame(rows_); MC_SCALE_VAL["score"] = (MC_SCALE_VAL.coverage80 - 0.8).abs() + (MC_SCALE_VAL.coverage90 - 0.9).abs()
        k_best = float(MC_SCALE_VAL.scale.values[int(MC_SCALE_VAL.score.values.argmin())])
        MC_CFG["scale"] = k_best; (SUB["results"] / "mc_config.json").write_text(json.dumps(MC_CFG))
        d_ = mc_eval("deploy_recipe", TEM, MC_CFG["nu"], CFG["mc"]["eval_rows"], seed=1, scale=k_best)
        MC_TEST_SCALED = pd.DataFrame([{"scale": k_best, "nu": MC_CFG["nu"] or "normal", "coverage80": float(d_.in80.mean()), "coverage90": float(d_.in90.mean()),
                                        "brier_dd_mc": float(np.mean((d_.p_dd_mc - d_.dd) ** 2)), "brier_dd_head": float(np.mean((d_.p_dd_head - d_.dd) ** 2)), "n": len(d_)}])
        _save_post_hoc(MC_SCALE_VAL, "mc_scale_val.csv", index=False); _save_post_hoc(MC_TEST_SCALED, "mc_calibration_test_scaled.csv", index=False)
        print(f"Chosen volatility scale on validation: {k_best}"); display(MC_SCALE_VAL.round(3)); display(MC_TEST_SCALED.round(3))
except Exception as e_:
    failed("phase_a_mc_scale", e_)

### 23A.5 Reproducibility check against the archived first run, and the post-hoc ledger

Before this addendum was run, the artifacts folder of the first (confirmatory) run was copied to `artifacts_full_research_v3.0_original`. The check below recomputes nothing: it compares the confirmatory tables written by this execution with the archived ones. Differences should be zero (up to floating-point noise).

In [ ]:
REPRO = pd.DataFrame()
try:
    ARCH = DRIVE_ROOT / "artifacts_full_research_v3.0_original" / "tables"
    rows_ = []
    if ARCH.exists():
        for f_, col_, keys_ in [("test_H1.csv", "skill", ["task"]), ("test_H2.csv", "skill", ["task"]), ("test_H2b.csv", "skill", ["task"]), ("test_H3.csv", "delta", ["task"]),
                                ("test_H4.csv", "delta", ["task"]), ("test_deploy.csv", "skill", ["task"]), ("test_H5.csv", "delta", ["model", "target"]),
                                ("cv_leaderboard_test.csv", "primary", ["model", "task"]), ("fusion_leaderboard_test.csv", "primary", ["model", "task"]),
                                ("nlp_leaderboard_test.csv", "primary", ["model", "task"]), ("nlp_panel_leaderboard_test.csv", "primary", ["model", "task"])]:
            pa_, pb_ = ARCH / f_, SUB["tables"] / f_
            if not (pa_.exists() and pb_.exists()):
                rows_.append({"table": f_, "rows_compared": 0, "max_abs_difference": np.nan, "status": "missing"}); continue
            a_df, b_df = pd.read_csv(pa_), pd.read_csv(pb_)
            m_ = a_df.dropna(subset=[col_]).merge(b_df.dropna(subset=[col_]), on=keys_, suffixes=("_old", "_new"))
            dmax = float(np.max(np.abs(m_[f"{col_}_old"] - m_[f"{col_}_new"]))) if len(m_) else np.nan
            status_ = ("identical" if dmax < 1e-9 else "identical up to float rounding (< 1e-6)" if dmax < 1e-6 else "DIFFERENT") if np.isfinite(dmax) else "DIFFERENT"
            rows_.append({"table": f_, "rows_compared": len(m_), "max_abs_difference": f"{dmax:.1e}" if np.isfinite(dmax) else "-", "status": status_})
    else:
        rows_.append({"table": "(archive folder not found)", "rows_compared": 0, "max_abs_difference": np.nan, "status": "not checked"})
    REPRO = pd.DataFrame(rows_); _save_post_hoc(REPRO, "phase_a_repro_check.csv", index=False); display(REPRO)
except Exception as e_:
    failed("phase_a_repro_check", e_)

_log = json.loads(VAULT_LOG.read_text()) if VAULT_LOG.exists() else []
PHASE_A = {"utc": now_utc(), "post_hoc_tables": POST_HOC_TABLES, "post_freeze_models": sorted(POST_FREEZE_IDS), "bootstrap_B_high": 5000 if not QUICK else CFG["bootstrap_B"],
           "mc_scale": MC_CFG.get("scale", 1.0), "vault_openings": len(_log), "executions_that_opened_the_vault": int(sum(1 for r_ in _log if r_.get("item") == "panel_labels")),
           "repro_status": REPRO["status"].value_counts().to_dict() if len(REPRO) else {}}
(SUB["results"] / "phase_a_results.json").write_text(json.dumps(PHASE_A, indent=1, default=str))
print("Post-hoc ledger:", json.dumps(PHASE_A, default=str)[:600])
tick("23A phase a")

## 24. The Streamlit dashboard (product)

**What it does.** Up to 4 stocks side by side. For each: upload the **1-week, 1-day and 1-hour chart screenshots** (any TradingView-style chart), optionally type the ticker, and paste a **recent 10-K / 10-Q / 8-K summary** with its report type. Press *Analyse*. For 5, 10, 20 and 40 sessions it shows P(UP / SIDEWAYS / DOWN), the expected volatility and the probability of a >10 % drawdown, then a **20-day Monte Carlo** fan chart of price paths.

**How the screenshots are used (and why).** The models were trained on charts drawn in one exact format (§6). A screenshot from a broker has different colours, fonts, indicators, zoom and aspect ratio, which §12.7 shows can move predictions. So the dashboard *reads* each screenshot (OCR, TriFrame's tested reader: ticker candidates, timeframe and the last bar's O/H/L/C legend), confirms the ticker and the as-of date against market data, then **redraws the three charts in the training format** from Yahoo bars (daily 3 years, hourly 60 days) and feeds those to the models. Both the screenshots and the redrawn model inputs are displayed, so the user can see what the model saw.

**What it runs.** Exactly the frozen `deploy_recipe` (§20) whose test-block results are shown in the app: the top-3 CV ensemble per task; when text is supplied, blended with the top-3 deployable NLP models using the validation-chosen weights. **No-call rule:** a direction call (UP / SIDEWAYS / DOWN) is printed only for horizons where the recipe's direction skill was significant on the locked test block; elsewhere the probabilities are shown with "no validated direction skill". Volatility and drawdown are always shown with their test scores.

In [ ]:
SCREEN_SRC = r'''
"""Reads ticker candidates, timeframe and the last-bar O/H/L/C legend from a TradingView-style chart screenshot.
From TriFrame (tested on 66 real TradingView dark-theme screenshots: timeframe 100 %, legend O/H/L/C 100 %,
true ticker among the candidates 66/66). The app confirms a candidate by matching the legend against market data."""
import re
import numpy as np
from PIL import Image

_OCR = None
TF_MAP = {"1W": "1W", "W": "1W", "1D": "1D", "D": "1D", "1H": "1H", "60": "1H"}
STOP = {"USD", "SELL", "BUY", "VOL", "SMA", "EMA", "NASDAQ", "NYSE", "AMEX", "ARCA", "BATS", "OTC", "TRADE", "SAVE", "ALERT",
        "PRE", "POST", "MACD", "RSI", "LOG", "AUTO", "INC", "CORP", "LTD", "CO", "PLC", "THE", "AND", "OF", "CLOSE", "OPEN",
        "HIGH", "LOW", "ADJ", "EXT", "UTC"}


def _ocr():
    global _OCR
    if _OCR is None:
        try:
            from rapidocr_onnxruntime import RapidOCR
            _OCR = RapidOCR()
        except ImportError:
            from rapidocr import RapidOCR
            _new = RapidOCR()
            def _OCR(arr):
                r = _new(arr)
                if r is None or getattr(r, "boxes", None) is None:
                    return [], 0.0
                return [[b.tolist() if hasattr(b, "tolist") else b, t, s] for b, t, s in zip(r.boxes, r.txts, r.scores)], 0.0
    return _OCR


def _num(s):
    try:
        return float(s.replace(",", ""))
    except Exception:
        return None


def _rows(items):
    boxes = []
    for box, text, _ in items:
        ys = [p[1] for p in box]; xs = [p[0] for p in box]
        boxes.append((sum(ys) / 4, max(ys) - min(ys), min(xs), text.strip()))
    boxes.sort()
    rows, cur = [], []
    for b in boxes:
        if cur and abs(b[0] - cur[-1][0]) > 0.6 * max(b[1], cur[-1][1], 1):
            rows.append(sorted(cur, key=lambda z: z[2])); cur = []
        cur.append(b)
    if cur:
        rows.append(sorted(cur, key=lambda z: z[2]))
    return [[z[3] for z in r if z[3]] for r in rows]


def _toolbar_candidates(rows):
    out = []
    for r in rows[:3]:
        for j, t in enumerate(r):
            if re.search(r"[OHLC0][\d,]+\.\d", t) or "·" in t:
                continue
            for tok in re.findall(r"[A-Za-z][A-Za-z\.\-]{0,5}", t):
                up = tok.upper().strip(".-")
                if not (1 <= len(up) <= 5) or up in STOP or up in TF_MAP:
                    continue
                if sum(c.islower() for c in tok) > 1:
                    continue
                if len(up) == 1 and j == 0 and len(r) > 1:
                    continue
                up = up.replace(".", "-")
                if up not in out:
                    out.append(up)
    return out


def _price_tag_candidates(img):
    w, h = img.size
    right = img.crop((int(w * 0.72), 0, w, h))
    if right.size[0] < 800:
        right = right.resize((right.size[0] * 2, right.size[1] * 2), Image.LANCZOS)
    res, _ = _ocr()(np.asarray(right))
    out = []
    for box, text, _ in sorted(res or [], key=lambda r: min(p[1] for p in r[0])):
        tok = text.strip()
        if re.fullmatch(r"[A-Z][A-Z\.\-]{0,5}", tok) and tok not in STOP and tok.replace(".", "-") not in out:
            out.append(tok.replace(".", "-"))
    return out


def read_header(img, deep=False):
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    img = img.convert("RGB")
    w, h = img.size
    top = img.crop((0, 0, int(w * 0.75), max(60, int(h * 0.12))))
    if top.size[0] < 1600:
        top = top.resize((top.size[0] * 2, top.size[1] * 2), Image.LANCZOS)
    res, _ = _ocr()(np.asarray(top))
    rows = _rows(res or [])
    joined = " | ".join(" ".join(r) for r in rows)
    tf = None
    m = re.search(r"[·•]\s*(1W|1D|1H|60)\s*[·•]", joined, re.I)
    if m:
        tf = TF_MAP.get(m.group(1).upper())
    if tf is None and rows:
        for t in rows[0]:
            if t.strip().upper() in TF_MAP:
                tf = TF_MAP[t.strip().upper()]; break
    ohlc = None
    flat = re.sub(r"\s+", "", joined)
    m = re.search(r"[O0]([\d,]*\d\.\d+)H([\d,]+\.\d+)L([\d,]+\.\d+)C([\d,]+\.\d+)", flat)
    if m:
        vals = [_num(x) for x in m.groups()]
        if all(v is not None for v in vals) and vals[1] >= max(vals[0], vals[3]) * 0.999 and vals[2] <= min(vals[0], vals[3]) * 1.001:
            ohlc = dict(zip(("open", "high", "low", "close"), vals))
    name = None
    m = re.search(r"([A-Za-z][^|·•]{2,80}?)\s*[·•]\s*(?:1W|1D|1H|60)\b", joined, re.I)
    if m:
        name = re.sub(r"^[^A-Za-z]+", "", m.group(1)).strip()
    cands = _toolbar_candidates(rows)
    if deep or not cands:
        cands += [c for c in _price_tag_candidates(img) if c not in cands]
    return dict(candidates=cands, timeframe=tf, ohlc=ohlc, name=name, header_text=joined[:300])
'''

ENGINE_SRC = r'''
"""NEXUS Alpha v3 dashboard engine: loads the frozen bundle and reproduces the notebook's deploy_recipe chain exactly."""
import os
import json
import math
import numpy as np
import pandas as pd
import torch
from nav3_core import (PALETTE_DARK, HORIZONS, CLASSES, ChartNet, TextNet, prepare_arrays, render_views, trailing_sigma,
                       chart_raw_outputs, text_raw_outputs, apply_calibration, raw_from_probs, blend, text_input, mc_simulate)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NY = "America/New_York"
_CACHE = {}


def load_bundle(path):
    with open(os.path.join(path, "bundle.json")) as f:
        B = json.load(f)
    B["dir"] = path
    return B


def _cv_model(B, mid):
    if ("cv", mid) not in _CACHE:
        ck = torch.load(B["cv_models"][mid]["path"], map_location="cpu", weights_only=False)
        net = ChartNet(ck["spec"], pretrained=False)
        net.load_state_dict(ck["state_dict"])
        _CACHE[("cv", mid)] = (net.to(DEV).eval(), np.asarray(ck["vol_mu"]), np.asarray(ck["vol_sd"]))
    return _CACHE[("cv", mid)]


def _nlp_model(B, mid):
    if ("nlp", mid) not in _CACHE:
        spec = B["nlp_models"][mid]
        if spec["kind"] == "tfidf":
            import joblib
            _CACHE[("nlp", mid)] = joblib.load(spec["path"])
        else:
            from transformers import AutoTokenizer
            ck = torch.load(spec["path"], map_location="cpu", weights_only=False)
            net = TextNet(ck["hf"], from_config=True)
            net.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()})
            _CACHE[("nlp", mid)] = (net.to(DEV).eval(), AutoTokenizer.from_pretrained(ck["hf"]), int(ck["max_len"]),
                                    np.asarray(ck["vol_mu"]), np.asarray(ck["vol_sd"]), bool(ck.get("amp", True)))
    return _CACHE[("nlp", mid)]


def cv_chain(B, img_idx):
    """img_idx: (n, 3, 224, 224) uint8 palette indices -> calibrated cv_top3 predictions."""
    pal = torch.tensor(PALETTE_DARK, dtype=torch.float32, device=DEV)
    cal = {}
    for mid in B["cv_union"]:
        net, mu, sd = _cv_model(B, mid)
        cal[mid] = apply_calibration(chart_raw_outputs(net, img_idx, pal, mu, sd, DEV), B["cal"][mid])
    avg = {t: np.mean([cal[m][t] for m in B["cv_members"][t]], axis=0) for t in B["tasks"]}
    return apply_calibration(raw_from_probs(avg), B["cal"]["cv_top3"])


def nlp_raw(B, mid, texts):
    spec = B["nlp_models"][mid]
    obj = _nlp_model(B, mid)
    if spec["kind"] == "tfidf":
        X = obj["vectorizer"].transform(texts)
        raw = {}
        for t in B["tasks"]:
            m = obj["models"][t]
            kd = t.split("_")[0]
            if kd == "vol":
                raw[t] = m.predict(X)
            elif kd == "dir":
                raw[t] = np.log(np.clip(m.predict_proba(X), 1e-9, 1))
            else:
                q = np.clip(m.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)
                raw[t] = np.log(q / (1 - q))
        return raw
    net, tok, L, mu, sd, amp = obj
    return text_raw_outputs(net, tok, [x[: L * 8] for x in texts], L, mu, sd, DEV, batch=4, use_amp=amp)


def nlp_chain(B, texts):
    tr = {}
    for mid in B["nlp_union"]:
        ev = apply_calibration(nlp_raw(B, mid, texts), B["cal_event"][mid])
        tr[mid] = apply_calibration(raw_from_probs(ev), B["cal"][B["nlp_models"][mid]["transfer_id"]])
    avg = {t: np.mean([tr[m][t] for m in B["nlp_members"][t]], axis=0) for t in B["tasks"]}
    return apply_calibration(raw_from_probs(avg), B["cal"]["nlp_deploy_top3"])


def final_chain(B, cv, nlp=None):
    mix = cv if nlp is None else blend(cv, nlp, B["w_fresh"])
    return apply_calibration(raw_from_probs(mix), B["cal"]["deploy_recipe"])


def fetch_market(ticker, asof=None):
    """Yahoo daily (3 years) and hourly (60 days) bars, split-adjusted and not dividend-adjusted (as in training);
    today's unfinished session is dropped; asof (date) truncates both."""
    import yfinance as yf
    tk = yf.Ticker(ticker.upper().replace(".", "-"))
    d = tk.history(period="3y", interval="1d", auto_adjust=False, actions=False)
    h = tk.history(period="60d", interval="60m", auto_adjust=False, actions=False, prepost=False)
    if d is None or d.empty or h is None or h.empty:
        raise ValueError(f"Yahoo returned no data for {ticker}")
    di = pd.DatetimeIndex(d.index)
    di = di.tz_convert(NY).tz_localize(None) if di.tz is not None else di
    D = pd.DataFrame({"date": di.normalize(), "open": d["Open"].values, "high": d["High"].values, "low": d["Low"].values,
                      "close": d["Close"].values, "volume": d["Volume"].values.astype(float)}).dropna().reset_index(drop=True)
    hi = pd.DatetimeIndex(h.index)
    hi = hi.tz_localize(NY) if hi.tz is None else hi.tz_convert(NY)
    Hh = pd.DataFrame({"timestamp": hi.tz_convert("UTC"), "session": hi.tz_localize(None).normalize(), "minute": hi.minute,
                       "open": h["Open"].values, "high": h["High"].values, "low": h["Low"].values, "close": h["Close"].values,
                       "volume": h["Volume"].values.astype(float)}).dropna()
    Hh = Hh[Hh["minute"] == 30].drop(columns="minute").reset_index(drop=True)
    now = pd.Timestamp.now(tz=NY)
    if len(D) and D["date"].iloc[-1] == now.tz_localize(None).normalize() and now.hour * 60 + now.minute < 16 * 60 + 15:
        D = D.iloc[:-1]
    if asof is not None:
        D = D[D["date"] <= pd.Timestamp(asof)]
    Hh = Hh[Hh["session"].isin(set(D["date"]))].reset_index(drop=True)
    return D.reset_index(drop=True), Hh


def match_legend(D, ohlc, tol=0.006):
    """Session whose daily O/H/L/C matches a screenshot legend (pins the as-of date), else None."""
    if not ohlc:
        return None
    x = D.tail(400)
    err = np.max(np.abs(x[["open", "high", "low", "close"]].values / np.array([ohlc["open"], ohlc["high"], ohlc["low"], ohlc["close"]]) - 1), axis=1)
    k = int(np.argmin(err))
    return x["date"].iloc[k] if err[k] < tol else None


def analyse(B, ticker, screenshots, form, text, n_paths=4000, seed=7):
    """screenshots: dict '1W'/'1D'/'1H' -> PIL image or None. Returns everything the page shows."""
    info, how = {}, "typed by the user"
    if any(v is not None for v in screenshots.values()):
        try:
            import screen_reader
            for slot, im in screenshots.items():
                if im is not None:
                    info[slot] = screen_reader.read_header(im)
        except Exception as e:
            info["error"] = f"{type(e).__name__}: {e}"
    cands = [c for s in ("1D", "1W", "1H") for c in (info.get(s) or {}).get("candidates", [])]
    legend = (info.get("1D") or {}).get("ohlc")
    tk = (ticker or "").strip().upper()
    D = Hh = None
    if not tk:
        for c in cands[:6]:
            try:
                D_, H_ = fetch_market(c)
            except Exception:
                continue
            if legend is None or match_legend(D_, legend) is not None:
                tk, D, Hh, how = c, D_, H_, ("read from the screenshot and confirmed by its price legend" if legend else "read from the screenshot (unconfirmed)")
                break
        if not tk:
            raise ValueError("Could not identify the ticker from the screenshots: type it in the ticker box.")
    if D is None:
        D, Hh = fetch_market(tk)
    asof = match_legend(D, legend)
    if asof is not None:
        D, Hh = fetch_market(tk, asof)
        how += f"; as-of date pinned by the daily screenshot's legend ({pd.Timestamp(asof).date()})"
    if len(D) < 300 or len(Hh) < 70:
        raise ValueError(f"Not enough history for {tk} (daily {len(D)}, hourly {len(Hh)} bars).")
    S = prepare_arrays(D, Hh)
    img = render_views(S, len(D) - 1)[None]
    cv = cv_chain(B, img)
    nlp = nlp_chain(B, [text_input(form, "summary", text)]) if (text or "").strip() and B["nlp_union"] else None
    fin = final_chain(B, cv, nlp)
    p0 = float(D["close"].iloc[-1]); sig = trailing_sigma(D["close"].values, 60)
    sim = mc_simulate(p0, sig, math.exp(float(fin["vol_20"][0])) * float(B.get("mc_scale", 1.0)), fin["dir_20"][0], nu=B["mc_nu"], n_days=20, n_paths=n_paths, seed=seed)
    rows = []
    for h in HORIZONS:
        p = fin[f"dir_{h}"][0]; k = int(np.argmax(p)); thr = B["k_sideways"] * sig * math.sqrt(h)
        call = CLASSES[k] if not B["no_call"].get(f"dir_{h}", True) else "NO CALL"
        rows.append({"horizon (sessions)": h, "P(UP)": float(p[2]), "P(SIDEWAYS)": float(p[1]), "P(DOWN)": float(p[0]), "call": call,
                     "sideways band": f"+/-{100 * (math.exp(thr) - 1):.1f}%", "expected vol (ann.)": float(math.exp(fin[f"vol_{h}"][0])),
                     "P(drawdown > 10%)": float(fin[f"dd_{h}"][0]),
                     "CV only P(UP)": float(cv[f"dir_{h}"][0][2]), "text only P(UP)": (float(nlp[f"dir_{h}"][0][2]) if nlp is not None else None)})
    return {"ticker": tk, "asof": str(pd.Timestamp(D["date"].iloc[-1]).date()), "how": how, "p0": p0, "sigma": sig, "img": img[0],
            "table": pd.DataFrame(rows), "sim": sim, "used_text": nlp is not None, "in_universe": tk.replace("-", ".") in B["universe"],
            "ocr": {k: {kk: v for kk, v in (vv or {}).items() if kk != "header_text"} if isinstance(vv, dict) else vv for k, vv in info.items()}}
'''

APP_SRC = r'''
import os
import sys
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from PIL import Image
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import nav3_engine as E
from nav3_core import PALETTE_DARK

st.set_page_config(page_title="NEXUS Alpha v3", page_icon="📈", layout="wide")


@st.cache_resource
def bundle():
    return E.load_bundle(os.environ.get("NAV3_BUNDLE", os.path.dirname(os.path.abspath(__file__))))


B = bundle()
st.title("NEXUS Alpha v3: charts + SEC filings → direction, volatility, drawdown")
st.caption(f"{B['stamp']} · models frozen {B['frozen_utc']} · trained on {len(B['universe'])} US stocks (2016-2021), chosen on 2022-2023, "
           "scored once on 2024-2026. Educational research, not financial advice.")
with st.expander("What the research found on the locked test block (read before using the numbers)"):
    for k, v in B["answers"].items():
        st.markdown(f"**{k}.** {v}")
    if B.get("deploy_skill"):
        st.dataframe(pd.DataFrame(B["deploy_skill"])[["task", "skill", "lo", "hi", "p_holm", "supported"]], hide_index=True)
    st.markdown("Direction calls are shown only where the recipe's direction skill was significant on the test block; "
                "otherwise the probabilities are shown as 'NO CALL'.")

n = st.radio("How many stocks?", [1, 2, 3, 4], horizontal=True)
inputs = []
for i, tab in enumerate(st.tabs([f"Stock {i + 1}" for i in range(n)])):
    with tab:
        ups = {}
        for col, slot in zip(st.columns(3), ["1W", "1D", "1H"]):
            with col:
                up = st.file_uploader(f"{slot} chart screenshot", type=["png", "jpg", "jpeg"], key=f"up_{i}_{slot}")
                if up is not None:
                    st.image(up, width="stretch")
                ups[slot] = up
        c1, c2 = st.columns([1, 3])
        tk = c1.text_input("Ticker (optional if the screenshot shows it)", key=f"tk_{i}")
        form = c1.selectbox("Report type of the text", ["8-K", "10-Q", "10-K"], key=f"form_{i}")
        txt = c2.text_area("Recent 10-K / 10-Q / 8-K summary (optional)", height=140, key=f"txt_{i}")
        inputs.append((ups, tk, form, txt))
with st.expander("Monte Carlo settings"):
    a1, a2, a3 = st.columns(3)
    n_paths = a1.select_slider("Simulated paths", [1000, 2000, 4000, 8000, 16000], value=4000)
    n_show = a2.slider("Paths drawn", 0, 100, 40, step=10)
    seed = int(a3.number_input("Random seed", 0, 10000, 7))

if st.button("Analyse", type="primary"):
    res = []
    bar = st.progress(0.0, text="Starting...")
    for j, (ups, tk, form, txt) in enumerate(inputs):
        bar.progress(j / len(inputs), text=f"Stock {j + 1}: reading screenshots, rebuilding charts, predicting...")
        if not tk.strip() and not any(u is not None for u in ups.values()):
            continue
        shots = {k: (Image.open(u).convert("RGB") if u is not None else None) for k, u in ups.items()}
        try:
            r = E.analyse(B, tk, shots, form, txt, n_paths=n_paths, seed=seed); r["slot"] = j + 1; r["shots"] = shots; res.append(r)
        except Exception as e:
            res.append({"slot": j + 1, "error": f"{type(e).__name__}: {e}"})
    bar.empty()
    st.session_state["res"] = res

def fan(r, n_show):
    sim = r["sim"]; x = list(range(0, 21)); q = sim["quantiles"]; p0 = r["p0"]
    fig = go.Figure()
    for k in range(min(n_show, sim["paths"].shape[0])):
        fig.add_trace(go.Scatter(x=x, y=[p0] + list(sim["paths"][k]), mode="lines", line=dict(width=0.6, color="rgba(150,150,150,0.35)"), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=x, y=[p0] + list(q[4]), line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=x, y=[p0] + list(q[0]), fill="tonexty", fillcolor="rgba(37,99,235,0.15)", line=dict(width=0), name="5-95 %"))
    fig.add_trace(go.Scatter(x=x, y=[p0] + list(q[3]), line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=x, y=[p0] + list(q[1]), fill="tonexty", fillcolor="rgba(37,99,235,0.3)", line=dict(width=0), name="25-75 %"))
    fig.add_trace(go.Scatter(x=x, y=[p0] + list(q[2]), line=dict(color="#2563eb", width=2), name="median"))
    fig.add_hline(y=p0 * 0.9, line_dash="dot", line_color="#dc2626", annotation_text="-10 %")
    fig.update_layout(height=340, margin=dict(l=10, r=10, t=30, b=10), xaxis_title="sessions ahead", yaxis_title="price",
                      title=f"{r['ticker']}: 20-session Monte Carlo ({sim['paths'].shape[0]:,} paths)")
    return fig

ok = [r for r in st.session_state.get("res", []) if "error" not in r]
for r in st.session_state.get("res", []):
    if "error" in r:
        st.error(f"Stock {r['slot']}: {r['error']}")
for r in ok:
    st.markdown(f"### {r['ticker']} — as of {r['asof']} (last close ${r['p0']:,.2f})")
    st.caption(f"Ticker {r['how']}. Text used: {'yes' if r['used_text'] else 'no'}." + ("" if r["in_universe"] else
               " Note: this stock is outside the 50 training stocks; treat the output as an out-of-sample extrapolation."))
    cA, cB = st.columns([1, 1])
    with cA:
        st.markdown("**Your screenshots**")
        cc = st.columns(3)
        for c_, k in zip(cc, ["1W", "1D", "1H"]):
            if r["shots"].get(k) is not None:
                c_.image(r["shots"][k], caption=k, width="stretch")
    with cB:
        st.markdown("**What the models saw (redrawn in the training format)**")
        cc = st.columns(3)
        for c_, j, k in zip(cc, range(3), ["weekly 52", "daily 63", "hourly 70"]):
            c_.image(PALETTE_DARK[r["img"][j]], caption=k, width="stretch")
    t = r["table"].copy()
    for c in ["P(UP)", "P(SIDEWAYS)", "P(DOWN)", "P(drawdown > 10%)", "CV only P(UP)", "text only P(UP)"]:
        t[c] = t[c].map(lambda v: "-" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{100 * v:.0f}%")
    t["expected vol (ann.)"] = t["expected vol (ann.)"].map(lambda v: f"{100 * v:.0f}%")
    st.dataframe(t, hide_index=True, width="stretch")
    sim = r["sim"]
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Simulated P(UP, 20 d)", f"{100 * sim['sim_dir'][2]:.0f}%"); c2.metric("Simulated P(DOWN, 20 d)", f"{100 * sim['sim_dir'][0]:.0f}%")
    c3.metric("Simulated P(drawdown > 10 %)", f"{100 * sim['p_dd']:.0f}%"); c4.metric("Median price in 20 d", f"${sim['quantiles'][2][-1]:,.2f}")
    st.plotly_chart(fan(r, n_show), width="stretch")
    st.markdown("---")
if len(ok) > 1:
    st.subheader("Comparison")
    st.dataframe(pd.DataFrame([{"stock": r["ticker"], "as of": r["asof"], "20d call": r["table"].iloc[2]["call"],
                                "P(UP) 20d": f"{100 * r['table'].iloc[2]['P(UP)']:.0f}%", "vol 20d": f"{100 * r['table'].iloc[2]['expected vol (ann.)']:.0f}%",
                                "P(DD>10%) 20d": f"{100 * r['table'].iloc[2]['P(drawdown > 10%)']:.0f}%",
                                "MC 5-95 % (20d)": f"${r['sim']['quantiles'][0][-1]:,.0f} - ${r['sim']['quantiles'][4][-1]:,.0f}"} for r in ok]), hide_index=True)
'''

In [ ]:
APP_DIR = SUB["app"]; (APP_DIR / ".streamlit").mkdir(parents=True, exist_ok=True)
CV_UNION = sorted({m for t in TASKS for m in TOP3_CV.get(t, [])})
NLP_UNION = sorted({m for t in TASKS for m in NLP_DEPLOY_TOP3.get(t, [])})
for m in CV_UNION + ["cv_top3", "nlp_deploy_top3", "deploy_recipe"] + [_transfer_id(x) for x in NLP_UNION]:
    load_pred("panel", m)
for m in NLP_UNION:
    load_pred("event", m)
BUNDLE = {"stamp": STAMP, "frozen_utc": FREEZE["frozen_utc"], "tasks": TASKS, "horizons": HORIZONS, "classes": CLASSES, "k_sideways": K_SIDE,
          "universe": TICKERS, "sector_of": SECTOR_OF,
          "cv_union": CV_UNION, "cv_members": TOP3_CV, "cv_models": {m: {"path": str(SUB["models"] / f"cv_{m}.pt"), "label": LABEL_OF.get(m, m)} for m in CV_UNION},
          "nlp_union": NLP_UNION, "nlp_members": NLP_DEPLOY_TOP3,
          "nlp_models": {m: {"kind": "tfidf" if m == "tfidf_lr" else "encoder",
                             "path": str(SUB["models"] / ("tfidf_lr.joblib" if m == "tfidf_lr" else f"nlp_{m}.pt")),
                             "transfer_id": _transfer_id(m), "label": LABEL_OF.get(m, m)} for m in NLP_UNION},
          "cal": {m: CALS["panel"][m] for m in CV_UNION + ["cv_top3", "nlp_deploy_top3", "deploy_recipe"] + [_transfer_id(x) for x in NLP_UNION] if m in CALS["panel"]},
          "cal_event": {m: CALS["event"][m] for m in NLP_UNION}, "w_fresh": FUSION_W.get("deploy_fresh", {t: 1.0 for t in TASKS}),
          "mc_nu": MC_CFG["nu"], "mc_scale": MC_CFG.get("scale", 1.0), "no_call": NO_CALL, "answers": ANSWERS, "deploy_skill": FINAL["deploy_skill"]}
(APP_DIR / "bundle.json").write_text(json.dumps(BUNDLE, indent=1, default=str))
(APP_DIR / "nav3_core.py").write_text(CORE_SRC); (APP_DIR / "nav3_engine.py").write_text(ENGINE_SRC)
(APP_DIR / "screen_reader.py").write_text(SCREEN_SRC); (APP_DIR / "app.py").write_text(APP_SRC)
(APP_DIR / ".streamlit" / "config.toml").write_text('[theme]\nbase = "dark"\nprimaryColor = "#26a69a"\n[server]\nmaxUploadSize = 20\n')
print("Dashboard written to", APP_DIR, "| CV members:", CV_UNION, "| NLP members:", NLP_UNION)

In [ ]:
# ---------------------------------------------------------------- parity test: the dashboard engine reproduces the notebook's test predictions
import importlib
sys.path.insert(0, str(APP_DIR))
for mod in ("nav3_core", "nav3_engine", "screen_reader"):
    sys.modules.pop(mod, None)
ENG = importlib.import_module("nav3_engine")
BE = ENG.load_bundle(str(APP_DIR))
PAR = []
if HAS_GPU and CV_UNION:
    dep = load_pred("panel", "deploy_recipe")
    tr_cand = np.where(HAS_8K & (~HAS_PER | (np.nan_to_num(AGE["8k"], nan=1e9) <= np.nan_to_num(AGE["per"], nan=1e9))), LATEST["8k"], LATEST["per"])
    rr_f = np.flatnonzero(TEM & FRESH); rr_n = np.flatnonzero(TEM & ~FRESH)
    pick = list(np.random.default_rng(4).choice(rr_f, min(4, len(rr_f)), replace=False)) + list(np.random.default_rng(5).choice(rr_n, min(4, len(rr_n)), replace=False))
    for r in pick:
        cv = ENG.cv_chain(BE, np.asarray(IMG[[r]]))
        nlp = ENG.nlp_chain(BE, [EV_TEXT[int(tr_cand[r])]]) if FRESH[r] and NLP_UNION else None
        fin = ENG.final_chain(BE, cv, nlp)
        PAR.append({"row": int(r), "fresh_text": bool(FRESH[r]), "max_abs_dprob_dir": float(max(np.abs(fin[f"dir_{h}"][0] - dep[f"dir_{h}"][r]).max() for h in HORIZONS)),
                    "max_abs_dlogvol": float(max(abs(fin[f"vol_{h}"][0] - dep[f"vol_{h}"][r]) for h in HORIZONS)),
                    "max_abs_dprob_dd": float(max(abs(fin[f"dd_{h}"][0] - dep[f"dd_{h}"][r]) for h in HORIZONS))})
    PAR_DF = pd.DataFrame(PAR); PAR_DF.to_csv(SUB["tables"] / "dashboard_parity.csv", index=False); display(PAR_DF.round(5))
    PARITY_OK = bool((PAR_DF[["max_abs_dprob_dir", "max_abs_dlogvol", "max_abs_dprob_dd"]] < 0.02).all().all())
    print("Dashboard parity passed: engine = notebook within 0.02." if PARITY_OK else
          "WARNING: dashboard parity above 0.02 (see dashboard_parity.csv). Reports still build; send the table to Claude Code.")
    ENG._CACHE.clear(); torch.cuda.empty_cache()

In [ ]:
# ---------------------------------------------------------------- launch (Colab port proxy; optional public tunnel, OFF by default)
PUBLIC_TUNNEL = False      # True = cloudflared quick tunnel: anyone with the link can use the app while this runtime lives
import urllib.request
log_f = open(APP_DIR / "streamlit.log", "w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP_DIR / "app.py"), "--server.port", "8501", "--server.headless", "true",
                         "--browser.gatherUsageStats", "false"], cwd=str(APP_DIR), env={**os.environ, "NAV3_BUNDLE": str(APP_DIR)}, stdout=log_f, stderr=log_f)
for _ in range(90):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read().decode().strip() == "ok":
            break
    except Exception:
        time.sleep(1)
else:
    print("Streamlit did not start; log:"); print((APP_DIR / "streamlit.log").read_text()[-3000:])
if IN_COLAB:
    from google.colab import output
    print("Open the dashboard:", output.eval_js("google.colab.kernel.proxyPort(8501)"))
    output.serve_kernel_port_as_window(8501)
if PUBLIC_TUNNEL:
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("/content/cloudflared", 0o755)
    tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
    for line in tun.stderr or []:
        m_ = re.search(r"https://\S+trycloudflare.com", line)
        if m_:
            print("PUBLIC URL (anyone with it can use the app):", m_.group(0)); break
tick("24 dashboard")

## 25. The three reports (Computer Vision, NLP, Multimodal)

Built with ReportLab **only from artifacts saved by this run** (tables, figures, registry, freeze manifest, final results). The fixed text describes the design; every number, including every number in the discussion and conclusion paragraphs, is read from a file or a table at build time. A model that did not run appears as "Not Run". Missing artifacts are listed on the first page and the title starts with "INCOMPLETE DRAFT". In QUICK_VERIFY mode every page carries the stamp. Tables and paragraphs marked *post-hoc* come from the addendum §23A, which was written after the test block had been opened. Filing text is never printed beyond short n-grams and tokens.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, KeepTogether
from reportlab.platypus import Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from PIL import Image as PILImage

FONT, FONT_B = "Helvetica", "Helvetica-Bold"
if Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf").exists():
    pdfmetrics.registerFont(TTFont("DejaVu", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"))
    pdfmetrics.registerFont(TTFont("DejaVu-Bold", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"))
    FONT, FONT_B = "DejaVu", "DejaVu-Bold"
SS = getSampleStyleSheet()
BODY = ParagraphStyle("b", parent=SS["BodyText"], fontName=FONT, fontSize=9.5, leading=12.8, spaceAfter=5)
H1s = ParagraphStyle("h1", parent=SS["Heading1"], fontName=FONT_B, fontSize=14, spaceBefore=10, spaceAfter=6)
H2s = ParagraphStyle("h2", parent=SS["Heading2"], fontName=FONT_B, fontSize=11, spaceBefore=6, spaceAfter=4)
TITLE = ParagraphStyle("t", parent=SS["Title"], fontName=FONT_B, fontSize=17, leading=21)
CAP = ParagraphStyle("c", parent=BODY, fontSize=8, textColor=colors.HexColor("#555555"), spaceAfter=9)
TCAP = ParagraphStyle("tc", parent=BODY, fontName=FONT_B, fontSize=8.3, leading=10.5, spaceBefore=4, spaceAfter=3)
CELL = ParagraphStyle("cell", parent=BODY, fontSize=6.8, leading=8.2, spaceAfter=0)
REQUIRED_MISSING = []

def Pp(text, style=BODY):
    return Paragraph(re.sub(r"&(?!amp;|lt;|gt;|quot;|#\d+;)", "&amp;", str(text)), style)
def fmt(v, d=3):
    if v is None or (isinstance(v, (float, np.floating)) and not np.isfinite(v)):
        return "-"
    if isinstance(v, (float, np.floating)):
        return f"{v:.{d}f}"
    return str(v)
P_COLS = {"p", "p_holm", "p_dev", "p_raw"}
INT_COLS = {"n", "size", "n_perm", "tests", "better", "worse", "no clear difference", "rows", "rows_compared", "B"}
def fmt_cell(col, v, d=3):
    """p-values as '<0.001' or three decimals; learning rates in scientific notation; counts as integers."""
    if v is None or (isinstance(v, (float, np.floating)) and not np.isfinite(v)):
        return "-"
    if col in P_COLS and isinstance(v, (int, float, np.floating)):
        return "<0.001" if float(v) < 0.001 else f"{float(v):.3f}"
    if col == "lr" and isinstance(v, (int, float, np.floating)):
        return f"{float(v):.1e}"
    if col in INT_COLS and isinstance(v, (float, np.floating)) and float(v).is_integer():
        return str(int(v))
    return fmt(v, d)

PHASE_A_TABLES = {"phase_a_baseline_choice.csv", "test_H1c.csv", "test_H2c_events.csv", "test_H2c_panel.csv", "test_H3c.csv", "test_H8.csv", "test_H8b.csv", "test_H8c.csv",
                  "hypotheses_phase_a.csv", "phase_a_panel_test.csv", "phase_a_event_test.csv", "cv_ablations_val_hiB.csv", "nlp_ablations_val_hiB.csv", "test_H5_hiB.csv",
                  "mc_scale_val.csv", "mc_calibration_test_scaled.csv", "phase_a_repro_check.csv",
                  "test_H8d.csv", "test_H8e.csv", "test_H8f.csv", "test_H1d.csv", "test_H2d_events.csv"}
OPTIONAL_IN_QUICK = {"cv_lr_probe.csv", "nlp_lr_probe.csv", "cv_ablations_val.csv"} | PHASE_A_TABLES      # QUICK skips the probes, two ablations and parts of the addendum
def rtab(name):
    p_ = SUB["tables"] / name
    if not p_.exists():
        if not (QUICK and name in OPTIONAL_IN_QUICK):
            REQUIRED_MISSING.append(name)
        return None
    try:
        return pd.read_csv(p_, dtype={"item": str})
    except Exception:
        return None
def tbl(df, cols=None, max_rows=28, d=3, index=False, caption=None):
    if df is None or len(df) == 0:
        return Pp("<i>Not available (the step was Not Run or produced no rows).</i>")
    x = df.reset_index() if index else df
    x = x[[c for c in cols if c in x.columns]] if cols else x
    x = x.head(max_rows).iloc[:, :11]
    data = [[Paragraph(html.escape(str(c)), CELL) for c in x.columns]] + \
           [[Paragraph(html.escape(fmt_cell(str(c), v, d)), CELL) for c, v in zip(x.columns, row)] for row in x.values.tolist()]
    t = Table(data, repeatRows=1, colWidths=[17 * cm / len(x.columns)] * len(x.columns))
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e5e7eb")), ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#9ca3af")),
                           ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    return KeepTogether([Pp(caption, TCAP), t]) if caption else t
def figp(name, caption, width=16 * cm):
    p_ = SUB["figures"] / name
    if not p_.exists():
        REQUIRED_MISSING.append(name); return Pp(f"<i>Figure {name} not available.</i>")
    w, h = PILImage.open(p_).size
    return KeepTogether([RLImage(str(p_), width=width, height=min(width * h / w, 22 * cm)), Pp(caption, CAP)])
def mdtext(name):
    p_ = SUB["results"] / name
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", html.escape(p_.read_text())) if p_.exists() else "Not available."

REFS = {
 "araci": "Araci, D. (2019). FinBERT: Financial sentiment analysis with pre-trained language models. arXiv:1908.10063.",
 "baltrusaitis": "Baltrusaitis, T., Ahuja, C., & Morency, L.-P. (2019). Multimodal machine learning: A survey and taxonomy. IEEE TPAMI, 41(2), 423-443.",
 "beltagy": "Beltagy, I., Peters, M. E., & Cohan, A. (2020). Longformer: The long-document transformer. arXiv:2004.05150.",
 "cohen": "Cohen, L., Malloy, C., & Nguyen, Q. (2020). Lazy prices. The Journal of Finance, 75(3), 1371-1415.",
 "corsi": "Corsi, F. (2009). A simple approximate long-memory model of realized volatility. Journal of Financial Econometrics, 7(2), 174-196.",
 "dettmers": "Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). QLoRA: Efficient finetuning of quantized LLMs. NeurIPS 36.",
 "dosovitskiy": "Dosovitskiy, A., et al. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. ICLR.",
 "glasserman": "Glasserman, P., & Lin, C. (2023). Assessing look-ahead bias in stock return predictions generated by GPT sentiment analysis. arXiv:2309.17322.",
 "guo": "Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. ICML, PMLR 70, 1321-1330.",
 "he_resnet": "He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. CVPR, 770-778.",
 "he_deberta": "He, P., Gao, J., & Chen, W. (2023). DeBERTaV3: Improving DeBERTa using ELECTRA-style pre-training with gradient-disentangled embedding sharing. ICLR.",
 "hendrycks": "Hendrycks, D., & Dietterich, T. (2019). Benchmarking neural network robustness to common corruptions and perturbations. ICLR.",
 "ho": "Ho, T.-T., & Huang, Y. (2021). Stock price movement prediction using sentiment analysis and candlestick chart representation. Sensors, 21(23), 7957.",
 "holm": "Holm, S. (1979). A simple sequentially rejective multiple test procedure. Scandinavian Journal of Statistics, 6(2), 65-70.",
 "hu": "Hu, E. J., et al. (2022). LoRA: Low-rank adaptation of large language models. ICLR.",
 "huang": "Huang, A. H., Wang, H., & Yang, Y. (2023). FinBERT: A large language model for extracting information from financial text. Contemporary Accounting Research, 40(2), 806-841.",
 "jkx": "Jiang, J., Kelly, B., & Xiu, D. (2023). (Re-)Imag(in)ing price trends. The Journal of Finance, 78(6), 3193-3249.",
 "jocher": "Jocher, G., Chaurasia, A., & Qiu, J. (2023). Ultralytics YOLOv8 (software). https://github.com/ultralytics/ultralytics",
 "ke": "Ke, Z. T., Kelly, B. T., & Xiu, D. (2019). Predicting returns with text data. NBER Working Paper 26186.",
 "kunsch": "Kunsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. The Annals of Statistics, 17(3), 1217-1241.",
 "lerman": "Lerman, A., & Livnat, J. (2010). The new Form 8-K disclosures. Review of Accounting Studies, 15(4), 752-778.",
 "liu_convnext": "Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., & Xie, S. (2022). A ConvNet for the 2020s. CVPR, 11976-11986.",
 "liu_roberta": "Liu, Y., et al. (2019). RoBERTa: A robustly optimized BERT pretraining approach. arXiv:1907.11692.",
 "liu_swin": "Liu, Z., et al. (2021). Swin Transformer: Hierarchical vision transformer using shifted windows. ICCV, 10012-10022.",
 "loughran11": "Loughran, T., & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. The Journal of Finance, 66(1), 35-65.",
 "loughran16": "Loughran, T., & McDonald, B. (2016). Textual analysis in accounting and finance: A survey. Journal of Accounting Research, 54(4), 1187-1230.",
 "loukas": "Loukas, L., et al. (2022). FiNER: Financial numeric entity recognition for XBRL tagging. ACL 2022, 4419-4431.",
 "patton": "Patton, A. J. (2011). Volatility forecast comparison using imperfect volatility proxies. Journal of Econometrics, 160(1), 246-256.",
 "redmon": "Redmon, J., Divvala, S., Girshick, R., & Farhadi, A. (2016). You only look once: Unified, real-time object detection. CVPR, 779-788.",
 "sarkar": "Sarkar, S. K., & Vafa, K. (2024). Lookahead bias in pretrained language models. SSRN Working Paper 4754678.",
 "selvaraju": "Selvaraju, R. R., et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. ICCV, 618-626.",
 "sezer": "Sezer, O. B., & Ozbayoglu, A. M. (2019). Financial trading model with stock bar chart image time series with deep convolutional neural networks. Intelligent Automation & Soft Computing, 26(2), 323-334.",
 "sundararajan": "Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic attribution for deep networks. ICML, PMLR 70, 3319-3328.",
 "tan": "Tan, M., & Le, Q. (2019). EfficientNet: Rethinking model scaling for convolutional neural networks. ICML, PMLR 97, 6105-6114.",
 "tetlock": "Tetlock, P. C. (2007). Giving content to investor sentiment: The role of media in the stock market. The Journal of Finance, 62(3), 1139-1168.",
 "touvron": "Touvron, H., et al. (2023). Llama 2: Open foundation and fine-tuned chat models. arXiv:2307.09288.",
 "yang": "Yang, H., Liu, X.-Y., & Wang, C. D. (2023). FinGPT: Open-source financial large language models. arXiv:2306.06031.",
}
def refs(keys):
    st_ = ParagraphStyle("r", parent=BODY, fontSize=8.3, leading=10.5, leftIndent=12, firstLineIndent=-12)
    return [Paragraph(html.escape(REFS[k]), st_) for k in sorted(keys)]

_n_exec = int(sum(1 for r_ in json.loads(VAULT_LOG.read_text()) if r_.get("item") == "panel_labels")) if VAULT_LOG.exists() else 0
RUN_LINE = (f"Run {RUN_MODE} ({STAMP}); configuration {cfg_hash(CFG)}; freeze {FREEZE['selections_sha256']} made {FREEZE.get('frozen_utc')} "
            f"({len(FREEZE['freeze_history'])} earlier freezes; resumed {len(FREEZE.get('resumed_utc', []))} time(s) with identical selections and configuration); "
            f"the locked test labels were opened in {_n_exec} notebook execution(s) ({FINAL['test_openings']} logged openings: panel and filing-event labels once per execution, "
            f"later executions only re-reading them to build the post-hoc addendum); GPU of the confirmatory run {FREEZE.get('gpu', 'n/a')}, "
            f"GPU of this execution {GPU_NAME}; built {now_utc()}.")
UNIVERSE_LINE = ("; ".join(f"{s}: {', '.join(t for t in TICKERS if SECTOR_OF[t] == s)}" for s in SECTORS) +
                 ". Replacements versus the brief (no verified 10-year hourly bars): " + ", ".join(f"{a}->{b}" for a, b in REPLACED.items()) + ".")
def common_front(kind):
    return [Pp(f"Master of Applied Artificial Intelligence, University of Wollongong in Dubai. {RUN_LINE}"),
            Pp("<b>Educational research; not financial advice. Every result is an out-of-sample predictive association, not a cause.</b>"),
            Pp("Data", H2s), Pp(f"Universe ({len(TICKERS)} stocks): {UNIVERSE_LINE}"),
            Pp(f"Prices: TriFrame tier-A verified bars (SHA-256 manifest {DATA_MANIFEST_SHA256[:12]}), split-adjusted, not dividend-adjusted; "
               "1-week, 1-day and regular-hours 1-hour bars; data end 2026-10-09. Decisions 2016-10-10 onward; blocks: train 2016-10 to 2020-12, "
               "inner 2021 (early stopping and calibration), validation 2022-2023 (all selection), locked test 2024-01 to 2026-08 (opened after "
               "the freeze; every opening is logged). 40-session purge between blocks. Targets at 5/10/20/40 sessions: 3-class direction with a "
               "0.5 sigma sqrt(h) band, log realised volatility from hourly bars, and a >10 % drawdown."),
            Pp("<b>Confirmatory and post-hoc results.</b> The hypotheses H1-H7 and every selection were fixed and frozen before the test block was opened. After the "
               "test a methodological review asked for stronger baselines (numbers-only models instead of only the prior), a test of whether charts and text add "
               "information to technical indicators (H8), and more precise p-values for the many-test ablations. These additions are labelled <i>post-hoc</i> "
               "wherever they appear; they use the same locked test rows, only models trained on the training block and tuned on 2021, and baselines chosen on "
               "validation. They do not replace the confirmatory results."),
            tbl(rtab("leakage_tests.csv"), caption="Leakage tests (all must pass)")]
def build_pdf(path, title, sections):
    REQUIRED_MISSING.clear()
    story_body = []
    for heading, items in (sections() if callable(sections) else sections):
        story_body.append(Pp(heading, H1s)); story_body += items
    full_title = ("INCOMPLETE DRAFT: " if REQUIRED_MISSING else "") + title
    story = [Pp(full_title, TITLE)]
    if QUICK:
        story.append(Pp("<b>QUICK_VERIFY - NOT RESEARCH RESULTS.</b> Reduced run (10 stocks, short training) that checks the pipeline. Do not cite any number."))
    if REQUIRED_MISSING:
        story.append(Pp("Missing artifacts: " + ", ".join(sorted(set(REQUIRED_MISSING))), CAP))
    story += story_body
    doc = SimpleDocTemplate(str(path), pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm, bottomMargin=1.8 * cm, title=full_title)
    def on_page(c_, d_):
        c_.saveState(); c_.setFont(FONT, 7); c_.drawString(2 * cm, 1.1 * cm, f"{title} | {STAMP} | page {d_.page}"); c_.restoreState()
    doc.build(story, onFirstPage=on_page, onLaterPages=on_page)
    return path

def hyp_line(df, label):
    if df is None or not len(df):
        return f"{label}: Not Run."
    return f"{label}: {int(df['supported'].sum())} of {len(df)} Holm-corrected tests supported on the locked test block."

# ---------------------------------------------------------------- data-driven text helpers (every number below is read from a table)
H1C, H2C_EV, H2C_PANEL, H3C, H8, H8B, H8C = (globals().get(n_) for n_ in ("H1C", "H2C_EV", "H2C_PANEL", "H3C", "H8", "H8B", "H8C"))
H5_HI, MC_TEST_SCALED = globals().get("H5_HI"), globals().get("MC_TEST_SCALED")
H8D, H8E, H8F, H1D, H2D = (globals().get(n_) for n_ in ("H8D", "H8E", "H8F", "H1D", "H2D"))
COLS_D = ["task", "A", "B", "skill_A", "skill_B", "delta", "lo", "hi", "p_holm", "verdict"]
COLS_V = ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"]
TNAME = {"dir": "direction", "vol": "volatility", "dd": "drawdown"}
def _ok(df, *cols):
    return df is not None and len(df) > 0 and all(c in df.columns for c in cols)
def vcounts(df, prefix=None):
    if not _ok(df, "task", "verdict"):
        return None
    d = df if prefix is None else df[df["task"].astype(str).str.startswith(prefix)]
    vc = d["verdict"].value_counts()
    return int(vc.get("better", 0)), int(vc.get("no clear difference", 0)), int(vc.get("worse", 0)), len(d)
def vphrase(df, prefix=None):
    c = vcounts(df, prefix)
    if c is None:
        return "not run"
    return f"better in {c[0]} of {c[3]} tasks, no clear difference in {c[1]}, worse in {c[2]}"
def sig_tasks(df, prefix):
    if not _ok(df, "task", "supported"):
        return []
    return [str(t) for t, s in zip(df["task"], df["supported"]) if str(t).startswith(prefix) and bool(s)]
def mean_of(df, col, prefix):
    if not _ok(df, "task", col):
        return float("nan")
    return float(df[df["task"].astype(str).str.startswith(prefix)][col].mean())
def best_epoch_text(models):
    cur = rtab("cv_training_curves.csv") if models is CV_MAIN else rtab("nlp_training_curves.csv")
    if not _ok(cur, "model", "epoch", "inner_loss"):
        return ""
    cur = cur[cur["model"].isin(models)]
    if not len(cur):
        return ""
    best = cur.loc[cur.groupby("model")["inner_loss"].idxmin()]
    n1, nm = int((best["epoch"] == 1).sum()), best["model"].nunique()
    return (f"For {n1} of {nm} fine-tuned networks the lowest loss on the early-stopping block (2021) was reached after the first epoch, so the checkpoint kept by early "
            f"stopping is the one from epoch 1 and later epochs only fitted the training block better. Short, early-stopped fine-tuning is therefore the regime these results describe.")
def text_vs_calendar_text():
    """Data-driven reading of H8c-H8f for volatility: is the text gain content, calendar or company identity?"""
    c, d, e, f = vcounts(H8C, "vol"), vcounts(H8D, "vol"), vcounts(H8E, "vol"), vcounts(H8F, "vol")
    if not (c and d and e and f):
        return "The calendar and company controls were not run."
    s = (f"For volatility, adding the filing-calendar columns alone to the technical model is {vphrase(H8D, 'vol')}; adding the text on top of the calendar (H8e) is "
         f"{vphrase(H8E, 'vol')}; adding the text on top of the calendar and the company identity (H8f) is {vphrase(H8F, 'vol')}. ")
    if f[0] > f[3] / 2:
        s += "So the text carries volatility information beyond knowing when filings happen and which company filed them."
    elif e[0] > e[3] / 2:
        s += "So the text adds volatility information beyond the filing calendar, but most of it is explained once the company is known: the embeddings largely identify the firm."
    elif d[0] > d[3] / 2:
        s += "So the volatility gain is explained by the filing calendar (proximity to earnings filings), not by the words in the filings."
    else:
        s += "The evidence on where the gain comes from is mixed; it should not be attributed to the content of the text."
    return s
def abl_delta(df, comparison, target):
    if not _ok(df, "comparison", "target", "delta"):
        return None
    r = df[(df["comparison"] == comparison) & (df["target"] == target)]
    return float(r["delta"].iloc[0]) if len(r) else None

In [ ]:
# ================================================================ Report 1: Computer Vision
cv_abs = (f"Seven vision models (Custom JKX-style CNN, ResNet-50, EfficientNet-B0, ConvNeXt-Tiny, ViT-B/16, Swin-T, YOLOv8s-cls) read weekly, "
          f"daily and hourly candlestick charts of {len(TICKERS)} US stocks and forecast 3-class direction, realised volatility and a >10 % "
          f"drawdown at 5, 10, 20 and 40 sessions. The validation-chosen best model was {CV_BEST} (mean rank over 12 tasks). " +
          hyp_line(H1, "H1 (CV skill against the prior and the shuffled null)") + " " + hyp_line(H5, "H5 (fine-tuning beats frozen)") + " " + hyp_line(H6, "H6 (sector-specific rankings)") +
          f" Post-hoc, against the strongest numbers-only baseline (HAR / momentum / trailing volatility, or LightGBM on 20 indicators, chosen on validation) the "
          f"best CV model per task is {vphrase(H1C)}; direction: {vphrase(H1C, 'dir')}; volatility: {vphrase(H1C, 'vol')}; drawdown: {vphrase(H1C, 'dd')}.")

def cv_training_text():
    return Pp("One backbone shared by the three charts; features concatenated; a multi-task head (4 horizons x [3 direction logits, "
              "volatility, drawdown]). ImageNet-pretrained timm backbones and the Ultralytics YOLOv8s-cls backbone are fine-tuned end-to-end (AdamW, "
              "warm-up + cosine, bfloat16, batch 64 rows, up to 6 epochs, early stopping on 2021); the JKX-style CNN is trained from scratch. Augmentation "
              "respects chart semantics (no flips; only vertical rescale, oldest-bar crop, brightness, noise). Calibration on 2021 only (temperature, Platt, linear). " +
              best_epoch_text(CV_MAIN))

def cv_discussion():
    out = []
    ds = sig_tasks(H1, "dir")
    out.append(Pp(f"<b>Direction.</b> Against a null that shuffles labels within months (same class shares, no link to the future), the validation-chosen models show skill "
                  f"in {len(ds)} of 4 horizons after Holm correction ({', '.join(ds) if ds else 'none'}); the mean skill over the four horizons is {mean_of(H1, 'skill', 'dir'):+.3f} macro-F1. "
                  + ("The effect is small, which is expected for a 3-class label whose sideways band is scaled by volatility. " if ds else "No horizon reaches significance. ")
                  + f"On raw macro-F1 against the strongest numbers-only baseline the best CV model is {vphrase(H1C, 'dir')}; that comparison flatters the CV model, because the "
                  f"baselines predict mostly one class and macro-F1 punishes this. Judged fairly, each model against its own shuffled null (H1d, post-hoc), the best CV model is "
                  f"{vphrase(H1D)}; read the H1d table for the size of the difference."))
    vs, dsig = sig_tasks(H1, "vol"), sig_tasks(H1, "dd")
    out.append(Pp(f"<b>Volatility and drawdown.</b> Against the uninformed benchmarks (R-squared 0, AUC 0.5) the CV models have skill in {len(vs)} of 4 volatility and {len(dsig)} of 4 drawdown tasks. "
                  f"That comparison is weak, because trailing volatility alone predicts future volatility well. Against the strongest numbers-only baseline the best CV model is "
                  f"{vphrase(H1C, 'vol')} for volatility and {vphrase(H1C, 'dd')} for drawdown (post-hoc, Holm-corrected)."))
    da = abl_delta(rtab("cv_ablations_val_hiB.csv"), "with vs without augmentation", "vol")
    expl = ""
    if da is not None:
        expl = (f" One candidate explanation, not a tested cause: each chart's price axis is scaled to that chart's own high and low, which removes the absolute volatility level from the image, "
                f"and the vertical-rescale augmentation changes candle heights; the ablation without augmentation differs from the augmented model by {da:+.3f} R-squared on volatility (validation).")
    out.append(Pp("<b>Why images may not carry the volatility level.</b>" + expl))
    h5n = int(H5_HI["supported"].sum()) if _ok(H5_HI, "supported") else (int(H5["supported"].sum()) if _ok(H5, "supported") else 0)
    h5t = len(H5_HI) if _ok(H5_HI, "supported") else (len(H5) if _ok(H5, "supported") else 0)
    h5df = H5_HI if _ok(H5_HI, "delta", "p_holm") else H5
    n_w = int(((h5df["delta"] < 0) & (h5df["p_holm"] < CFG["alpha"])).sum()) if _ok(h5df, "delta", "p_holm") else 0
    out.append(Pp(f"<b>Fine-tuning versus frozen features.</b> Fine-tuning beats the frozen backbone in {h5n} of {h5t} (model, target) tests on the locked test block "
                  f"({'5,000' if _ok(H5_HI, 'supported') else '1,000'} bootstrap draws) and is significantly worse in {n_w}. " +
                  ("A significant loss from fine-tuning means that adapting every weight to noisy financial labels is not a safe default for that network. " if n_w else "") + best_epoch_text(CV_MAIN)))
    k_t = int((CV_RANK_TEST.index == CV_BEST).argmax()) + 1 if (CV_BEST in CV_RANK_TEST.index) else None
    out.append(Pp(f"<b>Validation versus test.</b> The validation winner {CV_BEST} ranks {k_t if k_t else 'n/a'} of {len(CV_RANK_TEST)} on the locked test; the best on the test ex post was "
                  f"{CV_RANK_TEST.index[0] if len(CV_RANK_TEST) else 'n/a'}. Validation covers the 2022 bear market and the test a strong 2024-2026 market, so rankings are partly regime-dependent; "
                  f"averaging the top-3 models (cv_top3) is the more robust choice and is what the dashboard uses."))
    return out
def cv_conclusion():
    ds = sig_tasks(H1, "dir")
    bv, bd = vcounts(H1C, "vol"), vcounts(H1C, "dd")
    interp = ""
    if bv and bd:
        if bv[0] == 0 and bd[0] == 0:
            interp = "In this design charts do not beat simple numbers on volatility or drawdown risk."
        elif bv[0] + bd[0] < (bv[3] + bd[3]) / 2:
            interp = "In this design charts beat simple numbers on volatility or drawdown risk in only a minority of tasks."
        else:
            interp = "In this design charts beat simple numbers on volatility or drawdown risk in most tasks."
    dtxt = (f"Charts carry a statistically supported direction signal in {len(ds)} of 4 horizons (mean skill {mean_of(H1, 'skill', 'dir'):+.3f} macro-F1 over a shuffled null)."
            if ds else "No horizon shows a statistically supported direction signal over the shuffled null.")
    dtxt += f" Compared fairly with the numbers-only baselines (each against its own shuffled null), the best CV model is {vphrase(H1D)} on direction."
    return [Pp(f"Seven vision models were compared on identical, leakage-controlled rows. {dtxt} For volatility and drawdown risk, simple numbers are the benchmark to beat: the best CV model is "
               f"{vphrase(H1C, 'vol')} on volatility and {vphrase(H1C, 'dd')} on drawdown against the strongest numbers-only baseline. {interp} Multimodal results are in the third report.")]

CV_SECTIONS = lambda: [
    ("Abstract", [Pp(cv_abs)]),
    ("1. Problem, gap and contribution", [Pp("Can a vision model read a candlestick chart well enough to forecast direction, volatility and drawdown risk, which "
        "architecture does it best, do combinations help, and does the answer depend on the sector? Earlier work used one architecture, one timeframe or "
        "few stocks; we compare seven architectures on identical leakage-controlled rows with three timeframes, four horizons and three targets, with "
        "paired month-block bootstrap intervals and Holm correction, and test the validation choice once on 2024-2026."), *common_front("cv")]),
    ("2. Images and preprocessing", [figp("chart_gallery.png", "Model inputs: weekly (52 candles), daily (63) and hourly (70) charts, 224 x 224, no text or axes."),
        tbl(rtab("image_dataset.csv"), caption="Image dataset: rows, size and storage"), tbl(rtab("label_balance.csv"), max_rows=12, caption="Class balance and base rates by horizon and block"),
        figp("eda_labels.png", "Class balance, drawdown base rates and volatility.")]),
    ("3. Models and training", [cv_training_text(),
        tbl(rtab("cv_lr_probe.csv"), caption="Learning-rate probe: a short run on a subsample of the training block for two candidate rates per model; the lower inner-block loss is kept"),
        figp("cv_training_curves.png", "Training vs inner-block loss per epoch (early-stopping set).")]),
    ("4. Validation results (selection)", [figp("cv_leaderboard_val.png", "Validation primary metrics; colour = rank within each task."),
        tbl(rtab("cv_rank_val.csv"), caption="Mean validation rank over the 12 tasks (1 = best)"), Pp(mdtext("why_cv_best_val.md")),
        tbl(rtab("cv_skill_val.csv"), ["task", "model", "skill", "lo", "hi", "p", "n"], caption="Validation skill of the best model per task (direction: over the shuffled null; volatility: R-squared; drawdown: AUC minus 0.5)"),
        Pp("Combinations (equal-weight averages of calibrated outputs; nothing fitted):", H2s), tbl(rtab("cv_combo_rank_val.csv"), max_rows=12, caption="Best model combinations by mean validation rank"),
        tbl(rtab("cv_best_per_task_val.csv"), caption="Best model per task on validation"), Pp("Sectors:", H2s), tbl(rtab("cv_best_by_sector_val.csv"), caption="Best CV model per sector and target (validation)"),
        figp("cv_sector_interaction_val.png", "Sector x model interaction (centred metric) and permutation p-values."),
        Pp("Ablations (1,000 bootstrap draws; with this many draws the smallest attainable Holm-corrected p-value is about 0.07 for 36-44 tests, so none can be called significant here):", H2s),
        tbl(rtab("cv_ablations_val.csv"), ["comparison", "A", "target", "delta", "lo", "hi", "p_holm"], max_rows=44, caption="Ablations on validation, 1,000 draws (confirmatory run)"),
        Pp("Post-hoc re-run of the same ablations with 5,000 bootstrap draws; only the precision of the p-values changes:", H2s),
        tbl(rtab("cv_ablations_val_hiB.csv"), ["comparison", "A", "target", "delta", "lo", "hi", "p_holm"], max_rows=44, caption="Ablations on validation, 5,000 draws (post-hoc)"),
        figp("cv_view_ablation.png", "Which timeframe carries the signal (frozen probes).")]),
    ("5. Locked test results (2024-2026)", [
        tbl(rtab("test_H1.csv"), ["task", "model", "skill", "lo", "hi", "p_holm", "supported"], caption="H1 (confirmatory): skill of the validation-chosen CV model over the uninformed benchmark"),
        tbl(rtab("cv_rank_test.csv"), caption="Mean test rank of the fine-tuned CV models"), tbl(rtab("cv_best_per_target_test.csv"), caption="Validation choice, top-3 ensemble and the model that happened to be best on the test, per task"),
        tbl(rtab("cv_combo_rank_test.csv"), max_rows=10, caption="Best model combinations on the locked test"),
        tbl(rtab("test_H5.csv"), ["model", "target", "delta", "lo", "hi", "p_holm", "supported"], caption="H5 (confirmatory, 1,000 draws): fine-tuned minus frozen"),
        tbl(rtab("test_H5_hiB.csv"), ["model", "target", "delta", "lo", "hi", "p_holm", "supported"], caption="H5 re-run with 5,000 draws (post-hoc precision check)"),
        tbl(rtab("cv_best_by_sector_test.csv"), caption="Best CV model per sector and target on the locked test"), tbl(rtab("test_H6.csv"), caption="H6: sector x model interaction (permutation test)")]),
    ("5a. Against the strongest numbers-only baseline (post-hoc)", [
        Pp("The prior is a weak benchmark for volatility and drawdown. For every task the stronger of two numbers-only baselines was chosen on validation: <i>baseline_simple</i> "
           "(momentum for direction, HAR regression for volatility, trailing-volatility logistic model for drawdown) or <i>tech_gbm</i> (LightGBM on 20 technical indicators). "
           "The table gives the best CV model minus that baseline on the locked test."),
        tbl(rtab("phase_a_baseline_choice.csv"), caption="Strongest numbers-only baseline per task (chosen on validation)"),
        tbl(rtab("test_H1c.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H1c (post-hoc): best CV model minus strongest numbers-only baseline, paired month-block bootstrap, Holm"),
        tbl(rtab("test_H1d.csv"), COLS_D, caption="H1d (post-hoc, the fair direction comparison): skill = macro-F1 minus the model's own within-month shuffled null; delta = CV skill minus baseline skill"),
        tbl(rtab("test_H8b.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H8b (post-hoc): LightGBM on technical + CV features minus technical features alone"),
        tbl(rtab("phase_a_panel_test.csv"), index=False, max_rows=12, caption="Primary test metric per task for the reference models (post-hoc)")]),
    ("6. Error analysis, explainability, robustness, efficiency", [figp("cv_confusion_val.png", "Confusion matrices of the best CV model (validation)."),
        figp("cv_errors_val.png", "Accuracy by volatility decile, sector, quarter and true class."), figp("cv_confident_errors_val.png", "Most confident wrong 20-day calls."),
        figp("cv_gradcam.png", "Grad-CAM: sensitivity maps, not explanations of reasoning; coarse (7x7 / 14x14 cells)."),
        tbl(rtab("cv_occlusion_val.csv"), max_rows=30, caption="Occlusion: metric change when a view or the most recent bars are hidden"),
        tbl(rtab("cv_robustness_val.csv"), max_rows=24, caption="Robustness to noise, blur, JPEG compression, chart style and resizing"), tbl(rtab("cv_efficiency.csv"), caption="Parameters, compute and training time"),
        figp("cv_efficiency.png", "Accuracy vs compute."), figp("yolo_detections.png", "The zero-shot YOLOv8 chart-pattern detector on our weekly charts."),
        tbl(rtab("yolo_patterns_train.csv"), caption="Chart patterns detected by the zero-shot detector (training block)")]),
    ("7. Discussion", cv_discussion()),
    ("8. Threats to validity and limitations", [Pp("<b>Population.</b> Fifty large, surviving US stocks (survivorship bias); the original image-prediction literature found most signal in small caps. "
        "<b>Time.</b> One decade with a 2022 bear market in validation and a strong 2024-2026 market in test (regime shift); the sideways band is volatility-scaled, which makes direction hard by design. "
        "<b>Inputs.</b> Prices are not dividend-adjusted; charts are rebuilt from bars, so the models never saw real broker screenshots (the dashboard redraws them); each chart is min-max scaled, which "
        "hides absolute volatility. <b>Training.</b> One seed per model, two-value learning-rate grids and short early-stopped training; the differences between close models are within seed noise. "
        "<b>Inference.</b> Validation was used for many choices, so only the locked test numbers are unbiased; the post-hoc comparisons in section 5a were added after the test was opened and "
        "should be read as exploratory. <b>Explanations.</b> Grad-CAM and occlusion show sensitivity, not causation.")]),
    ("9. Conclusion", cv_conclusion()),
    ("References", refs(["jkx", "he_resnet", "tan", "liu_convnext", "dosovitskiy", "liu_swin", "redmon", "jocher", "sezer", "ho", "corsi", "patton",
                         "guo", "selvaraju", "hendrycks", "kunsch", "holm"])),
]

In [ ]:
# ================================================================ Report 2: NLP
_nlp_ran = [NLPC[e]["label"] for e in NLPC if any(REG.get(m, {}).get("status") == "RUN" for m in (f"{e}_ft", f"{e}_frozen"))]
_fg = "" if REG.get("fingpt_frozen", {}).get("status") == "RUN" else " FinGPT was not run: access to Meta's gated Llama-2 base model was not granted before the deadline."
def _dir_count_text():
    if not _ok(H2, "task", "supported"):
        return ""
    return f"direction {len(sig_tasks(H2, 'dir'))}/4, volatility {len(sig_tasks(H2, 'vol'))}/4, drawdown {len(sig_tasks(H2, 'dd'))}/4"
nlp_abs = (f"{len(_nlp_ran)} language models ({', '.join(_nlp_ran)}) and a TF-IDF baseline read the MD&A of "
           f"10-K and 10-Q reports and the text of 8-K filings of {len(TICKERS)} companies (point-in-time EDGAR timestamps) and forecast direction, "
           f"volatility and a >10 % drawdown from the first tradeable price after each filing. Validation-chosen best: {NLP_BEST}. " +
           hyp_line(H2, "H2 (NLP skill on filing events)") + f" Tasks with supported skill: {_dir_count_text()}. " + hyp_line(H7, "H7 (event study)") +
           f" Post-hoc, against the strongest no-text baseline on the same filings (per-company base rates, momentum/HAR/trailing volatility + form, or LightGBM on indicators + form, chosen on validation) "
           f"the best NLP model per task is {vphrase(H2C_EV)}; for drawdown it is {vphrase(H2C_EV, 'dd')}. "
           "The test period starts after the pre-training cut-off of every language model, so no model can remember test-period news." + _fg)

def nlp_training_text():
    return Pp("Frozen: chunked (4 x 512 tokens; Longformer 4,096) mean-pooled embeddings, 32 principal components, "
              "regularised linear heads. Fine-tuned: the same multi-task head as the CV models, AdamW, learning rate from a two-value probe, early stopping on "
              "2021 filings (up to 3 epochs). FinGPT: Llama-2-7B + FinGPT LoRA in 4-bit, frozen embeddings plus its own sentiment read-out. " + best_epoch_text(NLP_MAIN))

def _top_sector_text(df, tag):
    if not _ok(df, "sector", "form", "excess_over_non_filing_days"):
        return ""
    parts = []
    for f_, g in df.groupby("form"):
        hi_, lo_ = g.loc[g["excess_over_non_filing_days"].idxmax()], g.loc[g["excess_over_non_filing_days"].idxmin()]
        parts.append(f"{f_}: highest {hi_['sector']} ({hi_['excess_over_non_filing_days']:+.2f}), lowest {lo_['sector']} ({lo_['excess_over_non_filing_days']:+.2f})")
    return f"{tag}: " + "; ".join(parts) + "."

def nlp_discussion():
    out = []
    es = rtab("event_study_by_form_test.csv"); h7 = rtab("test_H7.csv")
    if _ok(es, "group", "mean_abs_z"):
        out.append(Pp("<b>Which report moves the market (Q6).</b> Mean absolute abnormal two-day move on the locked test, in normal-day units: " +
                      ", ".join(f"{r.group} {r.mean_abs_z:.2f} (n={int(r.n)})" for r in es.itertuples()) + ". " +
                      ("Filing days move more than matched non-filing days (Mann-Whitney p " + ("<0.001" if float(h7['p'].iloc[0]) < 0.001 else f"= {float(h7['p'].iloc[0]):.3f}") +
                       ") and the forms differ (Kruskal-Wallis p " + ("<0.001" if float(h7['p'].iloc[1]) < 0.001 else f"= {float(h7['p'].iloc[1]):.3f}") + ")." if _ok(h7, "p") and len(h7) > 1 else "")))
    out.append(Pp("<b>Which sectors (Q7, Q9).</b> " + _top_sector_text(rtab("event_study_sector_form_dev.csv"), "Development") + " " + _top_sector_text(rtab("event_study_sector_form_test.csv"), "Test") +
                  " A sector is called more sensitive to a form when its excess reaction (mean |z| minus the same sector's non-filing days) is larger; agreement between the two periods is the evidence that the pattern is stable."))
    it = rtab("event_study_8k_items_dev.csv"); wd = rtab("nlp_words_abs_z_dev.csv"); tn = rtab("nlp_tone_vs_reaction_dev.csv")
    q8 = "<b>What in the reports (Q8).</b> "
    if _ok(it, "item", "mean_abs_z"):
        q8 += "8-K items with the largest reactions: " + ", ".join(f"{r.item} ({r.mean_abs_z:.2f})" for r in it.head(3).itertuples()) + ". "
    if _ok(wd, "top_positive"):
        q8 += "Phrases most associated with large reactions: " + ", ".join(str(x) for x in wd["top_positive"].head(6)) + "; most associated with small reactions: " + ", ".join(str(x) for x in wd["top_negative"].head(6)) + ". "
    if _ok(tn, "form", "spearman_tone_vs_signed_AR"):
        r_all = tn[tn["form"] == "all"]
        if len(r_all):
            q8 += f"FinBERT sentence tone is almost unrelated to the sign of the reaction (Spearman {float(r_all['spearman_tone_vs_signed_AR'].iloc[0]):+.3f}) and weakly related to its size ({float(r_all['spearman_tone_vs_abs_z'].iloc[0]):+.3f}). "
    q8 += "Phrases that mark the filing type, the period (for example pandemic-era wording) or a company name are associations with the form, the time or the firm rather than with news; read the table with that in mind."
    out.append(Pp(q8))
    sd, sv, sk = sig_tasks(H2, "dir"), sig_tasks(H2, "vol"), sig_tasks(H2, "dd")
    n_neg = int((H2[H2["task"].astype(str).str.startswith("vol")]["skill"] < 0).sum()) if _ok(H2, "task", "skill") else 0
    out.append(Pp(f"<b>Can language models forecast from filings?</b> Against the uninformed benchmarks the validation-chosen NLP models have skill in {len(sd)} of 4 direction, {len(sv)} of 4 volatility and "
                  f"{len(sk)} of 4 drawdown tasks on the locked test. For volatility the skill is negative (worse than predicting the training mean) in {n_neg} of 4 tasks"
                  + (", which means that text alone does not track volatility regimes." if n_neg >= 2 else ".")))
    out.append(Pp(f"<b>Is the drawdown signal only company identity? (post-hoc)</b> A filing tells the model which company wrote it, and a company's usual volatility predicts its drawdown risk. "
                  f"Against the strongest no-text baseline on the same filings the best NLP model is {vphrase(H2C_EV, 'dd')} for drawdown, {vphrase(H2C_EV, 'vol')} for volatility and "
                  f"{vphrase(H2C_EV, 'dir')} for direction. A drawdown gain over the prior that disappears against the per-company or numbers-only baseline would indicate that the text mostly identifies the company; "
                  f"a gain that survives indicates information beyond identity. Judged fairly on direction (each model against its own shuffled null, H2d), the best NLP model is {vphrase(H2D)}."))
    out.append(Pp("<b>Does text add to simple numbers on the daily panel? (post-hoc)</b> Technical + text features versus technical alone (H8c) is "
                  f"{vphrase(H8C)}. " + text_vs_calendar_text()))
    out.append(Pp("<b>Fine-tuned versus frozen, length and domain.</b> The 5,000-draw ablation table shows, per comparison, the mean difference on each target; "
                  "reading chunked versus first-512-token inputs and fine-tuned versus frozen encoders there is more reliable than in the 1,000-draw table, whose smallest attainable "
                  "Holm-corrected p-value (about 0.07) is above the 5 % level. " + best_epoch_text(NLP_MAIN)))
    return out
def nlp_conclusion():
    es = rtab("event_study_by_form_test.csv"); ratio_txt = ""
    if _ok(es, "group", "mean_abs_z"):
        ctrl = es[es["group"].astype(str).str.startswith("no filing")]["mean_abs_z"]
        fil = es[~es["group"].astype(str).str.startswith("no filing")]["mean_abs_z"]
        if len(ctrl) and len(fil) and float(ctrl.iloc[0]) > 0:
            ratio_txt = f"On the locked test, filing days move prices {float(fil.mean()) / float(ctrl.iloc[0]):.1f} times as much as matched non-filing days, and the reaction differs by form and sector. "
    bd = vcounts(H2C_EV, "dd")
    interp = ""
    if bd:
        if bd[0] == 0:
            interp = "The drawdown gain over the prior does not survive against the strongest no-text baseline, so it should not be read as information in the text beyond the company and simple numbers."
        elif bd[0] == bd[3]:
            interp = "The drawdown gain over the prior survives against the strongest no-text baseline in every task, which indicates information in the text beyond the company and simple numbers."
        else:
            interp = "The drawdown gain over the prior survives against the strongest no-text baseline in some tasks only."
    return [Pp(f"{ratio_txt}Language-model forecasts of direction are supported in {len(sig_tasks(H2, 'dir'))} of 4 horizons, of volatility in {len(sig_tasks(H2, 'vol'))} of 4 and of drawdown risk in "
               f"{len(sig_tasks(H2, 'dd'))} of 4 tasks over the uninformed benchmarks. Against the strongest no-text baseline the best NLP model is {vphrase(H2C_EV)} (drawdown: {vphrase(H2C_EV, 'dd')}). "
               f"{interp} {text_vs_calendar_text()} Comparisons between fine-tuned transformers and the TF-IDF baseline should be read with the confidence intervals, not the ranks.")]

NLP_SECTIONS = lambda: [
    ("Abstract", [Pp(nlp_abs)]),
    ("1. Problem, gap and contribution", [Pp("Which SEC report moves prices, in which sectors, what in the text goes with the move, and can language models "
        "forecast the next 5-40 sessions from it? We combine an event study (abnormal reaction vs matched non-filing days) with a controlled comparison "
        "of transformer language models and a TF-IDF baseline on identical filings, evaluated on 2024-2026 after all selections were frozen."), *common_front("nlp")]),
    ("2. Filings and preprocessing", [tbl(rtab("sec_per_year.csv"), caption="Filings per year and form"), tbl(rtab("sec_per_sector.csv"), caption="Filings per sector and form"),
        tbl(rtab("sec_text_stats.csv"), index=False, caption="Text length, MD&A extraction success and empty documents by form"),
        figp("eda_sec.png", "Section lengths, acceptance timing and 8-K items."),
        Pp("Acceptance time from the SGML header (quality A) or the later of two JSON readings; availability = acceptance + 35 minutes; entry at the first "
           "hourly bar after availability. MD&A extraction for 10-K/10-Q (Exhibit 13 for banks), 8-K body + EX-99. Models read [form] [section] text.")]),
    ("3. Event study (Q6, Q7, Q9)", [tbl(rtab("event_study_by_form_dev.csv"), caption="Reaction by form, development blocks (train + inner + validation); |z| = absolute abnormal 2-day move in normal-day units"),
        tbl(rtab("event_study_by_form_test.csv"), caption="Reaction by form, locked test block"), tbl(rtab("test_H7.csv"), caption="H7: filing days move more than non-filing days; reaction size differs by form"),
        figp("event_study_dev.png", "Reaction size by form, sector x form excess over non-filing days, 8-K items (development blocks)."),
        tbl(rtab("event_study_sector_form_dev.csv"), max_rows=20, caption="Sector x form, development: mean |z|, number of filings, excess over the sector's non-filing days"),
        tbl(rtab("event_study_sector_form_test.csv"), max_rows=20, caption="Sector x form, locked test")]),
    ("4. Models, training and validation results", [nlp_training_text(),
        tbl(rtab("nlp_lr_probe.csv"), caption="Learning-rate probe: two candidate rates per model, a short run on a subsample; the lower inner-block loss is kept"),
        figp("nlp_training_curves.png", "Training vs inner loss."), figp("nlp_leaderboard_val.png", "Validation primary metrics."),
        tbl(rtab("nlp_rank_val.csv"), caption="Mean validation rank over the 12 tasks (1 = best)"), Pp(mdtext("why_nlp_best_val.md")),
        tbl(rtab("nlp_ablations_val.csv"), ["comparison", "A", "target", "delta", "lo", "hi", "p_holm"], max_rows=40, caption="Ablations on validation, 1,000 draws (confirmatory run; smallest attainable Holm p about 0.07)"),
        tbl(rtab("nlp_ablations_val_hiB.csv"), ["comparison", "A", "target", "delta", "lo", "hi", "p_holm"], max_rows=40, caption="Ablations on validation, 5,000 draws (post-hoc precision check)")]),
    ("5. Locked test results", [tbl(rtab("test_H2.csv"), ["task", "model", "skill", "lo", "hi", "p_holm", "supported"], caption="H2 (confirmatory): skill of the validation-chosen NLP model on filing events over the uninformed benchmark"),
        tbl(rtab("nlp_rank_test.csv"), caption="Mean test rank of the NLP models"),
        tbl(rtab("test_H2b.csv"), ["task", "model", "skill", "lo", "hi", "p_holm", "supported"], caption="H2b (confirmatory): the same models carried to the daily panel")]),
    ("5a. Against no-text baselines on the same filings (post-hoc)", [
        Pp("Three baselines see the same events but no text: <i>company_event</i> (each company's training-block base rates), <i>simple_event</i> (momentum, HAR, trailing volatility at the last close "
           "before the filing, plus the form) and <i>tech_event</i> (LightGBM on 20 indicators plus the form). For every task the strongest was chosen on validation. A language model that only "
           "recognises the company cannot beat <i>company_event</i>."),
        tbl(rtab("phase_a_baseline_choice.csv"), caption="Strongest baseline per task (chosen on validation; event_baseline column applies here)"),
        tbl(rtab("test_H2c_events.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H2c (post-hoc): best NLP model minus strongest no-text baseline on the locked test"),
        tbl(rtab("test_H2d_events.csv"), COLS_D, caption="H2d (post-hoc, fair direction comparison): NLP skill minus baseline skill, each over its own within-month shuffled null"),
        tbl(rtab("test_H8e.csv"), COLS_V, caption="H8e (post-hoc, daily panel): technical + text minus technical + filing-calendar columns (is it the words or the filing dates?)"),
        tbl(rtab("test_H8f.csv"), COLS_V, caption="H8f (post-hoc, daily panel): technical + text minus technical + calendar + company one-hot (words beyond the company's identity)"),
        tbl(rtab("phase_a_event_test.csv"), max_rows=14, caption="Primary test metric per task for the no-text baselines and the NLP models (post-hoc)"),
        tbl(rtab("test_H8c.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H8c (post-hoc, daily panel): LightGBM on technical + NLP features minus technical features alone")]),
    ("6. What in the filings goes with the move (Q8)", [tbl(rtab("event_study_8k_items_dev.csv"), caption="8-K items ranked by mean |z| (development blocks)"),
        tbl(rtab("event_study_8k_items_test.csv"), caption="8-K items on the locked test"),
        tbl(rtab("nlp_words_abs_z_dev.csv"), max_rows=15, caption="n-grams most associated with the SIZE of the reaction (ridge coefficients, TF-IDF)"),
        tbl(rtab("nlp_words_ar2_dev.csv"), max_rows=15, caption="n-grams most associated with the SIGN of the reaction (positive / negative abnormal return)"),
        tbl(rtab("nlp_tone_vs_reaction_dev.csv"), caption="FinBERT sentence tone vs the reaction (Spearman correlations)"),
        tbl(rtab("nlp_integrated_gradients_val.csv"), caption="Tokens that raise the output most (integrated gradients), largest-reaction validation filings; many are dates, company names and formatting tokens"),
        Pp("These are associations inside the sample; boilerplate phrases can mark filing types, periods or companies rather than news.")]),
    ("7. Errors and efficiency", [figp("nlp_errors_val.png", "Accuracy by form, sector, timing and length; confusion matrix."), tbl(rtab("nlp_efficiency.csv"), caption="Parameters, context length, training and inference speed")]),
    ("8. Discussion", nlp_discussion()),
    ("9. Threats to validity and limitations", [Pp("<b>Sample.</b> Only 50 large companies, so a few thousand filings; the model ranks between close competitors are not reliable. <b>Text.</b> MD&A extraction "
        "is heuristic (fallback to full text is reported); 512-token models read only the beginning of long sections; filings reveal the company, so company identity can masquerade as content "
        "(addressed post-hoc in section 5a). <b>Models.</b> FinGPT is used frozen; one seed per model; two-value learning-rate grids; short early-stopped fine-tuning. <b>Events.</b> Reactions mix the "
        "filing with same-day news; survivorship bias. <b>Inference.</b> Only the locked test numbers are unbiased; section 5a was added after the test was opened and is exploratory.")]),
    ("10. Conclusion", nlp_conclusion()),
    ("References", refs(["araci", "huang", "loukas", "liu_roberta", "beltagy", "he_deberta", "yang", "touvron", "dettmers", "hu", "loughran11", "loughran16",
                         "tetlock", "cohen", "ke", "lerman", "glasserman", "sarkar", "sundararajan", "kunsch", "holm"])),
]

In [ ]:
# ================================================================ Report 3: Multimodal
def answers_text():
    out = []
    def clean(s):
        return html.escape(str(s).replace("**", ""), quote=False)
    q1 = clean(ANSWERS['Q1']).replace("See q1_answers_test.csv: '*' marks Holm-significant skill on the locked test block. ", "")
    out.append(Pp(f"<b>Q1. Can CV, NLP or CV+NLP predict UP / SIDEWAYS / DOWN at 5-40 days?</b> {q1} The per-task table is in section 3. Post-hoc, against the strongest numbers-only baseline: "
                  f"best CV model {vphrase(H1C)}; best NLP model (panel) {vphrase(H2C_PANEL)}; best fusion {vphrase(H3C)}. For direction, judged fairly (each model against its own shuffled null): "
                  f"CV {vphrase(H1D)}; NLP on filing events {vphrase(H2D)}."))
    out.append(Pp(f"<b>Q2. Best CV model.</b> {clean(ANSWERS['Q2'])}"))
    cr = rtab("cv_combo_rank_test.csv")
    if _ok(cr, "combo", "mean_rank_test"):
        k = int((cr["combo"] == BEST_COMBO).argmax()) + 1 if BEST_COMBO in set(cr["combo"]) else None
        out.append(Pp(f"<b>Q3. Best CV combination.</b> The best combination on validation was {BEST_COMBO}; it ranks {k if k else 'n/a'} of {len(cr)} on the locked test. "
                      f"The three best combinations on the test were " + ", ".join(f"{r.combo} (mean rank {r.mean_rank_test:.1f})" for r in cr.head(3).itertuples()) +
                      ". A combination that is a single model means that no averaged ensemble beat the best single network on validation."))
    bt = rtab("cv_best_per_target_test.csv")
    if _ok(bt, "task", "val_choice", "test_best_ex_post"):
        same = int((bt["val_choice"] == bt["test_best_ex_post"]).sum())
        out.append(Pp(f"<b>Q4. Best model per target.</b> Table 'best model per task' lists the validation choice and its test score for direction, volatility and drawdown at each horizon. "
                      f"The validation choice was also the best on the test in {same} of {len(bt)} tasks; the top-3 ensemble is the more reliable choice when validation and test regimes differ."))
    sec = rtab("cv_best_by_sector_test.csv")
    h6 = ", ".join(f"{r.target}: p = {('<0.001' if r.p < 0.001 else f'{r.p:.3f}')}" for r in H6.itertuples()) if _ok(H6, "target", "p") else "not run"
    out.append(Pp(f"<b>Q5. Best model per sector; sector-specific networks?</b> The best model per sector and target is in the sector table. The interaction test (do rankings change by sector?) gives {h6}; "
                  f"a small p-value means the best network depends on the sector for that target."))
    es = rtab("event_study_by_form_test.csv")
    if _ok(es, "group", "mean_abs_z"):
        forms_ = es[~es["group"].astype(str).str.startswith("no filing")].sort_values("mean_abs_z", ascending=False)
        top_item = rtab("event_study_8k_items_dev.csv")
        item_txt = f" Among 8-K items the largest reaction is item {top_item['item'].iloc[0]} (mean |z| {float(top_item['mean_abs_z'].iloc[0]):.2f}, development blocks)." if _ok(top_item, "item", "mean_abs_z") else ""
        out.append(Pp("<b>Q6. Which report moves the market more?</b> Mean |z| on the test: " + ", ".join(f"{r.group} {r.mean_abs_z:.2f}" for r in es.itertuples()) +
                      f". By reaction size the forms rank: {' > '.join(str(g) for g in forms_['group'])}." + item_txt))
    out.append(Pp("<b>Q7 and Q9. Which sectors are sensitive to which report?</b> " + _top_sector_text(rtab("event_study_sector_form_dev.csv"), "Development") + " " +
                  _top_sector_text(rtab("event_study_sector_form_test.csv"), "Test")))
    it = rtab("event_study_8k_items_dev.csv"); wd = rtab("nlp_words_abs_z_dev.csv")
    q8 = "<b>Q8. What in the reports moves the market?</b> "
    if _ok(it, "item", "mean_abs_z"):
        q8 += "8-K items by reaction: " + ", ".join(f"{r.item} ({r.mean_abs_z:.2f})" for r in it.head(3).itertuples()) + ". "
    if _ok(wd, "top_positive"):
        q8 += "Phrases associated with large reactions: " + ", ".join(str(x) for x in wd["top_positive"].head(6)) + ". "
    q8 += "Sentence tone has almost no relation to the sign of the reaction; many top phrases mark the form, the period or the company."
    out.append(Pp(q8))
    wfile = SUB["results"] / "fusion_weights.json"
    wdir = ""
    if wfile.exists():
        wl = json.loads(wfile.read_text()).get("late_top3", {})
        dirw = [wl.get(f"dir_{h}") for h in HORIZONS]
        if dirw and all(w is not None for w in dirw) and all(float(w) == 1.0 for w in dirw):
            wdir = " For direction, validation assigned all weight to the charts in late fusion (so fusion and charts give identical direction predictions and the difference is exactly zero): text received zero weight."
    out.append(Pp(f"<b>Q10. Does combining the top-3 CV and top-3 NLP models add edge?</b> {clean(ANSWERS['Q10'])} By target: volatility {len(sig_tasks(H4, 'vol'))} of 4, "
                  f"drawdown {len(sig_tasks(H4, 'dd'))} of 4, direction {len(sig_tasks(H4, 'dir'))} of 4." + wdir +
                  f" Post-hoc, adding CV and NLP features to the technical LightGBM: both {vphrase(H8)}; CV only {vphrase(H8B)}; NLP only {vphrase(H8C)}."))
    return out

def mm_discussion():
    out = []
    fw = SUB["results"] / "fusion_weights.json"
    wtxt = ""
    if fw.exists():
        wd_ = json.loads(fw.read_text()).get("late_top3", {})
        if wd_:
            wtxt = " Validation weights on the CV side (1 = charts only, 0 = text only), late top-3 fusion: " + ", ".join(f"{k}: {v:.2f}" for k, v in wd_.items()) + "."
    out.append(Pp(f"<b>Does fusion help?</b> Fusion beats the better single modality in {int(H3['supported'].sum()) if _ok(H3, 'supported') else 0} of {len(H3) if _ok(H3, 'supported') else 0} tasks (H3) and "
                  f"top-3 CV + top-3 NLP beats top-3 CV in {int(H4['supported'].sum()) if _ok(H4, 'supported') else 0} of {len(H4) if _ok(H4, 'supported') else 0} (H4). By target, H3 is supported for "
                  f"direction in {len(sig_tasks(H3, 'dir'))} of 4 tasks, volatility {len(sig_tasks(H3, 'vol'))} of 4 and drawdown {len(sig_tasks(H3, 'dd'))} of 4." + wtxt))
    out.append(Pp(f"<b>Is the gain real once simple numbers are present? (post-hoc)</b> The fairest comparison keeps the learner fixed and adds features. Technical + CV + NLP features versus technical features alone: "
                  f"{vphrase(H8)} (direction: {vphrase(H8, 'dir')}; volatility: {vphrase(H8, 'vol')}; drawdown: {vphrase(H8, 'dd')}). Adding only CV features: {vphrase(H8B)}. "
                  f"Adding only NLP features: {vphrase(H8C)}. A gain here is information that charts or filings contain beyond the numbers; no gain means the late-fusion improvements above come from "
                  f"the CV and NLP models re-learning what the numbers already say. " + text_vs_calendar_text()))
    out.append(Pp(f"<b>Direction, compared fairly (post-hoc).</b> The numbers-only baselines predict mostly one class, which macro-F1 punishes, so raw macro-F1 differences flatter the "
                  f"image and text models. With every model judged against its own within-month shuffled null, the best CV model versus the strongest numbers-only baseline is {vphrase(H1D)} "
                  f"and the best NLP model versus the strongest no-text baseline on filing events is {vphrase(H2D)}."))
    mct = rtab("mc_calibration_test.csv"); mc_txt = ""
    if _ok(mct, "coverage80", "coverage90"):
        mc_txt = (f" In the confirmatory run the nominal 80 % / 90 % Monte Carlo ranges contained the realised 20-day return {100 * float(mct['coverage80'].iloc[0]):.0f} % / "
                  f"{100 * float(mct['coverage90'].iloc[0]):.0f} % of the time on the test; a single volatility scale chosen on validation is evaluated post-hoc below.")
    out.append(Pp("<b>Dashboard.</b> The recipe (top-3 CV ensemble, blended with top-3 deployable NLP models when a filing is fresh) is parity-tested against the notebook. Its skill on the locked test and the "
                  "per-task no-call rule are in section 5." + mc_txt))
    if _ok(MC_TEST_SCALED, "coverage80"):
        r = MC_TEST_SCALED.iloc[0]
        out.append(Pp(f"With the volatility scale {r['scale']:.2f} chosen on validation, the nominal 80 % / 90 % ranges contain the realised 20-day return {100 * float(r['coverage80']):.0f} % / {100 * float(r['coverage90']):.0f} % of the time on the test."))
    return out
def mm_conclusion():
    return [Pp(f"Charts and filing text were compared alone and combined on identical rows. On the locked test the best CV model is supported over the uninformed benchmark in "
               f"{len(sig_tasks(H1, 'dir'))} of 4 direction, {len(sig_tasks(H1, 'vol'))} of 4 volatility and {len(sig_tasks(H1, 'dd'))} of 4 drawdown tasks; the best NLP model (filings) in "
               f"{len(sig_tasks(H2, 'dir'))}, {len(sig_tasks(H2, 'vol'))} and {len(sig_tasks(H2, 'dd'))}; fusion beats the better single modality in {len(sig_tasks(H3, 'dir')) + len(sig_tasks(H3, 'vol')) + len(sig_tasks(H3, 'dd'))} of 12 tasks. "
               f"The post-hoc incremental test (technical features with CV and NLP features versus technical features alone) is {vphrase(H8)}, which is the answer to whether charts and filings add "
               f"information beyond simple numbers. {text_vs_calendar_text()} Compared fairly on direction, the best CV model versus the numbers-only baseline is {vphrase(H1D)}. The dashboard presents calibrated probabilities and a simulation range, withholds direction calls where the test does not support them, and is a "
               f"decision-support illustration, not a trading system.")]

mm_abs = ("Charts and filing text are aligned on identical decision rows (latest filing known at each close) and combined by late fusion (top-3 + top-3, "
          "best + best, weights chosen on validation) and early feature fusion. " + hyp_line(H3, "H3 (fusion beats the better single modality)") + " " +
          hyp_line(H4, "H4 / Q10 (top-3 CV + top-3 NLP beats top-3 CV)") +
          f" Post-hoc, adding CV and NLP features to a LightGBM on technical indicators is {vphrase(H8)} (H8). The dashboard recipe, its no-call rule and the Monte Carlo simulator "
          "are evaluated by the same rules.")
MM_SECTIONS = lambda: [
    ("Abstract", [Pp(mm_abs)]),
    ("1. Answers to the ten project questions (locked test)", [*answers_text(), tbl(rtab("hypotheses_test.csv"), caption="Hypothesis families (confirmatory): number of Holm-corrected tests supported on the locked test"),
        tbl(rtab("hypotheses_phase_a.csv"), caption="Post-hoc comparison families: verdicts against the strongest numbers-only baseline and the incremental-value tests")]),
    ("2. Design", [*common_front("mm"), tbl(rtab("panel_text_coverage.csv"), caption="Share of decision rows with a periodic filing in the last 90 sessions, an 8-K in the last 20, and fresh text (within 5 sessions)"),
        Pp("Fusion candidates: late fusion of calibrated probabilities (weights 0-1 in steps of 0.25 per task, chosen on validation), early fusion "
           "(linear heads on frozen CV + frozen NLP features), and a technical-indicator reference (LightGBM on technical + CV + NLP features).")]),
    ("3. Q1: CV vs NLP vs CV+NLP", [tbl(rtab("q1_answers_test.csv"), max_rows=12, caption="Q1 per task: skill of the best CV model, best NLP model (events and panel) and the fusion gain; * = supported after Holm"),
        tbl(rtab("q1_test.csv"), max_rows=40, caption="Primary test metric by role, all decision rows"),
        tbl(rtab("q1_test_fresh.csv"), max_rows=40, caption="Primary test metric by role, only rows with fresh text (a filing within 5 sessions)")]),
    ("4. Q10, H3 and the incremental-value tests", [
        tbl(rtab("test_H4.csv"), ["task", "delta", "lo", "hi", "p_holm", "supported"], caption="H4 / Q10: top-3 CV + top-3 NLP minus top-3 CV"),
        tbl(rtab("test_H3.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm"], caption="H3: best fusion minus the better single modality"),
        tbl(rtab("q10_val.csv"), ["task", "delta", "lo", "hi", "p_holm"], caption="Q10 on validation"),
        tbl(rtab("fusion_leaderboard_test.csv"), ["model", "task", "primary"], max_rows=40, caption="Fusion and reference models on the locked test"),
        tbl(rtab("test_H3c.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H3c (post-hoc): best fusion minus strongest numbers-only baseline"),
        tbl(rtab("test_H8.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H8 (post-hoc): LightGBM on technical + CV + NLP features minus technical features alone"),
        tbl(rtab("test_H8b.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H8b (post-hoc): technical + CV minus technical alone"),
        tbl(rtab("test_H8c.csv"), ["task", "A", "B", "delta", "lo", "hi", "p_holm", "verdict"], caption="H8c (post-hoc): technical + NLP minus technical alone"),
        tbl(rtab("test_H8d.csv"), COLS_V, caption="H8d (post-hoc): technical + filing-calendar columns minus technical alone (do the filing dates alone help?)"),
        tbl(rtab("test_H8e.csv"), COLS_V, caption="H8e (post-hoc): technical + text minus technical + filing calendar (the content of the text)"),
        tbl(rtab("test_H8f.csv"), COLS_V, caption="H8f (post-hoc): technical + text minus technical + calendar + company one-hot (text beyond the company's identity)"),
        tbl(rtab("test_H1d.csv"), COLS_D, caption="H1d (post-hoc): direction skill of the best CV model minus that of the strongest numbers-only baseline, each over its own shuffled null"),
        tbl(rtab("phase_a_panel_test.csv"), max_rows=12, caption="Primary test metric per task for the reference models (post-hoc)")]),
    ("5. The dashboard recipe and Monte Carlo", [tbl(rtab("test_deploy.csv"), ["task", "skill", "lo", "hi", "p_holm", "supported"], caption="Dashboard recipe: skill on the locked test over the uninformed benchmark"),
        Pp("No-call rule: " + ", ".join(f"{k}: {'NO CALL' if v else 'call shown'}" for k, v in NO_CALL.items()) + "."),
        tbl(rtab("mc_calibration_val.csv"), caption="Monte Carlo tail thickness nu chosen on validation (target coverage 80 % and 90 %)"),
        tbl(rtab("mc_calibration_test.csv"), caption="Monte Carlo coverage on the locked test (confirmatory)"),
        tbl(rtab("mc_scale_val.csv"), caption="Post-hoc: volatility scale factor chosen on validation"),
        tbl(rtab("mc_calibration_test_scaled.csv"), caption="Post-hoc: Monte Carlo coverage on the locked test with the chosen scale"),
        figp("mc_example_val.png", "Monte Carlo example."), figp("deploy_confusion_test.png", "Dashboard recipe: confusion matrices on the test block."),
        tbl(rtab("dashboard_parity.csv"), caption="Parity: the dashboard engine reproduces the notebook's predictions (maximum absolute differences)")]),
    ("6. Reproducibility", [Pp(f"Freeze manifest: {len(FREEZE['prediction_files'])} prediction files hashed at the original freeze; selections {FREEZE['selections_sha256']}; core code "
        f"{FREEZE['core_sha256'][:16]}; packages {json.dumps(FREEZE['packages'])}. Predictions are recomputed deterministically; the check below compares the confirmatory tables of this execution with "
        f"the archived first run."),
        tbl(rtab("phase_a_repro_check.csv"), caption="Reproducibility check: confirmatory tables of this execution against the archived first run"),
        tbl(pd.DataFrame([{"section": k, "minutes": v} for k, v in TIMER.items() if k != "_t0"]), caption="Runtime by section of this execution (resumed stages are fast)")]),
    ("7. Discussion", mm_discussion()),
    ("8. Limitations and what would change the conclusions", [Pp("A stronger universe (delisted firms, small caps), a longer test, intraday text timing (news), alternative chart encodings and "
        "more training seeds could change the answers. Validation-chosen weights can still over-fit validation: only the locked test numbers are unbiased, and the post-hoc comparisons were added after "
        "the test was opened. Survivorship, a single data vendor lineage and cost-free evaluation are stated limits.")]),
    ("9. Conclusion", mm_conclusion()),
    ("References", refs(["baltrusaitis", "jkx", "loughran16", "glasserman", "sarkar", "guo", "kunsch", "holm"])),
]

In [ ]:
REPORT_FILES = []
for fname, title, sections in [("NEXUS_Alpha_v3_Computer_Vision_Report.pdf", "NEXUS Alpha v3 - Computer Vision Report", CV_SECTIONS),
                               ("NEXUS_Alpha_v3_NLP_Report.pdf", "NEXUS Alpha v3 - NLP Report (SEC filings)", NLP_SECTIONS),
                               ("NEXUS_Alpha_v3_Multimodal_Report.pdf", "NEXUS Alpha v3 - Multimodal (CV + NLP) Report", MM_SECTIONS)]:
    try:
        REPORT_FILES.append(build_pdf(SUB["reports"] / fname, title, sections)); print("wrote", fname)
    except Exception as e:
        failed(f"report_{fname}", e)
ZIP_OUT = SUB["reports"] / f"NEXUS_Alpha_v3_{RUN_MODE.lower()}_results.zip"
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("reports", "tables", "figures", "results"):
        for p_ in SUB[folder].glob("*"):
            if p_.is_file() and p_ != ZIP_OUT:
                z.write(p_, f"{folder}/{p_.name}")
print("Results ZIP:", ZIP_OUT)
if IN_COLAB:
    from google.colab import files as _files
    for p_ in REPORT_FILES:
        try:
            _files.download(str(p_))
        except Exception:
            pass
tick("25 reports")

## 26. Viva preparation, limitations and future work

**Answer every question with the run's own numbers** (`results/final_results.json`, the three PDFs). Prepared answers to the checklist questions:

| Question | Short answer (where the evidence is) |
|---|---|
| Why this problem? | Chart and text "AI signals" are widely sold; a leakage-free, sector-aware comparison tells users what is real (§1.1). |
| What gap? | No study compares 7 vision and 6 language models on identical point-in-time rows across 4 horizons, 3 targets and 5 sectors, with a test period after all LM pre-training cut-offs (§1.2). |
| Why this dataset? | Verified hourly, daily and weekly bars (SHA-256 checked, 0 missing sessions) plus EDGAR filings with acceptance timestamps; 6 tickers replaced for lack of 10-year hourly data (§0, §3). |
| How was leakage prevented? | Decision at the close; images and indicators tested by truncation and future-bar perturbation; 40-session purge; test labels in a vault opened only after a hashed freeze; filings usable only after acceptance + 35 min; entry at the next hourly open; test after LM cut-offs (§4.1, §5, §6.2, §14, §19, §22). |
| Preprocessing / augmentation? | Fixed-format 224 px charts, per-backbone normalisation; augmentation that preserves time order (no flips). Text: HTML cleaning, MD&A extraction, [form][section] tags, chunking (§6, §10.1, §13). |
| How does convolution / attention work here? | §9.1 and §16.1. |
| Why these architectures? | Table §1.3: from scratch vs pretrained, CNN vs transformer, small vs large, short vs long context, encoder vs LLM. |
| Transfer learning? | Frozen features (§9.2, §16.2) vs full fine-tuning (§10, §17); H5 tests whether fine-tuning helps. |
| Loss, optimiser, tuning? | Multi-task CE + MSE + BCE; AdamW, warm-up + cosine; two-value learning-rate probe per model; regularisation chosen on 2021 (§10, §17). |
| Over-fitting? | Training vs inner-loss curves, early stopping, the label-shuffle control (§10.4, §17.4, §9.3). |
| Why these metrics? | Macro-F1 + log loss for 3 classes; R² + QLIKE for volatility; ROC-AUC + PR-AUC + Brier skill + ECE for rare drawdowns (§7). |
| Comparison with baselines? | Prior, momentum, HAR, trailing volatility, TF-IDF, technical LightGBM; random-weight CNN (§8, §9, §15). |
| Ablations? | Views, frozen vs fine-tuned, pretrained vs scratch vs random, augmentation, text length, domain pre-training (§9.3, §12.4, §18.2). |
| Which errors / conditions? | Volatility deciles, sectors, quarters, market direction, filing timing and length (§12.5, §18.4). |
| Robustness? | Noise, blur, JPEG, light theme, screenshot resizing, missing hourly chart (§12.7). |
| Why is your chart model worse than a trailing-volatility baseline on volatility? | Each chart's price axis is scaled to that chart's own high and low, which removes the absolute volatility level, and the vertical-rescale augmentation changes candle heights (augmentation ablation, §12.4). Post-hoc table H1c compares every task with the strongest numbers-only baseline (§23A). |
| Isn't the text's drawdown signal just the company name? | A filing names the company and a company's usual volatility predicts its drawdown risk. §23A.1 trains per-company, numbers-only and LightGBM baselines on the same filings without text; H2c reports whether the best NLP model still beats them. |
| Do charts and filings add anything to simple numbers? | H8 / H8b / H8c (§23A.2): the same LightGBM with technical features alone versus technical + CV, + NLP, + both. Added after the test was opened, so labelled post-hoc. |
| Is the text gain for volatility just the earnings calendar? | H8d-H8f (§23A.2b): technical + filing-date columns, and + company one-hot, versus the same with text. The reports state which explanation the test supports. |
| Your CV model beats the baselines on direction by up to 0.14 macro-F1 - is that fair? | No: the baselines predict mostly one class, which macro-F1 punishes. H1d compares each model's skill over its own shuffled null; quote that number. |
| Why were extra analyses added after the test? | A methodological review found the baselines too weak. They use the same locked test rows, models fitted on training and tuned on 2021 only, baselines chosen on validation; the original tables are unchanged (reproducibility check, §23A.5). |
| Contribution? | A controlled, reproducible benchmark that answers Q1-Q10 with intervals, an event study of filing types and 8-K items by sector, and a dashboard that only claims what the test supports. |
| Limitations? | Below. |
| Deployment? | §24: the frozen recipe, parity-tested against the notebook; the no-call rule; Monte Carlo coverage checked on test. |

**Limitations.** 50 large surviving US stocks (survivorship bias; results may not hold for small caps where Jiang, Kelly and Xiu found most image signal). One decade and one regime shift between validation (2022-23) and test (2024-26). Prices not dividend-adjusted. Direction labels are intrinsically noisy; a 3-class band hides magnitude. Charts are rebuilt from bars, so real screenshots are redrawn rather than read pixel by pixel. MD&A extraction is heuristic. FinGPT is used frozen by default. Event reactions mix filings with same-day news. Many choices were made on validation; only the locked test numbers are unbiased. No trading costs or portfolio evaluation (this is a forecasting study).

**Future work.** Delisted and small-cap stocks; walk-forward refits each year; news text (minute-time-stamped) beside filings; alternative chart encodings (Gramian angular fields, returns images); calibrated position sizing from the volatility and drawdown heads with costs.

In [ ]:
print("Run finished:", STAMP, "| reports:", [p_.name for p_ in REPORT_FILES])
print("Runtime by section (minutes):", {k: v for k, v in TIMER.items() if k != "_t0"})